# 04 · Markow-Textgeneratoren

Die Aufgaben sind in einzelne Stationen eingeteilt. Lies jeweils zuerst die Aufgabenstellung und mach dir eigene Gedanken und probiere dann das jeweilige Widget aus. Speichere deine Codeänderungen mit **Strg + S**.

---

Als Trainingstexte stehen zur Auswahl eine Sammlung von 20 Grimm-Märchen, *Alice im Wunderland*, *Die Verwandlung*, *Max und Moritz*, *Hänsel und Gretel* sowie das Leitbild der CAU Kiel.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.

<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [ ]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import random
import re
import html
from fractions import Fraction
from collections import Counter
from IPython.display import display
from functools import lru_cache

PANELS = {}


# Textmodelle und gemeinsame Widget-Hilfen.
@lru_cache(maxsize=24)
def korpus_modell(name, ordnung=1, klein=False, satzzeichen=False):
    # Jedes Märchen wird separat gezählt: keine erfundenen Übergänge zwischen Werken.
    teile = [zerlege(t, klein, satzzeichen) for t in TEXTTEILE[name]]
    modell = {}
    for ts in teile:
        if len(ts) > ordnung:
            for zustand, counts in trainiere(ts, ordnung).items():
                modell.setdefault(zustand, Counter()).update(counts)
    return tuple(w for ts in teile for w in ts), modell


def textvorschau(name):
    anfang = " ".join(TEXTE[name].split()[:120])
    link = (
        f'<a href="{html.escape(QUELLEN[name]["url"],quote=True)}" target="_blank" rel="noopener">Quelle / Volltext</a>'
        if QUELLEN[name]["url"]
        else "Bereitgestellter Trainingstext"
    )
    return (
        '<div style="max-height:190px;overflow:auto;line-height:1.5">'
        + html.escape(anfang)
        + " …</div>"
        + "<p><small>"
        + html.escape(QUELLEN[name]["umfang"])
        + " · "
        + link
        + "</small></p>"
    )


def erzwungener_weg(modell, zustand, maximum=30):
    zustand = (zustand,) if isinstance(zustand, str) else tuple(zustand)
    ordnung = len(zustand)
    weg = list(zustand)
    besucht = set()
    for schritt in range(maximum + 1):
        aktuell = tuple(weg[-ordnung:])
        counts = modell.get(aktuell, {})
        if not counts:
            return weg, "Ende: kein beobachteter Nachfolger."
        if len(counts) > 1:
            return weg, f"Hier gibt es wieder eine Wahl: {len(counts)} verschiedene Nachfolger."
        if aktuell in besucht:
            return (
                weg,
                "Deterministischer Kreislauf: Diese Zustände wiederholen sich unbegrenzt.",
            )
        if schritt == maximum:
            return weg, f"Anzeige nach {maximum} erzwungenen Schritten gekürzt."
        besucht.add(aktuell)
        weg.append(next(iter(counts)))


def zerlege(text, klein=False, satzzeichen=False):
    if klein:
        text = text.lower()
    return re.findall(r"\w+|[^\w\s]", text) if satzzeichen else text.split()


def trainiere(tokens, ordnung=1):
    if ordnung not in (1, 2, 3):
        raise ValueError("Die Ordnung muss 1, 2 oder 3 sein.")
    if len(tokens) < ordnung + 1:
        raise ValueError(
            f"Für Ordnung {ordnung} werden mindestens {ordnung+1} Einheiten benötigt."
        )
    modell = {}
    for i in range(len(tokens) - ordnung):
        zustand = tuple(tokens[i : i + ordnung])
        modell.setdefault(zustand, Counter())[tokens[i + ordnung]] += 1
    # Auch das letzte Wort/Paar/Tripel ist ein Zustand. Es kann ohne Nachfolger sein.
    modell.setdefault(tuple(tokens[-ordnung:]), Counter())
    return modell


def generiere(modell, anfang, laenge, seed=2026):
    ordnung = len(next(iter(modell)))
    if len(anfang) < ordnung:
        raise ValueError("Der Anfang ist kürzer als die Modellordnung.")
    if laenge < len(anfang):
        raise ValueError("Die gewünschte Länge ist kürzer als der Anfang.")
    rng = random.Random(seed)
    ausgabe = list(anfang)
    while len(ausgabe) < laenge:
        zustand = tuple(ausgabe[-ordnung:])
        counts = modell.get(zustand, {})
        if not counts:
            return ausgabe, "Kein beobachteter Nachfolger für: " + " | ".join(zustand)
        ausgabe.append(rng.choices(list(counts), weights=list(counts.values()), k=1)[0])
    return ausgabe, "Gewünschte Länge erreicht."


def statistik(modell):
    fortsetzungen = [c for c in modell.values() if c]
    return len(fortsetzungen), sum(len(c) == 1 for c in fortsetzungen) / len(fortsetzungen)


def feld(text, w):
    return W.VBox([W.HTML(html.escape(text)), w])


def regler(wert, maximum=200, minimum=5):
    return W.IntSlider(
        value=wert,
        min=minimum,
        max=maximum,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def textwahl(default="Grimms Märchen"):
    return W.Dropdown(
        options=list(TEXTE), value=default, layout=W.Layout(width="330px", max_width="100%")
    )


def knopf(text="Neu auslosen"):
    return W.Button(description=text, button_style="info", layout=W.Layout(width="200px"))


def tabelle(kopf, rows):
    head = "".join(
        f'<th style="padding:4px 10px;text-align:left">{html.escape(str(x))}</th>' for x in kopf
    )
    body = "".join(
        "<tr>"
        + "".join(f'<td style="padding:4px 10px">{html.escape(str(x))}</td>' for x in row)
        + "</tr>"
        for row in rows
    )
    return (
        '<div style="max-height:300px;overflow:auto"><table><tr>'
        + head
        + "</tr>"
        + body
        + "</table></div>"
    )


def ausgabe_html(tokens, grund):
    return (
        '<div style="border-left:3px solid #2875aa;padding:8px 12px;max-height:240px;overflow:auto;overflow-wrap:anywhere">'
        + html.escape(" ".join(tokens))
        + "</div><p><small>"
        + html.escape(f"{len(tokens)} Einheiten. {grund}")
        + "</small></p>"
    )


def quelle_box():
    text = W.HTML()
    box = W.Accordion(children=[text], selected_index=None)
    box.set_title(0, "Textanfang und Quelle (höchstens 120 Wörter)")
    return text, box


# Stationen.
def station_zaehlen():
    labels = [
        "1 · P(Hase | der)",
        "1 · P(Fuchs | der)",
        "2 · P(schläft nach zwei Schritten | Start: der)",
        "3 · Kleinste Anzahl angehängter Sätze",
    ]
    ws = [W.Text(placeholder="Bruch oder Zahl", layout=W.Layout(width="220px")) for _ in labels]
    btn = knopf("Eingaben prüfen")
    info = W.HTML("Rechne zuerst. Du kannst auch einzelne Felder prüfen.")
    r = {"fields": ws, "button": btn, "info": info, "last": None}

    def invalidate(_):
        r["last"] = None
        info.value = "Eingabe geändert; erneut prüfen."

    def check(_):
        expected = [Fraction(2, 3), Fraction(1, 3), Fraction(1, 3), Fraction(3)]
        hints = [
            "Zähle alle Übergänge von „der“.",
            "Zähle alle Übergänge von „der“.",
            "Welche Wege führen in genau zwei Schritten von „der“ zu „schläft“?",
            "Zähle nach dem Anhängen sowohl Fuchs → schläft als auch alle Übergänge von Fuchs.",
        ]
        rows = []
        correct = []
        for lab, w, target, hint in zip(labels, ws, expected, hints):
            if not w.value.strip():
                rows.append([lab, "Noch offen"])
                correct.append(None)
                continue
            try:
                v = Fraction(w.value.strip().replace(",", "."))
                ok = abs(float(v - target)) < 1e-9
                if lab.startswith("3"):
                    ok = v == target
                rows.append([lab, "Richtig." if ok else hint])
                correct.append(ok)
            except (ValueError, ZeroDivisionError, OverflowError):
                rows.append([lab, "Bitte eine Zahl oder einen Bruch eingeben."])
                correct.append(False)
        info.value = tabelle(["Rechnung", "Rückmeldung"], rows)
        r["last"] = {"correct": correct}

    for w in ws:
        w.observe(invalidate, names="value")
    btn.on_click(check)
    display(W.VBox([*(feld(l, w) for l, w in zip(labels, ws)), btn, info]))
    PANELS["zaehlen"] = r
    return r


def station_nachfolger():
    corpus = textwahl()
    order = W.ToggleButtons(
        options=[("1. Ordnung: Wort", 1), ("2. Ordnung: Wortpaar", 2)], value=1
    )
    art = W.Dropdown(
        options=["Alle Wörter", "Genau ein Nachfolger"],
        value="Alle Wörter",
        layout=W.Layout(width="330px", max_width="100%"),
    )
    state = W.Combobox(
        ensure_option=True,
        placeholder="Wort oder Wortpaar tippen",
        layout=W.Layout(width="420px", max_width="100%"),
    )
    info, belege, chain, vergleich = W.HTML(), W.HTML(), W.HTML(), W.HTML()
    btn = knopf("Fortsetzung prüfen")
    r = {
        "corpus": corpus,
        "order": order,
        "art": art,
        "state": state,
        "info": info,
        "belege": belege,
        "chain": chain,
        "vergleich": vergleich,
        "button": btn,
        "last": None,
        "lock": False,
        "selected": None,
    }

    def update(_=None):
        if r["lock"]:
            return
        chain.value = ""
        zustand = r["lookup"].get(state.value)
        if zustand is None:
            info.value = "Wähle einen Eintrag aus der Liste; Schreibweise und Satzzeichen müssen genau passen."
            belege.value = ""
            vergleich.value = ""
            r["last"] = None
            btn.disabled = True
            return
        r["selected"] = zustand
        btn.disabled = False
        counts = r["model"].get(zustand, Counter())
        total = sum(counts.values())
        info.value = (
            tabelle(
                ["Nachfolger", "Anzahl", "Geschätzte Wahrscheinlichkeit"],
                [[w, c, f"{c}/{total} = {c/total:.3f}"] for w, c in counts.most_common()],
            )
            if total
            else "Kein beobachteter Nachfolger: Hier endet die Generierung."
        )
        # All pair successors must also be successors of the final single word.
        single = r["model1"].get(zustand[-1:], Counter())
        vergleich.value = (
            tabelle(
                ["Zustand", "Verschiedene Nachfolger"],
                [[zustand[-1], len(single)], [" | ".join(zustand), len(counts)]],
            )
            if len(zustand) == 2
            else ""
        )
        snippets = []
        k = len(zustand)
        for titel, ts in r["teile"]:
            for i in range(len(ts) - k + 1):
                if tuple(ts[i : i + k]) == zustand:
                    left = html.escape(" ".join(ts[max(0, i - 5) : i]))
                    right = html.escape(" ".join(ts[i + k : i + k + 8]))
                    snippets.append(
                        "<li><small>"
                        + html.escape(titel)
                        + "</small>: … "
                        + left
                        + " <b>"
                        + html.escape(" ".join(zustand))
                        + "</b> "
                        + right
                        + " …</li>"
                    )
                    if len(snippets) == 4:
                        break
            if len(snippets) == 4:
                break
        belege.value = (
            "<p><b>Bis zu vier Fundstellen</b> (Ausschnitte, nicht alle Vorkommen):</p><ul>"
            + "".join(snippets)
            + "</ul>"
        )
        r["last"] = {
            "state": zustand,
            "counts": counts.copy(),
            "total": total,
            "single_counts": single.copy(),
            "forced": None,
        }

    def options(_=None):
        if r["lock"]:
            return
        r["lock"] = True
        old = r["selected"]
        keys = [
            k
            for k, c in r["model"].items()
            if art.value != "Genau ein Nachfolger" or len(c) == 1
        ]
        keys.sort()
        r["lookup"] = {" ".join(k): k for k in keys}
        state.options = list(r["lookup"])
        if old in keys:
            choice = old
        elif keys:
            matching = [k for k in keys if old and k[-1] == old[-1]]
            preferred = ("der", "Königstochter") if order.value == 2 else ("Königstochter",)
            if preferred in (matching or keys):
                choice = preferred
            elif matching:
                choice = matching[0]
            elif art.value == "Genau ein Nachfolger":
                choice = max(
                    keys, key=lambda k: len(erzwungener_weg(r["model"], k, 6)[0]) - len(k)
                )
            else:
                choice = max(keys, key=lambda k: sum(r["model"][k].values()))
        else:
            choice = None
        state.value = " ".join(choice) if choice else ""
        r["lock"] = False
        update()

    def change(_=None):
        r["lock"] = True
        was_single = art.value == "Genau ein Nachfolger"
        art.options = [
            "Alle Wörter" if order.value == 1 else "Alle Wortpaare",
            "Genau ein Nachfolger",
        ]
        art.value = "Genau ein Nachfolger" if was_single else art.options[0]
        _, r["model"] = korpus_modell(corpus.value, order.value)
        _, r["model1"] = korpus_modell(corpus.value, 1)
        titles = GRIMM_TITEL if corpus.value == "Grimms Märchen" else [corpus.value]
        r["teile"] = [(titel, zerlege(t)) for titel, t in zip(titles, TEXTTEILE[corpus.value])]
        r["lock"] = False
        options()

    def reveal(_):
        if r["last"] is None:
            return
        zustand = r["last"]["state"]
        weg, grund = erzwungener_weg(r["model"], zustand)
        # Show words once; the active state is always the last k words.
        chain.value = (
            "<p><b>Erzwungene Wortfolge:</b> "
            + html.escape(" → ".join(weg))
            + "</p><p>"
            + html.escape(grund)
            + "</p>"
        )
        r["last"]["forced"] = (weg, grund)

    corpus.observe(change, names="value")
    order.observe(change, names="value")
    art.observe(options, names="value")
    state.observe(update, names="value")
    btn.on_click(reveal)
    display(
        W.VBox(
            [
                corpus,
                order,
                art,
                feld("Zustand (Suche durch Tippen)", state),
                vergleich,
                info,
                belege,
                btn,
                chain,
            ]
        )
    )
    change()
    PANELS["nachfolger"] = r
    return r


def station_ordnungen():
    corpus = textwahl()
    n = regler(60)
    btn = knopf()
    out1, out2, info = W.HTML(), W.HTML(), W.HTML()
    klein = W.Checkbox(
        description="Groß-/Kleinschreibung ignorieren",
        value=False,
        indent=False,
        layout=W.Layout(width="310px"),
    )
    satz = W.Checkbox(
        description="Satzzeichen als eigene Einheiten",
        value=False,
        indent=False,
        layout=W.Layout(width="310px"),
    )
    source, box = quelle_box()
    r = {
        "corpus": corpus,
        "n": n,
        "button": btn,
        "klein": klein,
        "satz": satz,
        "source": source,
        "box": box,
        "seed": 2026,
        "last": None,
    }

    def update(_=None):
        results = []
        stats = []
        for order, out in [(1, out1), (2, out2)]:
            ts, m = korpus_modell(corpus.value, order, klein.value, satz.value)
            words, reason = generiere(m, ts[:2], n.value, r["seed"])
            out.value = ausgabe_html(words, reason)
            results.append(words)
            count, frac = statistik(m)
            stats.append([order, count, f"{frac:.1%}"])
        info.value = tabelle(
            [
                "Ordnung",
                "Zustände mit Fortsetzung",
                "Davon mit nur einem verschiedenen Nachfolger",
            ],
            stats,
        )
        r["last"] = {
            "outputs": results,
            "stats": stats,
            "klein": klein.value,
            "satz": satz.value,
        }

    def change(_=None):
        source.value = textvorschau(corpus.value)
        update()

    def new(_):
        r["seed"] += 1
        update()

    corpus.observe(change, names="value")
    for w in (n, klein, satz):
        w.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                corpus,
                box,
                klein,
                satz,
                feld("Gewünschte Gesamtlänge (Einheiten)", n),
                btn,
                W.HTML("<b>1. Ordnung</b>"),
                out1,
                W.HTML("<b>2. Ordnung</b>"),
                out2,
                info,
            ]
        )
    )
    change()
    PANELS["ordnungen"] = r
    return r


def station_eigener_text():
    editor = W.Textarea(
        value="Die Katze schläft . Der Hund schläft . Die Katze frisst . Der Hund läuft .",
        layout=W.Layout(width="650px", max_width="100%", height="110px"),
    )
    apply = knopf("Text übernehmen")
    order = regler(1, 3, 1)
    n = regler(40)
    btn = knopf()
    info, out = W.HTML(), W.HTML()
    r = {
        "editor": editor,
        "apply": apply,
        "order": order,
        "n": n,
        "button": btn,
        "info": info,
        "out": out,
        "seed": 2026,
        "training": None,
        "last": None,
    }

    def update(_=None):
        if r["training"] is None:
            return
        r["last"] = None
        try:
            ts = r["training"]
            m = trainiere(ts, order.value)
            words, reason = generiere(m, ts[: order.value], n.value, r["seed"])
        except ValueError as e:
            info.value = html.escape(str(e))
            out.value = ""
            return
        count, frac = statistik(m)
        info.value = f"{len(ts)} Trainingseinheiten; Ordnung {order.value}. {count} Zustände mit Fortsetzung; davon {frac:.1%} mit nur einem Nachfolger."
        out.value = ausgabe_html(words, reason)
        r["last"] = {"tokens": words, "reason": reason, "model": m}

    def invalidate(_=None):
        r["training"] = None
        r["last"] = None
        out.value = ""
        info.value = "Text zuerst übernehmen."
        for w in (order, n, btn):
            w.disabled = True

    def accept(_):
        ts = zerlege(editor.value)
        if len(ts) < 2:
            invalidate()
            info.value = "Bitte mindestens zwei durch Leerraum getrennte Einheiten eingeben."
            return
        r["training"] = ts
        for w in (order, n, btn):
            w.disabled = False
        update()

    def new(_):
        r["seed"] += 1
        update()

    editor.observe(invalidate, names="value")
    apply.on_click(accept)
    btn.on_click(new)
    for w in (order, n):
        w.observe(update, names="value")
    display(
        W.VBox(
            [
                editor,
                apply,
                feld("Ordnung: so viele letzte Wörter werden berücksichtigt", order),
                feld("Gewünschte Gesamtlänge", n),
                btn,
                info,
                out,
            ]
        )
    )
    invalidate()
    PANELS["eigen"] = r
    return r


# Trainingstexte und Quellen: eingebettet für die eigenständige Nutzung.

TEXTTEILE = {
    "Grimms Märchen": [
        "In den alten Zeiten, wo das Wünschen noch geholfen hat, lebte ein König, "
        "dessen Töchter waren alle schön, aber die jüngste war so schön, daß die "
        "Sonne selber, die doch so vieles gesehen hat, sich verwunderte, so oft "
        "sie ihr ins Gesicht schien. Nahe bei dem Schlosse des Königs lag ein "
        "großer dunkler Wald, und in dem Walde unter einer alten Linde war ein "
        "Brunnen: wenn nun der Tag sehr heiß war, so ging das Königskind hinaus in "
        "den Wald und setzte sich an den Rand des kühlen Brunnens: und wenn sie "
        "Langeweile hatte, so nahm sie eine goldene Kugel, warf sie in die Höhe "
        "und fing sie wieder; und das war ihr liebstes Spielwerk.\n"
        "\n"
        "Nun trug es sich einmal zu, daß die goldene Kugel der Königstochter nicht "
        "in ihr Händchen fiel, das sie in die Höhe gehalten hatte, sondern vorbei "
        "auf die Erde schlug und geradezu ins Wasser hineinrollte. Die "
        "Königstochter folgte ihr mit den Augen nach, aber die Kugel verschwand, "
        "und der Brunnen war tief, so tief, daß man keinen Grund sah. Da fing sie "
        "an zu weinen und weinte immer lauter und konnte sich gar nicht trösten. "
        "Und wie sie so klagte, rief ihr jemand zu: »Was hast du vor, "
        "Königstochter, du schreist ja, daß sich ein Stein erbarmen möchte.« Sie "
        "sah sich um, woher die Stimme käme, da erblickte sie einen Frosch, der "
        "seinen dicken häßlichen Kopf aus dem Wasser streckte. »Ach, du bist's, "
        "alter Wasserpatscher,« sagte sie, »ich weine über meine goldene Kugel, "
        "die mir in den Brunnen hinabgefallen ist.« — »Sei still und weine nicht,« "
        "antwortete der Frosch, »ich kann wohl Rat schaffen, aber was gibst du "
        "mir, wenn ich dein Spielwerk wieder heraufhole?« — »Was du haben willst, "
        "lieber Frosch,« sagte sie, »meine Kleider, meine Perlen und Edelsteine, "
        "auch noch die goldene Krone, die ich trage.« Der Frosch antwortete: "
        "»Deine Kleider, deine Perlen und Edelsteine, und deine goldene Krone, die "
        "mag ich nicht: aber wenn du mich lieb haben willst, und ich soll dein "
        "Geselle und Spielkamerad sein, an deinem Tischlein neben dir sitzen, von "
        "deinem goldenen Tellerlein essen, aus deinem Becherlein trinken, in "
        "deinem Bettlein schlafen: wenn du mir das versprichst, so will ich "
        "hinuntersteigen und dir die goldene Kugel wieder heraufholen.« — »Ach "
        "ja,« sagte sie, »ich verspreche dir alles, was du willst, wenn du mir nur "
        "die Kugel wiederbringst.« Sie dachte aber: »Was der einfältige Frosch "
        "schwätzt, der sitzt im Wasser bei seinesgleichen und quakt, und kann "
        "keines Menschen Geselle sein.«\n"
        "\n"
        "Der Frosch, als er die Zusage erhalten hatte, tauchte seinen Kopf unter, "
        "sank hinab und über ein Weilchen kam er wieder herauf gerudert; hatte die "
        "Kugel im Maul und warf sie ins Gras. Die Königstochter war voll Freude, "
        "als sie ihr schönes Spielwerk wieder erblickte, hob es auf und sprang "
        "damit fort. »Warte, warte,« rief der Frosch, »nimm mich mit, ich kann "
        "nicht so laufen wie du.« Aber was half es ihm, daß er ihr sein quak quak "
        "so laut nachschrie als er konnte; sie hörte nicht darauf, eilte nach "
        "Hause und hatte bald den armen Frosch vergessen, der wieder in seinen "
        "Brunnen hinabsteigen mußte.\n"
        "\n"
        "Am andern Tage, als sie mit dem König und allen Hofleuten sich zur Tafel "
        "gesetzt hatte und von ihrem goldenen Tellerlein aß, da kam, plitsch "
        "platsch, plitsch platsch, etwas die Marmortreppe heraufgekrochen und als "
        "es oben angelangt war, klopfte es an die Tür und rief: »Königstochter, "
        "jüngste, mach mir auf.« Sie lief und wollte sehen, wer draußen wäre, als "
        "sie aber aufmachte, so saß der Frosch davor. Da warf sie die Tür hastig "
        "zu, setzte sich wieder an den Tisch, und war ihr ganz angst. Der König "
        "sah wohl, daß ihr das Herz gewaltig klopfte und sprach: »Mein Kind, was "
        "fürchtest du dich, steht etwa ein Riese vor der Tür und will dich holen?« "
        "— »Ach nein,« antwortete sie, »es ist kein Riese, sondern ein garstiger "
        "Frosch.« — »Was will der Frosch von dir?« — »Ach, lieber Vater, als ich "
        "gestern im Wald bei dem Brunnen saß und spielte, da fiel meine goldene "
        "Kugel ins Wasser. Und weil ich so weinte, hat sie der Frosch wieder "
        "heraufgeholt, und weil er es durchaus verlangte, so versprach ich ihm, er "
        "sollte mein Geselle werden, ich dachte aber nimmermehr, daß er aus seinem "
        "Wasser heraus könnte. Nun ist er draußen und will zu mir herein.« Indem "
        "klopfte es zum zweitenmal und rief:\n"
        "\n"
        "»Königstochter, jüngste, mach mir auf, weißt du nicht, was gestern du zu "
        "mir gesagt bei dem kühlen Brunnenwasser? Königstochter, jüngste, mach mir "
        "auf.«\n"
        "\n"
        "Da sagte der König: »Was du versprochen hast, das mußt du auch halten; "
        "geh nur und mach ihm auf.« Sie ging und öffnete die Türe, da hüpfte der "
        "Frosch herein, ihr immer auf dem Fuße nach, bis zu ihrem Stuhl. Da saß er "
        "und rief: »Heb mich herauf zu dir.« Sie zauderte, bis es endlich der "
        "König befahl. Als der Frosch erst auf dem Stuhl war, wollte er auf den "
        "Tisch, und als er da saß, sprach er: »Nun schieb mir dein goldenes "
        "Tellerlein näher, damit wir zusammen essen.« Das tat sie zwar, aber man "
        "sah wohl, daß sie's nicht gerne tat. Der Frosch ließ sich's gutschmecken, "
        "aber ihr blieb fast jedes Bißlein im Halse. Endlich sprach er: »Ich habe "
        "mich satt gegessen und bin müde, nun trag mich in dein Kämmerlein und "
        "mach dein seiden Bettlein zurecht, da wollen wir uns schlafen legen.« Die "
        "Königstochter fing an zu weinen und fürchtete sich vor dem kalten Frosch, "
        "den sie nicht anzurühren getraute, und der nun in ihrem schönen reinen "
        "Bettlein schlafen sollte. Der König aber ward zornig und sprach: »Wer dir "
        "geholfen hat, als du in der Not warst, den sollst du hernach nicht "
        "verachten.« Da packte sie ihn mit zwei Fingern, trug ihn hinauf und "
        "setzte ihn in eine Ecke. Als sie aber im Bette lag, kam er gekrochen und "
        "sprach: »Ich bin müde, ich will schlafen so gut wie du: heb mich herauf, "
        "oder ich sag's deinem Vater.« Da ward sie erst bitterböse, holte ihn "
        "herauf und warf ihn aus allen Kräften wider die Wand, »nun wirst du Ruhe "
        "haben, du garstiger Frosch.«\n"
        "\n"
        "Als er aber herabfiel, war er kein Frosch, sondern ein Königssohn mit "
        "schönen freundlichen Augen. Der war nun nach ihres Vaters Willen ihr "
        "lieber Geselle und Gemahl. Da erzählte er ihr, er wäre von einer bösen "
        "Hexe verwünscht worden, und niemand hätte ihn aus dem Brunnen erlösen "
        "können als sie allein, und morgen wollten sie zusammen in sein Reich "
        "gehen. Dann schliefen sie ein, und am andern Morgen, als die Sonne sie "
        "aufweckte, kam ein Wagen herangefahren mit acht weißen Pferden bespannt, "
        "die hatten weiße Straußfedern auf dem Kopf und gingen in goldenen Ketten, "
        "und hinten stand der Diener des jungen Königs, das war der treue "
        "Heinrich. Der treue Heinrich hatte sich so betrübt, als sein Herr war in "
        "einen Frosch verwandelt worden, daß er drei eiserne Bande hatte um sein "
        "Herz legen lassen, damit es ihm nicht vor Weh und Traurigkeit zerspränge. "
        "Der Wagen aber sollte den jungen König in sein Reich abholen; der treue "
        "Heinrich hob beide hinein, stellte sich wieder hinten auf und war voller "
        "Freude über die Erlösung. Und als sie ein Stück Wegs gefahren waren, "
        "hörte der Königssohn, daß es hinter ihm krachte, als wäre etwas "
        "zerbrochen. Da drehte er sich um und rief:\n"
        "\n"
        "»Heinrich, der Wagen bricht,« »nein, Herr, der Wagen nicht, es ist ein "
        "Band von meinem Herzen, das da lag in großen Schmerzen, als Ihr in dem "
        "Brunnen saßt, als Ihr eine Fretsche wast.«\n"
        "\n"
        "Doch einmal und noch einmal krachte es auf dem Weg, und der Königssohn "
        "meinte immer, der Wagen bräche, und es waren doch nur die Bande, die vom "
        "Herzen des treuen Heinrich absprangen, weil sein Herr erlöst und "
        "glücklich war.",
        "Es war einmal eine alte Geiß, die hatte sieben junge Geißlein, und hatte "
        "sie lieb, wie eine Mutter ihre Kinder lieb hat. Eines Tages wollte sie in "
        "den Wald gehen und Futter holen, da rief sie alle sieben herbei und "
        "sprach: »Liebe Kinder, ich will hinaus in den Wald, seid auf eurer Hut "
        "vor dem Wolf; wenn er hereinkommt, so frißt er euch alle mit Haut und "
        "Haar. Der Bösewicht verstellt sich oft, aber an seiner rauhen Stimme und "
        "an seinen schwarzen Füßen werdet ihr ihn gleich erkennen.« — Die Geißlein "
        "sagten: »Liebe Mutter, wir wollen uns schon in acht nehmen, Ihr könnt "
        "ohne Sorge fortgehen.« Da meckerte die Alte und machte sich getrost auf "
        "den Weg.\n"
        "\n"
        "Es dauerte nicht lange, so klopfte jemand an die Haustür und rief: »Macht "
        "auf, ihr lieben Kinder, eure Mutter ist da und hat jedem von euch etwas "
        "mitgebracht.« Aber die Geißerchen hörten an der rauhen Stimme, daß es der "
        "Wolf war, »wir machen nicht auf,« riefen sie, »du bist unsere Mutter "
        "nicht, die hat eine feine liebliche Stimme, aber deine Stimme ist rauh; "
        "du bist der Wolf.« Da ging der Wolf fort zu einem Krämer und kaufte sich "
        "ein großes Stück Kreide: die aß er und machte seine Stimme damit fein. "
        "Dann kam er zurück, klopfte an die Haustür und rief: »Macht auf, ihr "
        "lieben Kinder, eure Mutter ist da und hat jedem von euch etwas "
        "mitgebracht.« Aber der Wolf hatte seine schwarze Pfote in das Fenster "
        "gelegt, das sahen die Kinder und riefen: »Wir machen nicht auf, unsere "
        "Mutter hat keinen schwarzen Fuß wie du: du bist der Wolf.« Da lief der "
        "Wolf zu einem Bäcker und sprach: »Ich habe mich an den Fuß gestoßen, "
        "streich mir Teig darüber.« Und als ihm der Bäcker die Pfote bestrichen "
        "hatte, so lief er zum Müller und sprach: »Streu mir weißes Mehl auf meine "
        "Pfote.« Der Müller dachte: »Der will einen betrügen,« und weigerte sich, "
        "aber der Wolf sprach: »Wenn du es nicht tust, so fresse ich dich.« Da "
        "fürchtete sich der Müller und machte ihm die Pfote weiß. Ja, so sind die "
        "Menschen.\n"
        "\n"
        "Nun ging der Bösewicht zum drittenmal zu der Haustür, klopfte an und "
        "sprach: »Macht mir auf, Kinder, euer liebes Mütterchen ist heimgekommen "
        "und hat jedem von euch etwas aus dem Walde mitgebracht.« Die Geißerchen "
        "riefen: »Zeig uns erst deine Pfote, damit wir wissen, daß du unser liebes "
        "Mütterchen bist.« Da legte er die Pfote ins Fenster, und als sie sahen, "
        "daß sie weiß war, so glaubten sie, es wäre alles wahr, was er sagte, und "
        "machten die Türe auf. Wer aber hereinkam, das war der Wolf. Sie "
        "erschraken und wollten sich verstecken. Das eine sprang unter den Tisch, "
        "das zweite ins Bett, das dritte in den Ofen, das vierte in die Küche, das "
        "fünfte in den Schrank, das sechste unter die Waschschüssel, das siebente "
        "in den Kasten der Wanduhr. Aber der Wolf fand sie alle und machte nicht "
        "langes Federlesen: eins nach dem andern schluckte er in seinen Rachen; "
        "nur das jüngste in dem Uhrkasten, das fand er nicht. Als der Wolf seine "
        "Lust gebüßt hatte, trollte er sich fort, legte sich draußen auf der "
        "grünen Wiese unter einen Baum und fing an zu schlafen.\n"
        "\n"
        "Nicht lange danach kam die alte Geiß aus dem Walde wieder heim. Ach, was "
        "mußte sie da erblicken! Die Haustüre stand sperrweit auf: Tisch, Stühle "
        "und Bänke waren umgeworfen, die Waschschüssel lag in Scherben, Decke und "
        "Kissen waren aus dem Bett gezogen. Sie suchte ihre Kinder, aber nirgends "
        "waren sie zu finden. Sie rief sie nacheinander bei Namen, aber niemand "
        "antwortete. Endlich, als sie an das jüngste kam, da rief eine feine "
        "Stimme: »Liebe Mutter, ich stecke im Uhrkasten.« Sie holte es heraus, und "
        "es erzählte ihr, daß der Wolf gekommen wäre und die andern alle gefressen "
        "hätte. Da könnt ihr denken, wie sie über ihre armen Kinder geweint hat.\n"
        "\n"
        "Endlich ging sie in ihrem Jammer hinaus, und das jüngste Geißlein lief "
        "mit. Als sie auf die Wiese kam, so lag da der Wolf an dem Baum und "
        "schnarchte, daß die Äste zitterten. Sie betrachtete ihn von allen Seiten "
        "und sah, daß in seinem angefüllten Bauch sich etwas regte und zappelte. "
        "»Ach Gott,« dachte sie, »sollten meine armen Kinder, die er zum Abendbrot "
        "hinuntergewürgt hat, noch am Leben sein?« Da mußte das Geißlein nach Haus "
        "laufen und Schere, Nadel und Zwirn holen. Dann schnitt sie dem Ungetüm "
        "den Wanst auf, und kaum hatte sie einen Schnitt getan, so streckte schon "
        "ein Geißlein den Kopf heraus, und als sie weiter schnitt, so sprangen "
        "nacheinander alle sechse heraus, und waren noch alle am Leben, und hatten "
        "nicht einmal Schaden gelitten, denn das Ungetüm hatte sie in der Gier "
        "ganz hinuntergeschluckt. Das war eine Freude! Da herzten sie ihre liebe "
        "Mutter, und hüpften wie ein Schneider, der Hochzeit hält. Die Alte aber "
        "sagte: »Jetzt geht und sucht Wackersteine, damit wollen wir dem gottlosen "
        "Tiere den Bauch füllen, solange es noch im Schlafe liegt.« Da schleppten "
        "die sieben Geißlein in aller Eile die Steine herbei und steckten sie ihm "
        "in den Bauch, so viel sie hineinbringen konnten. Dann nähte ihn die Alte "
        "in aller Geschwindigkeit wieder zu, daß er nichts merkte und sich nicht "
        "einmal regte.\n"
        "\n"
        "Als der Wolf endlich ausgeschlafen hatte, machte er sich auf die Beine, "
        "und weil ihm die Steine im Magen so großen Durst erregten, so wollte er "
        "zu einem Brunnen gehen und trinken. Als er aber anfing zu gehen und sich "
        "hin und her zu bewegen, so stießen die Steine in seinem Bauch aneinander "
        "und rappelten. Da rief er:\n"
        "\n"
        "»Was rumpelt und pumpelt in meinem Bauch herum? Ich meinte, es wären "
        "sechs Geißlein, so sind's lauter Wackerstein.«\n"
        "\n"
        "Und als er an den Brunnen kam und sich über das Wasser bückte und trinken "
        "wollte, da zogen ihn die schweren Steine hinein, und er mußte jämmerlich "
        "ersaufen. Als die sieben Geißlein das sahen, da kamen sie herbeigelaufen, "
        "riefen laut: »Der Wolf ist tot! der Wolf ist tot!« und tanzten mit ihrer "
        "Mutter vor Freude um den Brunnen herum.",
        "Es hatte ein Mann einen Esel, der schon lange Jahre die Säcke "
        "unverdrossen zur Mühle getragen hatte, dessen Kräfte aber nun zu Ende "
        "gingen, so daß er zur Arbeit immer untauglicher ward. Da dachte der Herr "
        "daran, ihn aus dem Futter zu schaffen, aber der Esel merkte, daß kein "
        "guter Wind wehte, lief fort und machte sich auf den Weg nach Bremen; "
        "dort, meinte er, könnte er ja Stadtmusikant werden. Als er ein Weilchen "
        "fortgegangen war, fand er einen Jagdhund auf dem Wege liegen, der jappte "
        "wie einer, der sich müde gelaufen hat. »Nun, was jappst du so, Packan?« "
        "fragte der Esel. — »Ach,« sagte der Hund, »weil ich alt bin und jeden Tag "
        "schwächer werde, auch auf der Jagd nicht mehr fort kann, hat mich mein "
        "Herr wollen totschlagen, da hab ich Reißaus genommen; aber womit soll ich "
        "nun mein Brot verdienen?« — »Weißt du was,« sprach der Esel, »ich gehe "
        "nach Bremen und werde dort Stadtmusikant, geh mit und laß dich auch bei "
        "der Musik annehmen. Ich spiele die Laute, und du schlägst die Pauken.« "
        "Der Hund war's zufrieden, und sie gingen weiter. Es dauerte nicht lange, "
        "so saß da eine Katze an dem Weg und machte ein Gesicht wie drei Tage "
        "Regenwetter. »Nun, was ist dir in die Quere gekommen, alter Bartputzer?« "
        "sprach der Esel. — »Wer kann da lustig sein, wenn's einem an den Kragen "
        "geht,« antwortete die Katze, »weil ich nun zu Jahren komme, meine Zähne "
        "stumpf werden und ich lieber hinter dem Ofen sitze und spinne, als nach "
        "Mäusen herumjage, hat mich meine Frau ersäufen wollen; ich habe mich zwar "
        "noch fortgemacht, aber nun ist guter Rat teuer; wo soll ich hin?« — »Geh "
        "mit uns nach Bremen, du verstehst dich doch auf die Nachtmusik, da kannst "
        "du ein Stadtmusikant werden.« Die Katze hielt das für gut und ging mit. "
        "Darauf kamen die drei Landesflüchtigen an einem Hof vorbei, da saß auf "
        "dem Tor der Haushahn und schrie aus Leibeskräften. »Du schreist einem "
        "durch Mark und Bein,« sprach der Esel, »was hast du vor?« — »Da hab ich "
        "gut Wetter prophezeit,« sprach der Hahn, »weil unserer lieben Frauen Tag "
        "ist, wo sie dem Christkindlein die Hemdchen gewaschen hat und sie "
        "trocknen will; aber weil morgen zum Sonntag Gäste kommen, so hat die "
        "Hausfrau doch kein Erbarmen und hat der Köchin gesagt, sie wollte mich "
        "morgen in der Suppe essen, und da soll ich mir heut abend den Kopf "
        "abschneiden lassen. Nun schrei ich aus vollem Hals, solang ich noch "
        "kann.« — »Ei was, du Rotkopf,« sagte der Esel, »zieh lieber mit uns fort, "
        "wir gehen nach Bremen, etwas Besseres als den Tod findest du überall; du "
        "hast eine gute Stimme, und wenn wir zusammen musizieren, so muß es eine "
        "Art haben.« Der Hahn ließ sich den Vorschlag gefallen, und sie gingen "
        "alle viere zusammen fort.\n"
        "\n"
        "Sie konnten aber die Stadt Bremen in einem Tag nicht erreichen und kamen "
        "abends in einen Wald, wo sie übernachten wollten. Der Esel und der Hund "
        "legten sich unter einen großen Baum, die Katze und der Hahn machten sich "
        "in die Äste, der Hahn aber flog bis in die Spitze, wo es am sichersten "
        "für ihn war. Ehe er einschlief, sah er sich noch einmal nach allen vier "
        "Winden um, da deuchte ihn, er sähe in der Ferne ein Fünkchen brennen und "
        "rief seinen Gesellen zu, es müßte nicht gar weit ein Haus sein, denn es "
        "scheine ein Licht. Sprach der Esel: »So müssen wir uns aufmachen und noch "
        "hingehen, denn hier ist die Herberge schlecht.« Der Hund meinte, ein paar "
        "Knochen und etwas Fleisch dran, täten ihm auch gut. Also machten sie sich "
        "auf den Weg nach der Gegend, wo das Licht war, und sahen es bald heller "
        "schimmern, und es ward immer größer, bis sie vor ein hell erleuchtetes "
        "Räuberhaus kamen. Der Esel, als der größte, näherte sich dem Fenster und "
        "schaute hinein. »Was siehst du, Grauschimmel?« fragte der Hahn. »Was ich "
        "sehe?« antwortete der Esel, »einen gedeckten Tisch mit schönem Essen und "
        "Trinken, und Räuber sitzen daran und lassen's sich wohl sein.« — »Das "
        "wäre was für uns,« sprach der Hahn. »Ja, ja, ach, wären wir da!« sagte "
        "der Esel. Da ratschlagten die Tiere, wie sie es anfangen müßten, um die "
        "Räuber hinauszujagen und fanden endlich ein Mittel. Der Esel mußte sich "
        "mit den Vorderfüßen auf das Fenster stellen, der Hund auf des Esels "
        "Rücken springen, die Katze auf den Hund klettern, und endlich flog der "
        "Hahn hinauf und setzte sich der Katze auf den Kopf. Wie das geschehen "
        "war, fingen sie auf ein Zeichen insgesamt an ihre Musik zu machen. Der "
        "Esel schrie, der Hund bellte, die Katze miaute und der Hahn krähte; dann "
        "stürzten sie durch das Fenster in die Stube hinein, daß die Scheiben "
        "klirrten. Die Räuber fuhren bei dem entsetzlichen Geschrei in die Höhe, "
        "meinten nicht anders als ein Gespenst käme herein und flohen in größter "
        "Furcht in den Wald hinaus. Nun setzten sich die vier Gesellen an den "
        "Tisch, nahmen mit dem vorlieb, was übrig geblieben war, und aßen, als "
        "wenn sie vier Wochen hungern sollten. Wie die vier Spielleute fertig "
        "waren, löschten sie das Licht aus und suchten sich eine Schlafstätte, "
        "jeder nach seiner Natur und Bequemlichkeit. Der Esel legte sich auf den "
        "Mist, der Hund hinter die Türe, die Katze auf den Herd bei die warme "
        "Asche, und der Hahn setzte sich auf den Hahnenbalken. Und weil sie müde "
        "waren von ihrem langen Weg, schliefen sie auch bald ein. Als Mitternacht "
        "vorbei war und die Räuber von weitem sahen, daß kein Licht mehr im Haus "
        "brannte, auch alles ruhig schien, sprach der Hauptmann: »Wir hätten uns "
        "doch nicht sollen ins Bockshorn jagen lassen,« und hieß einen hingehen "
        "und das Haus untersuchen. Der Abgeschickte fand alles still, ging in die "
        "Küche, ein Licht anzuzünden, und weil er die glühenden, feurigen Augen "
        "der Katze für lebendige Kohlen ansah, hielt er ein Schwefelhölzchen "
        "daran, daß es Feuer fangen sollte. Aber die Katze verstand keinen Spaß, "
        "sprang ihm ins Gesicht, spie und kratzte. Da erschrak er gewaltig, lief "
        "und wollte zur Hintertüre hinaus, aber der Hund, der da lag, sprang auf "
        "und biß ihn ins Bein; und als er über den Hof an dem Miste vorbei rannte, "
        "gab ihm der Esel noch einen tüchtigen Schlag mit dem Hinterfuß; der Hahn "
        "aber, der vom Lärmen aus dem Schlaf geweckt und munter geworden war, rief "
        "vom Balken herab: »Kikeriki!« Da lief der Räuber, was er konnte, zu "
        "seinem Hauptmann zurück und sprach: »Ach, in dem Haus sitzt eine "
        "greuliche Hexe, die hat mich angehaucht und mit ihren langen Fingern mir "
        "das Gesicht zerkratzt. Und vor der Türe steht ein Mann mit einem Messer, "
        "der hat mich ins Bein gestochen; und auf dem Hof liegt ein schwarzes "
        "Ungetüm, das hat mit einer Holzkeule auf mich losgeschlagen. Und oben auf "
        "dem Dache, da sitzt der Richter, der rief, bringt mir den Schelm her. Da "
        "machte ich, daß ich fortkam.« Von nun an getrauten sich die Räuber nicht "
        "weiter in das Haus, den vier Bremer Musikanten gefiel's aber so wohl "
        "darin, daß sie nicht wieder heraus wollten. Und der das zuletzt erzählt "
        "hat, dem ist der Mund noch warm.",
        "An einem Sommermorgen saß ein Schneiderlein auf seinem Tisch am Fenster, "
        "war guter Dinge und nähte aus Leibeskräften. Da kam eine Bauersfrau die "
        "Straße herab und rief: »Gut Mus feil! Gut Mus feil!« Das klang dem "
        "Schneiderlein lieblich in die Ohren, er steckte sein zartes Haupt zum "
        "Fenster hinaus und rief: »Hier herauf, liebe Frau, hier wird Sie ihre "
        "Ware los.« Die Frau stieg die drei Treppen mit ihrem schweren Korbe zu "
        "dem Schneider herauf und mußte die Töpfe sämtlich vor ihm auspacken. Er "
        "besah sie alle, hob sie in die Höhe, hielt die Nase dran und sagte "
        "endlich: »Das Mus scheint mir gut, wieg Sie mir doch vier Lot ab, liebe "
        "Frau, wenn's auch ein Viertelpfund ist, kommt es mir nicht darauf an.« "
        "Die Frau, welche gehofft hatte, einen guten Absatz zu finden, gab ihm, "
        "was er verlangte, ging aber ganz ärgerlich und brummig fort. »Nun das Mus "
        "soll mir Gott gesegnen,« rief das Schneiderlein, »und soll mir Kraft und "
        "Stärke geben,« holte das Brot aus dem Schrank, schnitt sich ein Stück "
        "über den ganzen Laib und strich das Mus darüber. »Das wird nicht bitter "
        "schmecken,« sprach er, »aber erst will ich den Wams fertig machen, eh ich "
        "anbeiße.« Er legte das Brot neben sich, nähte weiter und machte vor "
        "Freude immer größere Stiche. Indes stieg der Geruch von dem süßen Mus "
        "hinauf an die Wand, wo die Fliegen in großer Menge saßen, so daß sie "
        "herangelockt wurden und sich scharenweis darauf niederließen. »Ei, wer "
        "hat euch eingeladen?« sprach das Schneiderlein und jagte die ungebetenen "
        "Gäste fort. Die Fliegen aber, die kein Deutsch verstanden, ließen sich "
        "nicht abweisen, sondern kamen in immer größerer Gesellschaft wieder. Da "
        "lief dem Schneiderlein endlich, wie man sagt, die Laus über die Leber, es "
        "langte aus seiner Hölle nach einem Tuchlappen und: »Wart, ich will es "
        "euch geben!« schlug es unbarmherzig drauf. Als es abzog und zählte, so "
        "lagen nicht weniger als sieben vor ihm tot und streckten die Beine. »Bist "
        "du so ein Kerl?« sprach er, und mußte selbst seine Tapferkeit bewundern, "
        "»das soll die ganze Stadt erfahren.« Und in der Hast schnitt sich das "
        "Schneiderlein einen Gürtel, nähte ihn und stickte mit großen Buchstaben "
        "darauf: »Siebene auf einen Streich!« — »Ei, was Stadt!« sprach er weiter, "
        "»die ganze Welt soll's erfahren!« und sein Herz wackelte ihm vor Freude "
        "wie ein Lämmerschwänzchen.\n"
        "\n"
        "Der Schneider band sich den Gürtel um den Leib und wollte in die Welt "
        "hinaus, weil er meinte, die Werkstätte sei zu klein für seine Tapferkeit. "
        "Eh er abzog, suchte er im Haus herum, ob nichts da wäre, was er mitnehmen "
        "könnte, er fand aber nichts als einen alten Käs, den steckte er ein. Vor "
        "dem Tore bemerkte er einen Vogel, der sich im Gesträuch gefangen hatte, "
        "der mußte zu dem Käse in die Tasche. Nun nahm er den Weg tapfer zwischen "
        "die Beine, und weil er leicht und behend war, fühlte er keine Müdigkeit. "
        "Der Weg führte ihn auf einen Berg, und als er den höchsten Gipfel "
        "erreicht hatte, so saß da ein gewaltiger Riese und schaute sich ganz "
        "gemächlich um. Das Schneiderlein ging beherzt auf ihn zu, redete ihn an "
        "und sprach: »Guten Tag, Kamerad, gelt, du sitzest da und besiehst dir die "
        "weitläufige Welt? Ich bin eben auf dem Wege dahin und will mich "
        "versuchen. Hast du Lust, mitzugehen?« Der Riese sah den Schneider "
        "verächtlich an und sprach: »Du Lump! du miserabler Kerl!« — »Das wäre!« "
        "antwortete das Schneiderlein, knöpfte den Rock auf und zeigte dem Riesen "
        "den Gürtel, »da kannst du lesen, was ich für ein Mann bin.« Der Riese "
        "las: »Siebene auf einen Streich,« meinte, das wären Menschen gewesen, die "
        "der Schneider erschlagen hätte, und kriegte ein wenig Respekt vor dem "
        "kleinen Kerl. Doch wollte er ihn erst prüfen, nahm einen Stein in die "
        "Hand und drückte ihn zusammen, daß das Wasser heraustropfte. »Das mach "
        "mir nach,« sprach der Riese, »wenn du Stärke hast.« — »Ist's weiter "
        "nichts?« sagte das Schneiderlein, »das ist bei unsereinem Spielwerk,« "
        "griff in die Tasche, holte den weichen Käs und drückte ihn, daß der Saft "
        "herauslief. »Gelt,« sprach er, »das war ein wenig besser?« Der Riese "
        "wußte nicht, was er sagen sollte, und konnte es von dem Männlein nicht "
        "glauben. Da hob der Riese einen Stein auf und warf ihn so hoch, daß man "
        "ihn mit Augen kaum noch sehen konnte: »Nun, du Erpelmännchen, das tu mir "
        "nach.« — »Gut geworfen,« sagte der Schneider, »aber der Stein hat doch "
        "wieder zur Erde herabfallen müssen, ich will dir einen werfen, der soll "
        "gar nicht wieder kommen,« griff in die Tasche, nahm den Vogel und warf "
        "ihn in die Luft. Der Vogel, froh über seine Freiheit, stieg auf, flog "
        "fort und kam nicht wieder. »Wie gefällt dir das Stückchen, Kamerad?« "
        "fragte der Schneider. »Werfen kannst du wohl,« sagte der Riese, »aber nun "
        "wollen wir sehen, ob du imstande bist, etwas Ordentliches zu tragen.« Er "
        "führte das Schneiderlein zu einem mächtigen Eichbaum, der da gefällt auf "
        "dem Boden lag, und sagte: »Wenn du stark genug bist, so hilf mir den Baum "
        "aus dem Walde heraustragen.« — »Gerne,« antwortete der kleine Mann, »nimm "
        "du nur den Stamm auf deine Schulter, ich will die Äste mit dem Gezweig "
        "aufheben und tragen, das ist doch das schwerste.« Der Riese nahm den "
        "Stamm auf die Schulter, der Schneider aber setzte sich auf einen Ast, und "
        "der Riese, der sich nicht umsehen konnte, mußte den ganzen Baum und das "
        "Schneiderlein noch obendrein forttragen. Es war da hinten ganz lustig und "
        "guter Dinge, pfiff das Liedchen: »Es ritten drei Schneider zum Tore "
        "hinaus,« als wäre das Baumtragen ein Kinderspiel. Der Riese, nachdem er "
        "ein Stück Wegs die schwere Last fortgeschleppt hatte, konnte nicht weiter "
        "und rief: »Hör, ich muß den Baum fallen lassen.« Der Schneider sprang "
        "behendiglich herab, faßte den Baum mit beiden Armen, als wenn er ihn "
        "getragen hätte, und sprach zum Riesen: »Du bist ein so großer Kerl und "
        "kannst den Baum nicht einmal tragen.«\n"
        "\n"
        "Sie gingen zusammen weiter, und als sie an einem Kirschbaum vorbeikamen, "
        "faßte der Riese die Krone des Baums, wo die zeitigsten Früchte hingen, "
        "bog sie herab, gab sie dem Schneider in die Hand und hieß ihn essen. Das "
        "Schneiderlein aber war viel zu schwach, um den Baum zu halten, und als "
        "der Riese los ließ, fuhr der Baum in die Höhe, und der Schneider ward mit "
        "in die Luft geschnellt. Als er wieder ohne Schaden herabgefallen war, "
        "sprach der Riese: »Was ist das, hast du nicht Kraft, die schwache Gerte "
        "zu halten?« — »An der Kraft fehlt es nicht,« antwortete das "
        "Schneiderlein, »meinst du, das wäre etwas für einen, der siebene mit "
        "einem Streich getroffen hat? Ich bin über den Baum gesprungen, weil die "
        "Jäger da unten in das Gebüsch schießen. Spring nach, wenn du's vermagst.« "
        "Der Riese machte den Versuch, konnte aber nicht über den Baum kommen, "
        "sondern blieb in den Ästen hängen, also, daß das Schneiderlein auch hier "
        "die Oberhand behielt.\n"
        "\n"
        "Der Riese sprach: »Wenn du so ein tapferer Kerl bist, so komm mit in "
        "unsere Höhle und übernachte bei uns.« Das Schneiderlein war bereit und "
        "folgte ihm. Als sie in der Höhle anlangten, saßen da noch andere Riesen "
        "beim Feuer, und jeder hatte ein gebratenes Schaf in der Hand und aß "
        "davon. Das Schneiderlein sah sich um und dachte: »Es ist doch hier viel "
        "weitläufiger als in meiner Werkstatt.« Der Riese wies ihm ein Bett an und "
        "sagte, er sollte sich hineinlegen und ausschlafen. Dem Schneiderlein war "
        "aber das Bett zu groß, er legte sich nicht hinein, sondern kroch in eine "
        "Ecke. Als es Mitternacht war und der Riese meinte, das Schneiderlein läge "
        "in tiefem Schlafe, so stand er auf, nahm eine große Eisenstange und "
        "schlug das Bett mit einem Schlag durch und meinte, er hätte dem "
        "Grashüpfer den Garaus gemacht. Mit dem frühesten Morgen gingen die Riesen "
        "in den Wald und hatten das Schneiderlein ganz vergessen, da kam es auf "
        "einmal ganz lustig und verwegen daher geschritten. Die Riesen erschraken, "
        "fürchteten, es schlüge sie alle tot und liefen in einer Hast fort.\n"
        "\n"
        "Das Schneiderlein zog weiter, immer seiner spitzen Nase nach. Nachdem es "
        "lange gewandert war, kam es in den Hof eines königlichen Palastes, und da "
        "es Müdigkeit empfand, so legte es sich ins Gras und schlief ein. Während "
        "es da lag, kamen die Leute, betrachteten es von allen Seiten und lasen "
        "auf dem Gürtel: »Siebene auf einen Streich.« — »Ach,« sprachen sie, »was "
        "will der große Kriegsheld hier mitten im Frieden? Das muß ein mächtiger "
        "Herr sein.« Sie gingen und meldeten es dem König und meinten, wenn Krieg "
        "ausbrechen sollte, wäre das ein wichtiger und nützlicher Mann, den man um "
        "keinen Preis fortlassen dürfte. Dem König gefiel der Rat, und er schickte "
        "einen von seinen Hofleuten an das Schneiderlein ab, der sollte ihm, wenn "
        "es aufgewacht wäre, Kriegsdienste anbieten. Der Abgesandte blieb bei dem "
        "Schläfer stehen, wartete, bis er seine Glieder streckte und die Augen "
        "aufschlug, und brachte dann seinen Antrag vor. »Eben deshalb bin ich "
        "hierher gekommen,« antwortete er, »ich bin bereit, in des Königs Dienste "
        "zu treten.« Also ward er ehrenvoll empfangen und ihm eine besondere "
        "Wohnung angewiesen.\n"
        "\n"
        "Die Kriegsleute aber waren dem Schneiderlein aufgesessen und wünschten, "
        "es wäre tausend Meilen weit weg. »Was soll daraus werden?« sprachen sie "
        "untereinander, »wenn wir Zank mit ihm kriegen und er haut zu, so fallen "
        "auf jeden Streich siebene. Da kann unsereiner nicht bestehen.« Also "
        "faßten sie einen Entschluß, begaben sich allesamt zum König und baten um "
        "ihren Abschied. »Wir sind nicht gemacht,« sprachen sie, »neben einem Mann "
        "auszuhalten, der siebene auf einen Streich schlägt.« Der König war "
        "traurig, daß er um des einen willen alle seine treuen Diener verlieren "
        "sollte, wünschte, daß seine Augen ihn nie gesehen hätten, und wäre ihn "
        "gerne wieder losgewesen. Aber er getraute sich nicht, ihm den Abschied zu "
        "geben, weil er fürchtete, er möchte ihn samt seinem Volke totschlagen und "
        "sich auf den königlichen Thron setzen. Er sann lange hin und her, endlich "
        "fand er einen Rat. Er schickte zu dem Schneiderlein und ließ ihm sagen, "
        "weil er ein so großer Kriegsheld wäre, so wollte er ihm ein Anerbieten "
        "machen. In einem Walde seines Landes hausten zwei Riesen, die mit Rauben, "
        "Morden, Sengen und Brennen großen Schaden stifteten, niemand dürfte sich "
        "ihnen nahen, ohne sich in Lebensgefahr zu setzen. Wenn er diese beiden "
        "Riesen überwände und tötete, so wollte er ihm seine einzige Tochter zur "
        "Gemahlin geben und das halbe Königreich zur Ehesteuer; auch sollten "
        "hundert Reiter mit ziehen und ihm Beistand leisten. »Das wäre so etwas "
        "für einen Mann, wie du bist,« dachte das Schneiderlein, »eine schöne "
        "Königstochter und ein halbes Königreich wird einem nicht alle Tage "
        "angeboten.« — »O ja,« gab er zur Antwort, »die Riesen will ich schon "
        "bändigen, und habe die hundert Reiter dabei nicht nötig. Wer siebene auf "
        "einen Streich trifft, braucht sich vor zweien nicht zu fürchten.«\n"
        "\n"
        "Das Schneiderlein zog aus, und die hundert Reiter folgten ihm. Als er zu "
        "dem Rand des Waldes kam, sprach er zu seinen Begleitern: »Bleibt hier nur "
        "halten, ich will schon allein mit den Riesen fertig werden.« Dann sprang "
        "er in den Wald hinein und schaute sich rechts und links um. Über ein "
        "Weilchen erblickte er beide Riesen: sie lagen unter einem Baume und "
        "schliefen und schnarchten dabei, daß sich die Äste auf und nieder bogen. "
        "Das Schneiderlein, nicht faul, las beide Taschen voll Steine und stieg "
        "damit auf den Baum. Als es in der Mitte war, rutschte es auf einen Ast, "
        "bis es gerade über die Schläfer zu sitzen kam, und ließ dem einen Riesen "
        "einen Stein nach dem andern auf die Brust fallen. Der Riese spürte lange "
        "nichts, doch endlich wachte er auf, stieß seinen Gesellen an und sprach: "
        "»Was schlägst du mich?« — »Du träumst,« sagte der andere, »ich schlage "
        "dich nicht.« Sie legten sich wieder zum Schlaf, da warf der Schneider auf "
        "den zweiten einen Stein herab. »Was soll das?« rief der andere, »warum "
        "wirfst du mich.« — »Ich werfe dich nicht,« antwortete der erste und "
        "brummte. Sie zankten sich eine Weile herum, doch weil sie müde waren, "
        "ließen sie's gut sein, und die Augen fielen ihnen wieder zu. Das "
        "Schneiderlein fing sein Spiel von neuem an, suchte den dicksten Stein aus "
        "und warf ihn dem ersten Riesen mit aller Gewalt auf die Brust. »Das ist "
        "zu arg!« schrie er, sprang wie ein Unsinniger auf und stieß seinen "
        "Gesellen wider den Baum, daß dieser zitterte. Der andere zahlte mit "
        "gleicher Münze, und sie gerieten in solche Wut, daß sie Bäume ausrissen, "
        "aufeinander losschlugen, so lang, bis sie endlich beide zugleich tot auf "
        "die Erde fielen. Nun sprang das Schneiderlein herab. »Ein Glück nur,« "
        "sprach es, »daß sie den Baum, auf dem ich saß, nicht ausgerissen haben, "
        "sonst hätte ich wie ein Eichhörnchen auf einen andern springen müssen; "
        "doch unsereiner ist flüchtig!« Er zog sein Schwert und versetzte jedem "
        "ein paar tüchtige Hiebe in die Brust, dann ging es hinaus zu den Reitern "
        "und sprach: »Die Arbeit ist getan, ich habe beiden den Garaus gemacht. "
        "Aber hart ist es hergegangen, sie haben in der Not Bäume ausgerissen und "
        "sich gewehrt, doch das hilft alles nichts, wenn einer kommt wie ich, der "
        "siebene auf einen Streich schlägt.« — »Seid Ihr denn nicht verwundet?« "
        "fragten die Reiter. »Das hat gute Wege,« antwortete der Schneider, »kein "
        "Haar haben sie mir gekrümmt.« Die Reiter wollten ihm keinen Glauben "
        "beimessen und ritten in den Wald hinein: da fanden sie die Riesen in "
        "ihrem Blute schwimmend, und rings herum lagen die ausgerissenen Bäume.\n"
        "\n"
        "Das Schneiderlein verlangte von dem König die versprochene Belohnung, den "
        "aber reute sein Versprechen und er sann aufs neue, wie er sich den Helden "
        "vom Halse schaffen könnte. »Ehe du meine Tochter und das halbe Reich "
        "erhältst,« sprach er zu ihm, »mußt du noch eine Heldentat vollbringen. In "
        "dem Walde läuft ein Einhorn, das großen Schaden anrichtet, das mußt du "
        "erst einfangen.« — »Vor einem Einhorne fürchte ich mich noch weniger als "
        "vor zwei Riesen; siebene auf einen Streich, das ist meine Sache.« Er nahm "
        "sich einen Strick und eine Axt mit, ging hinaus in den Wald, und hieß "
        "abermals die, welche ihm zugeordnet waren, außen warten. Er brauchte "
        "nicht lange zu suchen, das Einhorn kam bald daher und sprang geradezu auf "
        "den Schneider los, als wollte es ihn ohne Umstände aufspießen. »Sachte, "
        "sachte,« sprach er, »so geschwind geht das nicht,« blieb stehen und "
        "wartete, bis das Tier ganz nahe war, dann sprang er behendiglich hinter "
        "den Baum. Das Einhorn rannte mit aller Kraft gegen den Baum und spießte "
        "sein Horn so fest in den Stamm, daß es nicht Kraft genug hatte, es wieder "
        "herauszuziehen, und so war es gefangen. »Jetzt hab ich das Vöglein,« "
        "sagte der Schneider, kam hinter dem Baum hervor, legte dem Einhorn den "
        "Strick erst um den Hals, dann hieb er mit der Axt das Horn aus dem Baum, "
        "und als alles in Ordnung war, führte er das Tier ab und brachte es dem "
        "König.\n"
        "\n"
        "Der König wollte ihm den verheißenen Lohn noch nicht gewähren und machte "
        "eine dritte Forderung. Der Schneider sollte ihm vor der Hochzeit erst ein "
        "Wildschwein fangen, das in dem Wald großen Schaden tat; die Jäger sollten "
        "ihm Beistand leisten. »Gerne,« sprach der Schneider, »das ist ein "
        "Kinderspiel.« Die Jäger nahm er nicht mit in den Wald, und sie waren's "
        "wohl zufrieden, denn das Wildschwein hatte sie schon mehrmals so "
        "empfangen, daß sie keine Lust hatten, ihm nachzustellen. Als das Schwein "
        "den Schneider erblickte, lief es mit schäumendem Munde und wetzenden "
        "Zähnen auf ihn zu und wollte ihn zur Erde werfen. Der flüchtige Held aber "
        "sprang in eine Kapelle, die in der Nähe war, und gleich oben zum Fenster "
        "in einem Satze wieder hinaus. Das Schwein war hinter ihm hergelaufen, er "
        "aber hüpfte außen herum und schlug die Türe hinter ihm zu; da war das "
        "wütende Tier gefangen, das viel zu schwer und unbehilflich war, um zu dem "
        "Fenster hinauszuspringen. Das Schneiderlein rief die Jäger herbei, die "
        "mußten den Gefangenen mit eigenen Augen sehen. Der Held aber begab sich "
        "zum Könige, der nun, er mochte wollen oder nicht, sein Versprechen halten "
        "mußte und ihm seine Tochter und das halbe Königreich übergab. Hätte er "
        "gewußt, daß kein Kriegsheld, sondern ein Schneiderlein vor ihm stand, es "
        "wäre ihm noch mehr zu Herzen gegangen. Die Hochzeit ward also mit großer "
        "Pracht und kleiner Freude gehalten und aus einem Schneider ein König "
        "gemacht.\n"
        "\n"
        "Nach einiger Zeit hörte die junge Königin in der Nacht, wie ihr Gemahl im "
        "Traume sprach: »Junge, mach mir den Wams und flick mir die Hosen oder ich "
        "will dir die Elle über die Ohren schlagen.« Da merkte sie, in welcher "
        "Gasse der junge Herr geboren war, klagte am andern Morgen ihrem Vater ihr "
        "Leid und bat, er möchte ihr von dem Manne helfen, der nichts anders als "
        "ein Schneider wäre. Der König sprach ihr Trost zu und sagte: »Laß in der "
        "nächsten Nacht deine Schlafkammer offen, meine Diener sollen außen stehen "
        "und, wenn er eingeschlafen ist, hineingehen, ihn binden und auf ein "
        "Schiff tragen, das ihn in die weite Welt führt.« Die Frau war damit "
        "zufrieden, des Königs Waffenträger aber, der alles mit angehört hatte, "
        "war dem jungen Herrn gewogen und hinterbrachte ihm den ganzen Anschlag. "
        "»Dem Ding will ich einen Riegel vorschieben,« sagte das Schneiderlein. "
        "Abends legte es sich zu gewöhnlicher Zeit mit seiner Frau zu Bett. Als "
        "sie glaubte, er sei eingeschlafen, stand sie auf, öffnete die Türe und "
        "legte sich wieder. Das Schneiderlein, das sich nur stellte, als wenn es "
        "schlief, fing an mit heller Stimme zu rufen: »Junge, mach mir den Wams "
        "und flick mir die Hosen oder ich will dir die Elle über die Ohren "
        "schlagen! Ich habe siebene mit einem Streich getroffen, zwei Riesen "
        "getötet, ein Einhorn fortgeführt und ein Wildschwein gefangen, und sollte "
        "mich vor denen fürchten, die draußen vor der Kammer stehen!« Als diese "
        "den Schneider also sprechen hörten, überkam sie eine große Furcht, sie "
        "liefen, als wenn das wilde Heer hinter ihnen wäre, und keiner wollte sich "
        "mehr an ihn wagen. Also war und blieb das Schneiderlein sein Lebtag ein "
        "König.",
        "Es war ein armer Bauersmann, der saß abends beim Herd und schürte das "
        "Feuer, und die Frau saß und spann. Da sprach er: »Wie ist's so traurig, "
        "daß wir keine Kinder haben! Es ist so still bei uns, und in den andern "
        "Häusern ist's so laut und lustig.« — »Ja,« antwortete die Frau und "
        "seufzte, »wenn's nur ein einziges wäre, und wenn's auch ganz klein wäre, "
        "nur Daumens groß, so wollt ich schon zufrieden sein; wir hätten's doch "
        "von Herzen lieb.« Nun geschah es, daß die Frau kränklich ward und nach "
        "sieben Monaten ein Kind gebar, das zwar an allen Gliedern vollkommen, "
        "aber nicht länger als ein Daumen war. Da sprachen sie: »Es ist, wie wir "
        "es gewünscht haben, und es soll unser liebes Kind sein,« und nannten es "
        "nach seiner Gestalt Daumesdick . Sie ließen's nicht an Nahrung fehlen, "
        "aber das Kind ward nicht größer, sondern blieb wie es in der ersten "
        "Stunde gewesen war; doch schaute es verständig aus den Augen und zeigte "
        "sich bald als ein kluges und behendes Ding, dem alles glückte, was es "
        "anfing.\n"
        "\n"
        "Der Bauer machte sich eines Tages fertig, in den Wald zu gehen und Holz "
        "zu fällen, da sprach er so vor sich hin: »Nun wollt ich, daß einer da "
        "wäre, der mir den Wagen nachbrächte.« — »O Vater,« rief Daumesdick, »den "
        "Wagen will ich schon bringen, verlaßt Euch drauf, er soll zur bestimmten "
        "Zeit im Walde sein.« Da lachte der Mann und sprach: »Wie sollte das "
        "zugehen, du bist viel zu klein, um das Pferd mit dem Zügel zu leiten.« — "
        "»Das tut nichts, Vater, wenn nur die Mutter anspannen will, ich setze "
        "mich dem Pferd ins Ohr und rufe ihm zu, wie es gehen soll.« — »Nun,« "
        "antwortete der Vater, »einmal wollen wir's versuchen.« Als die Stunde "
        "kam, spannte die Mutter an und setzte Daumesdick ins Ohr des Pferdes, und "
        "dann rief der Kleine, wie das Pferd gehen sollte, »jüh und joh! hott und "
        "har!« Da ging es ganz ordentlich als wie bei einem Meister, und der Wagen "
        "fuhr den rechten Weg nach dem Walde. Es trug sich zu, als er eben um eine "
        "Ecke bog und der Kleine »har, har!« rief, daß zwei fremde Männer daher "
        "kamen. »Mein,« sprach der eine, »was ist das? Da fährt ein Wagen, und ein "
        "Fuhrmann ruft dem Pferde zu, und ist doch nicht zu sehen.« — »Das geht "
        "nicht mit rechten Dingen zu,« sagte der andere, »wir wollen dem Karren "
        "folgen und sehen, wo er anhält.« Der Wagen aber fuhr vollends in den Wald "
        "hinein und richtig zu dem Platze, wo das Holz gehauen ward. Als "
        "Daumesdick seinen Vater erblickte, rief er ihm zu: »Siehst du, Vater, da "
        "bin ich mit dem Wagen, nun hol mich herunter.« Der Vater faßte das Pferd "
        "mit der Linken und holte mit der Rechten sein Söhnlein aus dem Ohr, das "
        "sich ganz lustig auf einen Strohhalm niedersetzte. Als die beiden fremden "
        "Männer den Daumesdick erblickten, wußten sie nicht, was sie vor "
        "Verwunderung sagen sollten. Da nahm der eine den andern beiseite und "
        "sprach: »Hör, der kleine Kerl könnte unser Glück machen, wenn wir ihn in "
        "einer großen Stadt für Geld sehen ließen. Wir wollen ihn kaufen.« Sie "
        "gingen zu dem Bauer und sprachen: »Verkauft uns den kleinen Mann, er "
        "soll's gut bei uns haben.« — »Nein,« antwortete der Vater, »es ist mein "
        "Herzblatt und ist mir für alles Gold in der Welt nicht feil.« Daumesdick "
        "aber, als er von dem Handel gehört, war an den Rockfalten seines Vaters "
        "hinaufgekrochen, stellte sich ihm auf die Schulter und wisperte ihm ins "
        "Ohr: »Vater, gib mich nur hin, ich will schon wieder zurückkommen.« Da "
        "gab ihn der Vater für ein schönes Stück Geld den beiden Männern hin. »Wo "
        "willst du sitzen?« sprachen sie zu ihm. »Ach, setzt mich nur auf den Rand "
        "von eurem Hut, da kann ich auf und ab spazieren und die Gegend betrachten "
        "und falle doch nicht herunter.« Sie taten ihm den Willen, und als "
        "Daumesdick Abschied von seinem Vater genommen hatte, machten sie sich mit "
        "ihm fort. So gingen sie, bis es dämmrig ward, da sprach der Kleine: »Hebt "
        "mich einmal herunter, es ist nötig.« — »Bleib nur droben,« sprach der "
        "Mann, auf dessen Kopf er saß, »ich will mir nichts draus machen, die "
        "Vögel lassen mir auch manchmal was drauf fallen.« — »Nein,« sprach "
        "Daumesdick, »ich weiß auch, was sich schickt; hebt mich nur geschwind "
        "herab.« Der Mann nahm den Hut ab, und setzte den Kleinen auf einen Acker "
        "am Weg, da sprang und kroch er ein wenig zwischen den Schollen hin und "
        "her, dann schlüpfte er plötzlich in ein Mausloch, das er sich ausgesucht "
        "hatte. »Guten Abend, ihr Herren, geht nur ohne mich heim,« rief er ihnen "
        "zu und lachte sie aus. Sie liefen herbei und stachen mit Stöcken in das "
        "Mausloch, aber das war vergebliche Mühe. Daumesdick kroch immer weiter "
        "zurück, und da es bald ganz dunkel ward, so mußten sie mit Ärger und mit "
        "leerem Beutel wieder heim wandern.\n"
        "\n"
        "Als Daumesdick merkte, daß sie fort waren, kroch er aus dem "
        "unterirdischen Gang wieder hervor. »Es ist auf dem Acker in der "
        "Finsternis so gefährlich gehen,« sprach er, »wie leicht bricht einer Hals "
        "und Bein!« Zum Glück stieß er an ein leeres Schneckenhaus. »Gottlob,« "
        "sagte er, »da kann ich die Nacht sicher zubringen,« und setzte sich "
        "hinein. Nicht lange, als er eben einschlafen wollte, so hörte er zwei "
        "Männer vorüber gehen, davon sprach der eine: »Wie wir's nur anfangen, um "
        "dem reichen Pfarrer sein Geld und sein Silber zu holen?« — »Das könnt ich "
        "dir sagen,« rief Daumesdick dazwischen. »Was war das?« sprach der eine "
        "Dieb erschrocken, »ich hörte jemand sprechen.« Sie blieben stehen und "
        "horchten, da sprach Daumesdick wieder: »Nehmt mich mit, so will ich euch "
        "helfen.« — »Wo bist du denn?« — »Sucht nur auf der Erde und merkt, wo die "
        "Stimme herkommt,« antwortete er. Da fanden ihn endlich die Diebe und "
        "hoben ihn in die Höhe. »Du kleiner Wicht, was willst du uns helfen!« "
        "sprachen sie. »Seht,« antwortete er, »ich krieche zwischen den "
        "Eisenstäben in die Kammer des Pfarrers und reiche euch heraus, was ihr "
        "haben wollt.« — »Wohlan,« sagten sie, »wir wollen sehen, was du kannst.« "
        "Als sie bei dem Pfarrhaus ankamen, kroch Daumesdick in die Kammer, schrie "
        "aber gleich aus Leibeskräften: »Wollt ihr alles haben, was hier ist?« Die "
        "Diebe erschraken und sagten: »So sprich doch leise, damit niemand "
        "aufwacht.« Aber Daumesdick tat als hätte er sie nicht verstanden und "
        "schrie von neuem: »Was wollt ihr? Wollt ihr alles haben, was hier ist?« "
        "Das hörte die Köchin, die in der Stube daran schlief, richtete sich im "
        "Bett auf und horchte. Die Diebe aber waren vor Schrecken ein Stück Wegs "
        "zurückgelaufen, endlich faßten sie wieder Mut und dachten: »Der kleine "
        "Kerl will uns necken.« Sie kamen zurück und flüsterten ihm zu: »Nun mach "
        "Ernst und reich uns etwas heraus.« Da schrie Daumesdick noch einmal so "
        "laut er konnte: »Ich will euch ja alles geben, reicht nur die Hände "
        "herein.« Das hörte die horchende Magd ganz deutlich, sprang aus dem Bett "
        "und stolperte zur Tür herein. Die Diebe liefen fort und rannten, als wäre "
        "der wilde Jäger hinter ihnen. Die Magd aber, als sie nichts bemerken "
        "konnte, ging ein Licht anzünden. Wie sie damit herbeikam, machte sich "
        "Daumesdick, ohne daß er gesehen wurde, hinaus in die Scheune. Die Magd "
        "aber, nachdem sie alle Winkel durchgesucht und nichts gefunden hatte, "
        "legte sich endlich wieder zu Bett und glaubte, sie hätte mit offenen "
        "Augen und Ohren doch nur geträumt.\n"
        "\n"
        "Daumesdick war in den Heuhälmchen herumgeklettert und hatte einen schönen "
        "Platz zum Schlafen gefunden. Da wollte er sich ausruhen, bis es Tag wäre, "
        "und dann zu seinen Eltern wieder heim gehen. Aber er mußte andere Dinge "
        "erfahren! Ja, es gibt viel Trübsal und Not auf der Welt! Die Magd stieg, "
        "als der Tag graute, schon aus dem Bett, um das Vieh zu füttern. Ihr "
        "erster Gang war in die Scheune, wo sie einen Arm voll Heu packte, und "
        "gerade dasjenige, worin der arme Daumesdick lag und schlief. Er schlief "
        "aber so fest, daß er nichts gewahr ward, und nicht eher aufwachte, als "
        "bis er in dem Maul der Kuh war, die ihn mit dem Heu aufgerafft hatte. "
        "»Ach Gott,« rief er, »wie bin ich in die Walkmühle geraten!« merkte aber "
        "bald, wo er war. Da hieß es aufpassen, daß er nicht zwischen die Zähne "
        "kam und zermalmt ward, und hernach mußte er doch mit in den Magen "
        "hinabrutschen. »In dem Stübchen sind die Fenster vergessen,« sprach er, "
        "»und scheint keine Sonne hinein; ein Licht wird auch nicht gebracht.« "
        "Überhaupt gefiel ihm das Quartier schlecht, und was das schlimmste war, "
        "es kam immer mehr neues Heu zur Türe hinein, und der Platz ward immer "
        "enger. Da rief er endlich in der Angst, so laut er konnte: »Bringt mir "
        "kein frisch Futter mehr, bringt mir kein frisch Futter mehr.« Die Magd "
        "melkte gerade die Kuh, und als sie sprechen hörte, ohne jemand zu sehen, "
        "und es dieselbe Stimme war, die sie auch in der Nacht gehört hatte, "
        "erschrak sie so, daß sie von ihrem Stühlchen herabglitschte und die Milch "
        "verschüttete. Sie lief in der größten Hast zu ihrem Herrn und rief: »Ach "
        "Gott, Herr Pfarrer, die Kuh hat geredet.« — »Du bist verrückt,« "
        "antwortete der Pfarrer, ging aber doch selbst in den Stall und wollte "
        "nachsehen, was es da gäbe. Kaum aber hatte er den Fuß hineingesetzt, so "
        "rief Daumesdick aufs neue: »Bringt mir kein frisch Futter mehr, bringt "
        "mir kein frisch Futter mehr.« Da erschrak der Pfarrer selbst, meinte, es "
        "wäre ein böser Geist in die Kuh gefahren und hieß sie töten. Sie ward "
        "geschlachtet, der Magen aber, worin Daumesdick steckte, auf den Mist "
        "geworfen. Daumesdick hatte große Mühe, sich hindurch zu arbeiten und "
        "hatte große Mühe damit, doch brachte er's so weit, daß er Platz bekam, "
        "aber als er eben sein Haupt herausstrecken wollte, kam ein neues Unglück. "
        "Ein hungriger Wolf lief heran und verschlang den ganzen Magen mit einem "
        "Schluck. Daumesdick verlor den Mut nicht: »Vielleicht,« dachte er, »läßt "
        "der Wolf mit sich reden,« und rief ihm aus dem Wanste zu, »lieber Wolf, "
        "ich weiß dir einen herrlichen Fraß.« — »Wo ist der zu holen?« sprach der "
        "Wolf. »In dem und dem Haus, da mußt du durch die Gosse hineinkriechen und "
        "wirst Kuchen, Speck und Wurst finden, soviel du essen willst,« und "
        "beschrieb ihm genau seines Vaters Haus. Der Wolf ließ sich das nicht "
        "zweimal sagen, drängte sich in der Nacht zur Gosse hinein und fraß in der "
        "Vorratskammer nach Herzenslust. Als er sich gesättigt hatte, wollte er "
        "wieder fort, aber er war so dick geworden, daß er denselben Weg nicht "
        "wieder hinaus konnte. Darauf hatte Daumesdick gerechnet und fing nun an "
        "in dem Leib des Wolfs einen gewaltigen Lärm zu machen, tobte und schrie, "
        "was er konnte. »Willst du stille sein,« sprach der Wolf, »du weckst die "
        "Leute auf.« — »Ei was,« antwortete der Kleine, »du hast dich satt "
        "gefressen, ich will mich auch lustig machen,« und fing von neuem an, aus "
        "allen Kräften zu schreien. Davon erwachte endlich sein Vater und seine "
        "Mutter, liefen an die Kammer und schauten durch die Spalte hinein. Wie "
        "sie sahen, daß ein Wolf darin hauste, liefen sie davon, und der Mann "
        "holte die Axt und die Frau die Sense. »Bleib dahinten,« sprach der Mann, "
        "als sie in die Kammer traten, »wenn ich ihm einen Schlag gegeben habe und "
        "er davon noch nicht tot ist, so mußt du auf ihn einhauen und ihm den Leib "
        "zerschneiden.« Da hörte Daumesdick die Stimme seines Vaters und rief: "
        "»Lieber Vater, ich bin hier, ich stecke im Leibe des Wolfs.« Sprach der "
        "Vater voll Freuden: »Gottlob, unser liebes Kind hat sich wieder "
        "gefunden,« und hieß die Frau die Sense wegtun, damit Daumesdick nicht "
        "beschädigt würde. Danach holte er aus und schlug dem Wolf einen Schlag "
        "auf den Kopf, daß er tot niederstürzte, dann suchten sie Messer und "
        "Schere, schnitten ihm den Leib auf und zogen den Kleinen wieder hervor. "
        "»Ach,« sprach der Vater, »was haben wir für Sorge um dich ausgestanden!« "
        "— »Ja, Vater, ich bin viel in der Welt herumgekommen; gottlob, daß ich "
        "wieder frische Luft schöpfe!« — »Wo bist du denn all gewesen?« — »Ach, "
        "Vater, ich war in einem Mauseloch, in einer Kuh Bauch und in eines Wolfes "
        "Wanst; nun bleib ich bei euch.« — »Und wir verkaufen dich um alle "
        "Reichtümer der Welt nicht wieder,« sprachen die Eltern, herzten und "
        "küßten ihren lieben Daumesdick. Sie gaben ihm zu essen und trinken und "
        "ließen ihm neue Kleider machen, denn die seinigen waren ihm auf der Reise "
        "verdorben.",
        "Es war einmal ein Mann und eine Frau, die wünschten sich schon lange "
        "vergeblich ein Kind, endlich machte sich die Frau Hoffnung, der liebe "
        "Gott werde ihren Wunsch erfüllen. Die Leute hatten in ihrem Hinterhaus "
        "ein kleines Fenster, daraus konnte man in einen prächtigen Garten sehen, "
        "der voll der schönsten Blumen und Kräuter stand; er war aber von einer "
        "hohen Mauer umgeben, und niemand wagte hineinzugehen, weil er einer "
        "Zauberin gehörte, die große Macht hatte und von aller Welt gefürchtet "
        "ward. Eines Tages stand die Frau an diesem Fenster und sah in den Garten "
        "hinab, da erblickte sie ein Beet, das mit den schönsten Rapunzeln "
        "bepflanzt war; und sie sahen so frisch und grün aus, daß sie lüstern ward "
        "und das größte Verlangen empfand von den Rapunzeln zu essen. Das "
        "Verlangen nahm jeden Tag zu, und da sie wußte, daß sie keine davon "
        "bekommen konnte, so fiel sie ganz ab, sah blaß und elend aus. Da erschrak "
        "der Mann und fragte: »Was fehlt dir, liebe Frau!« — »Ach,« antwortete "
        "sie, »wenn ich keine Rapunzeln aus dem Garten hinter unserm Hause zu "
        "essen kriege, so sterbe ich.« Der Mann, der sie lieb hatte, dachte: »Eh "
        "du deine Frau sterben lässest, holst du ihr von den Rapunzeln, es mag "
        "kosten was es will.« In der Abenddämmerung stieg er also über die Mauer "
        "in den Garten der Zauberin, stach in aller Eile eine Handvoll Rapunzeln "
        "und brachte sie seiner Frau. Sie machte sich sogleich Salat daraus und aß "
        "sie in voller Begierde auf. Sie hatten ihr aber so gut, so gut "
        "geschmeckt, daß sie den andern Tag noch dreimal soviel Lust bekam. Sollte "
        "sie Ruhe haben, so mußte der Mann noch einmal in den Garten steigen. Er "
        "machte sich also in der Abenddämmerung wieder hinab, als er aber die "
        "Mauer herabgeklettert war, erschrak er gewaltig, denn er sah die Zauberin "
        "vor sich stehen. »Wie kannst du es wagen,« sprach sie mit zornigem Blick, "
        "»in meinen Garten zu steigen und wie ein Dieb mir meine Rapunzeln zu "
        "stehlen? Das soll dir schlecht bekommen.« — »Ach,« antwortete er, »laßt "
        "Gnade für Recht ergehen, ich habe mich nur aus Not dazu entschlossen. "
        "Meine Frau hat Eure Rapunzeln aus dem Fenster erblickt, und empfindet ein "
        "so großes Gelüsten, daß sie sterben würde, wenn sie nicht davon zu essen "
        "bekäme.« Da ließ die Zauberin in ihrem Zorne nach und sprach zu ihm: "
        "»Verhält es sich so, wie du sagst, so will ich dir gestatten Rapunzeln "
        "mitzunehmen soviel du willst, allein ich mache eine Bedingung: du mußt "
        "mir das Kind geben, das deine Frau zur Welt bringen wird. Es soll ihm gut "
        "gehen, und ich will für es sorgen wie eine Mutter.« Der Mann sagte in der "
        "Angst alles zu, und als die Frau in Wochen kam, so erschien sogleich die "
        "Zauberin, gab dem Kinde den Namen Rapunzel und nahm es mit sich fort.\n"
        "\n"
        "Rapunzel ward das schönste Kind unter der Sonne. Als es zwölf Jahr alt "
        "war, schloß es die Zauberin in einen Turm, der in einem Walde lag, und "
        "weder Treppe noch Türe hatte, nur ganz oben war ein kleines Fensterchen. "
        "Wenn die Zauberin hinein wollte, so stellte sie sich unten hin und rief:\n"
        "\n"
        "»Rapunzel, Rapunzel, laß mir dein Haar herunter.«\n"
        "\n"
        "Rapunzel hatte lange prächtige Haare, fein wie gesponnen Gold. Wenn sie "
        "nun die Stimme der Zauberin vernahm, so band sie ihre Zöpfe los, wickelte "
        "sie oben um einen Fensterhaken, und dann fielen die Haare zwanzig Ellen "
        "tief herunter, und die Zauberin stieg daran hinauf.\n"
        "\n"
        "Nach ein paar Jahren trug es sich zu, daß der Sohn des Königs durch den "
        "Wald ritt und an dem Turm vorüber kam. Da hörte er einen Gesang, der war "
        "so lieblich, daß er stille hielt und horchte. Das war Rapunzel, die in "
        "ihrer Einsamkeit sich die Zeit damit vertrieb, ihre süße Stimme "
        "erschallen zu lassen. Der Königssohn wollte zu ihr hinauf steigen und "
        "suchte nach einer Türe des Turms, aber es war keine zu finden. Er ritt "
        "heim, doch der Gesang hatte ihm so sehr das Herz gerührt, daß er jeden "
        "Tag hinaus in den Wald ging und zuhörte. Als er einmal so hinter einem "
        "Baum stand, sah er, daß eine Zauberin heran kam und hörte, wie sie hinauf "
        "rief:\n"
        "\n"
        "»Rapunzel, Rapunzel, laß dein Haar herunter.«\n"
        "\n"
        "Da ließ Rapunzel die Haarflechten herab, und die Zauberin stieg zu ihr "
        "hinauf. »Ist das die Leiter, auf welcher man hinauf kommt, so will ich "
        "auch einmal mein Glück versuchen.« Und den folgenden Tag, als es anfing "
        "dunkel zu werden, ging er zu dem Turme und rief:\n"
        "\n"
        "»Rapunzel, Rapunzel, laß dein Haar herunter.«\n"
        "\n"
        "Alsbald fielen die Haare herab und der Königssohn stieg hinauf.\n"
        "\n"
        "Anfangs erschrak Rapunzel gewaltig, als ein Mann zu ihr herein kam, wie "
        "ihre Augen noch nie einen erblickt hatten, doch der Königssohn fing an "
        "ganz freundlich mit ihr zu reden und erzählte ihr, daß von ihrem Gesang "
        "sein Herz so sehr sei bewegt worden, daß es ihm keine Ruhe gelassen, und "
        "er sie selbst habe sehen müssen. Da verlor Rapunzel ihre Angst, und als "
        "er sie fragte, ob sie ihn zum Manne nehmen wollte, und sie sah, daß er "
        "jung und schön war, so dachte sie: »Der wird mich lieber haben als die "
        "alte Frau Gothel,« und sagte ja, und legte ihre Hand in seine Hand. Sie "
        "sprach: »Ich will gerne mit dir gehen, aber ich weiß nicht, wie ich "
        "herabkommen kann. Wenn du kommst, so bring jedesmal einen Strang Seide "
        "mit, daraus will ich eine Leiter flechten, und wenn die fertig ist, so "
        "steige ich herunter und du nimmst mich auf dein Pferd.« Sie verabredeten, "
        "daß er bis dahin alle Abend zu ihr kommen sollte, denn bei Tag kam die "
        "Alte. Die Zauberin merkte auch nichts davon, bis einmal Rapunzel anfing "
        "und zu ihr sagte: »Sag' Sie mir doch, Frau Gothel, wie kommt es nur, Sie "
        "wird mir viel schwerer heraufzuziehen, als der junge Königssohn, der ist "
        "in einem Augenblick bei mir.« — »Ach, du gottloses Kind,« rief die "
        "Zauberin, »was muß ich von dir hören, ich dachte, ich hätte dich von "
        "aller Welt geschieden, und du hast mich doch betrogen!« In ihrem Zorne "
        "packte sie die schönen Haare der Rapunzel, schlug sie ein paarmal um ihre "
        "linke Hand, griff eine Schere mit der rechten und ritsch, ratsch, waren "
        "sie abgeschnitten, und die schönen Flechten lagen auf der Erde. Und sie "
        "war so unbarmherzig, daß sie die arme Rapunzel in eine Wüstenei brachte, "
        "wo sie in großem Jammer und Elend leben mußte.\n"
        "\n"
        "Denselben Tag aber, wo sie Rapunzel verstoßen hatte, machte abends die "
        "Zauberin die abgeschnittenen Flechten oben am Fensterhaken fest, und als "
        "der Königssohn kam und rief:\n"
        "\n"
        "»Rapunzel, Rapunzel, laß dein Haar herunter,«\n"
        "\n"
        "so ließ sie die Haare hinab. Der Königssohn stieg hinauf, aber er fand "
        "oben nicht seine liebste Rapunzel, sondern die Zauberin, die ihn mit "
        "bösen und giftigen Blicken ansah. »Aha,« rief sie höhnisch, »du willst "
        "die Frau Liebste holen, aber der schöne Vogel sitzt nicht mehr im Nest "
        "und singt nicht mehr, die Katze hat ihn geholt und wird dir auch noch die "
        "Augen auskratzen. Für dich ist Rapunzel verloren, du wirst sie nie wieder "
        "erblicken.« Der Königssohn geriet außer sich vor Schmerzen, und in der "
        "Verzweiflung sprang er den Turm herab. Das Leben brachte er davon, aber "
        "die Dornen, in die er fiel, zerstachen ihm die Augen. Da irrte er blind "
        "im Walde umher, aß nichts als Wurzeln und Beeren, und tat nichts als "
        "jammern und weinen über den Verlust seiner liebsten Frau. So wanderte er "
        "einige Jahre im Elend umher und geriet endlich in die Wüstenei, wo "
        "Rapunzel mit den Zwillingen, die sie geboren hatte, einem Knaben und "
        "Mädchen, kümmerlich lebte. Er vernahm eine Stimme, und sie deuchte ihm so "
        "bekannt. Da ging er darauf zu, und wie er heran kam, erkannte ihn "
        "Rapunzel und fiel ihm um den Hals und weinte. Zwei von ihren Tränen aber "
        "benetzten seine Augen, da wurden sie wieder klar, und er konnte damit "
        "sehen wie sonst. Er führte sie in sein Reich, wo er mit Freude empfangen "
        "ward, und sie lebten noch lange glücklich und vergnügt.\n"
        "\n"
        "Tischchen Goldesel, Knüppel aus deck dich, und dem Sack\n"
        "\n"
        "Vorzeiten war ein Schneider, der drei Söhne hatte und nur eine einzige "
        "Ziege. Aber die Ziege, weil sie alle zusammen mit ihrer Milch ernährte, "
        "mußte ihr gutes Futter haben und täglich hinaus auf die Weide geführt "
        "werden. Die Söhne taten das auch nach der Reihe. Einmal brachte sie der "
        "älteste auf den Kirchhof, wo die schönsten Kräuter standen, ließ sie da "
        "fressen und herumspringen. Abends, als es Zeit war, heimzugehen, fragte "
        "er: »Ziege, bist du satt?« Die Ziege antwortete:\n"
        "\n"
        "»Ich bin so satt, ich mag kein Blatt: meh! meh!«\n"
        "\n"
        "»So komm nach Haus,« sprach der Junge, faßte sie am Strickchen, führte "
        "sie in den Stall und band sie fest. »Nun,« sagte der alte Schneider, »hat "
        "die Ziege ihr gehöriges Futter?« — »O,« antwortete der Sohn, »die ist so "
        "satt, sie mag kein Blatt.« Der Vater aber wollte sich selbst überzeugen, "
        "ging hinab in den Stall, streichelte das liebe Tier und fragte: »Ziege, "
        "bist du auch satt?« Die Ziege antwortete:\n"
        "\n"
        "»Wovon sollt ich satt sein? Ich sprang nur über Gräbelein, und fand kein "
        "einzig Blättelein: meh! meh!«\n"
        "\n"
        "»Was muß ich hören?« rief der Schneider, lief hinauf und sprach zu dem "
        "Jungen: »Ei, du Lügner, sagst die Ziege wäre satt und hast sie hungern "
        "lassen?« Und in seinem Zorne nahm er die Elle von der Wand und jagte ihn "
        "mit Schlägen hinaus.\n"
        "\n"
        "Am andern Tag war die Reihe am zweiten Sohn, der suchte an der "
        "Gartenhecke einen Platz aus, wo lauter gute Kräuter standen, und die "
        "Ziege fraß sie rein ab. Abends als er heim wollte, fragte er: »Ziege, "
        "bist du satt?« Die Ziege antwortete:\n"
        "\n"
        "»Ich bin so satt, ich mag kein Blatt: meh! meh!«\n"
        "\n"
        "»So komm nach Haus,« sprach der Junge, zog sie heim und band sie im Stall "
        "fest. »Nun,« sagte der alte Schneider, »hat die Ziege ihr gehöriges "
        "Futter?« — »O,« antwortete der Sohn, »die ist so satt, sie mag kein "
        "Blatt.« Der Schneider wollte sich darauf nicht verlassen, ging hinab in "
        "den Stall und fragte: »Ziege, bist du auch satt?« Die Ziege antwortete:\n"
        "\n"
        "»Wovon sollt ich satt sein? Ich sprang nur über Gräbelein, und fand kein "
        "einzig Blättelein: meh! meh!«\n"
        "\n"
        "»Der gottlose Bösewicht!« schrie der Schneider, »so ein frommes Tier "
        "hungern zu lassen!« lief hinauf und schlug mit der Elle den Jungen zur "
        "Haustüre hinaus.\n"
        "\n"
        "Die Reihe kam jetzt an den dritten Sohn, der wollte seine Sache gut "
        "machen, suchte Buschwerk mit dem schönsten Laube aus und ließ die Ziege "
        "daran fressen. Abends, als er heim wollte, fragte er: »Ziege, bist du "
        "auch satt?« Die Ziege antwortete:\n"
        "\n"
        "»Ich bin so satt, ich mag kein Blatt: meh! meh!«\n"
        "\n"
        "»So komm nach Haus,« sagte der Junge, führte sie in den Stall und band "
        "sie fest. »Nun,« sagte der alte Schneider, »hat die Ziege ihr gehöriges "
        "Futter?« — »O,« antwortete der Sohn, »die ist so satt, sie mag kein "
        "Blatt.« Der Schneider traute nicht, ging hinab und fragte: »Ziege, bist "
        "du auch satt?« Das boshafte Tier antwortete:\n"
        "\n"
        "»Wovon sollt ich satt sein? Ich sprang nur über Gräbelein, und fand kein "
        "einzig Blättelein: meh! meh!«\n"
        "\n"
        "»O du Lügenbrut!« rief der Schneider, »einer so gottlos und "
        "pflichtvergessen wie der andere! Ihr sollt mich nicht länger zum Narren "
        "haben!« Und vor Zorn ganz außer sich, sprang er hinauf und gerbte dem "
        "armen Jungen mit der Elle den Rücken so gewaltig, daß er zum Haus hinaus "
        "sprang.\n"
        "\n"
        "Der alte Schneider war nun mit seiner Ziege allein. Am andern Morgen ging "
        "er hinab in den Stall, liebkoste die Ziege und sprach: »Komm, mein liebes "
        "Tierlein, ich will dich selbst zur Weide führen.« Er nahm sie am Strick "
        "und brachte sie zu grünen Hecken und unter Schafrippe und was sonst die "
        "Ziegen gerne fressen. »Da kannst du dich einmal nach Herzenslust "
        "sättigen,« sprach er zu ihr, und ließ sie weiden bis zum Abend. Da fragte "
        "er: »Ziege, bist du satt?« Sie antwortete:\n"
        "\n"
        "»Ich bin so satt, ich mag kein Blatt: meh! meh!«\n"
        "\n"
        "»So komm nach Haus,« sagte der Schneider, führte sie in den Stall und "
        "band sie fest. Als er wegging, kehrte er sich noch einmal um und sagte: "
        "»Nun bist du doch einmal satt?« Aber die Ziege machte es mit ihm nicht "
        "besser und rief:\n"
        "\n"
        "»Wie sollt ich satt sein? Ich sprang nur über Gräbelein, und fand kein "
        "einzig Blättelein: meh! meh!«\n"
        "\n"
        "Als der Schneider das hörte, stutzte er und sah wohl, daß er seine drei "
        "Söhne ohne Ursache verstoßen hatte. »Wart,« rief er, »du undankbares "
        "Geschöpf, dich fortzujagen, ist noch zu wenig, ich will dich zeichnen, "
        "daß du dich unter ehrbaren Schneidern nicht mehr darfst sehen lassen.« In "
        "einer Hast sprang er hinauf, holte sein Bartmesser, seifte der Ziege den "
        "Kopf ein und schor sie so glatt wie seine flache Hand. Und weil die Elle "
        "zu ehrenvoll gewesen wäre, holte er die Peitsche und versetzte ihr solche "
        "Hiebe, daß sie in gewaltigen Sprüngen davon lief.\n"
        "\n"
        "Der Schneider, als er so ganz einsam in seinem Hause saß, verfiel in "
        "große Traurigkeit und hätte seine Söhne gerne wieder gehabt, aber niemand "
        "wußte, wo sie hingeraten waren. Der älteste war zu einem Schreiner in die "
        "Lehre gegangen, da lernte er fleißig und unverdrossen, und als seine Zeit "
        "herum war, daß er wandern sollte, schenkte ihm der Meister ein Tischchen, "
        "das gar kein besonderes Ansehen hatte und von gewöhnlichem Holz war: aber "
        "es hatte eine gute Eigenschaft. Wenn man es hinstellte und sprach: "
        "»Tischchen, deck dich,« so war das gute Tischchen auf einmal mit einem "
        "saubern Tüchlein bedeckt und stand da ein Teller und Messer und Gabel "
        "daneben und Schüsseln mit Gesottenem und Gebratenem, soviel Platz hatten, "
        "und ein großes Glas mit rotem Wein leuchtete, daß einem das Herz lachte. "
        "Der junge Gesell dachte: »Damit hast du genug für dein Lebtag,« zog guter "
        "Dinge in der Welt umher und bekümmerte sich gar nicht darum, ob ein "
        "Wirtshaus gut oder schlecht und ob etwas darin zu finden war oder nicht. "
        "Wenn es ihm gefiel, so kehrte er gar nicht ein, sondern im Felde, im "
        "Wald, auf einer Wiese, wo er Lust hatte, nahm er sein Tischchen vom "
        "Rücken, stellte es vor sich hin und sprach: »Deck dich,« so war alles da, "
        "was sein Herz begehrte. Endlich kam es ihm in den Sinn, er wollte zu "
        "seinem Vater zurückkehren, sein Zorn würde sich gelegt haben, und mit dem "
        "Tischchen deck dich würde er ihn gerne wieder aufnehmen. Es trug sich zu, "
        "daß er auf dem Heimweg abends in ein Wirtshaus kam, das mit Gästen "
        "angefüllt war. Sie hießen ihn willkommen und luden ihn ein, sich zu ihnen "
        "zu setzen und mit ihnen zu essen, sonst würde er schwerlich noch etwas "
        "bekommen. »Nein,« antwortete der Schreiner, »die paar Bissen will ich "
        "euch nicht vor dem Munde nehmen, lieber sollt ihr meine Gäste sein.« Sie "
        "lachten und meinten, er triebe seinen Spaß mit ihnen. Er aber stellte "
        "sein hölzernes Tischchen mitten in die Stube und sprach: »Tischchen, deck "
        "dich.« Augenblicklich war es mit Speisen besetzt, so gut wie sie der Wirt "
        "nicht hätte herbeischaffen können und wovon der Geruch den Gästen "
        "lieblich in die Nase stieg. »Zugegriffen, liebe Freunde,« sprach der "
        "Schreiner, und die Gäste, als sie sahen, wie es gemeint war, ließen sich "
        "nicht zweimal bitten, rückten heran, zogen ihre Messer und griffen tapfer "
        "zu. Und was sie am meisten verwunderte, wenn eine Schüssel leer geworden "
        "war, so stellte sich gleich von selbst eine volle an ihren Platz. Der "
        "Wirt stand in einer Ecke und sah dem Dinge zu. Er wußte gar nicht, was er "
        "sagen sollte, dachte aber: »Einen solchen Koch könntest du in deiner "
        "Wirtschaft wohl brauchen.« Der Schreiner und seine Gesellschaft waren "
        "lustig bis in die späte Nacht, endlich legten sie sich schlafen, und der "
        "junge Geselle ging auch zu Bett und stellte sein Wünschtischchen an die "
        "Wand. Dem Wirte aber ließen seine Gedanken keine Ruhe, es fiel ihm ein, "
        "daß in seiner Rumpelkammer ein altes Tischchen stände, das gerade so "
        "aussähe. Das holte er ganz sachte herbei und vertauschte es mit dem "
        "Wünschtischchen. Am andern Morgen zahlte der Schreiner sein Schlafgeld, "
        "packte sein Tischchen auf, dachte gar nicht daran, daß er ein falsches "
        "hätte und ging seiner Wege. Zu Mittag kam er bei seinem Vater an, der ihn "
        "mit großer Freude empfing. »Nun, mein lieber Sohn, was hast du gelernt?« "
        "sagte er zu ihm. »Vater, ich bin ein Schreiner geworden.« — »Ein gutes "
        "Handwerk,« erwiderte der Alte, »aber was hast du von deiner Wanderschaft "
        "mitgebracht?« — »Vater, das beste, was ich mitgebracht habe, ist das "
        "Tischchen.« Der Schneider betrachtete es von allen Seiten und sagte: "
        "»Daran hast du kein Meisterstück gemacht, das ist ein altes und "
        "schlechtes Tischchen.« — »Aber es ist ein Tischchen deck dich,« "
        "antwortete der Sohn, »wenn ich es hinstelle und sage ihm, es solle sich "
        "decken, so stehen gleich die schönsten Gerichte darauf und ein Wein "
        "dabei, der das Herz erfreut. Ladet nur alle Verwandte und Freunde ein, "
        "die sollen sich einmal laben und erquicken, denn das Tischchen macht sie "
        "alle satt.« Als die Gesellschaft beisammen war, stellte er sein Tischchen "
        "mitten in die Stube und sprach: »Tischchen, deck dich.« Aber das "
        "Tischchen regte sich nicht und blieb so leer wie ein anderer Tisch, der "
        "die Sprache nicht versteht. Da merkte der arme Geselle, daß ihm das "
        "Tischchen vertauscht war und schämte sich, daß er wie ein Lügner da "
        "stand. Die Verwandten aber lachten ihn aus und mußten ungetrunken und "
        "ungegessen wieder heim wandern. Der Vater holte seine Lappen wieder "
        "herbei und schneiderte fort, der Sohn aber ging bei einem Meister in die "
        "Arbeit.\n"
        "\n"
        "Der zweite Sohn war zu einem Müller gekommen und bei ihm in die Lehre "
        "gegangen. Als er seine Jahre herum hatte, sprach der Meister: »Weil du "
        "dich so wohl gehalten hast, so schenke ich dir einen Esel von einer "
        "besonderen Art, er zieht nicht am Wagen und trägt auch keine Säcke.« — "
        "»Wozu ist er denn nütze?« fragte der junge Geselle. »Er speit Gold,« "
        "antwortete der Müller, »wenn du ihn auf ein Tuch stellst und sprichst "
        ">Bricklebrit<, so speit dir das gute Tier Goldstücke aus, hinten und "
        "vorn.« — »Das ist eine schöne Sache,« sprach der Geselle, dankte dem "
        "Meister und zog in die Welt. Wenn er Gold nötig hatte, brauchte er nur zu "
        "seinem Esel »Bricklebrit« zu sagen, so regnete es Goldstücke, und er "
        "hatte weiter keine Mühe, als sie von der Erde aufzuheben. Wo er hinkam, "
        "war ihm das Beste gut genug, und je teurer, je lieber, denn er hatte "
        "immer einen vollen Beutel. Als er sich eine Zeitlang in der Welt "
        "umgesehen hatte, dachte er: »Du mußt deinen Vater aufsuchen, wenn du mit "
        "dem Goldesel kommst, so wird er seinen Zorn vergessen und dich gut "
        "aufnehmen.« Es trug sich zu, daß er in dasselbe Wirtshaus geriet, in "
        "welchem seinem Bruder das Tischchen vertauscht war. Er führte seinen Esel "
        "an der Hand, und der Wirt wollte ihm das Tier abnehmen und anbinden, der "
        "junge Geselle aber sprach: »Gebt Euch keine Mühe, meinen Grauschimmel "
        "führe ich selbst in den Stall und binde ihn auch selbst an, denn ich muß "
        "wissen, wo er steht.« Dem Wirt kam das wunderlich vor und er meinte, "
        "einer, der seinen Esel selbst besorgen müßte, hätte nicht viel zu "
        "verzehren. Als aber der Fremde in die Tasche griff, zwei Goldstücke "
        "herausholte und sagte, er sollte nur etwas Gutes für ihn einkaufen, so "
        "machte er große Augen, lief und suchte das Beste, das er auftreiben "
        "konnte. Nach der Mahlzeit fragte der Gast, was er schuldig wäre, der Wirt "
        "wollte die doppelte Kreide nicht sparen und sagte, noch ein paar "
        "Goldstücke müßte er zulegen. Der Geselle griff in die Tasche, aber sein "
        "Gold war eben zu Ende. »Wartet einen Augenblick, Herr Wirt,« sprach er, "
        "»ich will nur gehen und Gold holen,« nahm aber das Tischtuch mit. Der "
        "Wirt wußte nicht, was das heißen sollte, war neugierig, schlich ihm nach, "
        "und da der Gast die Stalltüre zuriegelte, so guckte er durch ein Astloch. "
        "Der Fremde breitete unter dem Esel das Tuch aus, rief »Bricklebrit«, und "
        "augenblicklich fing das Tier an Gold zu speien von hinten und vorn, daß "
        "es ordentlich auf die Erde herabregnete. »Ei der tausend,« sagte der "
        "Wirt, »da sind die Dukaten bald geprägt! So ein Geldbeutel ist nicht "
        "übel!« Der Gast bezahlte seine Zeche und legte sich schlafen, der Wirt "
        "aber schlich in der Nacht herab in den Stall, führte den Münzmeister weg "
        "und band einen andern Esel an seine Stelle. Den folgenden Morgen in der "
        "Frühe zog der Geselle mit seinem Esel ab und meinte, er hätte seinen "
        "Goldesel. Mittags kam er bei seinem Vater an, der sich freute, als er ihn "
        "wiedersah und ihn gerne aufnahm. »Was ist aus dir geworden, mein Sohn?« "
        "fragte der Alte. »Ein Müller, lieber Vater,« antwortete er. »Was hast du "
        "von deiner Wanderschaft mitgebracht?« — »Weiter nichts als einen Esel.« — "
        "»Esel gibt's hier genug,« sagte der Vater, »da wäre mir doch eine gute "
        "Ziege lieber gewesen.« — »Ja,« antwortete der Sohn, »aber es ist kein "
        "gemeiner Esel, sondern ein Goldesel. Wenn ich sage: >Bricklebrit<, so "
        "speit Euch das gute Tier ein ganzes Tuch voll Goldstücke. Laßt nur alle "
        "Verwandte herbeirufen, ich mache sie alle zu reichen Leuten.« — »Das laß "
        "ich mir gefallen,« sagte der Schneider, »dann brauch ich mich mit der "
        "Nadel nicht weiter zu quälen,« sprang selbst fort und rief die Verwandten "
        "herbei. Sobald sie beisammen waren, hieß sie der Müller Platz machen, "
        "breitete sein Tuch aus und brachte den Esel in die Stube. »Jetzt gebt "
        "acht,« sagte er und rief »Bricklebrit«, aber es waren keine Goldstücke, "
        "was herabfiel, und es zeigte sich, daß das Tier nichts von der Kunst "
        "verstand, denn es bringt's nicht jeder Esel so weit. Da machte der arme "
        "Müller ein langes Gesicht, sah, daß er betrogen war und bat die "
        "Verwandten um Verzeihung, die so arm heim gingen als sie gekommen waren. "
        "Es blieb nichts übrig, der Alte mußte wieder nach der Nadel greifen und "
        "der Junge sich bei einem Müller verdingen.\n"
        "\n"
        "Der dritte Bruder war zu einem Drechsler in die Lehre gegangen, und weil "
        "es ein kunstreiches Handwerk ist, mußte er am längsten lernen. Seine "
        "Brüder aber meldeten ihm in einem Briefe, wie schlimm es ihnen ergangen "
        "wäre und wie sie der Wirt noch am letzten Abend um ihre schönen "
        "Wünschdinge gebracht hätte. Als der Drechsler nun ausgelernt hatte und "
        "wandern sollte, so schenkte ihm sein Meister, weil er sich so wohl "
        "gehalten, einen Sack und sagte: »Es liegt ein Knüppel darin.« — »Den Sack "
        "kann ich umhängen, und er kann mir gute Dienste leisten, aber was soll "
        "der Knüppel darin? Der macht ihn nur schwer.« — »Das will ich dir sagen,« "
        "antwortete der Meister, »hat dir jemand etwas zu leid getan, so sprich "
        "nur: >Knüppel, aus dem Sack<, so springt dir der Knüppel heraus unter die "
        "Leute und tanzt ihnen so lustig auf dem Rücken herum, daß sie sich acht "
        "Tage lang nicht regen und bewegen können; und eher läßt er nicht ab, als "
        "bis du sagst: >Knüppel, in den Sack<. Der Gesell dankte ihm, hing den "
        "Sack um, und wenn ihm jemand zu nahe kam und auf den Leib wollte, so "
        "sprach er: »Knüppel, aus dem Sack,« alsbald sprang der Knüppel heraus und "
        "klopfte einem nach dem andern den Rock oder Wams gleich auf dem Rücken "
        "aus und wartete nicht erst bis er ihn ausgezogen hatte; und das ging so "
        "geschwind, daß eh sich's einer versah, die Reihe schon an ihm war. Der "
        "junge Drechsler langte zur Abendzeit in dem Wirtshaus an, wo seine Brüder "
        "waren betrogen worden. Er legte seinen Ranzen vor sich auf den Tisch und "
        "fing an zu erzählen, was er alles Merkwürdiges in der Welt gesehen habe. "
        "»Ja,« sagte er, »man findet wohl ein Tischchen deck dich, einen Goldesel "
        "und dergleichen: lauter gute Dinge, die ich nicht verachte, aber das ist "
        "alles nichts gegen den Schatz, den ich mir erworben habe und mit mir da "
        "in meinem Sack führe.« Der Wirt spitzte die Ohren: »Was in aller Welt mag "
        "das sein?« dachte er, »der Sack ist wohl mit lauter Edelsteinen "
        "angefüllt; den sollte ich billig auch noch haben, denn aller guten Dinge "
        "sind drei.« Als Schlafenszeit war, streckte sich der Gast auf die Bank "
        "und legte seinen Sack als Kopfkissen unter. Der Wirt, als er meinte, der "
        "Gast läge in tiefem Schlaf, ging herbei, rückte und zog ganz sachte und "
        "vorsichtig an dem Sack, ob er ihn vielleicht wegziehen und einen andern "
        "unterlegen könnte. Der Drechsler aber hatte schon lange darauf gewartet. "
        "Wie nun der Wirt eben einen herzhaften Ruck tun wollte, rief er: "
        "»Knüppel, aus dem Sack.« Alsbald fuhr das Knüppelchen heraus, dem Wirt "
        "auf den Leib und rieb ihm die Nähte, daß es eine Art hatte. Der Wirt "
        "schrie zum Erbarmen, aber je lauter er schrie, desto kräfter schlug der "
        "Knüppel ihm den Takt dazu auf dem Rücken, bis er endlich erschöpft zur "
        "Erde fiel. Da sprach der Drechsler: »Wo du das Tischchen deck dich und "
        "den Goldesel nicht wieder heraus gibst, so soll der Tanz von neuem "
        "angeben.« — »Ach nein,« rief der Wirt ganz kleinlaut, »ich gebe alles "
        "gerne wieder heraus, laßt nur den verwünschten Kobold wieder in den Sack "
        "kriechen.« Da sprach der Geselle: »Ich will Gnade für Recht ergehen "
        "lassen, aber hüte dich vor Schaden!« Dann rief er: »Knüppel, in den "
        "Sack!« und ließ ihn ruhen.\n"
        "\n"
        "Der Drechsler zog am andern Morgen mit dem Tischchen deck dich und dem "
        "Goldesel heim zu seinem Vater. Der Schneider freute sich, als er ihn "
        "wieder sah und fragte auch ihn, was er in der Fremde gelernt hätte. "
        "»Lieber Vater,« antwortete er, »ich bin ein Drechsler geworden.« — »Ein "
        "kunstreiches Handwerk,« sagte der Vater, »was hast du von der "
        "Wanderschaft mitgebracht?« — »Ein kostbares Stück, lieber Vater,« "
        "antwortete der Sohn, »einen Knüppel in dem Sack.« — »Was!« rief der "
        "Vater, »einen Knüppel! das ist der Mühe wert! Den kannst du dir von jedem "
        "Baume abhauen.« — »Aber einen solchen nicht, lieber Vater; sage ich: "
        ">Knüppel, aus dem Sack<, so springt der Knüppel heraus und macht mit dem, "
        "der es nicht gut meint mit mir, einen schlimmen Tanz und läßt nicht eher "
        "nach, als bis er auf der Erde liegt und um gut Wetter bittet. Seht ihr, "
        "mit diesem Knüppel habe ich das Tischchen deck dich und den Goldesel "
        "wieder herbeigeschafft, die der diebische Wirt meinen Brüdern abgenommen "
        "hatte. Jetzt laßt sie beide rufen und ladet alle Verwandten ein, ich will "
        "sie speisen und tränken und will ihnen die Taschen noch mit Gold füllen.« "
        "Der alte Schneider wollte nicht recht trauen, brachte aber doch die "
        "Verwandten zusammen. Da deckte der Drechsler ein Tuch in die Stube, führe "
        "den Goldesel herein und sagte zu seinem Bruder: »Nun, lieber Bruder "
        "sprich mit ihm.« Der Müller sagte: »Bricklebrit,« und augenblicklich "
        "sprangen die Goldstücke auf das Tuch herab, als käme ein Platzregen, und "
        "der Esel hörte nicht eher auf, als bis alle so viel hatten, daß sie nicht "
        "mehr tragen konnten. (Ich sehe dir's an, du wärst auch gerne dabei "
        "gewesen.) Dann holte der Drechsler das Tischchen und sagte: »Lieber "
        "Bruder, nun sprich mit ihm.« Und kaum hatte der Schreiner »Tischchen, "
        "deck dich« gesagt, so war es gedeckt und mit den schönsten Schüsseln "
        "reichlich besetzt. Da ward eine Mahlzeit gehalten, wie der gute Schneider "
        "noch keine in seinem Hause erlebt hatte, und die ganze Verwandtschaft "
        "blieb beisammen bis in die Nacht, und waren alle lustig und vergnügt. Der "
        "Schneider verschloß Nadel und Zwirn, Elle und Bügeleisen in einen Schrank "
        "und lebte mit seinen drei Söhnen in Freude und Herrlichkeit.\n"
        "\n"
        "Wo ist aber die Ziege hingekommen, die schuld war, daß der Schneider "
        "seine drei Söhne fortjagte? Das will ich dir sagen. Sie schämte sich, daß "
        "sie einen kahlen Kopf hatte, lief in eine Fuchshöhle und verkroch sich "
        "hinein. Als der Fuchs nach Haus kam, funkelten ihm ein paar große Augen "
        "aus der Dunkelheit entgegen, daß er erschrak und wieder zurücklief. Der "
        "Bär begegnete ihm, und da der Fuchs ganz verstört aussah, so sprach er: "
        "»Was ist dir, Bruder Fuchs, was machst du für ein Gesicht?« — »Ach,« "
        "antwortete der Rote, »ein grimmig Tier sitzt in meiner Höhle und hat mich "
        "mit feurigen Augen angeglotzt.« — »Das wollen wir bald austreiben,« "
        "sprach der Bär, ging mit zu der Höhle und schaute hinein; als er aber die "
        "feurigen Augen erblickte, wandelte ihn ebenfalls Furcht an. Er wollte mit "
        "dem grimmigen Tiere nichts zu tun haben und nahm Reißaus. Die Biene "
        "begegnete ihm, und da sie merkte, daß es ihm in seiner Haut nicht wohl "
        "zumute war, sprach sie: »Bär, du machst ja ein gewaltig verdrießlich "
        "Gesicht, wo ist deine Lustigkeit geblieben?« — »Du hast gut reden,« "
        "antwortete der Bär, »es sitzt ein grimmiges Tier mit Glotzaugen in dem "
        "Hause des Roten, und wir können es nicht herausjagen.« Die Biene sprach: "
        "»Du dauerst mich, Bär, ich bin ein armes schwaches Geschöpf, das ihr im "
        "Wege nicht anguckt, aber ich glaube doch, daß ich euch helfen kann.« Sie "
        "flog in die Fuchshöhle, setzte sich der Ziege auf den glatten geschorenen "
        "Kopf und stach sie so gewaltig, daß sie aufsprang, »meh! meh!« schrie und "
        "wie toll in die Welt hineinlief; und weiß niemand auf diese Stunde, wo "
        "sie hingelaufen ist.",
        "Brüderchen nahm sein Schwesterchen an der Hand und sprach »Seit die "
        "Mutter tot ist, haben wir keine gute Stunde mehr; die Stiefmutter schlägt "
        "uns alle Tage, und wenn wir zu ihr kommen, stößt sie uns mit den Füßen "
        "fort. Die harten Brotkrusten, die übrig bleiben, sind unsere Speise, und "
        "dem Hündlein unter dem Tisch geht's besser: dem wirft sie doch manchmal "
        "einen guten Bissen zu. Daß Gott erbarm, wenn das unsere Mutter wüßte! "
        "Komm, wir wollen miteinander in die weite Welt gehen.« Sie gingen den "
        "ganzen Tag über Wiesen, Felder und Steine, und wenn es regnete, sprach "
        "das Schwesterchen: »Gott und unsere Herzen die weinen zusammen!« Abends "
        "kamen sie in einen großen Wald und waren so müde von Jammer, Hunger und "
        "dem langen Weg, daß sie sich in einen hohlen Baum setzten und "
        "einschliefen.\n"
        "\n"
        "Am andern Morgen, als sie aufwachten, stand die Sonne schon hoch am "
        "Himmel und schien heiß in den Baum hinein. Da sprach das Brüderchen: "
        "»Schwesterchen, mich dürstet, wenn ich ein Brünnlein wüßte, ich ging und "
        "tränk einmal; ich mein, ich hört eins rauschen.« Brüderchen stand auf, "
        "nahm Schwesterchen an der Hand, und sie wollten das Brünnlein suchen. Die "
        "böse Stiefmutter aber war eine Hexe und hatte wohl gesehen, wie die "
        "beiden Kinder fortgegangen waren, war ihnen nachgeschlichen, heimlich, "
        "wie die Hexen schleichen, und hatte alle Brünnlein im Walde verwünscht. "
        "Als sie nun einen Brunnen fanden, der so glitzerig über die Steine "
        "sprang, wollte das Brüderchen daraus trinken, aber das Schwesterchen "
        "hörte, wie er im Rauschen sprach: »Wer aus mir trinkt, wird ein Tiger, "
        "wer aus mir trinkt, wird ein Tiger.« — Da rief das Schwesterchen: »Ich "
        "bitte dich, Brüderchen, trink nicht, sonst wirst du ein wildes Tier und "
        "zerreißest mich.« Das Brüderchen trank nicht, ob es gleich so großen "
        "Durst hatte, und sprach: »Ich will warten bis zur nächsten Quelle.« Als "
        "sie zum zweiten Brünnlein kamen, hörte das Schwesterchen, wie auch dieses "
        "sprach: »Wer aus mir trinkt, wird ein Wolf, wer aus mir trinkt, wird ein "
        "Wolf.« — Da rief das Schwesterchen: »Brüderchen, ich bitte dich, trink "
        "nicht, sonst wirst du ein Wolf und frissest mich.« — Das Brüderchen trank "
        "nicht, und sprach: »Ich will warten, bis wir zur nächsten Quelle kommen, "
        "aber dann muß ich trinken, du magst sagen, was du willst; mein Durst ist "
        "gar zu groß.« Und als sie zum dritten Brünnlein kamen, hörte das "
        "Schwesterlein, wie es im Rauschen sprach: »Wer aus mir trinkt, wird ein "
        "Reh, wer aus mir trinkt, wird ein Reh.« — Das Schwesterchen sprach: »Ach, "
        "Brüderchen, ich bitte dich, trink nicht, sonst wirst du ein Reh und "
        "läufst mir fort.« Aber das Brüderchen hatte sich gleich beim Brünnlein "
        "niedergekniet, hinabgebeugt und von dem Wasser getrunken, und wie die "
        "ersten Tropfen auf seine Lippen gekommen waren, lag es da als ein "
        "Rehkälbchen.\n"
        "\n"
        "Nun weinte das Schwesterchen über das arme verwünschte Brüderchen, und "
        "das Rehchen weinte auch und saß so traurig neben ihm. Da sprach das "
        "Mädchen endlich: »Sei still, liebes Rehchen, ich will dich ja nimmermehr "
        "verlassen.« Dann band es sein goldenes Strumpfband ab und tat es dem "
        "Rehchen um den Hals, und rupfte Binsen und flocht ein weiches Seil "
        "daraus. Daran band es das Tierchen und führte es weiter, und ging immer "
        "tiefer in den Wald hinein. Und als sie lange lange gegangen waren, kamen "
        "sie endlich an ein kleines Haus, und das Mädchen schaute hinein, und weil "
        "es leer war, dachte es: »Hier können wir bleiben und wohnen.« Da suchte "
        "es dem Rehchen Laub und Moos zu einem weichen Lager, und jeden Morgen "
        "ging es aus und sammelte sich Wurzeln, Beeren und Nüsse, und für das "
        "Rehchen brachte es zartes Gras mit, das fraß es ihm aus der Hand, war "
        "vergnügt und spielte vor ihm herum. Abends wenn Schwesterchen müde war "
        "und sein Gebet gesagt hatte, legte es seinen Kopf auf den Rücken des "
        "Rehkälbchens, das war sein Kissen, darauf es sanft einschlief. Und hätte "
        "das Brüderchen nur seine menschliche Gestalt gehabt, es wäre ein "
        "herrliches Leben gewesen.\n"
        "\n"
        "Das dauerte eine Zeitlang, daß sie so allein in der Wildnis waren. Es "
        "trug sich aber zu, daß der König des Landes eine große Jagd in dem Wald "
        "hielt. Da schallte das Hörnerblasen, Hundegebell und das lustige Geschrei "
        "der Jäger durch die Bäume, und das Rehlein hörte es und wäre gar zu gerne "
        "dabei gewesen. »Ach,« sprach es zum Schwesterlein, »laß mich hinaus in "
        "die Jagd, ich kann's nicht länger mehr aushalten,« und bat so lange, bis "
        "es einwilligte. »Aber,« sprach es zu ihm, »komm mir ja abends wieder, vor "
        "den wilden Jägern schließ ich mein Türlein; und damit ich dich kenne, so "
        "klopf und sprich: mein Schwesterlein, laß mich herein. Und wenn du nicht "
        "so sprichst, so schließ ich mein Türlein nicht auf.« Nun sprang das "
        "Rehchen hinaus, und war ihm so wohl und war so lustig in freier Luft. Der "
        "König und seine Jäger sahen das schöne Tier und setzten ihm nach, aber "
        "sie konnten es nicht einholen, und wenn sie meinten, sie hätten es gewiß, "
        "da sprang es über das Gebüsch weg und war verschwunden. Als es dunkel "
        "ward, lief es zu dem Häuschen, klopfte und sprach: »Mein Schwesterlein, "
        "laß mich herein.« Da ward ihm die kleine Tür aufgetan, es sprang hinein "
        "und ruhete sich die ganze Nacht auf seinem weichen Lager aus. Am andern "
        "Morgen ging die Jagd von neuem an, und als das Rehlein wieder das "
        "Hifthorn hörte und das ho, ho! der Jäger, da hatte es keine Ruhe, und "
        "sprach: »Schwesterchen, mach mir auf, ich muß hinaus.« Das Schwesterchen "
        "öffnete ihm die Türe und sprach: »Aber zum Abend mußt du wieder da sein "
        "und dein Sprüchlein sagen.« Als der König und seine Jäger das Rehlein mit "
        "dem goldenen Halsband wieder sahen, jagten sie ihm alle nach, aber es war "
        "ihnen zu schnell und behend. Das währte den ganzen Tag, endlich aber "
        "hatten es die Jäger abends umzingelt, und einer verwundete es ein wenig "
        "am Fuß, so daß es hinken mußte und langsam fortlief. Da schlich ihm ein "
        "Jäger nach bis zu dem Häuschen und hörte wie es rief: »Mein "
        "Schwesterlein, laß mich herein,« und sah, daß die Tür ihm aufgetan und "
        "alsbald wieder zugeschlossen ward. Der Jäger behielt das alles wohl im "
        "Sinn, ging zum König und erzählte ihm, was er gesehen und gehört hatte. "
        "Da sprach der König: »Morgen soll noch einmal gejagt werden.«\n"
        "\n"
        "Das Schwesterchen aber erschrak gewaltig, als es sah, daß sein "
        "Rehkälbchen verwundet war. Es wusch ihm das Blut ab, legte Kräuter auf "
        "und sprach: »Geh auf dein Lager, lieb Rehchen, daß du wieder heil wirst.« "
        "Die Wunde aber war so gering, daß das Rehchen am Morgen nichts mehr davon "
        "spürte. Und als es die Jagdlust wieder draußen hörte, sprach es: »Ich "
        "kann's nicht aushalten, ich muß dabei sein; so bald soll mich keiner "
        "kriegen.« Das Schwesterchen weinte und sprach: »Nun werden sie dich "
        "töten, und ich bin hier allein im Wald und bin verlassen von aller Welt. "
        "Ich laß dich nicht hinaus.« — »So sterb ich dir hier vor Betrübnis,« "
        "antwortete das Rehchen, »wenn ich das Hifthorn höre, so mein ich, ich "
        "müßt aus den Schuhen springen!« Da konnte das Schwesterchen nicht anders "
        "und schloß ihm mit schwerem Herzen die Tür auf, und das Rehchen sprang "
        "gesund und fröhlich in den Wald. Als es der König erblickte, sprach er zu "
        "seinen Jägern: »Nun jagt ihm nach den ganzen Tag bis in die Nacht, aber "
        "daß ihm keiner etwas zuleide tut.« Sobald die Sonne untergegangen war, "
        "sprach der König zum Jäger: »Nun komm und zeige mir das Waldhäuschen.« "
        "Und als er vor dem Türlein war, klopfte er an und rief: »Lieb "
        "Schwesterlein, laß mich herein.« Da ging die Tür auf, und der König trat "
        "herein, und da stand ein Mädchen, das war so schön, wie er noch keins "
        "gesehen hatte. Das Mädchen erschrak, als es sah, daß nicht sein Rehlein, "
        "sondern ein Mann herein kam, der eine goldene Krone auf dem Haupt hatte. "
        "Aber der König sah es freundlich an, reichte ihm die Hand und sprach: "
        "»Willst du mit mir gehen auf mein Schloß und meine liebe Frau sein?« — "
        "»Ach ja,« antwortete das Mädchen, »aber das Rehchen muß auch mit, das "
        "verlaß ich nicht.« Sprach der König: »Es soll bei dir bleiben, so lange "
        "du lebst, und soll ihm an nichts fehlen.« Indem kam es hereingesprungen, "
        "da band es das Schwesterchen wieder an das Binsenseil, nahm es selbst in "
        "die Hand und ging mit ihm aus dem Waldhäuschen fort.\n"
        "\n"
        "Der König nahm das schöne Mädchen auf sein Pferd und führte es in sein "
        "Schloß, wo die Hochzeit mit großer Pracht gefeiert wurde, und war es nun "
        "die Frau Königin, und lebten sie lange Zeit vergnügt zusammen; das "
        "Rehlein ward gehegt und gepflegt und sprang in dem Schloßgarten herum. "
        "Die böse Stiefmutter aber, um derentwillen die Kinder in die Welt "
        "hineingegangen waren, die meinte nicht anders, als Schwesterchen wäre von "
        "den wilden Tieren im Walde zerrissen worden und Brüderchen als ein "
        "Rehkalb von den Jägern totgeschossen. Als sie nun hörte, daß sie so "
        "glücklich waren und es ihnen so wohlging, da wurden Neid und Mißgunst in "
        "ihrem Herzen rege und ließen ihr keine Ruhe, und sie hatte keinen andern "
        "Gedanken, als wie sie die beiden doch noch ins Unglück bringen könnte. "
        "Ihre rechte Tochter, die häßlich war wie die Nacht, und nur ein Auge "
        "hatte, die machte ihr Vorwürfe und sprach: »Eine Königin zu werden, das "
        "Glück hätte mir gebührt.« — »Sei nur still,« sagte die Alte und sprach "
        "sie zufrieden, »wenn's Zeit ist, will ich schon bei der Hand sein.« Als "
        "nun die Zeit herangerückt war, und die Königin ein schönes Knäblein zur "
        "Welt gebracht hatte, und der König gerade auf der Jagd war, nahm die alte "
        "Hexe die Gestalt der Kammerfrau an, trat in die Stube, wo die Königin lag "
        "und sprach zu der Kranken: »Kommt, das Bad ist fertig, das wird Euch wohl "
        "tun und frische Kräfte geben, geschwind, eh es kalt wird.« Ihre Tochter "
        "war auch bei der Hand, sie trugen die schwache Königin in die Badestube "
        "und legten sie in die Wanne. Dann schlossen sie die Tür ab und liefen "
        "davon. In der Badestube aber hatten sie ein rechtes Höllenfeuer "
        "angemacht, daß die schöne junge Königin bald ersticken mußte.\n"
        "\n"
        "Als das vollbracht war, nahm die Alte ihre Tochter, setzte ihr eine Haube "
        "auf und legte sie ins Bett an der Königin Stelle. Sie gab ihr auch die "
        "Gestalt und das Ansehen der Königin, nur das verlorene Auge konnte sie "
        "ihr nicht wiedergeben. Damit es aber der König nicht merkte, mußte sie "
        "sich auf die Seite legen, wo sie kein Auge hatte. Am Abend, als er heim "
        "kam und hörte, daß ihm ein Söhnlein geboren war, freute er sich herzlich, "
        "und wollte ans Bett seiner lieben Frau gehen und sehen, was sie machte. "
        "Da rief die Alte geschwind: »Beileibe, laßt die Vorhänge zu, die Königin "
        "darf noch nicht ins Licht sehen und muß Ruhe haben.« Der König ging "
        "zurück und wußte nicht, daß eine falsche Königin im Bette lag.\n"
        "\n"
        "Als es aber Mitternacht war und alles schlief, da sah die Kinderfrau, die "
        "in der Kinderstube neben der Wiege saß und allein noch wachte, wie die "
        "Türe aufging, und die rechte Königin herein trat. Sie nahm das Kind aus "
        "der Wiege, legte es in ihren Arm und gab ihm zu trinken. Dann schüttelte "
        "sie ihm sein Kißchen, legte es wieder hinein und deckte es mit dem "
        "Deckbettchen zu. Sie vergaß aber auch das Rehchen nicht, ging in die "
        "Ecke, wo es lag, und streichelte ihm über den Rücken. Darauf ging sie "
        "ganz stillschweigend wieder zur Türe hinaus, und die Kinderfrau fragte am "
        "andern Morgen die Wächter, ob jemand während der Nacht ins Schloß "
        "gegangen wäre, aber sie antworteten: »Nein, wir haben niemand gesehen.« "
        "So kam sie viele Nächte und sprach niemals ein Wort dabei; die Kinderfrau "
        "sah sie immer, aber sie getraute sich nicht jemand etwas davon zu sagen.\n"
        "\n"
        "Als nun so eine Zeit verflossen war, da hub die Königin in der Nacht an "
        "zu reden und sprach:\n"
        "\n"
        "»Was macht mein Kind? was macht mein Reh? Nun komm ich noch zweimal und "
        "dann nimmermehr.«\n"
        "\n"
        "Die Kinderfrau antwortete ihr nicht, aber als sie wieder verschwunden "
        "war, ging sie zum König und erzählte ihm alles. Sprach der König: »Ach "
        "Gott, was ist das! Ich will in der nächsten Nacht bei dem Kinde wachen.« "
        "Abends ging er in die Kinderstube, aber um Mitternacht erschien die "
        "Königin wieder und sprach:\n"
        "\n"
        "»Was macht mein Kind? was macht mein Reh? Nun komm ich noch einmal und "
        "dann nimmermehr.«\n"
        "\n"
        "Und pflegte dann des Kindes wie sie gewöhnlich tat, ehe sie verschwand. "
        "Der König getraute sich nicht sie anzureden, aber er wachte auch in der "
        "folgenden Nacht. Sie sprach abermals:\n"
        "\n"
        "»Was macht mein Kind? was macht mein Reh? Nun komm ich noch diesmal und "
        "dann nimmermehr.«\n"
        "\n"
        "Da konnte sich der König nicht zurückhalten, sprang zu ihr und sprach: "
        "»Du kannst niemand anders sein, als meine liebe Frau.« Da antwortete sie: "
        "»Ja, ich bin deine liebe Frau,« und hatte in dem Augenblicke durch Gottes "
        "Gnade das Leben wieder erhalten, war frisch, rot und gesund. Darauf "
        "erzählte sie dem König den Frevel, den die böse Hexe und ihre Tochter an "
        "ihr verübt hatten. Der König ließ beide vor Gericht führen, und es ward "
        "ihnen das Urteil gesprochen. Die Tochter ward in den Wald geführt, wo sie "
        "die wilden Tiere zerrissen, die Hexe aber ward ins Feuer gelegt und mußte "
        "jammervoll verbrennen. Und wie sie zu Asche verbrannt war, verwandelte "
        "sich das Rehkälbchen und erhielt seine menschliche Gestalt wieder; "
        "Schwesterchen und Brüderchen aber lebten glücklich zusammen bis an ihr "
        "Ende.",
        "Einem reichen Manne, dem wurde seine Frau krank, und als sie fühlte, daß "
        "ihr Ende herankam, rief sie ihr einziges Töchterlein zu sich ans Bett und "
        "sprach: »Liebes Kind, bleib fromm und gut, so wird dir der liebe Gott "
        "immer beistehen, und ich will vom Himmel auf dich herabblicken und will "
        "um dich sein.« Darauf tat sie die Augen zu und verschied. Das Mädchen "
        "ging jeden Tag hinaus zu dem Grabe der Mutter und weinte und blieb fromm "
        "und gut. Als der Winter kam, deckte der Schnee ein weißes Tüchlein auf "
        "das Grab, und als die Sonne im Frühjahr es wieder herabgezogen hatte, "
        "nahm sich der Mann eine andere Frau.\n"
        "\n"
        "Die Frau hatte zwei Töchter mit ins Haus gebracht, die schön und weiß von "
        "Angesicht waren, aber garstig und schwarz von Herzen. Da ging eine "
        "schlimme Zeit für das arme Stiefkind an. »Soll die dumme Gans bei uns in "
        "der Stube sitzen!« sprachen sie, »wer Brot essen will, muß es verdienen; "
        "hinaus mit der Küchenmagd.« Sie nahmen ihm seine schönen Kleider weg, "
        "zogen ihm einen grauen alten Kittel an und gaben ihm hölzerne Schuhe. "
        "»Seht einmal die stolze Prinzessin, wie sie geputzt ist!« riefen sie, "
        "lachten und führten es in die Küche. Da mußte es von Morgen bis Abend "
        "schwere Arbeit tun, früh vor Tag aufstehn, Wasser tragen, Feuer anmachen, "
        "kochen und waschen. Obendrein taten ihm die Schwestern alles ersinnliche "
        "Herzeleid an, verspotteten es und schütteten ihm die Erbsen und Linsen in "
        "die Asche, so daß es sitzen und sie wieder auslesen mußte. Abends, wenn "
        "es sich müde gearbeitet hatte, kam es in kein Bett, sondern mußte sich "
        "neben den Herd in die Asche legen. Und weil es darum immer staubig und "
        "schmutzig aussah, nannten sie es Aschenputtel.\n"
        "\n"
        "Es trug sich zu, daß der Vater einmal in die Messe ziehen wollte, da "
        "fragte er die beiden Stieftöchter, was er ihnen mitbringen sollte? "
        "»Schöne Kleider,« sagte die eine, »Perlen und Edelsteine,« die zweite. — "
        "»Aber du, Aschenputtel,« sprach er, »was willst du haben?« — »Vater, das "
        "erste Reis, das Euch auf Eurem Heimweg an den Hut stößt, das brecht für "
        "mich ab.« Er kaufte nun für die beiden Stiefschwestern schöne Kleider, "
        "Perlen und Edelsteine, und auf dem Rückweg, als er durch einen grünen "
        "Busch ritt, streifte ihn ein Haselreis und stieß ihm den Hut ab. Da brach "
        "er das Reis ab und nahm es mit. Als er nach Haus kam, gab er den "
        "Stieftöchtern, was sie sich gewünscht hatten, und dem Aschenputtel gab er "
        "das Reis von dem Haselbusch. Aschenputtel dankte ihm, ging zu seiner "
        "Mutter Grab und pflanzte das Reis darauf und weinte so sehr, daß die "
        "Tränen darauf niederfielen und es begossen. Es wuchs aber und ward ein "
        "schöner Baum. Aschenputtel ging alle Tage dreimal darunter, weinte und "
        "betete, und allemal kam ein weißes Vöglein auf den Baum, und wenn es "
        "einen Wunsch aussprach, so warf ihm das Vöglein herab, was es sich "
        "gewünscht hatte.\n"
        "\n"
        "Es begab sich aber, daß der König ein Fest anstellte, das drei Tage "
        "dauern sollte und wozu alle schönen Jungfrauen im Lande eingeladen "
        "wurden, damit sich sein Sohn eine Braut aussuchen möchte. Die zwei "
        "Stiefschwestern, als sie hörten, daß sie auch dabei erscheinen sollten, "
        "waren guter Dinge, riefen Aschenputtel und sprachen: »Kämm uns die Haare, "
        "bürste uns die Schuhe und mache uns die Schnallen fest, wir gehen zur "
        "Hochzeit auf des Königs Schloß.« Aschenputtel gehorchte, weinte aber, "
        "weil es auch gern zum Tanz mitgegangen wäre, und bat die Stiefmutter, sie "
        "möchte es ihm erlauben. »Du, Aschenputtel,« sprach sie, »bist voll Staub "
        "und Schmutz und willst zur Hochzeit? Du hast keine Kleider und Schuhe und "
        "willst tanzen!« Als es aber mit Bitten anhielt, sprach sie endlich, »da "
        "habe ich dir eine Schüssel Linsen in die Asche geschüttet, wenn du die "
        "Linsen in zwei Stunden wieder ausgelesen hast, so sollst du mitgehen.« "
        "Das Mädchen ging durch die Hintertür nach dem Garten und rief: »Ihr "
        "zahmen Täubchen, ihr Turteltäubchen, all ihr Vöglein unter dem Himmel, "
        "kommt und helft mir lesen,\n"
        "\n"
        "Die guten ins Töpfchen, die schlechten ins Kröpfchen.«\n"
        "\n"
        "Da kamen zum Küchenfenster zwei weiße Täubchen herein, und danach die "
        "Turteltäubchen, und endlich schwirrten und schwärmten alle Vögel unter "
        "dem Himmel herein und ließen sich um die Asche nieder. Und die Täubchen "
        "nickten mit den Köpfchen und fingen an pik, pik, pik, pik und da fingen "
        "die übrigen auch an pik, pik, pik, pik und lasen alle guten Körnlein in "
        "die Schüssel. Kaum war eine Stunde herum, so waren sie schon fertig und "
        "flogen alle wieder hinaus. Da brachte das Mädchen die Schüssel der "
        "Stiefmutter, freute sich und glaubte, es dürfte nun mit auf die Hochzeit "
        "gehen. Aber sie sprach: »Nein, Aschenputtel, du hast keine Kleider und "
        "kannst nicht tanzen: du wirst nur ausgelacht.« — Als es nun weinte, "
        "sprach sie: »Wenn du mir zwei Schüsseln voll Linsen in einer Stunde aus "
        "der Asche rein lesen kannst, so sollst du mitgehen,« und dachte: »Das "
        "kann es ja nimmermehr.« Als sie die zwei Schüsseln Linsen in die Asche "
        "geschüttet hatte, ging das Mädchen durch die Hintertür nach dem Garten "
        "und rief: »Ihr zahmen Täubchen, ihr Turteltäubchen, all ihr Vöglein unter "
        "dem Himmel, kommt und helft mir lesen,\n"
        "\n"
        "Die guten ins Töpfchen, die schlechten ins Kröpfchen.«\n"
        "\n"
        "Da kamen zum Küchenfenster zwei weiße Täubchen herein und danach die "
        "Turteltäubchen, und endlich schwirrten und schwärmten alle Vögel unter "
        "dem Himmel herein, und ließen sich um die Asche nieder. Und die Täubchen "
        "nickten mit ihren Köpfchen und fingen an pik, pik, pik, pik, und da "
        "fingen die übrigen auch an pik, pik, pik, pik und lasen alle guten Körner "
        "in die Schüsseln. Und eh eine halbe Stunde herum war, waren sie schon "
        "fertig und flogen alle wieder hinaus. Da trug das Mädchen die Schüsseln "
        "zu der Stiefmutter, freute sich und glaubte, nun dürfte es mit auf die "
        "Hochzeit gehen. Aber sie sprach: »Es hilft dir alles nichts; du kommst "
        "nicht mit, denn du hast keine Kleider und kannst nicht tanzen; wir müßten "
        "uns deiner schämen.« Darauf kehrte sie ihm den Rücken zu und eilte mit "
        "ihren zwei stolzen Töchtern fort.\n"
        "\n"
        "Als nun niemand mehr daheim war, ging Aschenputtel zu seiner Mutter Grab "
        "unter den Haselbaum und rief:\n"
        "\n"
        "»Bäumchen, rüttel dich und schüttel dich, wirf Gold und Silber über "
        "mich.«\n"
        "\n"
        "Da warf ihm der Vogel ein golden und silbern Kleid herunter und mit Seide "
        "und Silber ausgestickte Pantoffeln. In aller Eile zog es das Kleid an und "
        "ging zur Hochzeit. Seine Schwestern aber und die Stiefmutter kannten es "
        "nicht und meinten, es müsse eine fremde Königstochter sein, so schön sah "
        "es in dem goldenen Kleide aus. An Aschenputtel dachten sie gar nicht und "
        "dachten, es säße daheim im Schmutz und suchte die Linsen aus der Asche. "
        "Der Königssohn kam ihm entgegen, nahm es bei der Hand und tanzte mit ihm. "
        "Er wollte auch sonst mit niemand tanzen, also daß er ihm die Hand nicht "
        "los ließ, und wenn ein anderer kam, es aufzufordern, sprach er: »Das ist "
        "meine Tänzerin.«\n"
        "\n"
        "Es tanzte bis es Abend war, da wollte es nach Haus gehen. Der Königssohn "
        "aber sprach: »Ich gehe mit und begleite dich,« denn er wollte sehen, wem "
        "das schöne Mädchen angehörte. Sie entwischte ihm aber und sprang in das "
        "Taubenhaus. Nun wartete der Königssohn, bis der Vater kam und sagte ihm, "
        "das fremde Mädchen wäre in das Taubenhaus gesprungen. Der Alte dachte: "
        "»Sollte es Aschenputtel sein,« und sie mußten ihm Axt und Hacken bringen, "
        "damit er das Taubenhaus entzwei schlagen konnte. Aber es war niemand "
        "darin. Und als sie ins Haus kamen, lag Aschenputtel in seinen schmutzigen "
        "Kleidern in der Asche, und ein trübes Öllämpchen brannte im Schornstein; "
        "denn Aschenputtel war geschwind aus dem Taubenhaus hinten herabgesprungen "
        "und war zu dem Haselbäumchen gelaufen. Da hatte es die schönen Kleider "
        "abgezogen und aufs Grab gelegt, und der Vogel hatte sie wieder "
        "weggenommen, und dann hatte es sich in seinem grauen Kittelchen in die "
        "Küche zur Asche gesetzt.\n"
        "\n"
        "Am andern Tag, als das Fest von neuem anhub und die Eltern und "
        "Stiefschwestern wieder fort waren, ging Aschenputtel zu dem Haselbaum und "
        "sprach:\n"
        "\n"
        "»Bäumchen, rüttel dich und schüttel dich, wirf Gold und Silber über "
        "mich.«\n"
        "\n"
        "Da warf der Vogel ein noch viel stolzeres Kleid herab, als am vorigen "
        "Tag. Und als es mit diesem Kleide auf der Hochzeit erschien, erstaunte "
        "jedermann über seine Schönheit. Der Königssohn aber hatte gewartet bis es "
        "kam, nahm es gleich bei der Hand und tanzte nur allein mit ihm. Wenn die "
        "andern kamen und es aufforderten, sprach er: »Das ist meine Tänzerin.« "
        "Als es nun Abend war, wollte es fort, und der Königssohn ging ihm nach "
        "und wollte sehen, in welches Haus es ging: aber es sprang ihm fort und in "
        "den Garten hinter dem Haus. Darin stand ein schöner großer Baum, an dem "
        "die herrlichsten Birnen hingen, es kletterte so behend wie ein "
        "Eichhörnchen zwischen die Äste, und der Königssohn wußte nicht, wo es "
        "hingekommen war. Er wartete aber bis der Vater kam und sprach zu ihm: "
        "»Das fremde Mädchen ist mir entwischt, und ich glaube, es ist auf den "
        "Birnbaum gesprungen.« Der Vater dachte: »Sollte es Aschenputtel sein,« "
        "ließ sich die Axt holen und hieb den Baum um, aber es war niemand darauf. "
        "Und als sie in die Küche kamen, lag Aschenputtel da in der Asche, wie "
        "sonst auch, denn es war auf der andern Seite vom Baum herabgesprungen, "
        "hatte dem Vogel auf dem Haselbäumchen die schönen Kleider wieder gebracht "
        "und sein graues Kittelchen angezogen.\n"
        "\n"
        "Am dritten Tag, als die Eltern und Schwestern fort waren, ging "
        "Aschenputtel wieder zu seiner Mutter Grab und sprach zu dem Bäumchen:\n"
        "\n"
        "»Bäumchen, rüttel dich und schüttel dich, wirf Gold und Silber über "
        "mich.«\n"
        "\n"
        "Nun warf ihm der Vogel ein Kleid herab, das war so prächtig und glänzend, "
        "wie es noch keins gehabt hatte, und die Pantoffeln waren ganz golden. Als "
        "es in dem Kleid zu der Hochzeit kam, wußten sie alle nicht, was sie vor "
        "Verwunderung sagen sollten. Der Königssohn tanzte ganz allein mit ihm, "
        "und wenn es einer aufforderte, sprach er: »Das ist meine Tänzerin.«\n"
        "\n"
        "Als es nun Abend war, wollte Aschenputtel fort, und der Königssohn wollte "
        "es begleiten, aber es entsprang ihm so geschwind, daß er nicht folgen "
        "konnte. Der Königssohn hatte aber eine List gebraucht und hatte die ganze "
        "Treppe mit Pech bestreichen lassen. Da war, als es hinabsprang, der linke "
        "Pantoffel des Mädchens hängen geblieben. Der Königssohn hob ihn auf, und "
        "er war klein und zierlich und ganz golden. Am nächsten Morgen ging er "
        "damit zu dem Mann und sagte zu ihm: »Keine andere soll meine Gemahlin "
        "werden als die, an deren Fuß dieser goldene Schuh paßt.« Da freuten sich "
        "die beiden Schwestern, denn sie hatten schöne Füße. Die älteste ging mit "
        "dem Schuh in die Kammer und wollte ihn anprobieren, und die Mutter stand "
        "dabei. Aber sie konnte mit der großen Zehe nicht hineinkommen, und der "
        "Schuh war ihr zu klein, da reichte ihr die Mutter ein Messer und sprach. "
        "»Hau die Zehe ab; wann du Königin bist, so brauchst du nicht mehr zu Fuß "
        "zu gehen.« Das Mädchen hieb die Zehe ab, zwängte den Fuß in den Schuh, "
        "verbiß den Schmerz und ging heraus zum Königssohn. Da nahm er sie als "
        "seine Braut aufs Pferd und ritt mit ihr fort. Sie mußten aber an dem "
        "Grabe vorbei, da saßen die zwei Täubchen auf dem Haselbäumchen und "
        "riefen:\n"
        "\n"
        "»Rucke di guck, rucke di guck, Blut ist im Schuck: Der Schuck ist zu "
        "klein, die rechte Braut sitzt noch daheim.«\n"
        "\n"
        "Da blickte er auf ihren Fuß und sah, wie das Blut herausquoll. Er wendete "
        "sein Pferd um, brachte die falsche Braut wieder nach Haus und sagte, das "
        "wäre nicht die rechte, die andere Schwester solle den Schuh anziehen. Da "
        "ging diese in die Kammer und kam mit den Zehen glücklich in den Schuh, "
        "aber die Ferse war zu groß. Da reichte ihr die Mutter ein Messer und "
        "sprach. »Hau ein Stück von der Ferse ab. Wann du Königin bist, brauchst "
        "du nicht mehr zu Fuß zu gehen.« Das Mädchen hieb ein Stück von der Ferse "
        "ab, zwängte den Fuß in den Schuh, verbiß den Schmerz und ging heraus zum "
        "Königssohn. Da nahm er sie als seine Braut aufs Pferd und ritt mit ihr "
        "fort. Als sie an dem Haselbäumchen vorbeikamen, saßen die zwei Täubchen "
        "darauf und riefen:\n"
        "\n"
        "»Rucke di guck, rucke di guck, Blut ist im Schuck: Der Schuck ist zu "
        "klein, die rechte Braut sitzt noch daheim.«\n"
        "\n"
        "Er blickte nieder auf ihren Fuß und sah, wie das Blut aus dem Schuh quoll "
        "und an den weißen Strümpfen ganz rot heraufgestiegen war. Da wendete er "
        "sein Pferd und brachte die falsche Braut wieder nach Haus. »Das ist auch "
        "nicht die rechte,« sprach er, »habt ihr keine andere Tochter?« — »Nein,« "
        "sagte der Mann, »nur von meiner verstorbenen Frau ist noch ein kleines "
        "verbuttetes Aschenputtel da. Das kann unmöglich die Braut sein.« Der "
        "Königssohn sprach: er sollte es heraufschicken, die Mutter aber "
        "antwortete: »Ach nein, das ist viel zu schmutzig, das darf sich nicht "
        "sehen lassen.« Er wollte es aber durchaus haben, und Aschenputtel mußte "
        "gerufen werden. Da wusch es sich erst Hände und Angesicht rein, ging dann "
        "hin und neigte sich vor dem Königssohn, der ihm den goldenen Schuh "
        "reichte. Dann setzte es sich auf einen Schemel, zog den Fuß aus dem "
        "schweren Holzschuh und steckte ihn in den Pantoffel, der war wie "
        "angegossen. Und als es sich in die Höhe richtete und der König ihm ins "
        "Gesicht sah, so erkannte er das schöne Mädchen, das mit ihm getanzt "
        "hatte, und rief: »Das ist die rechte Braut!« Die Stiefmutter und die "
        "beiden Schwestern erschraken und wurden bleich vor Ärger. Er aber nahm "
        "Aschenputtel aufs Pferd und ritt mit ihm fort. Als sie an dem "
        "Haselbäumchen vorbeikamen, riefen die zwei weißen Täubchen:\n"
        "\n"
        "»Rucke di guck, rucke di guck, Kein Blut im Schuck: Der Schuck ist nicht "
        "zu klein, die rechte Braut, die führt er heim.«\n"
        "\n"
        "Und als sie das gerufen hatten, kamen sie beide herabgeflogen und setzten "
        "sich dem Aschenputtel auf die Schultern, eine rechts, die andere links, "
        "und blieben da sitzen.\n"
        "\n"
        "Als die Hochzeit mit dem Königssohn sollte gehalten werden, kamen die "
        "falschen Schwestern, wollten sich einschmeicheln und teil an seinem Glück "
        "nehmen. Als die Brautleute nun zur Kirche gingen, war die älteste zur "
        "rechten, die jüngste zur linken Seite. Da pickten die Tauben einer jeden "
        "das eine Auge aus. Hernach, als sie herausgingen, war die älteste zur "
        "linken und die jüngste zur rechten. Da pickten die Tauben einer jeden das "
        "andere Auge aus. Und waren sie also für ihre Bosheit und Falschheit mit "
        "Blindheit auf ihr Lebtag gestraft.",
        "Vor einem großen Walde wohnte ein armer Holzhacker mit seiner Frau und "
        "seinen zwei Kindern; das Bübchen hieß Hänsel und das Mädchen Gretel. Er "
        "hatte wenig zu beißen und zu brechen, und einmal, als große Teuerung ins "
        "Land kam, konnte er auch das tägliche Brot nicht mehr schaffen. Wie er "
        "sich nun abends im Bette Gedanken machte und sich vor Sorgen herumwälzte, "
        "seufzte er und sprach zu seiner Frau: »Was soll aus uns werden? Wie "
        "können wir unsere armen Kinder ernähren, da wir für uns selbst nichts "
        "mehr haben?« — »Weißt du was, Mann,« antwortete die Frau, »wir wollen "
        "morgen in aller Frühe die Kinder hinaus in den Wald führen, wo er am "
        "dicksten ist. Da machen wir ihnen ein Feuer an und geben jedem noch ein "
        "Stückchen Brot, dann gehen wir an unsere Arbeit und lassen sie allein. "
        "Sie finden den Weg nicht wieder nach Hause und wir sind sie los.« — "
        "»Nein, Frau,« sagte der Mann, »das tue ich nicht; wie sollt ich's übers "
        "Herz bringen, meine Kinder im Walde allein zu lassen, die wilden Tiere "
        "würden bald kommen und sie zerreißen.« — »O du Narr,« sagte sie, »dann "
        "müssen wir alle viere Hungers sterben, du kannst nur die Bretter für die "
        "Särge hobeln,« und ließ ihm keine Ruhe, bis er einwilligte. »Aber die "
        "armen Kinder dauern mich doch,« sagte der Mann.\n"
        "\n"
        "Die zwei Kinder hatten vor Hunger auch nicht einschlafen können und "
        "hatten gehört, was die Stiefmutter zum Vater gesagt hatte. Gretel weinte "
        "bittere Tränen und sprach zu Hänsel: »Nun ist's um uns geschehen.« — "
        "»Still, Gretel,« sprach Hänsel, »gräme dich nicht, ich will uns schon "
        "helfen.« Und als die Alten eingeschlafen waren, stand er auf, zog sein "
        "Röcklein an, machte die Untertüre auf und schlich sich hinaus. Da schien "
        "der Mond ganz helle, und die weißen Kieselsteine, die vor dem Hause "
        "lagen, glänzten wie lauter Batzen. Hänsel bückte sich und steckte so viel "
        "in sein Rocktäschlein, als nur hinein wollten. Dann ging er wieder "
        "zurück, sprach zu Gretel: »Sei getrost, liebes Schwesterchen, und schlaf "
        "nur ruhig ein, Gott wird uns nicht verlassen,« und legte sich wieder in "
        "sein Bett.\n"
        "\n"
        "Als der Tag anbrach, noch ehe die Sonne aufgegangen war, kam schon die "
        "Frau und weckte die beiden Kinder, »steht auf, ihr Faulenzer, wir wollen "
        "in den Wald gehen und Holz holen.« Dann gab sie jedem ein Stückchen Brot "
        "und sprach: »Da habt ihr etwas für den Mittag, aber eßt's nicht vorher "
        "auf, weiter kriegt ihr nichts.« Gretel nahm das Brot unter die Schürze, "
        "weil Hänsel die Steine in der Tasche hatte. Danach machten sie sich alle "
        "zusammen auf den Weg nach dem Wald. Als sie ein Weilchen gegangen waren, "
        "stand Hänsel still und guckte nach dem Haus zurück und tat das wieder und "
        "immer wieder. Der Vater sprach: »Hänsel, was guckst du da und bleibst "
        "zurück, hab acht und vergiß deine Beine nicht.« — »Ach, Vater,« sagte "
        "Hänsel, »ich sehe nach meinem weißen Kätzchen, das sitzt oben auf dem "
        "Dach und will mir Ade sagen.« Die Frau sprach: »Narr, das ist dein "
        "Kätzchen nicht, das ist die Morgensonne, die auf den Schornstein "
        "scheint.« Hänsel aber hatte nicht nach dem Kätzchen gesehen, sondern "
        "immer einen von den blanken Kieselsteinen aus seiner Tasche auf den Weg "
        "geworfen.\n"
        "\n"
        "Als sie mitten in den Wald gekommen waren, sprach der Vater: »Nun sammelt "
        "Holz, ihr Kinder, ich will ein Feuer anmachen, damit ihr nicht friert.« "
        "Hänsel und Gretel trugen Reisig zusammen, einen kleinen Berg hoch. Das "
        "Reisig ward angezündet, und als die Flamme recht hoch brannte, sagte die "
        "Frau: »Nun legt euch ans Feuer, ihr Kinder, und ruht euch aus, wir gehen "
        "in den Wald und hauen Holz. Wenn wir fertig sind, kommen wir wieder und "
        "holen euch ab.«\n"
        "\n"
        "Hänsel und Gretel saßen am Feuer, und als der Mittag kam, aß jedes sein "
        "Stücklein Brot. Und weil sie die Schläge der Holzaxt hörten, so glaubten "
        "sie, ihr Vater wäre in der Nähe. Es war aber nicht die Holzaxt, es war "
        "ein Ast, den er an einen dürren Baum gebunden hatte und den der Wind hin "
        "und her schlug. Und als sie so lange gesessen hatten, fielen ihnen die "
        "Augen vor Müdigkeit zu, und sie schliefen fest ein. Als sie endlich "
        "erwachten, war es schon finstere Nacht. Gretel fing an zu weinen und "
        "sprach: »Wie sollen wir nun aus dem Wald kommen!« Hänsel aber tröstete "
        "sie, »wart nur ein Weilchen, bis der Mond aufgegangen ist, dann wollen "
        "wir den Weg schon finden.« Und als der volle Mond aufgestiegen war, so "
        "nahm Hänsel sein Schwesterchen an der Hand und ging den Kieselsteinen "
        "nach, die schimmerten wie neu geschlagene Batzen und zeigten ihnen den "
        "Weg. Sie gingen die ganze Nacht hindurch und kamen bei anbrechendem Tag "
        "wieder zu ihres Vaters Haus. Sie klopften an die Tür, und als die Frau "
        "aufmachte und sah, daß es Hänsel und Gretel war, sprach sie: »Ihr bösen "
        "Kinder, was habt ihr so lange im Walde geschlafen, wir haben geglaubt, "
        "ihr wolltet gar nicht wiederkommen.« Der Vater aber freute sich, denn es "
        "war ihm zu Herzen gegangen, daß er sie so allein zurückgelassen hatte.\n"
        "\n"
        "Nicht lange danach war wieder Not in allen Ecken, und die Kinder hörten, "
        "wie die Mutter nachts im Bette zu dem Vater sprach: »Alles ist wieder "
        "aufgezehrt, wir haben noch einen halben Laib Brot, hernach hat das Lied "
        "ein Ende. Die Kinder müssen fort, wir wollen sie tiefer in den Wald "
        "hineinführen, damit sie den Weg nicht wieder herausfinden; es ist sonst "
        "keine Rettung für uns.« Dem Mann fiel's schwer aufs Herz und er dachte: "
        "»Es wäre besser, daß du den letzten Bissen mit deinen Kindern teiltest.« "
        "Aber die Frau hörte auf nichts, was er sagte, schalt ihn und machte ihm "
        "Vorwürfe. Wer A sagt muß auch B sagen, und weil er das erste Mal "
        "nachgegeben hatte, so mußte er es auch zum zweitenmal.\n"
        "\n"
        "Die Kinder waren aber noch wach gewesen und hatten das Gespräch mit "
        "angehört. Als die Alten schliefen, stand Hänsel wieder auf, wollte hinaus "
        "und Kieselsteine auflesen, wie das vorige Mal, aber die Frau hatte die "
        "Tür verschlossen, und Hänsel konnte nicht hinaus. Aber er tröstete sein "
        "Schwesterchen und sprach: »Weine nicht, Gretel, und schlaf nur ruhig, der "
        "liebe Gott wird uns schon helfen.«\n"
        "\n"
        "Am frühen Morgen kam die Frau und holte die Kinder aus dem Bette. Sie "
        "erhielten ihr Stückchen Brot, das war aber noch kleiner als das vorige "
        "Mal. Auf dem Wege nach dem Wald bröckelte es Hänsel in der Tasche, stand "
        "oft still und warf ein Bröcklein auf die Erde. »Hänsel, was stehst du und "
        "guckst dich um,« sagte der Vater, »geh deiner Wege.« — »Ich sehe nach "
        "meinem Täubchen, das sitzt auf dem Dache und will mir Ade sagen,« "
        "antwortete Hänsel. »Narr,« sagte die Frau, »das ist dein Täubchen nicht, "
        "das ist die Morgensonne, die auf den Schornstein oben scheint.« Hänsel "
        "aber warf nach und nach alle Bröcklein auf den Weg.\n"
        "\n"
        "Die Frau führte die Kinder noch tiefer in den Wald, wo sie ihr Lebtag "
        "noch nicht gewesen waren. Da ward wieder ein großes Feuer angemacht, und "
        "die Mutter sagte: »Bleibt nur da sitzen, ihr Kinder, und wenn ihr müde "
        "seid, könnt ihr ein wenig schlafen. Wir gehen in den Wald und hauen Holz, "
        "und abends, wenn wir fertig sind, kommen wir und holen euch ab.« Als es "
        "Mittag war, teilte Gretel ihr Brot mit Hänsel, der sein Stück auf den Weg "
        "gestreut hatte. Dann schliefen sie ein, und der Abend verging, aber "
        "niemand kam zu den armen Kindern. Sie erwachten erst in der finstern "
        "Nacht, und Hänsel tröstete sein Schwesterchen und sagte: »Wart nur, "
        "Gretel, bis der Mond aufgeht, dann werden wir die Brotbröcklein sehen, "
        "die ich ausgestreut habe, die zeigen uns den Weg nach Haus.« Als der Mond "
        "kam, machten sie sich auf, aber sie fanden kein Bröcklein mehr, denn die "
        "viel tausend Vögel, die im Walde und im Felde umherfliegen, die hatten "
        "sie weggepickt. Hänsel sagte zu Gretel: »Wir werden den Weg schon "
        "finden,« aber sie fanden ihn nicht. Sie gingen die ganze Nacht und noch "
        "einen Tag von Morgen bis Abend, aber sie kamen aus dem Wald nicht heraus, "
        "und waren so hungrig, denn sie hatten nichts als die paar Beeren, die auf "
        "der Erde standen. Und weil sie so müde waren, daß die Beine sie nicht "
        "mehr tragen wollten, so legten sie sich unter einen Baum und schliefen "
        "ein.\n"
        "\n"
        "Nun war's schon der dritte Morgen, daß sie ihres Vaters Haus verlassen "
        "hatten. Sie fingen wieder an zu gehen, aber sie gerieten immer tiefer in "
        "den Wald, und wenn nicht bald Hilfe kam, so mußten sie verschmachten. Als "
        "es Mittag war, sahen sie ein schönes schneeweißes Vöglein auf einem Ast "
        "sitzen, das sang so schön, daß sie stehen blieben und ihm zuhörten. Und "
        "als es fertig war, schwang es seine Flügel und flog vor ihnen her, und "
        "sie gingen ihm nach, bis sie zu einem Häuschen gelangten, auf dessen Dach "
        "es sich setzte, und als sie ganz nah herankamen, so sahen sie, daß das "
        "Häuslein aus Brot gebaut war und mit Kuchen gedeckt; aber die Fenster "
        "waren von hellem Zucker. »Da wollen wir uns dran machen,« sprach Hänsel, "
        "»und eine gesegnete Mahlzeit halten. Ich will ein Stück vom Dach essen, "
        "Gretel, du kannst vom Fenster essen, das schmeckt süß.« Hänsel reichte in "
        "die Höhe und brach sich ein wenig vom Dach ab, um zu versuchen, wie es "
        "schmeckte, und Gretel stellte sich an die Scheiben und knuperte daran. Da "
        "rief eine feine Stimme aus der Stube heraus:\n"
        "\n"
        "»Knuper, knuper, kneischen, wer knupert an meinem Häuschen?«\n"
        "\n"
        "die Kinder antworteten:\n"
        "\n"
        "»Der Wind, der Wind, das himmlische Kind,«\n"
        "\n"
        "und aßen weiter, ohne sich irre machen zu lassen. Hänsel, dem das Dach "
        "sehr gut schmeckte, riß sich ein großes Stück davon herunter, und Gretel "
        "stieß eine ganze runde Fensterscheibe heraus, setzte sich nieder und tat "
        "sich wohl damit. Da ging auf einmal die Türe auf, und eine steinalte "
        "Frau, die sich auf eine Krücke stützte, kam herausgeschlichen. Hänsel und "
        "Gretel erschraken so gewaltig, daß sie fallen ließen, was sie in den "
        "Händen hielten. Die Alte aber wackelte mit dem Kopfe und sprach: »Ei, ihr "
        "lieben Kinder, wer hat euch hierher gebracht? Kommt nur herein und bleibt "
        "bei mir, es geschieht euch kein Leid.« Sie faßte beide an der Hand und "
        "führte sie in ihr Häuschen. Da ward gutes Essen aufgetragen, Milch und "
        "Pfannekuchen mit Zucker, Äpfel und Nüsse. Hernach wurden zwei schöne "
        "Bettlein weiß gedeckt, und Hänsel und Gretel legten sich hinein und "
        "meinten, sie wären im Himmel.\n"
        "\n"
        "Die Alte hatte sich nur so freundlich angestellt, sie war aber eine böse "
        "Hexe, die den Kindern auflauerte, und hatte das Brothäuslein bloß gebaut, "
        "um sie herbeizulocken. Wenn eins in ihre Gewalt kam, so machte sie es "
        "tot, kochte es und aß es, und das war ihr ein Festtag. Die Hexen haben "
        "rote Augen und können nicht weit sehen, aber sie haben eine feine "
        "Witterung, wie die Tiere, und merken's, wenn Menschen herankommen. Als "
        "Hänsel und Gretel in ihre Nähe kamen, da lachte sie boshaft und sprach "
        "höhnisch: »Die habe ich, die sollen mir nicht wieder entwischen.« "
        "Frühmorgens, ehe die Kinder erwacht waren, stand sie schon auf, und als "
        "sie beide so lieblich ruhen sah, mit den vollen roten Backen, so murmelte "
        "sie vor sich hin: »Das wird ein guter Bissen werden.« Da packte sie "
        "Hänsel mit ihrer dürren Hand und trug ihn in einen kleinen Stall und "
        "sperrte ihn mit einer Gittertüre ein. Er mochte schreien, wie er wollte, "
        "es half ihm nichts. Dann ging sie zur Gretel, rüttelte sie wach und rief: "
        "»Steh auf, Faulenzerin, trag Wasser und koch deinem Bruder etwas Gutes, "
        "der sitzt draußen im Stall und soll fett werden. Wenn er fett ist, so "
        "will ich ihn essen.« Gretel fing an, bitterlich zu weinen, aber es war "
        "alles vergeblich, sie mußte tun, was die böse Hexe verlangte.\n"
        "\n"
        "Nun ward dem armen Hänsel das beste Essen gekocht, aber Gretel bekam "
        "nichts als Krebsschalen. Jeden Morgen schlich die Alte zu dem Ställchen "
        "und rief: »Hänsel, streck deine Finger heraus, damit ich fühle, ob du "
        "bald fett bist.« Hänsel streckte ihr aber ein Knöchlein heraus, und die "
        "Alte, die trübe Augen hatte, konnte es nicht sehen, und meinte, es wären "
        "Hänsels Finger, und verwunderte sich, daß er gar nicht fett werden "
        "wollte. Als vier Wochen herum waren und Hänsel immer mager blieb, da "
        "übernahm sie die Ungeduld, und sie wollte nicht länger warten. »Heda, "
        "Gretel,« rief sie dem Mädchen zu, »sei flink und trag Wasser. Hänsel mag "
        "fett oder mager sein, morgen will ich ihn schlachten und kochen.« Ach, "
        "wie jammerte das arme Schwesterchen, als es das Wasser tragen mußte, und "
        "wie flossen ihm die Tränen über die Backen herunter! »Lieber Gott, hilf "
        "uns doch,« rief sie aus, »hätten uns nur die wilden Tiere im Wald "
        "gefressen, so wären wir doch zusammen gestorben.« — »Spar nur dein "
        "Geplärre,« sagte die Alte, »es hilft dir alles nichts.«\n"
        "\n"
        "Frühmorgens mußte Gretel heraus, den Kessel mit Wasser aufhängen und "
        "Feuer anzünden. »Erst wollen wir backen,« sagte die Alte, »ich habe den "
        "Backofen schon eingeheizt und den Teig geknetet.« Sie stieß das arme "
        "Gretel hinaus zu dem Backofen, aus dem die Feuerflammen schon "
        "herausschlugen. »Kriech hinein,« sagte die Hexe, »und sieh zu, ob recht "
        "eingeheizt ist, damit wir das Brot hineinschießen können.« Und wenn "
        "Gretel darin war, wollte sie den Ofen zumachen, und Gretel sollte darin "
        "braten, und dann wollte sie's auch aufessen. Aber Gretel merkte, was sie "
        "im Sinn hatte und sprach: »Ich weiß nicht, wie ich's machen soll; wie "
        "komm ich da hinein?« — »Dumme Gans,« sagte die Alte, »die Öffnung ist "
        "groß genug, siehst du wohl, ich könnte selbst hinein,« krabbelte heran "
        "und steckte den Kopf in den Backofen. Da gab ihr Gretel einen Stoß, daß "
        "sie weit hineinfuhr, machte die eiserne Tür zu und schob den Riegel vor. "
        "Hu! da fing sie an zu heulen, ganz grauselig; aber Gretel lief fort, und "
        "die gottlose Hexe mußte elendiglich verbrennen.\n"
        "\n"
        "Gretel aber lief schnurstracks zum Hänsel, öffnete sein Ställchen und "
        "rief: »Hänsel, wir sind erlöst, die alte Hexe ist tot.« Da sprang Hänsel "
        "heraus, wie ein Vogel aus dem Käfig, wenn ihm die Türe aufgemacht wird. "
        "Wie haben sie sich gefreut, sind sich um den Hals gefallen, sind "
        "herumgesprungen und haben sich geküßt! Und weil sie sich nicht mehr zu "
        "fürchten brauchten, so gingen sie in das Haus der Hexe hinein, da standen "
        "in allen Ecken Kasten mit Perlen und Edelsteinen. »Die sind noch besser "
        "als Kieselsteine,« sagte Hänsel und steckte in seine Taschen, was hinein "
        "wollte, und Gretel sagte: »Ich will auch etwas mit nach Haus bringen,« "
        "und füllte sich sein Schürzchen voll. — »Aber jetzt wollen wir fort,« "
        "sagte Hänsel, »damit wir aus dem Hexenwald herauskommen.« Als sie aber "
        "ein paar Stunden gegangen waren, gelangten sie an ein großes Wasser. »Wir "
        "können nicht hinüber,« sprach Hänsel, »ich seh keinen Steg und keine "
        "Brücke.« — »Hier fährt auch kein Schiffchen,« antwortete Gretel, »aber da "
        "schwimmt eine weiße Ente, wenn ich die bitte, so hilft sie uns hinüber.« "
        "Da rief sie:\n"
        "\n"
        "»Entchen, Entchen, da steht Gretel und Hänsel. Kein Steg und keine "
        "Brücke, nimm uns auf deinen weißen Rücken.«\n"
        "\n"
        "Das Entchen kam auch heran, und Hänsel setzte sich auf und bat sein "
        "Schwesterchen sich zu ihm zu setzen. »Nein,« antwortete Gretel, »es wird "
        "dem Entchen zu schwer, es soll uns nacheinander hinüber bringen.« Das tat "
        "das gute Tierchen, und als sie glücklich drüben waren und ein Weilchen "
        "fortgingen, da kam ihnen der Wald immer bekannter und immer bekannter "
        "vor, und endlich erblickten sie von weitem ihres Vaters Haus. Da fingen "
        "sie an zu laufen, stürzten in die Stube hinein und fielen ihrem Vater um "
        "den Hals. Der Mann hatte keine frohe Stunde gehabt, seitdem er die Kinder "
        "im Walde gelassen hatte, die Frau aber war gestorben. Gretel schüttelte "
        "sein Schürzchen aus, daß die Perlen und Edelsteine in der Stube "
        "herumsprangen, und Hänsel warf eine Handvoll nach der andern aus seiner "
        "Tasche dazu. Da hatten alle Sorgen ein Ende, und sie lebten in lauter "
        "Freude zusammen. Mein Märchen ist aus, dort lauft eine Maus, wer sie "
        "fängt, darf sich eine große, große Pelzkappe daraus machen.",
        "Vorzeiten war ein König und eine Königin, die sprachen jeden Tag: »Ach, "
        "wenn wir doch ein Kind hätten!« und kriegten immer keins. Da trug sich "
        "zu, als die Königin einmal im Bade saß, daß ein Frosch aus dem Wasser ans "
        "Land kroch und zu ihr sprach: »Dein Wunsch wird erfüllt werden, ehe ein "
        "Jahr vergeht, wirst du eine Tochter zur Welt bringen.« Was der Frosch "
        "gesagt hatte, das geschah, und die Königin gebar ein Mädchen, das war so "
        "schön, daß der König vor Freude sich nicht zu lassen wußte und ein großes "
        "Fest anstellte. Er ladete nicht bloß seine Verwandte, Freunde und "
        "Bekannte, sondern auch die weisen Frauen dazu ein, damit sie dem Kind "
        "hold und gewogen wären. Es waren ihrer dreizehn in seinem Reiche, weil er "
        "aber nur zwölf goldene Teller hatte, von welchen sie essen sollten, so "
        "mußte eine von ihnen daheim bleiben. Das Fest ward mit aller Pracht "
        "gefeiert, und als es zu Ende war, beschenkten die weisen Frauen das Kind "
        "mit ihren Wundergaben: Die eine mit Tugend, die andere mit Schönheit, die "
        "dritte mit Reichtum, und so mit allem, was auf der Welt zu wünschen ist. "
        "Als elfe ihre Sprüche eben getan hatten, trat plötzlich die dreizehnte "
        "herein. Sie wollte sich dafür rächen, daß sie nicht eingeladen war, und "
        "ohne jemand zu grüßen oder nur anzusehen, rief sie mit lauter Stimme: "
        "»Die Königstochter soll sich in ihrem fünfzehnten Jahr an einer Spindel "
        "stechen und tot hinfallen.« Und ohne ein Wort weiter zu sprechen, kehrte "
        "sie sich um und verließ den Saal. Alle waren erschrocken, da trat die "
        "zwölfte hervor, die ihren Wunsch noch übrig hatte, und weil sie den bösen "
        "Spruch nicht aufheben, sondern nur ihn mildern konnte, so sagte sie: »Es "
        "soll aber kein Tod sein, sondern ein hundertjähriger tiefer Schlaf, in "
        "welchen die Königstochter fällt.«\n"
        "\n"
        "Der König, der sein liebes Kind vor dem Unglück gern bewahren wollte, "
        "ließ den Befehl ausgehen, daß alle Spindeln im ganzen Königreiche sollten "
        "verbrannt werden. An dem Mädchen aber wurden die Gaben der weisen Frauen "
        "sämtlich erfüllt, denn es war so schön, sittsam, freundlich und "
        "verständig, daß es jedermann, der es ansah, lieb haben mußte. Es geschah, "
        "daß an dem Tage, wo es gerade fünfzehn Jahr alt ward, der König und die "
        "Königin nicht zu Haus waren und das Mädchen ganz allein im Schloß "
        "zurückblieb. Da ging es allerorten herum, besah Stuben und Kammern, wie "
        "es Lust hatte, und kam endlich auch an einen alten Turm. Es stieg die "
        "enge Wendeltreppe hinauf und gelangte zu einer kleinen Türe. In dem "
        "Schloß steckte ein verrosteter Schlüssel, und als es umdrehte, sprang die "
        "Türe auf, und saß da in einem kleinen Stübchen eine alte Frau mit einer "
        "Spindel und spann emsig ihren Flachs. »Guten Tag, du altes Mütterchen,« "
        "sprach die Königstochter, »was machst du da?« — »Ich spinne,« sagte die "
        "Alte und nickte mit dem Kopf. »Was ist das für ein Ding, das so lustig "
        "herumspringt?« sprach das Mädchen, nahm die Spindel und wollte auch "
        "spinnen. Kaum hatte sie aber die Spindel angerührt, so ging der "
        "Zauberspruch in Erfüllung, und sie stach sich damit in den Finger.\n"
        "\n"
        "In dem Augenblick aber, wo sie den Stich empfand, fiel sie auf das Bett "
        "nieder, das da stand, und lag in einem tiefen Schlaf. Und dieser Schlaf "
        "verbreitete sich über das ganze Schloß. Der König und die Königin, die "
        "eben heim gekommen waren und in den Saal getreten waren, fingen an "
        "einzuschlafen, und der ganze Hofstaat mit ihnen. Da schliefen auch die "
        "Pferde im Stall, die Hunde im Hofe, die Tauben auf dem Dache, die Fliegen "
        "an der Wand, ja, das Feuer, das auf dem Herde flackerte, ward still und "
        "schlief ein, und der Braten hörte auf zu brutzeln, und der Koch, der den "
        "Küchenjungen, weil er etwas versehen hatte, in den Haaren ziehen wollte, "
        "ließ ihn los und schlief. Und der Wind legte sich, und auf den Bäumen vor "
        "dem Schloß regte sich kein Blättchen mehr.\n"
        "\n"
        "Rings um das Schloß aber begann eine Dornenhecke zu wachsen, die jedes "
        "Jahr höher ward und endlich das ganze Schloß umzog und darüber "
        "hinauswuchs, daß gar nichts mehr davon zu sehen war, selbst nicht die "
        "Fahne auf dem Dach. Es ging aber die Sage in dem Land von dem schönen "
        "schlafenden Dornröschen, denn so ward die Königstochter genannt, also daß "
        "von Zeit zu Zeit Königssöhne kamen und durch die Hecke in das Schloß "
        "dringen wollten. Es war ihnen aber nicht möglich, denn die Dornen, als "
        "hätten sie Hände, hielten fest zusammen, und die Jünglinge blieben darin "
        "hängen, konnten sich nicht wieder los machen und starben eines "
        "jämmerlichen Todes. Nach langen, langen Jahren kam wieder ein Königssohn "
        "in das Land und hörte, wie ein alter Mann von der Dornhecke erzählte, es "
        "sollte ein Schloß dahinter stehen, in welchem eine wunderschöne "
        "Königstochter, Dornröschen genannt, schon seit hundert Jahren schliefe, "
        "und mit ihr schliefe der König und die Königin und der ganze Hofstaat. Er "
        "wußte auch von seinem Großvater, daß schon viele Königssöhne gekommen "
        "wären und versucht hätten durch die Dornenhecke zu dringen, aber sie "
        "wären darin hängen geblieben und eines traurigen Todes gestorben. Da "
        "sprach der Jüngling: »Ich fürchte mich nicht, ich will hinaus und das "
        "schöne Dornröschen sehen.« Der gute Alte mochte ihm abraten, wie er "
        "wollte, er hörte nicht auf seine Worte.\n"
        "\n"
        "Nun waren aber gerade die hundert Jahre verflossen, und der Tag war "
        "gekommen, wo Dornröschen wieder erwachen sollte. Als der Königssohn sich "
        "der Dornenhecke näherte, waren es lauter schöne große Blumen, die taten "
        "sich von selbst auseinander und ließen ihn unbeschädigt hindurch, und "
        "hinter ihm taten sie sich wieder als eine Hecke zusammen. Im Schloßhof "
        "sah er die Pferde und scheckigen Jagdhunde liegen und schlafen, auf dem "
        "Dache saßen die Tauben und hatten das Köpfchen unter den Flügel gesteckt. "
        "Und als er ins Haus kam, schliefen die Fliegen an der Wand, der Koch in "
        "der Küche hielt noch die Hand, als wollte er den Jungen anpacken, und die "
        "Magd saß vor dem schwarzen Huhn, das sollte gerupft werden. Da ging er "
        "weiter und sah im Saale den ganzen Hofstaat liegen und schlafen, und oben "
        "bei dem Throne lag der König und die Königin. Da ging er noch weiter, und "
        "alles war so still, daß einer seinen Atem hören konnte, und endlich kam "
        "er zu dem Turm und öffnete die Türe zu der kleinen Stube, in welcher "
        "Dornröschen schlief. Da lag es und war so schön, daß er die Augen nicht "
        "abwenden konnte, und er bückte sich und gab ihm einen Kuß. Wie er es mit "
        "dem Kuß berührt hatte, schlug Dornröschen die Augen auf, erwachte und "
        "blickte ihn ganz freundlich an. Da gingen sie zusammen herab, und der "
        "König erwachte und die Königin und der ganze Hofstaat, und sahen einander "
        "mit großen Augen an. Und die Pferde im Hof standen auf und rüttelten "
        "sich, die Jagdhunde sprangen und wedelten; die Tauben auf dem Dache zogen "
        "die Köpfchen unterm Flügel hervor, sahen umher und flogen ins Feld. Die "
        "Fliegen an den Wänden krochen weiter, das Feuer in der Küche erhob sich, "
        "flackerte und kochte das Essen. Der Braten fing wieder an zu brutzeln: "
        "und der Koch gab dem Jungen eine Ohrfeige, daß er schrie; und die Magd "
        "rupfte das Huhn fertig. Und da wurde die Hochzeit des Königssohns mit dem "
        "Dornröschen in aller Pracht gefeiert, und sie lebten vergnügt bis an ihr "
        "Ende.",
        "Es war einmal mitten im Winter, und die Schneeflocken fielen wie Federn "
        "vom Himmel herab, da saß eine Königin an einem Fenster, das einen Rahmen "
        "von schwarzem Ebenholz hatte, und nähte. Und wie sie so nähte und nach "
        "dem Schnee aufblickte, stach sie sich mit der Nadel in den Finger, und es "
        "fielen drei Tropfen Blut in den Schnee. Und weil das Rote im weißen "
        "Schnee so schön aussah, dachte sie bei sich: »Hätt ich ein Kind so weiß "
        "wie Schnee, so rot wie Blut und so schwarz wie das Holz an dem Rahmen.« "
        "Bald darauf bekam sie ein Töchterlein, das war so weiß wie Schnee, so rot "
        "wie Blut und so schwarzhaarig wie Ebenholz, und ward darum das "
        "Sneewittchen (Schneeweißchen) genannt. Und wie das Kind geboren war, "
        "starb die Königin.\n"
        "\n"
        "Über ein Jahr nahm sich der König eine andere Gemahlin. Es war eine "
        "schöne Frau, aber sie war stolz und übermütig und konnte nicht leiden, "
        "daß sie an Schönheit von jemand sollte übertroffen werden. Sie hatte "
        "einen wunderbaren Spiegel, wenn sie vor den trat und sich darin "
        "beschaute, sprach sie:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "so antwortete der Spiegel:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste im Land.«\n"
        "\n"
        "Da war sie zufrieden, denn sie wußte, daß der Spiegel die Wahrheit "
        "sagte.\n"
        "\n"
        "Sneewittchen aber wuchs heran und wurde immer schöner, und als es sieben "
        "Jahre alt war, war es so schön, wie der klare Tag und schöner als die "
        "Königin selbst. Als diese einmal ihren Spiegel fragte:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "so antwortete er:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste hier, aber Sneewittchen ist "
        "tausendmal schöner als Ihr.«\n"
        "\n"
        "Da erschrak die Königin und ward gelb und grün vor Neid. Von Stund an, "
        "wenn sie Sneewittchen erblickte, kehrte sich ihr das Herz im Leibe herum, "
        "so haßte sie das Mädchen. Und der Neid und der Hochmut wuchsen wie ein "
        "Unkraut in ihrem Herzen immer höher, daß sie Tag und Nacht keine Ruhe "
        "mehr hatte. Da rief sie einen Jäger und sprach: »Bring das Kind hinaus in "
        "den Wald, ich will's nicht mehr vor meinen Augen sehen. Du sollst es "
        "töten und mir Lunge und Leber zum Wahrzeichen mitbringen.« Der Jäger "
        "gehorchte und führte es hinaus, und als er den Hirschfänger gezogen hatte "
        "und Sneewittchens unschuldiges Herz durchbohren wollte, fing es an zu "
        "weinen und sprach: »Ach, lieber Jäger, laß mir mein Leben; ich will in "
        "den wilden Wald laufen und nimmermehr wieder heim kommen.« Und weil es so "
        "schön war, hatte der Jäger Mitleiden und sprach: »So lauf hin, du armes "
        "Kind.« — »Die wilden Tiere werden dich bald gefressen haben,« dachte er, "
        "und doch war's ihm, als wär ein Stein von seinem Herzen gewälzt, weil er "
        "es nicht zu töten brauchte. Und als gerade ein junger Frischling daher "
        "gesprungen kam, stach er ihn ab, nahm Lunge und Leber heraus und brachte "
        "sie als Wahrzeichen der Königin mit. Der Koch mußte sie in Salz kochen, "
        "und das boshafte Weib aß sie auf und meinte, sie hätte Sneewittchens "
        "Lunge und Leber gegessen.\n"
        "\n"
        "Nun war das arme Kind in dem großen Wald mutterseelenallein und ward ihm "
        "so angst, daß es alle Blätter an den Bäumen ansah und nicht wußte, wie es "
        "sich helfen sollte. Da fing es an zu laufen und lief über die spitzen "
        "Steine und durch die Dornen, und die wilden Tiere sprangen an ihm vorbei, "
        "aber sie taten ihm nichts. Es lief so lange nur die Füße noch fort "
        "konnten, bis es bald Abend werden wollte, da sah es ein kleines Häuschen "
        "und ging hinein, sich zu ruhen. In dem Häuschen war alles klein, aber so "
        "zierlich und reinlich, daß es nicht zu sagen ist. Da stand ein "
        "weißgedecktes Tischlein mit sieben kleinen Tellern, jedes Tellerlein mit "
        "seinem Löffelein, ferner sieben Messerlein und Gäblein und sieben "
        "Becherlein. An der Wand waren sieben Bettlein nebeneinander aufgestellt "
        "und schneeweiße Laken darüber gedeckt. Sneewittchen, weil es so hungrig "
        "und durstig war, aß von jedem Tellerlein ein wenig Gemüs und Brot, und "
        "trank aus jedem Becherlein einen Tropfen Wein; denn es wollte nicht einem "
        "allein alles wegnehmen. Hernach, weil es so müde war, legte es sich in "
        "ein Bettchen, aber keins paßte; das eine war zu lang, das andere zu kurz, "
        "bis endlich das siebente recht war; und darin blieb es liegen, befahl "
        "sich Gott und schlief ein.\n"
        "\n"
        "Als es ganz dunkel geworden war, kamen die Herren von dem Häuslein, das "
        "waren die sieben Zwerge, die in den Bergen nach Erz hackten und gruben. "
        "Sie zündeten ihre sieben Lichtlein an, und wie es nun hell im Häuslein "
        "ward, sahen sie, daß jemand darin gewesen war, denn es stand nicht alles "
        "so in der Ordnung, wie sie es verlassen hatten. Der erste sprach: »Wer "
        "hat auf meinem Stühlchen gesessen?« Der zweite: »Wer hat von meinem "
        "Tellerchen gegessen?« Der dritte: »Wer hat von meinem Brötchen genommen?« "
        "Der vierte: »Wer hat von meinem Gemüschen gegessen?« Der fünfte: »Wer hat "
        "mit meinem Gäbelchen gestochen?« Der sechste: »Wer hat mit meinem "
        "Messerchen geschnitten?« Der siebente: »Wer hat aus meinem Becherlein "
        "getrunken?« Dann sah sich der erste um und sah, daß auf seinem Bett eine "
        "kleine Dälle war, da sprach er: »Wer hat in mein Bettchen getreten?« Die "
        "andern kamen gelaufen und riefen: »In meinem hat auch jemand gelegen.« "
        "Der siebente aber, als er in sein Bett sah, erblickte Sneewittchen, das "
        "lag darin und schlief. Nun rief er die andern, die kamen herbeigelaufen "
        "und schrien vor Verwunderung, holten ihre sieben Lichtlein und "
        "beleuchteten Sneewittchen. »Ei, du mein Gott! Ei, du mein Gott!« riefen "
        "sie, »was ist das Kind so schön!« Und hatten so große Freude, daß sie es "
        "nicht aufweckten, sondern im Bettlein fortschlafen ließen. Der siebente "
        "Zwerg aber schlief bei seinen Gesellen, bei jedem eine Stunde, da war die "
        "Nacht herum.\n"
        "\n"
        "Als es Morgen war, erwachte Sneewittchen, und wie es die sieben Zwerge "
        "sah, erschrak es. Sie waren aber freundlich und fragten. »Wie heißt du?« "
        "— »Ich heiße Sneewittchen,« antwortete es. »Wie bist du in unser Haus "
        "gekommen?« sprachen weiter die Zwerge. Da erzählte es ihnen, daß seine "
        "Stiefmutter es hätte wollen umbringen lassen, der Jäger hätte ihm aber "
        "das Leben geschenkt, und da wär es gelaufen den ganzen Tag, bis es "
        "endlich ihr Häuslein gefunden hätte. Die Zwerge sprachen: »Willst du "
        "unsern Haushalt versehen, kochen, betten, waschen, nähen und stricken, "
        "und willst du alles ordentlich und reinlich halten, so kannst du bei uns "
        "bleiben, und es soll dir an nichts fehlen.« — »Ja,« sagte Sneewittchen, "
        "»von Herzen gern,« und blieb bei ihnen. Es hielt ihnen das Haus in "
        "Ordnung. Morgens gingen sie in die Berge und suchten Erz und Gold, abends "
        "kamen sie wieder, und da mußte ihr Essen bereit sein. Den Tag über war "
        "das Mädchen allein, da warnten es die guten Zwerglein und sprachen: »Hüte "
        "dich vor deiner Stiefmutter, die wird bald wissen, daß du hier bist; laß "
        "ja niemand herein.«\n"
        "\n"
        "Die Königin aber, nachdem sie Sneewittchens Lunge und Leber glaubte "
        "gegessen zu haben, dachte nicht anders, als sie wäre wieder die Erste und "
        "Allerschönste, trat vor ihren Spiegel und sprach:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "Da antwortete der Spiegel:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste hier, aber Sneewittchen über den "
        "Bergen bei den sieben Zwergen ist noch tausendmal schöner als Ihr.«\n"
        "\n"
        "Da erschrak sie, denn sie wußte, daß der Spiegel keine Unwahrheit sprach, "
        "und merkte, daß der Jäger sie betrogen hatte und Sneewittchen noch am "
        "Leben war. Und da sann und sann sie aufs neue, wie sie es umbringen "
        "wollte; denn solange sie nicht die Schönste war im ganzen Land, ließ ihr "
        "der Neid keine Ruhe. Und als sie sich endlich etwas ausgedacht hatte, "
        "färbte sie sich das Gesicht und kleidete sich wie eine alte Krämerin und "
        "war ganz unkenntlich. In dieser Gestalt ging sie über die sieben Berge zu "
        "den sieben Zwergen, klopfte an die Türe und rief: »Schöne Ware feil! "
        "feil!« Sneewittchen guckte zum Fenster heraus und rief: »Guten Tag, liebe "
        "Frau, was habt Ihr zu verkaufen?« — »Gute Ware, schöne Ware,« antwortete "
        "sie, »Schnürriemen von allen Farben,« und holte einen hervor, der aus "
        "bunter Seide geflochten war. »Die ehrliche Frau kann ich herein lassen,« "
        "dachte Sneewittchen, riegelte die Türe auf und kaufte sich den hübschen "
        "Schnürriemen. »Kind,« sprach die Alte, »wie du aussiehst! Komm, ich will "
        "dich einmal ordentlich schnüren.« Sneewittchen hatte kein Arg, stellte "
        "sich vor sie und ließ sich mit dem neuen Schnürriemen schnüren. Aber die "
        "Alte schnürte geschwind und schnürte so fest, daß dem Sneewittchen der "
        "Atem verging und es für tot hinfiel. »Nun bist du die Schönste gewesen,« "
        "sprach sie und eilte hinaus.\n"
        "\n"
        "Nicht lange darauf, zur Abendzeit, kamen die sieben Zwerge nach Haus, "
        "aber wie erschraken sie, als sie ihr liebes Sneewittchen auf der Erde "
        "liegen sahen; und es regte und bewegte sich nicht, als wäre es tot. Sie "
        "hoben es in die Höhe, und weil sie sahen, daß es zu fest geschnürt war, "
        "schnitten sie den Schnürriemen entzwei. Da fing es an ein wenig zu atmen "
        "und ward nach und nach wieder lebendig. Als die Zwerge hörten, was "
        "geschehen war, sprachen sie, »die alte Krämerfrau war niemand als die "
        "gottlose Königin. Hüte dich und laß keinen Menschen herein, wenn wir "
        "nicht bei dir sind.«\n"
        "\n"
        "Das böse Weib aber, als es nach Haus gekommen war, ging vor den Spiegel "
        "und fragte:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "Da antwortete er wie sonst:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste hier, aber Sneewittchen über den "
        "Bergen bei den sieben Zwergen ist noch tausendmal schöner als Ihr.«\n"
        "\n"
        "Als sie das hörte, lief ihr alles Blut zum Herzen, so erschrak sie, denn "
        "sie sah wohl, daß Sneewittchen wieder lebendig geworden war. »Nun aber,« "
        "sprach sie, »will ich etwas aussinnen, das dich zugrunde richten soll,« "
        "und mit Hexenkünsten, die sie verstand, machte sie einen giftigen Kamm. "
        "Dann verkleidete sie sich und nahm die Gestalt eines andern alten Weibes "
        "an. So ging sie hin über die sieben Berge zu den sieben Zwergen, klopfte "
        "an die Türe und rief: »Gute Ware feil! feil!« Sneewittchen schaute heraus "
        "und sprach: »Geht nur weiter, ich darf niemand hereinlassen.« — »Das "
        "Ansehen wird dir doch erlaubt sein,« sprach die Alte, zog den giftigen "
        "Kamm heraus und hielt ihn in die Höhe. Da gefiel er dem Kinde so gut, daß "
        "es sich betören ließ und die Tür öffnete. Als sie des Kaufs einig waren, "
        "sprach die Alte: »Nun will ich dich einmal ordentlich kämmen.« Das arme "
        "Sneewittchen dachte an nichts und ließ die Alte gewähren, aber kaum hatte "
        "sie den Kamm in die Haare gesteckt, als das Gift darin wirkte und das "
        "Mädchen ohne Besinnung niederfiel. »Du Ausbund von Schönheit,« sprach das "
        "boshafte Weib, »jetzt ist's um dich geschehen,« und ging fort. Zum Glück "
        "aber war es bald Abend, wo die sieben Zwerglein nach Haus kamen. Als sie "
        "Sneewittchen wie tot auf der Erde liegen sahen, hatten sie gleich die "
        "Stiefmutter in Verdacht, suchten nach und fanden den giftigen Kamm, und "
        "kaum hatten sie ihn herausgezogen, so kam Sneewittchen wieder zu sich und "
        "erzählte, was vorgegangen war. Da warnten sie es noch einmal auf seiner "
        "Hut zu sein und niemand die Türe zu öffnen.\n"
        "\n"
        "Die Königin stellte sich daheim vor den Spiegel und sprach:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "Da antwortete er wie vorher:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste hier, aber Sneewittchen über den "
        "Bergen bei den sieben Zwergen ist noch tausendmal schöner als Ihr.«\n"
        "\n"
        "Als sie den Spiegel so reden hörte, zitterte und bebte sie vor Zorn. "
        "»Sneewittchen soll sterben,« rief sie, »und wenn es mein eigenes Leben "
        "kostet.« Darauf ging sie in eine ganz verborgene einsame Kammer, wo "
        "niemand hinkam, und machte da einen giftigen, giftigen Apfel. Äußerlich "
        "sah er schön aus, weiß mit roten Backen, daß jeder, der ihn erblickte, "
        "Lust danach bekam, aber wer ein Stückchen davon aß, der mußte sterben. "
        "Als der Apfel fertig war, färbte sie sich das Gesicht und verkleidete "
        "sich in eine Bauersfrau, und so ging sie über die sieben Berge zu den "
        "sieben Zwergen. Sie klopfte an, Sneewittchen streckte den Kopf zum "
        "Fenster heraus und sprach: »Ich darf keinen Menschen einlassen, die "
        "sieben Zwerge haben mir's verboten.« — »Mir auch recht,« antwortete die "
        "Bäuerin, »meine Äpfel will ich schon los werden. Da, einen will ich dir "
        "schenken.« — »Nein,« sprach Sneewittchen, »ich darf nichts annehmen.« — "
        "»Fürchtest du dich vor Gift?« sprach die Alte, »siehst du, da schneide "
        "ich den Apfel in zwei Teile; den roten Backen iß du, den weißen will ich "
        "essen.« Der Apfel war aber so künstlich gemacht, daß der rote Backen "
        "allein vergiftet war. Sneewittchen lusterte den schönen Apfel an, und als "
        "es sah, daß die Bäuerin davon aß, so konnte es nicht länger widerstehen, "
        "streckte die Hand hinaus und nahm die giftige Hälfte. Kaum aber hatte es "
        "einen Bissen davon im Mund, so fiel es tot zur Erde nieder. Da "
        "betrachtete es die Königin mit grausigen Blicken und lachte überlaut und "
        "sprach: »Weiß wie Schnee, rot wie Blut, schwarz wie Ebenholz! Diesmal "
        "können dich die Zwerge nicht wieder erwecken.« Und als sie daheim den "
        "Spiegel befragte:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "so antwortete er endlich:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste im Land.«\n"
        "\n"
        "Da hatte ihr neidisches Herz Ruhe, so gut ein neidisches Herz Ruhe haben "
        "kann.\n"
        "\n"
        "Die Zwerge, wie sie abends nach Haus kamen, fanden Sneewittchen auf der "
        "Erde liegen, und es ging kein Atem mehr aus seinem Mund, und es war tot. "
        "Sie hoben es auf, suchten ob sie was Giftiges fänden, schnürten es auf, "
        "kämmten ihm die Haare, wuschen es mit Wasser und Wein, aber es half alles "
        "nichts; das liebe Kind war tot und blieb tot. Sie legten es auf eine "
        "Bahre und setzten sich alle siebene daran und beweinten es, und weinten "
        "drei Tage lang. Da wollten sie es begraben, aber es sah noch so frisch "
        "aus wie ein lebender Mensch und hatte noch seine schönen roten Backen. "
        "Sie sprachen: »Das können wir nicht in die schwarze Erde versenken,« und "
        "ließen einen durchsichtigen Sarg von Glas machen, daß man es von allen "
        "Seiten sehen konnte, legten es hinein und schrieben mit goldenen "
        "Buchstaben seinen Namen darauf, und daß es eine Königstochter wäre. Dann "
        "setzten sie den Sarg hinaus auf den Berg, und einer von ihnen blieb immer "
        "dabei und bewachte ihn. Und die Tiere kamen auch und beweinten "
        "Sneewittchen, erst eine Eule, dann ein Rabe, zuletzt ein Täubchen.\n"
        "\n"
        "Nun lag Sneewittchen lange, lange Zeit in dem Sarg und verweste nicht, "
        "sondern sah aus als wenn es schliefe, denn es war noch so weiß als "
        "Schnee, so rot als Blut und so schwarzhaarig wie Ebenholz. Es geschah "
        "aber, daß ein Königssohn in den Wald geriet und zu dem Zwergenhaus kam, "
        "da zu übernachten. Er sah auf dem Berg den Sarg und das schöne "
        "Sneewittchen darin und las, was mit goldenen Buchstaben darauf "
        "geschrieben war. Da sprach er zu den Zwergen: »Laßt mir den Sarg, ich "
        "will euch geben, was ihr dafür haben wollt.« Aber die Zwerge antworteten: "
        "»Wir geben ihn nicht um alles Gold in der Welt.« Da sprach er: »So "
        "schenkt mir ihn, denn ich kann nicht leben ohne Sneewittchen zu sehen, "
        "ich will es ehren und hochachten wie mein Liebstes.« Wie er so sprach, "
        "empfanden die guten Zwerglein Mitleiden mit ihm und gaben ihm den Sarg. "
        "Der Königssohn ließ ihn nun von seinen Dienern auf den Schultern "
        "forttragen. Da geschah es, daß sie über einen Strauch stolperten, und von "
        "dem Schüttern fuhr ihr der giftige Apfelgrütz, den Sneewittchen "
        "abgebissen hatte, aus dem Hals. Und nicht lange, so öffnete es die Augen, "
        "hob den Deckel vom Sarg in die Höhe und richtete sich auf und war wieder "
        "lebendig. »Ach Gott, wo bin ich?« rief es. Der Königssohn sagte voll "
        "Freude: »Du bist bei mir,« und erzählte, was sich zugetragen hatte und "
        "sprach: »Ich habe dich lieber als alles auf der Welt; komm mit mir in "
        "meines Vaters Schloß, du sollst meine Gemahlin werden.« Da war ihm "
        "Sneewittchen gut und ging mit ihm, und ihre Hochzeit ward mit großer "
        "Pracht und Herrlichkeit angeordnet.\n"
        "\n"
        "Zu dem Fest wurde aber auch Sneewittchens gottlose Stiefmutter "
        "eingeladen. Wie sie sich nun mit schönen Kleidern angetan hatte, trat sie "
        "vor den Spiegel und sprach:\n"
        "\n"
        "»Spieglein, Spieglein an der Wand, wer ist die Schönste im ganzen Land?«\n"
        "\n"
        "Der Spiegel antwortete:\n"
        "\n"
        "»Frau Königin, Ihr seid die Schönste hier, aber die junge Königin ist "
        "tausendmal schöner als Ihr.«\n"
        "\n"
        "Da stieß das böse Weib einen Fluch aus und ward ihr so angst, so angst, "
        "daß sie sich nicht zu lassen wußte. Sie wollte zuerst gar nicht auf die "
        "Hochzeit kommen. Doch ließ es ihr keine Ruhe, sie mußte fort und die "
        "junge Königin sehen. Und wie sie hineintrat, erkannte sie Sneewittchen, "
        "und vor Angst und Schrecken stand sie da und konnte sich nicht regen. "
        "Aber es waren schon eiserne Pantoffeln über Kohlenfeuer gestellt und "
        "wurden mit Zangen hereingetragen und vor sie hingestellt. Da mußte sie in "
        "die rotglühenden Schuhe treten und so lange tanzen, bis sie tot zur Erde "
        "fiel.",
        "Es hatte ein Bauer ein treues Pferd, das war alt geworden und konnte "
        "keine Dienste mehr tun, da wollte ihm sein Herr nichts mehr zu fressen "
        "geben und sprach: »Brauchen kann ich dich freilich nicht mehr, indes mein "
        "ich es gut mit dir, zeigst du dich noch so stark, daß du mir einen Löwen "
        "hierher bringst, so will ich dich behalten, jetzt aber mach dich fort aus "
        "meinem Stall,« und jagte es damit ins weite Feld. Das Pferd war traurig "
        "und ging nach dem Wald zu, dort ein wenig Schutz vor dem Wetter zu "
        "suchen. Da begegnete ihm der Fuchs und sprach: »Was hängst du so den Kopf "
        "und gehst so einsam herum?« »Ach,« antwortete das Pferd, »Geiz und Treue "
        "wohnen nicht beisammen in einem Haus; mein Herr hat vergessen, was ich "
        "ihm für Dienste in so vielen Jahren geleistet habe, und weil ich nicht "
        "recht mehr ackern kann, will er mir kein Futter mehr geben, und hat mich "
        "fortgejagt.« — »Ohne allen Trost?« fragte der Fuchs. »Der Trost war "
        "schlecht, er hat gesagt, wenn ich noch so stark wäre, daß ich ihm einen "
        "Löwen brächte, wollt er mich behalten, aber er weiß wohl, daß ich das "
        "nicht vermag.« Der Fuchs sprach: »Da will ich dir helfen, leg dich nur "
        "hin, strecke dich aus und rege dich nicht, als wärst du tot.« Das Pferd "
        "tat, was der Fuchs verlangte, der Fuchs aber ging zum Löwen, der seine "
        "Höhle nicht weit davon hatte und sprach. »Da draußen liegt ein totes "
        "Pferd, komm doch mit hinaus, da kannst du eine fette Mahlzeit halten.« "
        "Der Löwe ging mit, und wie sie bei dem Pferd standen, sprach der Fuchs: "
        "»Hier hast du's doch nicht nach deiner Gemächlichkeit, weißt du was? Ich "
        "will's mit dem Schweif an dich binden, so kannst du's in deine Höhle "
        "ziehen und in aller Ruhe verzehren.« Dem Löwen gefiel der Rat, er stellte "
        "sich hin, und damit ihm der Fuchs das Pferd festknüpfen könnte, hielt er "
        "ganz still. Der Fuchs aber band mit des Pferdes Schweif dem Löwen die "
        "Beine zusammen und drehte und schnürte alles so wohl und stark, daß es "
        "mit keiner Kraft zu zerreißen war. Als er nun sein Werk vollendet hatte, "
        "klopfte er dem Pferd auf die Schulter und sprach: »Zieh, Schimmel, zieh.« "
        "Da sprang das Pferd mit einmal auf und zog den Löwen mit sich fort. Der "
        "Löwe fing an zu brüllen, daß die Vögel in dem ganzen Wald vor Schrecken "
        "aufflogen, aber das Pferd ließ ihn brüllen, zog und schleppte ihn über "
        "das Feld vor seines Herrn Tür. Wie der Herr das sah, besann er sich eines "
        "bessern und sprach zu dem Pferd: »Du sollst bei mir bleiben und es gut "
        "haben,« und gab ihm satt zu fressen bis es starb.",
        "Eine Witwe hatte zwei Töchter, davon war die eine schön und fleißig, die "
        "andere häßlich und faul. Sie hatte aber die häßliche und faule, weil sie "
        "ihre rechte Tochter war, viel lieber, und die andere mußte alle Arbeit "
        "tun und der Aschenputtel im Hause sein. Das arme Mädchen mußte sich "
        "täglich auf die große Straße bei einem Brunnen setzen und mußte so viel "
        "spinnen, daß ihm das Blut aus den Fingern sprang. Nun trug es sich zu, "
        "daß die Spule einmal ganz blutig war, da bückte es sich damit in den "
        "Brunnen und wollte sie abwaschen. Sie sprang ihm aber aus der Hand und "
        "fiel hinab. Es weinte, lief zur Stiefmutter und erzählte ihr das Unglück. "
        "Sie schalt es aber so heftig und war so unbarmherzig, daß sie sprach: "
        "»Hast du die Spule hinunterfallen lassen, so hol sie auch wieder herauf.« "
        "Da ging das Mädchen zu dem Brunnen zurück und wußte nicht was es anfangen "
        "sollte, und in seiner Herzensangst sprang es in den Brunnen hinein, um "
        "die Spule zu holen. Es verlor die Besinnung, und als es erwachte und "
        "wieder zu sich selber kam, war es auf einer schönen Wiese, wo die Sonne "
        "schien und viel tausend Blumen standen. Auf dieser Wiese ging es fort und "
        "kam zu einem Backofen, der war voller Brot; das Brot aber rief: »Ach, "
        "zieh mich raus, zieh mich raus, sonst verbrenn ich, ich bin schon längst "
        "ausgebacken.« Da trat es herzu, und holte mit dem Brotschieber alles "
        "nacheinander heraus. Danach ging es weiter und kam zu einem Baum, der "
        "hing voll Äpfel, und rief ihm zu: »Ach, schüttel mich, schüttel mich, wir "
        "Äpfel sind alle miteinander reif.« Da schüttelte es den Baum, daß die "
        "Äpfel fielen als regneten sie, und schüttelte bis keiner mehr oben war; "
        "und als es alle in einen Haufen zusammengelegt hatte, ging es wieder "
        "weiter. Endlich kam es zu einem kleinen Haus, daraus guckte eine alte "
        "Frau, weil sie aber so große Zähne hatte, ward ihm angst, und es wollte "
        "fortlaufen. Die alte Frau aber rief ihm nach: »Was fürchtest du dich, "
        "liebes Kind? Bleib bei mir, wenn du alle Arbeit im Hause ordentlich tun "
        "willst, so soll dir's gut gehn. Du mußt nur acht geben, daß du mein Bett "
        "gut machst und es fleißig aufschüttelst, daß die Federn fliegen, dann "
        "schneit es in der Welt; ich bin die Frau Holle.« Weil die Alte ihm so gut "
        "zusprach, so faßte sich das Mädchen ein Herz, willigte ein und begab sich "
        "in ihren Dienst. Es besorgte auch alles nach ihrer Zufriedenheit und "
        "schüttelte ihr das Bett immer gewaltig auf, daß die Federn wie "
        "Schneeflocken umherflogen; dafür hatte es auch ein gutes Leben bei ihr, "
        "kein böses Wort und alle Tage Gesottenes und Gebratenes. Nun war es eine "
        "Zeitlang bei der Frau Holle, da ward es traurig und wußte anfangs selbst "
        "nicht, was ihm fehlte, endlich merkte es, daß es Heimweh war; ob es ihm "
        "hier gleich viel tausendmal besser ging als zu Haus, so hatte es doch ein "
        "Verlangen dahin. Endlich sagte es zu ihr: »Ich habe den Jammer nach Haus "
        "kriegt, und wenn es mir auch noch so gut hier unten geht, so kann ich "
        "doch nicht länger bleiben, ich muß wieder hinauf zu den Meinigen.« Die "
        "Frau Holle sagte: »Es gefällt mir, daß du wieder nach Hause verlangst, "
        "und weil du mir so treu gedient hast, so will ich dich selbst wieder "
        "hinaufbringen.« Sie nahm es darauf bei der Hand und führte es vor ein "
        "großes Tor. Das Tor ward aufgetan, und wie das Mädchen gerade darunter "
        "stand, fiel ein gewaltiger Goldregen, und alles Gold blieb an ihm hängen, "
        "so daß es über und über davon bedeckt war. »Das sollst du haben, weil du "
        "so fleißig gewesen bist,« sprach die Frau Holle und gab ihm auch die "
        "Spule wieder, die ihm in den Brunnen gefallen war. Darauf ward das Tor "
        "verschlossen, und das Mädchen befand sich oben auf der Welt, nicht weit "
        "von seiner Mutter Haus. Und als es in den Hof kam, saß der Hahn auf dem "
        "Brunnen und rief:\n"
        "\n"
        "»Kikeriki, unsere goldene Jungfrau ist wieder hie.«\n"
        "\n"
        "Da ging es hinein zu seiner Mutter, und weil es so mit Gold bedeckt "
        "ankam, ward es von ihr und der Schwester gut aufgenommen.\n"
        "\n"
        "Das Mädchen erzählte alles, was ihm begegnet war, und als die Mutter "
        "hörte, wie es zu dem großen Reichtum gekommen war, wollte sie der andern "
        "häßlichen und faulen Tochter gerne dasselbe Glück verschaffen. Sie mußte "
        "sich an den Brunnen setzen und spinnen; und damit ihre Spule blutig ward, "
        "stach sie sich in die Finger und stieß sich die Hand in die Dornhecke. "
        "Dann warf sie die Spule in den Brunnen und sprang selber hinein. Sie kam, "
        "wie die andere, auf die schöne Wiese und ging auf demselben Pfade weiter. "
        "Als sie zu dem Backofen gelangte, schrie das Brot wieder: »Ach, zieh mich "
        "raus, zieh mich raus, sonst verbrenn ich, ich bin schon längst "
        "ausgebacken.« Die Faule aber antwortete: »Da hätt ich Lust, mich "
        "schmutzig zu machen,« und ging fort. Bald kam sie zu dem Apfelbaum, der "
        "rief: »Ach, schüttel mich, schüttel mich, wir Äpfel sind alle miteinander "
        "reif.« Sie antwortete aber: »Du kommst mir recht, es könnte mir einer auf "
        "den Kopf fallen,« und ging damit weiter. Als sie vor der Frau Holle Haus "
        "kam, fürchtete sie sich nicht, weil sie von ihren großen Zähnen schon "
        "gehört hatte und verdingte sich gleich zu ihr. Am ersten Tag tat sie sich "
        "Gewalt an, war fleißig und folgte der Frau Holle, wenn sie ihr etwas "
        "sagte, denn sie dachte an das viele Gold, das sie ihr schenken würde; am "
        "zweiten Tag aber fing sie schon an zu faulenzen, am dritten noch mehr, da "
        "wollte sie morgens gar nicht aufstehen. Sie machte auch der Frau Holle "
        "das Bett nicht wie sich's gebührte und schüttelte es nicht, daß die "
        "Federn aufflogen. Das ward die Frau Holle bald müde und sagte ihr den "
        "Dienst auf. Die Faule war das wohl zufrieden und meinte, nun würde der "
        "Goldregen kommen; die Frau Holle führte sie auch zu dem Tor, als sie aber "
        "darunter stand, ward statt des Goldes ein großer Kessel voll Pech "
        "ausgeschüttet. »Das ist zur Belohnung deiner Dienste,« sagte die Frau "
        "Holle und schloß das Tor zu. Da kam die Faule heim, aber sie war ganz mit "
        "Pech bedeckt, und der Hahn auf dem Brunnen, als er sie sah, rief:\n"
        "\n"
        "»Kikeriki, unsere schmutzige Jungfrau ist wieder hie.«\n"
        "\n"
        "Das Pech aber blieb fest an ihr hängen und wollte, so lange sie lebte, "
        "nicht abgehen.",
        "Ein König hatte eine Tochter, die war über alle Maßen schön, aber dabei "
        "so stolz und übermütig, daß ihr kein Freier gut genug war. Sie wies einen "
        "nach dem andern ab und trieb noch dazu Spott mit ihnen. Einmal ließ der "
        "König ein großes Fest anstellen und ladete dazu aus der Nähe und Ferne "
        "die heiratslustigen Männer ein. Sie wurden alle in eine Reihe nach Rang "
        "und Stand geordnet; erst kamen die Könige, dann die Herzöge, die Fürsten, "
        "Grafen und Freiherren, zuletzt die Edelleute. Nun ward die Königstochter "
        "durch die Reihen geführt, aber an jedem hatte sie etwas auszusetzen. Der "
        "eine war ihr zu dick: »Das Weinfaß!« sprach sie. Der andere zu lang: "
        "»Lang und schwank hat keinen Gang.« Der dritte zu kurz: »Kurz und dick "
        "hat kein Geschick.« Der vierte zu blaß: »Der bleiche Tod!« Der fünfte zu "
        "rot: »Der Zinshahn!« Der sechste war nicht gerad genug: »Grünes Holz, "
        "hinterm Ofen getrocknet!« Und so hatte sie an einem jeden etwas "
        "auszusetzen, besonders aber machte sie sich über einen guten König "
        "lustig, der ganz oben stand und dem das Kinn ein wenig krumm gewachsen "
        "war. »Ei,« rief sie und lachte, »der hat ein Kinn, wie die Drossel einen "
        "Schnabel.« Und seit der Zeit bekam er den Namen Drosselbart. Der alte "
        "König aber, als er sah, daß seine Tochter nichts tat als über die Leute "
        "spotten und alle Freier, die da versammelt waren, verschmähte, ward "
        "zornig und schwur, sie sollte den ersten besten Bettler zum Manne nehmen, "
        "der vor seine Türe käme.\n"
        "\n"
        "Ein paar Tage darauf hub ein Spielmann an unter dem Fenster zu singen, um "
        "damit ein geringes Almosen zu verdienen. Als es der König hörte, sprach "
        "er: »Laßt ihn heraufkommen.« Da trat der Spielmann in seinen schmutzigen, "
        "verlumpten Kleidern herein, sang vor dem König und seiner Tochter und "
        "bat, als er fertig war, um eine milde Gabe. Der König sprach: »Dein "
        "Gesang hat mir so wohl gefallen, daß ich dir meine Tochter da zur Frau "
        "geben will.« Die Königstochter erschrak, aber der König sagte: »Ich habe "
        "den Eid getan, dich dem ersten besten Bettelmann zu geben, den will ich "
        "auch halten.« Es half keine Einrede, der Pfarrer ward geholt, und sie "
        "mußte sich gleich mit dem Spielmann trauen lassen. Als das geschehen war, "
        "sprach der König: »Nun schickt sich's nicht, daß du als Bettelweib noch "
        "länger in meinem Schloß bleibst, du kannst nur mit deinem Manne "
        "fortziehen.«\n"
        "\n"
        "Der Bettelmann führte sie an der Hand hinaus, und sie mußte mit ihm zu "
        "Fuß fortgehen. Als sie in einen großen Wald kamen, da fragte sie:\n"
        "\n"
        "»Ach, wem gehört der schöne Wald?« »Der gehört dem König Drosselbart; "
        "hättst du'n genommen, so wär er dein.« »Ich arme Jungfer zart, ach, hätt "
        "ich genommen den König Drosselbart!«\n"
        "\n"
        "Darauf kamen sie über eine Wiese, da fragte sie wieder:\n"
        "\n"
        "»Wem gehört die schöne grüne Wiese?« »Sie gehört dem König Drosselbart; "
        "hättst du'n genommen, so wär sie dein.« »Ich arme Jungfer zart, ach, hätt "
        "ich genommen den König Drosselbart!«\n"
        "\n"
        "Dann kamen sie durch eine große Stadt, da fragte sie wieder:\n"
        "\n"
        "»Wem gehört diese schöne große Stadt?« »Sie gehört dem König Drosselbart; "
        "hättst du'n genommen, so wär sie dein.« »Ich arme Jungfer zart, ach, hätt "
        "ich genommen den König Drosselbart!«\n"
        "\n"
        "»Es gefällt mir gar nicht,« sprach der Spielmann, »daß du dir immer einen "
        "andern zum Manne wünschest; bin ich dir nicht gut genug?« Endlich kamen "
        "sie an ein ganz kleines Häuschen, da sprach sie:\n"
        "\n"
        "»Ach Gott, was ist das Haus so klein! Wem mag das elende winzige Häuschen "
        "sein?«\n"
        "\n"
        "Der Spielmann antwortete: »Das ist mein und dein Haus, wo wir zusammen "
        "wohnen.« Sie mußte sich bücken, damit sie zu der niedrigen Tür hineinkam. "
        "»Wo sind die Diener?« sprach die Königstochter. »Was, Diener,« antwortete "
        "der Bettelmann, »du mußt selber tun, was du willst getan haben. Mach nur "
        "gleich Feuer an und stell Wasser auf, daß du mir mein Essen kochst; ich "
        "bin ganz müde.« Die Königstochter verstand aber nichts vom Feueranmachen "
        "und Kochen, und der Bettelmann mußte selber mit Hand anlegen, daß es noch "
        "so leidlich ging. Als sie die schmale Kost verzehrt hatten, legten sie "
        "sich zu Bett. Aber am Morgen trieb er sie schon ganz früh heraus, weil "
        "sie das Haus besorgen sollte. Ein paar Tage lebten sie auf diese Art "
        "schlecht und recht und zehrten ihren Vorrat auf. Da sprach der Mann: "
        "»Frau, so geht's nicht länger, daß wir hier zehren und nichts verdienen. "
        "Du sollst Körbe flechten.« Er ging aus, schnitt Weiden und brachte sie "
        "heim. Da fing sie an zu flechten, aber die harten Weiden stachen ihr die "
        "zarten Hände wund. »Ich sehe, das geht nicht,« sprach der Mann, »spinn "
        "lieber, vielleicht kannst du das besser.« Sie setzte sich hin und "
        "versuchte zu spinnen, aber der harte Faden schnitt ihr bald in die "
        "weichen Finger, daß das Blut daran herunter lief. »Siehst du,« sprach der "
        "Mann, »du taugst zu keiner Arbeit, mit dir bin ich schlimm angekommen. "
        "Nun will ich's versuchen und einen Handel mit Töpfen und irdenem Geschirr "
        "anfangen: du sollst dich auf den Markt setzen und die Ware feilhalten.« — "
        "»Ach,« dachte sie, »wenn auf den Markt Leute aus meines Vaters Reich "
        "kommen und sehen mich da sitzen und feilhalten, wie werden sie mich "
        "verspotten!« Aber es half nichts, sie mußte sich fügen, wenn sie nicht "
        "Hungers sterben wollten. Das erstemal ging's gut, denn die Leute kauften "
        "der Frau, weil sie schön war, gern ihre Ware ab und bezahlten, was sie "
        "forderte; ja, viele gaben ihr das Geld und ließen ihr die Töpfe noch "
        "dazu. Nun lebten sie von dem Erworbenen, solange es dauerte, da handelte "
        "der Mann wieder eine Menge neues Geschirr ein. Sie setzte sich damit an "
        "eine Ecke des Marktes und stellte es um sich her und hielt feil. Da kam "
        "plötzlich ein trunkener Husar daher gejagt und ritt geradezu in die Töpfe "
        "hinein, daß alles in tausend Scherben zersprang. Sie fing an zu weinen "
        "und wußte vor Angst nicht, was sie anfangen sollte. »Ach, wie wird mir's "
        "ergehen!« rief sie, »was wird mein Mann dazu sagen!« Sie lief heim und "
        "erzählte ihm das Unglück. »Wer setzt sich auch an die Ecke des Marktes "
        "mit irdenem Geschirr!« sprach der Mann, »laß nur das Weinen, ich sehe "
        "wohl, du bist zu keiner ordentlichen Arbeit zu gebrauchen. Da bin ich in "
        "unseres Königs Schloß gewesen und habe gefragt, ob sie nicht eine "
        "Küchenmagd brauchen könnten, und sie haben mir versprochen, sie wollten "
        "dich dazu nehmen; dafür bekommst du freies Essen.«\n"
        "\n"
        "Nun ward die Königstochter eine Küchenmagd, mußte dem Koch zur Hand gehen "
        "und die sauerste Arbeit tun. Sie machte sich in beiden Taschen ein "
        "Töpfchen fest, darin brachte sie nach Haus, was ihr von dem "
        "Übriggebliebenen zuteil ward, und davon nährten sie sich. Es trug sich "
        "zu, daß die Hochzeit des ältesten Königssohnes sollte gefeiert werden, da "
        "ging die arme Frau hinauf, stellte sich vor die Saaltüre und wollte "
        "zusehen. Als nun die Lichter angezündet waren, und immer einer schöner "
        "als der andere hereintrat und alles voll Pracht und Herrlichkeit war, da "
        "dachte sie mit betrübtem Herzen an ihr Schicksal und verwünschte ihren "
        "Stolz und Übermut, der sie erniedrigt und in so große Armut gestürzt "
        "hatte. Von den köstlichsten Speisen, die da ein und aus getragen wurden, "
        "und von welchen der Geruch zu ihr aufstieg, warfen ihr Diener manchmal "
        "ein paar Brocken zu, die tat sie in ihr Töpfchen und wollte es "
        "heimtragen. Auf einmal trat der Königssohn herein, war in Sammet und "
        "Seide gekleidet und hatte goldene Ketten um den Hals. Und als er die "
        "schöne Frau in der Türe stehen sah, ergriff er sie bei der Hand und "
        "wollte mit ihr tanzen, aber sie weigerte sich und erschrak, denn sie sah, "
        "daß es der König Drosselbart war, der um sie gefreit und den sie mit "
        "Spott abgewiesen hatte. Ihr Sträuben half nichts, er zog sie in den Saal: "
        "da zerriß das Band, an welchem die Taschen hingen, und die Töpfe fielen "
        "heraus, daß die Suppe floß und die Brocken umhersprangen. Und wie das die "
        "Leute sahen, entstand ein allgemeines Gelächter und Spotten, und sie war "
        "so beschämt, daß sie sich lieber tausend Klafter unter die Erde gewünscht "
        "hätte. Sie sprang zur Türe hinaus und wollte entfliehen, aber auf der "
        "Treppe holte sie ein Mann ein und brachte sie zurück. Und wie sie ihn "
        "ansah, war es wieder der König Drosselbart. Er sprach ihr freundlich zu: "
        "»Fürchte dich nicht, ich und der Spielmann, der mit dir in dem elenden "
        "Häuschen gewohnt hat, sind eins. Dir zuliebe habe ich mich so verstellt, "
        "und der Husar, der dir die Töpfe entzwei geritten hat, bin ich auch "
        "gewesen. Das alles ist geschehen, um deinen stolzen Sinn zu beugen und "
        "dich für deinen Hochmut zu strafen, womit du mich verspottest hast.« Da "
        "weinte sie bitterlich und sagte: »Ich habe großes Unrecht gehabt und bin "
        "nicht wert, deine Frau zu sein.« Er aber sprach: »Tröste dich, die bösen "
        "Tage sind vorüber, jetzt wollen wir unsere Hochzeit feiern.« Da kamen die "
        "Kammerfrauen und taten ihr die prächtigsten Kleider an, und ihr Vater kam "
        "und der ganze Hof und wünschten ihr Glück zu ihrer Vermählung mit dem "
        "König Drosselbart, und die rechte Freude fing jetzt erst an. Ich wollte, "
        "du und ich, wir wären auch dabei gewesen.",
        "Es war einmal ein Mann, der verstand allerlei Künste. Er diente im Krieg "
        "und hielt sich brav und tapfer, aber als der Krieg zu Ende war, bekam er "
        "den Abschied und drei Heller Zehrgeld auf den Weg. »Wart,« sprach er, "
        "»das laß ich mir nicht gefallen, finde ich die rechten Leute, so soll mir "
        "der König noch die Schätze des ganzen Landes herausgeben.« Da ging er "
        "voll Zorn in den Wald und sah einen darin stehen, der hatte sechs Bäume "
        "ausgerupft, als wären's Kornhalme. Sprach er zu ihm: »Willst du mein "
        "Diener sein und mit mir ziehen?« — »Ja,« antwortete er, »aber erst will "
        "ich meiner Mutter das Wellchen Holz heimbringen,« und nahm einen von den "
        "Bäumen und wickelte ihn um die fünf andern, hob die Welle auf die "
        "Schulter und trug sie fort. Dann kam er wieder und ging mit seinem Herrn, "
        "der sprach: »Wir zwei sollten wohl durch die ganze Welt kommen.« Und als "
        "sie ein Weilchen gegangen waren, fanden sie einen Jäger, der lag auf den "
        "Knien, hatte die Büchse angelegt und zielte. Sprach der Herr zu ihm: "
        "»Jäger, was willst du schießen?« Er antwortete. »Zwei Meilen von hier "
        "sitzt eine Fliege auf dem Ast eines Eichbaums, der will ich das linke "
        "Auge herausschießen.« — »Oh, geh mit mir,« sprach der Mann, »wenn wir "
        "drei zusammen sind, sollten wir wohl durch die ganze Welt kommen.« Der "
        "Jäger war bereit und ging mit ihm, und sie kamen zu sieben Windmühlen, "
        "deren Flügel trieben ganz hastig herum und ging doch links und rechts "
        "kein Wind und bewegte sich kein Blättchen. Da sprach der Mann: »Ich weiß "
        "nicht, was die Windmühlen treibt, es regt sich ja kein Lüftchen,« und "
        "ging mit seinen Dienern weiter, und als sie zwei Meilen fortgegangen "
        "waren, sahen sie einen auf einem Baum sitzen, der hielt das eine "
        "Nasenloch zu und blies aus dem andern. »Mein, was treibst du da oben?« "
        "fragte der Mann. Er antwortete: »Zwei Meilen von hier stehen sieben "
        "Windmühlen, seht, die blase ich an, daß sie laufen.« — »Oh, geh mit mir,« "
        "sprach der Mann, »wenn wir vier zusammen sind, sollten wir wohl durch die "
        "ganze Welt kommen.« Da stieg der Bläser herab und ging mit, und über eine "
        "Zeit sahen sie einen, der stand da auf einem Bein und hatte das andere "
        "abgeschnallt und neben sich gelegt. Da sprach der Herr: »Du hast dir's ja "
        "bequem gemacht zum Ausruhen.« — »Ich bin ein Laufer,« antwortete er, »und "
        "damit ich nicht gar zu schnell springe, habe ich mir das eine Bein "
        "abgeschnallt; wenn ich mit zwei Beinen laufe, so geht's geschwinder als "
        "ein Vogel fliegt.« — »Oh, geh mit mir, wenn wir fünf zusammen sind, "
        "sollten wir wohl durch die ganze Welt kommen.« Da ging er mit, und gar "
        "nicht lang, so begegneten sie einem, der hatte ein Hütchen auf, hatte es "
        "aber ganz auf dem einen Ohre sitzen. Da sprach der Herr zu ihm: "
        "»Manierlich! Manierlich! Häng deinen Hut doch nicht auf ein Ohr, du "
        "siehst ja aus wie ein Hans Narr.« — »Ich darf's nicht tun,« sprach der "
        "andere, »denn setz ich meinen Hut gerad, so kommt ein gewaltiger Frost, "
        "und die Vögel unter dem Himmel erfrieren und fallen tot zur Erde.« — »Oh, "
        "geh mit mir,« sprach der Herr, »wenn wir sechs zusammen sind, sollten wir "
        "wohl durch die ganze Welt kommen.«\n"
        "\n"
        "Nun gingen die sechse in eine Stadt, wo der König hatte bekannt machen "
        "lassen, wer mit seiner Tochter in die Wette laufen wollte und den Sieg "
        "davon trüge, der sollte ihr Gemahl werden; wer aber verlöre, müßte auch "
        "seinen Kopf hergeben. Da meldete sich der Mann und sprach: »Ich will aber "
        "meinen Diener für mich laufen lassen.« Der König antwortete: »Dann mußt "
        "du auch noch dessen Leben zum Pfand setzen, also daß sein und dein Kopf "
        "für den Sieg haften.« Als das verabredet und festgemacht war, schnallte "
        "der Mann dem Laufer das andere Bein an und sprach zu ihm: »Nun sei hurtig "
        "und hilf, daß wir siegen.« Es war aber bestimmt, daß wer am ersten Wasser "
        "aus einem weit abgelegenen Brunnen brächte, der sollte Sieger sein. Nun "
        "bekam der Laufer einen Krug und die Königstochter auch einen, und sie "
        "fingen zu gleicher Zeit zu laufen an. Aber in einem Augenblick, als die "
        "Königstochter erst eine kleine Strecke fort war, konnte den Laufer schon "
        "kein Zuschauer mehr sehen, und es war nicht anders, als wäre der Wind "
        "vorbeigesaust. In kurzer Zeit langte er bei dem Brunnen an, schöpfte den "
        "Krug voll Wasser und kehrte wieder um. Mitten aber auf dem Heimweg "
        "überkam ihn eine Müdigkeit, da setzte er den Krug hin, legte sich nieder "
        "und schlief ein. Er hatte aber einen Pferdeschädel, der da auf der Erde "
        "lag, zum Kopfkissen gemacht, damit er hart läge und bald wieder erwachte. "
        "Indessen war die Königstochter, die auch gut laufen konnte, so gut es ein "
        "gewöhnlicher Mensch vermag, bei dem Brunnen angelangt, und eilte mit "
        "ihrem Krug voll Wasser zurück; und als sie den Laufer daliegen und "
        "schlafen sah, war sie froh und sprach: »Der Feind ist in meine Hände "
        "gegeben,« leerte seinen Krug aus und sprang weiter. Nun wäre alles "
        "verloren gewesen, wenn nicht zu gutem Glück der Jäger mit seinen scharfen "
        "Augen oben auf dem Schloß gestanden und alles mit angesehen hätte. Da "
        "sprach er: »Die Königstochter soll doch gegen uns nicht aufkommen,« lud "
        "seine Büchse und schoß so geschickt, daß er dem Laufer den Pferdeschädel "
        "unter dem Kopf wegschoß, ohne ihm weh zu tun. Da erwachte der Laufer, "
        "sprang in die Höhe und sah, daß sein Krug leer und die Königstochter "
        "schon weit voraus war. Aber er verlor den Mut nicht, lief mit dem Krug "
        "wieder zum Brunnen zurück, schöpfte aufs neue Wasser und war noch zehn "
        "Minuten eher als die Königstochter daheim. »Seht ihr,« sprach er, »jetzt "
        "hab' ich erst die Beine aufgehoben, vorher war's gar kein Laufen zu "
        "nennen.«\n"
        "\n"
        "Den König aber kränkte es, und seine Tochter noch mehr, daß sie so ein "
        "gemeiner abgedankter Soldat davon tragen sollte; sie ratschlagten "
        "miteinander, wie sie ihn samt seinen Gesellen los würden. Da sprach der "
        "König zu ihr: »Ich habe ein Mittel gefunden, laß dir nicht bang sein, sie "
        "sollen nicht wieder heim kommen.« Und sprach zu ihnen: »Ihr sollt euch "
        "nun zusammen lustig machen, essen und trinken,« und führte sie zu einer "
        "Stube, die hatte einen Boden von Eisen und die Türen waren auch von "
        "Eisen, und die Fenster waren mit eisernen Stäben verwahrt. In der Stube "
        "war eine Tafel mit köstlichen Speisen besetzt, da sprach der König zu "
        "ihnen: »Geht hinein und laßt's euch wohl sein.« Und wie sie darinnen "
        "waren, ließ er die Türe verschließen und verriegeln. Dann ließ er den "
        "Koch kommen und befahl ihm, ein Feuer so lange unter die Stube zu machen, "
        "bis das Eisen glühend würde. Das tat der Koch, und es fing an und ward "
        "den sechsen in der Stube, während sie an der Tafel saßen, ganz warm, und "
        "sie meinten, das käme vom Essen; als aber die Hitze immer größer ward und "
        "sie hinaus wollten, Türe und Fenster aber verschlossen fanden, da merkten "
        "sie, daß der König Böses im Sinne gehabt hatte und sie ersticken wollte. "
        "»Es soll ihm aber nicht gelingen,« sprach der mit dem Hütchen, »ich will "
        "einen Frost kommen lassen, vor dem sich das Feuer schämen und verkriechen "
        "soll.« Da setzte er sein Hütchen gerade, und alsobald fiel ein Frost, daß "
        "alle Hitze verschwand und die Speisen auf den Schüsseln anfingen zu "
        "frieren. Als nun ein paar Stunden herum waren und der König glaubte, sie "
        "wären in der Hitze verschmachtet, ließ er die Türe öffnen und wollte "
        "selbst nach ihnen sehen. Aber wie die Türe aufging, standen sie alle "
        "sechse da, frisch und gesund, und sagten, es wäre ihnen lieb, daß sie "
        "heraus könnten, sich zu wärmen, denn bei der großen Kälte in der Stube "
        "frören die Speisen an den Schüsseln fest. Da ging der König voll Zorn "
        "hinab zu dem Koch, schalt ihn und fragte, warum er nicht getan hätte, was "
        "ihm wäre befohlen worden. Der Koch aber antwortete: »Es ist Glut genug "
        "da, seht nur selbst.« Da sah der König, daß ein gewaltiges Feuer unter "
        "der Eisenstube brannte, und merkte, daß er den sechsen auf diese Weise "
        "nichts anhaben könnte.\n"
        "\n"
        "Nun sann der König aufs neue, wie er die bösen Gäste los würde, ließ den "
        "Meister kommen und sprach: »Willst du Gold nehmen und dein Recht auf "
        "meine Tochter aufgeben, so sollst du haben, soviel du willst.« — »O ja, "
        "Herr König,« antwortete er, »gebt mir soviel als mein Diener tragen kann, "
        "so verlange ich Eure Tochter nicht.« Das war der König zufrieden, und "
        "jener sprach weiter: »So will ich in vierzehn Tagen kommen und es holen.« "
        "Darauf rief er alle Schneider aus dem ganzen Reich herbei, die mußten "
        "vierzehn Tage lang sitzen und einen Sack nähen. Und als er fertig war, "
        "mußte der Starke, welcher Bäume ausrupfen konnte, den Sack auf die "
        "Schulter nehmen und mit ihm zu dem König gehen. Da sprach der König: »Was "
        "ist das für ein gewaltiger Kerl, der den hausgroßen Ballen Leinwand auf "
        "der Schulter trägt?« erschrak und dachte, »was wird der für Gold "
        "wegschleppen!« Da hieß er eine Tonne Gold herbringen, die mußten sechzehn "
        "der stärksten Männer tragen, aber der Starke packte sie mit einer Hand, "
        "steckte sie in den Sack und sprach: »Warum bringt ihr nicht gleich mehr, "
        "das deckt ja kaum den Boden.« Da ließ der König nach und nach seinen "
        "ganzen Schatz herbeitragen, den schob der Starke in den Sack hinein, und "
        "der Sack ward davon noch nicht zur Hälfte voll. »Schafft mehr herbei,« "
        "rief er, »die paar Brocken füllen nicht.« Da mußten noch siebentausend "
        "Wagen mit Gold in dem ganzen Reich zusammengefahren werden. Die schob der "
        "Starke samt den vorgespannten Ochsen in seinen Sack. »Ich will's nicht "
        "lange besehen,« sprach er, »und nehmen was kommt, damit der Sack nur voll "
        "wird.« Wie alles darin stak, ging doch noch viel hinein, da sprach er: "
        "»Ich will dem Ding ein Ende machen, man bindet wohl einmal einen Sack zu, "
        "wenn er auch noch nicht voll ist.« Dann huckte er ihn auf den Rücken und "
        "ging mit seinen Gesellen fort.\n"
        "\n"
        "Als der König nun sah, wie der einzige Mann des ganzen Landes Reichtum "
        "forttrug, ward er zornig und ließ seine Reiterei aufsitzen, die sollten "
        "den sechsen nachjagen, und hatten Befehl, dem Starken den Sack wieder "
        "abzunehmen. Zwei Regimenter holten sie bald ein und riefen ihnen zu: »Ihr "
        "seid Gefangene, legt den Sack mit dem Gold nieder oder ihr werdet "
        "zusammengehauen.« — »Was sagt ihr?« sprach der Bläser, »wir wären "
        "Gefangene? Eher sollt ihr sämtlich in der Luft herumtanzen,« hielt das "
        "eine Nasenloch zu und blies mit dem andern die beiden Regimenter an; da "
        "fuhren sie auseinander und in die blaue Luft über alle Berge weg, der "
        "eine hierhin, der andere dorthin. Ein Feldwebel rief um Gnade, er hätte "
        "neun Wunden und wäre ein braver Kerl, der den Schimpf nicht verdiente. Da "
        "ließ der Bläser ein wenig nach, so daß er ohne Schaden wieder herab kam, "
        "dann sprach er zu ihm: »Nun geh heim zum König und sag, er sollte nur "
        "noch mehr Reiterei schicken, ich wollte sie alle in die Luft blasen.« Der "
        "König, als er den Bescheid vernahm, sprach: »Laßt die Kerle gehen, die "
        "haben etwas an sich.« Da brachten die sechs den Reichtum heim, teilten "
        "ihn unter sich und lebten vergnügt bis an ihr Ende.",
        "Es war einmal ein altes Schloß mitten in einem großen dicken Wald, "
        "darinnen wohnte eine alte Frau ganz allein, das war eine Erzzauberin. Am "
        "Tage machte sie sich zur Katze oder zur Nachteule, des Abends aber wurde "
        "sie wieder ordentlich wie ein Mensch gestaltet. Sie konnte das Wild und "
        "die Vögel herbeilocken, und dann schlachtete sie, kochte und briet es. "
        "Wenn jemand auf hundert Schritte dem Schloß nahe kam, so mußte er stille "
        "stehen und konnte sich nicht von der Stelle bewegen, bis sie ihn "
        "lossprach. Wenn aber eine keusche Jungfrau in diesen Kreis kam, so "
        "verwandelte sie dieselbe in einen Vogel und sperrte sie dann in einen "
        "Korb ein und trug den Korb in eine Kammer des Schlosses. Sie hatte wohl "
        "siebentausend solcher Körbe mit so raren Vögeln im Schlosse.\n"
        "\n"
        "Nun war einmal eine Jungfrau, die hieß Jorinde; sie war schöner als alle "
        "anderen Mädchen. Die und dann ein gar schöner Jüngling, namens Joringel, "
        "hatten sich zusammen versprochen. Sie waren in den Brauttagen und sie "
        "hatten ihr größtes Vergnügen eins am andern. Damit sie nun einsmalen "
        "vertraut zusammen reden könnten, gingen sie in den Wald spazieren. »Hüte "
        "dich,« sagte Joringel, »daß du nicht so nahe ans Schloß kommst.« Es war "
        "ein schöner Abend, die Sonne schien zwischen den Stämmen der Bäume hell "
        "ins dunkle Grün des Waldes, und die Turteltaube sang kläglich auf den "
        "alten Maibuchen.\n"
        "\n"
        "Jorinde weinte zuweilen, setzte sich hin im Sonnenschein und klagte; "
        "Joringel klagte auch. Sie waren so bestürzt, als wenn sie hätten sterben "
        "sollen. Sie sahen sich um, waren irre und wußten nicht, wohin sie nach "
        "Hause gehen sollten. Noch halb stand die Sonne über dem Berg und halb war "
        "sie unter. Joringel sah durchs Gebüsch und sah die alte Mauer des "
        "Schlosses nahe bei sich; er erschrak und wurde todbang. Jorinde sang:\n"
        "\n"
        "»Mein Vöglein mit dem Ringlein rot singt Leide, Leide, Leide: Es singt "
        "dem Täubelein seinen Tod, singt Leide, Lei — zucküth, zicküth, zicküth.«\n"
        "\n"
        "Joringel sah nach Jorinde. Jorinde war in eine Nachtigall verwandelt, die "
        "sang »zicküth, zicküth«. Eine Nachteule mit glühenden Augen flog dreimal "
        "um sie herum und schrie dreimal »schuh, hu, hu, hu«. Joringel konnte sich "
        "nicht regen; er stand da wie ein Stein, konnte nicht weinen, nicht reden, "
        "nicht Hand noch Fuß regen. Nun war die Sonne unter: die Eule flog in "
        "einen Strauch, und gleich darauf kam eine alte krumme Frau aus diesem "
        "hervor, gelb und mager: große rote Augen, krumme Nase, die mit der Spitze "
        "ans Kinn reichte. Sie murmelte, fing die Nachtigall und trug sie auf der "
        "Hand fort. Joringel konnte nichts sagen, nicht von der Stelle kommen; die "
        "Nachtigall war fort. Endlich kam das Weib wieder und sagte mit dumpfer "
        "Stimme: »Grüß dich, Zachiel, wenns Möndel ins Körbel scheint, bind los, "
        "Zachiel, zu guter Stund.« Da wurde Joringel los. Er fiel vor dem Weib auf "
        "die Knie und bat, sie möchte ihm seine Jorinde wiedergeben, aber sie "
        "sagte, er sollte sie nie wieder haben, und ging fort. Er rief, er weinte, "
        "er jammerte, aber alles umsonst. »Uu, was soll mir geschehen?« Joringel "
        "ging fort und kam endlich in ein fremdes Dorf. Da hütete er die Schafe "
        "lange Zeit. Oft ging er rund um das Schloß herum, aber nicht zu nahe "
        "dabei. Endlich träumte er einmal des Nachts, er fände eine blutrote "
        "Blume, in deren Mitte eine schöne große Perle war. Die Blume brach er ab, "
        "ging damit zum Schlosse. Alles, was er mit der Blume berührte, ward von "
        "der Zauberei frei; auch träumte er, er hätte seine Jorinde dadurch wieder "
        "bekommen. Des Morgens, als er erwachte, fing er an durch Berg und Tal zu "
        "suchen, ob er eine solche Blume fände. Er suchte bis an den neunten Tag, "
        "da fand er die blutrote Blume am Morgen früh. In der Mitte war ein großer "
        "Tautropfe, so groß wie die schönste Perle. Diese Blume trug er Tag und "
        "Nacht bis zum Schloß. Wie er auf hundert Schritt nahe bis zum Schloß kam, "
        "da ward er nicht fest, sondern ging fort bis ans Tor. Joringel freute "
        "sich hoch, berührte die Pforte mit der Blume, und sie sprang auf. Er ging "
        "hinein, durch den Hof, horchte, wo er die vielen Vögel vernähme; endlich "
        "hörte er's. Er ging und fand den Saal, darauf war die Zauberin und "
        "fütterte die Vögel in den siebentausend Körben. Wie sie den Joringel sah, "
        "ward sie bös, sehr bös, schalt, spie Gift und Galle gegen ihn aus, aber "
        "sie konnte auf zwei Schritte nicht an ihn kommen. Er kehrte sich nicht an "
        "sie und ging, besah die Körbe mit den Vögeln; da waren aber viele hundert "
        "Nachtigallen, wie sollte er nun seine Jorinde wieder finden? Indem er so "
        "zusah, merkte er, daß die Alte heimlich ein Körbchen mit einem Vogel "
        "wegnahm und damit nach der Türe ging. Flugs sprang er hinzu, berührte das "
        "Körbchen mit der Blume und auch das alte Weib. Nun konnte es nicht mehr "
        "zaubern, und Jorinde stand da, hatte ihn um den Hals gefaßt, so schön wie "
        "sie ehemals war. Da machte er auch alle die andern Vögel wieder zu "
        "Jungfrauen, und da ging er mit seiner Jorinde nach Hause, und sie lebten "
        "lange vergnügt zusammen.",
        "Es war einmal eine kleine süße Dirne, die hatte jedermann lieb, der sie "
        "nur ansah, am allerliebsten aber ihre Großmutter, die wußte gar nicht, "
        "was sie alles dem Kinde geben sollte. Einmal schenkte sie ihm ein "
        "Käppchen von rotem Sammet, und weil ihm das so wohl stand und es nichts "
        "anders mehr tragen wollte, hieß es nur das Rotkäppchen. Eines Tages "
        "sprach seine Mutter zu ihm: »Komm, Rotkäppchen, da hast du ein Stück "
        "Kuchen und eine Flasche Wein, bring das der Großmutter hinaus; sie ist "
        "krank und schwach und wird sich daran laben. Mach dich auf, bevor es heiß "
        "wird, und wenn du hinauskommst, so geh hübsch sittsam und lauf nicht vom "
        "Weg ab, sonst fällst du und zerbrichst das Glas, und die Großmutter hat "
        "nichts. Und wenn du in ihre Stube kommst, so vergiß nicht guten Morgen zu "
        "sagen, und guck nicht erst in allen Ecken herum.«\n"
        "\n"
        "»Ich will schon alles gut machen,« sagte Rotkäppchen zur Mutter und gab "
        "ihr die Hand darauf. Die Großmutter aber wohnte draußen im Wald, eine "
        "halbe Stunde vom Dorf. Wie nun Rotkäppchen in den Wald kam, begegnete ihm "
        "der Wolf. Rotkäppchen aber wußte nicht, was das für ein böses Tier war "
        "und fürchtete sich nicht vor ihm. »Guten Tag, Rotkäppchen,« sprach er. "
        "»Schönen Dank, Wolf.« — »Wo hinaus so früh, Rotkäppchen?« — »Zur "
        "Großmutter.« — »Was trägst du unter der Schürze?« — »Kuchen und Wein. "
        "Gestern haben wir gebacken, da soll sich die kranke und schwache "
        "Großmutter etwas zugut tun und sich damit stärken.« — »Rotkäppchen, wo "
        "wohnt deine Großmutter?« — »Noch eine gute Viertelstunde weiter im Wald, "
        "unter den drei großen Eichbäumen, da steht ihr Haus, unten sind die "
        "Nußhecken, das wirst du ja wissen,« sagte Rotkäppchen. Der Wolf dachte "
        "bei sich: »Das junge zarte Ding, das ist ein fetter Bissen, der wird noch "
        "besser schmecken als die Alte. Du mußt es listig anfangen, damit du beide "
        "erschnappst.« Da ging er ein Weilchen neben Rotkäppchen her, dann sprach "
        "er: »Rotkäppchen, sieh einmal die schönen Blumen, die rings umher stehen, "
        "warum guckst du dich nicht um? Ich glaube, du hörst gar nicht, wie die "
        "Vöglein lieblich singen? Du gehst ja für dich hin, als wenn du zur Schule "
        "gingst, und ist so lustig haußen in dem Wald.«\n"
        "\n"
        "Rotkäppchen schlug die Augen auf, und als es sah, wie die Sonnenstrahlen "
        "durch die Bäume hin und her tanzten und alles voll schöner Blumen stand, "
        "dachte es: »Wenn ich der Großmutter einen frischen Strauß mitbringe, der "
        "wird ihr auch Freude machen; es ist so früh am Tag, daß ich doch zu "
        "rechter Zeit ankomme,« lief vom Wege ab in den Wald hinein und suchte "
        "Blumen. Und wenn es eine gebrochen hatte, meinte es, weiter hinaus stände "
        "eine schönere, und lief danach und geriet immer tiefer in den Wald "
        "hinein. Der Wolf aber ging geradeswegs nach dem Haus der Großmutter und "
        "klopfte an die Türe. »Wer ist draußen?« — »Rotkäppchen, das bringt Kuchen "
        "und Wein, mach auf.« — »Drück nur auf die Klinke,« rief die Großmutter, "
        "»ich bin zu schwach und kann nicht aufstehen.« Der Wolf drückte auf die "
        "Klinke, die Türe sprang auf, und er ging, ohne ein Wort zu sprechen, "
        "gerade zum Bett der Großmutter und verschluckte sie. Dann tat er ihre "
        "Kleider an, setzte ihre Haube auf, legte sich in ihr Bett und zog die "
        "Vorhänge vor.\n"
        "\n"
        "Rotkäppchen aber war nach den Blumen herumgelaufen, und als es soviel "
        "zusammen hatte, daß es keine mehr tragen konnte, fiel ihm die Großmutter "
        "wieder ein, und es machte sich auf den Weg zu ihr. Es wunderte sich, daß "
        "die Türe aufstand, und wie es in die Stube trat, so kam es ihm so seltsam "
        "darin vor, daß es dachte: »Ei, du mein Gott, wie ängstlich wird mir's "
        "heute zumut und bist sonst so gerne bei der Großmutter!« — Es rief: "
        "»Guten Morgen,« bekam aber keine Antwort. Darauf ging es zum Bett und zog "
        "die Vorhänge zurück. Da lag die Großmutter und hatte die Haube tief ins "
        "Gesicht gesetzt und sah so wunderlich aus. »Ei, Großmutter, was hast du "
        "für große Ohren!« — »Daß ich dich besser hören kann.« — »Ei, Großmutter, "
        "was hast du für große Augen!« — »Daß ich dich besser sehen kann.« — »Ei, "
        "Großmutter, was hast du für große Hände!« — »Daß ich dich besser packen "
        "kann.« — »Aber, Großmutter, was hast du für ein entsetzlich großes Maul!« "
        "— »Daß ich dich besser fressen kann.« Kaum hatte der Wolf das gesagt, so "
        "tat er einen Satz aus dem Bette und verschlang das arme Rotkäppchen.\n"
        "\n"
        "Wie der Wolf sein Gelüsten gestillt hatte, legte er sich wieder ins Bett, "
        "schlief ein und fing an überlaut zu schnarchen. Der Jäger ging eben an "
        "dem Haus vorbei und dachte: »Wie die alte Frau schnarcht, du mußt doch "
        "sehen, ob ihr etwas fehlt.« Da trat er in die Stube, und wie er vor das "
        "Bette kam, so sah er, daß der Wolf darin lag. »Finde ich dich hier, du "
        "alter Sünder,« sagte er, »ich habe dich lange gesucht.« Nun wollte er "
        "seine Büchse anlegen, da fiel ihm ein, der Wolf könnte die Großmutter "
        "gefressen haben und sie wäre noch zu retten: schoß nicht, sondern nahm "
        "eine Schere und fing an, dem schlafenden Wolf den Bauch aufzuschneiden. "
        "Wie er ein paar Schnitte getan hatte, da sah er das rote Käppchen "
        "leuchten, und noch ein paar Schnitte, da sprang das Mädchen heraus und "
        "rief: »Ach, wie war ich erschrocken, wie war's so dunkel in dem Wolf "
        "seinem Leib!« Und dann kam die alte Großmutter auch noch lebendig heraus "
        "und konnte kaum atmen. Rotkäppchen aber holte geschwind große Steine, "
        "damit füllten sie dem Wolf den Leib, und wie er aufwachte, wollte er "
        "fortspringen, aber die Steine waren so schwer, daß er gleich niedersank "
        "und sich tot fiel.\n"
        "\n"
        "Da waren alle drei vergnügt; der Jäger zog dem Wolf den Pelz ab und ging "
        "damit heim, die Großmutter aß den Kuchen und trank den Wein, den "
        "Rotkäppchen gebracht hatte, und erholte sich wieder, Rotkäppchen aber "
        "dachte: »Du willst dein Lebtag nicht wieder allein vom Wege ab in den "
        "Wald laufen, wenn dir's die Mutter verboten hat.«",
        "Es war einmal ein Müller, der war arm, aber er hatte eine schöne Tochter. "
        "Nun traf es sich, daß er mit dem König zu sprechen kam, und um sich ein "
        "Ansehen zu geben, sagte er zu ihm: »Ich habe eine Tochter, die kann Stroh "
        "zu Gold spinnen.« Der König sprach zum Müller: »Das ist eine Kunst, die "
        "mir wohl gefällt, wenn deine Tochter so geschickt ist, wie du sagst, so "
        "bring sie morgen in mein Schloß, da will ich sie auf die Probe stellen.« "
        "Als nun das Mädchen zu ihm gebracht ward, führte er es in eine Kammer, "
        "die ganz voll Stroh lag, gab ihr Rad und Haspel und sprach: »Jetzt mache "
        "dich an die Arbeit, und wenn du diese Nacht durch bis morgen früh dieses "
        "Stroh nicht zu Gold versponnen hast, so mußt du sterben.« Darauf schloß "
        "er die Kammer selbst zu, und sie blieb allein darin.\n"
        "\n"
        "Da saß nun die arme Müllerstochter und wußte um ihr Leben keinen Rat. Sie "
        "verstand gar nichts davon, wie man Stroh zu Gold spinnen konnte, und ihre "
        "Angst ward immer größer, daß sie endlich zu weinen anfing. Da ging auf "
        "einmal die Türe auf und trat ein kleines Männchen herein und sprach: "
        "»Guten Abend, Jungfer Müllerin, warum weint sie so sehr?« — »Ach,« "
        "antwortete das Mädchen, »ich soll Stroh zu Gold spinnen und verstehe das "
        "nicht.« Sprach das Männchen: »Was gibst du mir, wenn ich dir's spinne?« — "
        "»Mein Halsband,« sagte das Mädchen. Das Männchen nahm das Halsband, "
        "setzte sich vor das Rädchen, und schnurr, schnurr, schnurr, dreimal "
        "gezogen, war die Spule voll. Dann steckte es eine andere auf, und "
        "schnurr, schnurr, schnurr, dreimal gezogen, war auch die zweite voll: und "
        "so ging's fort bis zum Morgen, da war alles Stroh versponnen, und alle "
        "Spulen waren voll Gold. Bei Sonnenaufgang kam schon der König, und als er "
        "das Gold erblickte, erstaunte er und freute sich, aber sein Herz ward nur "
        "noch goldgieriger. Er ließ die Müllerstochter in eine andere Kammer voll "
        "Stroh bringen, die noch viel größer war, und befahl ihr das auch in einer "
        "Nacht zu spinnen, wenn ihr das Leben lieb wäre. Das Mädchen wußte sich "
        "nicht zu helfen und weinte, da ging abermals die Türe auf, und das kleine "
        "Männchen erschien und sprach: »Was gibst du mir, wenn ich dir das Stroh "
        "zu Gold spinne?« — »Meinen Ring von dem Finger,« antwortete das Mädchen. "
        "Das Männchen nahm den Ring, fing wieder an zu schnurren mit dem Rade und "
        "hatte bis zum Morgen alles Stroh zu glänzendem Gold gesponnen. Der König "
        "freute sich über die Maßen bei dem Anblick, war aber noch immer nicht "
        "Goldes satt, sondern ließ die Müllerstochter in eine noch größere Kammer "
        "voll Stroh bringen und sprach: »Die mußt du noch in dieser Nacht "
        "verspinnen: gelingt dir's aber, so sollst du meine Gemahlin werden.« — "
        "»Wenn's auch eine Müllerstochter ist,« dachte er, »eine reichere Frau "
        "finde ich in der ganzen Welt nicht.« Als das Mädchen allein war, kam das "
        "Männlein zum drittenmal wieder und sprach: »Was gibst du mir, wenn ich "
        "dir noch diesmal das Stroh spinne?« — »Ich habe nichts mehr, das ich "
        "geben könnte,« antwortete das Mädchen. »So versprich mir, wenn du Königin "
        "wirst, dein erstes Kind.« — »Wer weiß wie das noch geht,« dachte die "
        "Müllerstochter und wußte sich auch in der Not nicht anders zu helfen; sie "
        "versprach also dem Männchen, was es verlangte, und das Männchen spann "
        "dafür noch einmal das Stroh zu Gold. Und als am Morgen der König kam und "
        "alles fand, wie er gewünscht hatte, so hielt er Hochzeit mit ihr, und die "
        "schöne Müllerstochter ward eine Königin.\n"
        "\n"
        "Über ein Jahr brachte sie ein schönes Kind zur Welt und dachte gar nicht "
        "mehr an das Männchen. Da trat es plötzlich in ihre Kammer und sprach: "
        "»Nun gib mir, was du versprochen hast.« Die Königin erschrak und bot dem "
        "Männchen alle Reichtümer des Königreiches an, wenn es ihr das Kind lassen "
        "wollte. Aber das Männchen sprach: »Nein, etwas Lebendes ist mir lieber "
        "als alle Schätze der Welt.« Da fing die Königin so an zu jammern und zu "
        "weinen, daß das Männchen Mitleiden mit ihr hatte: »Drei Tage will ich dir "
        "Zeit lassen,« sprach er, »wenn du bis dahin meinen Namen weißt, so sollst "
        "du dein Kind behalten.«\n"
        "\n"
        "Nun besann sich die Königin die ganze Nacht über auf alle Namen, die sie "
        "jemals gehört hatte, und schickte einen Boten über Land, der sollte sich "
        "erkundigen weit und breit, was es sonst noch für Namen gäbe. Als am "
        "andern Tag das Männchen kam, fing sie an mit Kaspar, Melchior, Balzer, "
        "und sagte alle Namen, die sie wußte, nach der Reihe her, aber bei jedem "
        "sprach das Männlein: »So heiß ich nicht.« Den zweiten Tag ließ sie in der "
        "Nachbarschaft herumfragen, wie die Leute da genannt würden, und sagte dem "
        "Männlein die ungewöhnlichsten und seltsamsten Namen vor: »Heißt du "
        "vielleicht Rippenbiest oder Hammelswade oder Schnürbein?« Aber es "
        "antwortete immer: »So heiß ich nicht.« Den dritten Tag kam der Bote "
        "wieder zurück und erzählte: »Neue Namen habe ich keinen einzigen finden "
        "können, aber wie ich an einen hohen Berg um die Waldecke kam, wo Fuchs "
        "und Has sich gute Nacht sagen, so sah ich da ein kleines Haus, und vor "
        "dem Haus brannte ein Feuer, und um das Feuer sprang ein gar zu "
        "lächerliches Männchen, hüpfte auf einem Bein und schrie:\n"
        "\n"
        "»Heute back ich, morgen brau ich, übermorgen hol ich der Königin ihr "
        "Kind; ach, wie gut ist, daß niemand weiß, daß ich Rumpelstilzchen heiß!«\n"
        "\n"
        "Da könnt ihr denken, wie die Königin froh war, als sie den Namen hörte, "
        "und als bald hernach das Männlein hereintrat und fragte: »Nun, Frau "
        "Königin, wie heiß ich?« fragte sie erst: »Heißest du Kunz?« — »Nein.« — "
        "»Heißest du Heinz?« — »Nein.«\n"
        "\n"
        "»Heißt du etwa Rumpelstilzchen?«\n"
        "\n"
        "»Das hat dir der Teufel gesagt, das hat dir der Teufel gesagt,« schrie "
        "das Männlein und stieß mit dem rechten Fuß vor Zorn so tief in die Erde, "
        "daß es bis an den Leib hineinfuhr, dann packte es in seiner Wut den "
        "linken Fuß mit beiden Händen und riß sich selbst mitten entzwei.",
        "Eine arme Witwe, die lebte einsam in einem Hüttchen, und vor dem Hüttchen "
        "war ein Garten, darin standen zwei Rosenbäumchen, davon trug das eine "
        "weiße, das andere rote Rosen; und sie hatte zwei Kinder, die glichen den "
        "beiden Rosenbäumchen, und das eine hieß Schneeweißchen, das andere "
        "Rosenrot. Sie waren aber so fromm und gut, so arbeitsam und unverdrossen, "
        "als je zwei Kinder auf der Welt gewesen sind. Schneeweißchen war nur "
        "stiller und sanfter als Rosenrot. Rosenrot sprang lieber in den Wiesen "
        "und Feldern umher, suchte Blumen und fing Sommervögel; Schneeweißchen "
        "aber saß daheim bei der Mutter, half ihr im Hauswesen oder las ihr vor, "
        "wenn nichts zu tun war. Die beiden Kinder hatten einander so lieb, daß "
        "sie sich immer an den Händen faßten, so oft sie zusammen ausgingen; und "
        "wenn Schneeweißchen sagte: »Wir wollen uns nicht verlassen,« so "
        "antwortete Rosenrot: »Solange wir leben nicht,« und die Mutter setzte "
        "hinzu: »Was das eine hat, soll's mit dem andern teilen.« Oft liefen sie "
        "im Walde allein umher und sammelten rote Beeren, aber kein Tier tat ihnen "
        "etwas zuleid, sondern sie kamen vertraulich herbei: das Häschen fraß ein "
        "Kohlblatt aus ihren Händen, das Reh graste an ihrer Seite, der Hirsch "
        "sprang ganz lustig vorbei und die Vögel blieben auf den Ästen sitzen und "
        "sangen, was sie nur wußten. Kein Unfall traf sie; wenn sie sich im Walde "
        "verspätet hatten und die Nacht sie überfiel, so legten sie sich "
        "nebeneinander auf das Moos und schliefen bis der Morgen kam, und die "
        "Mutter wußte das und hatte ihretwegen keine Sorge. Einmal, als sie im "
        "Walde übernachtet hatten und das Morgenrot sie aufweckte, da sahen sie "
        "ein schönes Kind in einem weißen, glänzenden Kleidchen neben ihrem Lager "
        "sitzen. Es stand auf und blickte sie ganz freundlich an, sprach aber "
        "nichts und ging in den Wald hinein. Und als sie sich umsahen, so hatten "
        "sie ganz nahe bei einem Abgrunde geschlafen und wären gewiß "
        "hineingefallen, wenn sie in der Dunkelheit noch ein paar Schritte "
        "weitergegangen wären. Die Mutter aber sagte ihnen, das müßte der Engel "
        "gewesen sein, der gute Kinder bewache.\n"
        "\n"
        "Schneeweißchen und Rosenrot hielten das Hüttchen der Mutter so reinlich, "
        "daß es eine Freude war hineinzuschauen. Im Sommer besorgte Rosenrot das "
        "Haus und stellte der Mutter jeden Morgen, ehe sie aufwachte, einen "
        "Blumenstrauß vors Bett, darin war von jedem Bäumchen eine Rose. Im Winter "
        "zündete Schneeweißchen das Feuer an und hing den Kessel an den "
        "Feuerhaken, und der Kessel war von Messing, glänzte aber wie Gold, so "
        "rein war er gescheuert. Abends, wenn die Flocken fielen, sagte die "
        "Mutter: »Geh, Schneeweißchen, und schieb den Riegel vor,« und dann "
        "setzten sie sich an den Herd, und die Mutter nahm die Brille und las aus "
        "einem großen Buche vor, und die beiden Mädchen hörten zu, saßen und "
        "spannen; neben ihnen lag ein Lämmchen auf dem Boden und hinter ihnen auf "
        "einer Stange saß ein weißes Täubchen und hatte seinen Kopf unter den "
        "Flügel gesteckt.\n"
        "\n"
        "Eines Abends, als sie so vertraulich beisammen saßen, klopfte jemand an "
        "die Türe, als wollte er eingelassen sein. Die Mutter sprach: »Geschwind, "
        "Rosenrot, mach auf, es wird ein Wanderer sein, der Obdach sucht.« "
        "Rosenrot ging und schob den Riegel weg und dachte, es wäre ein armer "
        "Mann, aber der war es nicht, es war der Bär, der seinen dicken schwarzen "
        "Kopf zur Tür hereinsteckte. Rosenrot schrie laut und sprang zurück; das "
        "Lämmchen blökte, das Täubchen flatterte auf, und Schneeweißchen "
        "versteckte sich hinter der Mutter Bett. Der Bär aber fing an zu sprechen "
        "und sagte: »Fürchtet euch nicht, ich tue euch nichts zuleid, ich bin halb "
        "erfroren und will mich nur ein wenig bei euch wärmen.« — »Du armer Bär,« "
        "sprach die Mutter, »leg dich ans Feuer und gib nur acht, daß dir dein "
        "Pelz nicht brennt.« Dann rief sie: »Schneeweißchen, Rosenrot, kommt "
        "hervor, der Bär tut euch nichts, er meint's ehrlich.« Da kamen sie beide "
        "heran, und nach und nach näherten sich auch das Lämmchen und Täubchen und "
        "hatten keine Furcht vor ihm. Der Bär sprach: »Ihr Kinder, klopft mir den "
        "Schnee ein wenig aus dem Pelzwerk,« und sie holten den Besen und kehrten "
        "dem Bären das Fell rein; er aber streckte sich ans Feuer und brummte ganz "
        "vergnügt und behaglich. Nicht lange, so wurden sie ganz vertraut und "
        "trieben Mutwillen mit dem unbeholfenen Gast. Sie zausten ihm das Fell mit "
        "den Händen, setzten ihre Füßchen auf seinen Rücken und walgerten ihn hin "
        "und her, oder sie nahmen eine Haselrute und schlugen auf ihn los, und "
        "wenn er brummte, so lachten sie. Der Bär ließ sich's aber gerne gefallen, "
        "nur wenn sie's gar zu arg machten, rief er: »Laßt mich am Leben, ihr "
        "Kinder:\n"
        "\n"
        "Schneeweißchen, Rosenrot, schlägst dir den Freier tot.«\n"
        "\n"
        "Als Schlafenszeit war und die andern zu Bett gingen, sagte die Mutter zu "
        "dem Bär: »Du kannst in Gottes Namen da am Herde liegen bleiben, so bist "
        "du vor der Kälte und dem bösen Wetter geschützt.« Sobald der Tag graute, "
        "ließen ihn die beiden Kinder hinaus und er trabte über den Schnee in den "
        "Wald hinein. Von nun an kam der Bär jeden Abend zu der bestimmten Stunde, "
        "legte sich an den Herd und erlaubte den Kindern, Kurzweil zu treiben, "
        "soviel sie wollten; und sie waren so gewöhnt an ihn, daß die Türe nicht "
        "eher zugeriegelt ward, als bis der schwarze Gesell angelangt war.\n"
        "\n"
        "Als das Frühjahr herangekommen und draußen alles grün war, sagte der Bär "
        "eines Morgens zu Schneeweißchen: »Nun muß ich fort und darf den ganzen "
        "Sommer nicht wiederkommen.« — »Wo gehst du denn hin, lieber Bär?« fragte "
        "Schneeweißchen. »Ich muß in den Wald und meine Schätze vor den bösen "
        "Zwergen hüten; im Winter, wenn die Erde hart gefroren ist, müssen sie "
        "wohl unten bleiben und können sich nicht durcharbeiten, aber jetzt, wenn "
        "die Sonne die Erde aufgetaut und erwärmt hat, da brechen sie durch, "
        "steigen herauf, suchen und stehlen; was einmal in ihren Händen ist und in "
        "ihren Höhlen liegt, das kommt so leicht nicht wieder an des Tages Licht.« "
        "Schneeweißchen war ganz traurig über den Abschied, und als es ihm die "
        "Türe aufriegelte und der Bär sich hinausdrängte, blieb er an dem Türhaken "
        "hängen und ein Stück seiner Haut riß auf, und da war es Schneeweißchen, "
        "als hätte es Gold durchschimmern gesehen; aber es war seiner Sache nicht "
        "gewiß. Der Bär lief eilig fort und war bald hinter den Bäumen "
        "verschwunden.\n"
        "\n"
        "Nach einiger Zeit schickte die Mutter die Kinder in den Wald, Reisig zu "
        "sammeln. Da fanden sie draußen einen großen Baum, der lag gefällt auf dem "
        "Boden, und an dem Stamme sprang zwischen dem Gras etwas auf und ab, sie "
        "konnten aber nicht unterscheiden, was es war. Als sie näherkamen, sahen "
        "sie einen Zwerg mit einem alten verwelkten Gesicht und einem ellenlangen "
        "schneeweißen Bart. Das Ende des Bartes war in eine Spalte des Baumes "
        "eingeklemmt, und der Kleine sprang hin und her wie ein Hündchen an einem "
        "Seil und wußte nicht, wie er sich helfen sollte. Er glotzte die Mädchen "
        "mit seinen roten, feurigen Augen an und schrie: »Was steht ihr da! Könnt "
        "ihr nicht herbeigehen und mir Beistand leisten?« — »Was hast du "
        "angefangen, kleines Männchen?« fragte Rosenrot. »Dumme, neugierige Gans,« "
        "antwortete der Zwerg, »den Baum habe ich mir spalten wollen, um kleines "
        "Holz in der Küche zu haben; bei den dicken Klötzen verbrennt gleich das "
        "bißchen Speise, das unsereiner braucht, der nicht so viel "
        "hinunterschlingt als ihr grobes, gieriges Volk. Ich hatte den Keil schon "
        "glücklich hineingetrieben, und es wäre alles nach Wunsch gegangen, aber "
        "das verwünschte Holz war zu glatt und sprang unversehens heraus, und der "
        "Baum fuhr so geschwind zusammen, daß ich meinen schönen, weißen Bart "
        "nicht mehr herausziehen konnte; nun steckt er drin, und ich kann nicht "
        "fort. Da lachen die albernen, glatten Milchgesichter! Pfui, was seid ihr "
        "garstig!« Die Kinder gaben sich alle Mühe, aber sie konnten den Bart "
        "nicht herausziehen, er steckte zu fest. Ich will laufen und Leute "
        "herbeiholen,« sagte Rosenrot. »Wahnsinnige Schafsköpfe,« schnarrte der "
        "Zwerg, »wer wird gleich Leute herbeirufen, ihr seid mir schon um zwei "
        "zuviel; fällt euch nichts Besseres ein?« — »Sei nur nicht ungeduldig,« "
        "sagte Schneeweißchen, »ich will schon Rat schaffen,« holte sein Scherchen "
        "aus der Tasche und schnitt das Ende des Bartes ab. Sobald der Zwerg sich "
        "frei fühlte, griff er nach einem Sack, der zwischen den Wurzeln des "
        "Baumes steckte und mit Gold gefüllt war, hob ihn heraus und brummte vor "
        "sich hin: »Ungehobeltes Volk, schneidet mir ein Stück von meinem stolzen "
        "Barte ab! Lohn's euch der Kuckuck!« Damit schwang er seinen Sack auf den "
        "Rücken und ging fort, ohne die Kinder nur einmal anzusehen.\n"
        "\n"
        "Einige Zeit danach wollten Schneeweißchen und Rosenrot ein Gericht Fische "
        "angeln. Als sie nahe bei dem Bache waren, sahen sie, daß etwas wie eine "
        "große Heuschrecke nach dem Wasser zu hüpfte, als wollte es "
        "hineinspringen. Sie liefen heran und erkannten den Zwerg. »Wo willst du "
        "hin?« sagte Rosenrot, »du willst doch nicht ins Wasser?« — »Solch ein "
        "Narr bin ich nicht,« schrie der Zwerg, »seht ihr nicht, der verwünschte "
        "Fisch will mich hineinziehen?« Der Kleine hatte dagesessen und geangelt, "
        "und unglücklicherweise hatte der Wind seinen Bart mit der Angelschnur "
        "verflochten; als gleich darauf ein großer Fisch anbiß, fehlten dem "
        "schwachen Geschöpf die Kräfte, ihn herauszuziehen; der Fisch behielt die "
        "Oberhand und riß den Zwerg zu sich hin. Zwar hielt er sich an allen "
        "Halmen und Binsen, aber das half nicht viel, er mußte den Bewegungen des "
        "Fisches folgen und war in beständiger Gefahr, ins Wasser gezogen zu "
        "werden. Die Mädchen kamen zu rechter Zeit, hielten ihn fest und "
        "versuchten den Bart von der Schnur loszumachen, aber vergebens, Bart und "
        "Schnur waren fest ineinander verwirrt. Es blieb nichts übrig, als das "
        "Scherchen hervorzuholen und den Bart abzuschneiden, wobei ein kleiner "
        "Teil desselben verloren ging. Als der Zwerg das sah, schrie er sie an: "
        "»Ist das Manier, ihr Lorche, einem das Gesicht zu schänden? Nicht genug, "
        "daß ihr mir den Bart unten abgestutzt habt, jetzt schneidet ihr mir den "
        "besten Teil davon ab; ich darf mich vor den Meinigen gar nicht sehen "
        "lassen. Daß ihr laufen müßtet und die Schuhsohlen verloren hättet!« Dann "
        "holte er einen Sack Perlen, der im Schilfe lag, und ohne ein Wort weiter "
        "zu sagen, schleppte er ihn fort und verschwand hinter einem Stein.\n"
        "\n"
        "Es trug sich zu, daß bald hernach die Mutter die beiden Mädchen nach der "
        "Stadt schickte, Zwirn, Nadeln, Schnüre und Bänder einzukaufen. Der Weg "
        "führte sie über eine Heide, auf der hier und da mächtige Felsenstücke "
        "zerstreut lagen. Da sahen sie einen großen Vogel in der Luft schweben, "
        "der langsam über ihnen kreiste, sich immer tiefer herabsenkte und endlich "
        "nicht weit bei einem Felsen niederstieß. Gleich darauf hörten sie einen "
        "durchdringenden, jämmerlichen Schrei. Sie liefen herzu und sahen mit "
        "Schrecken, daß der Adler ihren alten Bekannten, den Zwerg, gepackt hatte "
        "und ihn forttragen wollte. Die mitleidigen Kinder hielten gleich das "
        "Männchen fest und zerrten sich so lange mit dem Adler herum, bis er seine "
        "Beute fahren ließ. Als der Zwerg sich von dem ersten Schrecken erholt "
        "hatte, schrie er mit seiner kreischenden Stimme: »Konntet ihr nicht "
        "säuberlicher mit mir umgehen? Gerissen habt ihr an meinem dünnen "
        "Röckchen, daß es überall zerfetzt und durchlöchert ist, unbeholfenes und "
        "täppisches Gesindel, das ihr seid!« Dann nahm er einen Sack mit "
        "Edelsteinen und schlüpfte wieder unter den Felsen in seine Höhle. Die "
        "Mädchen waren an seinen Undank schon gewöhnt, setzten ihren Weg fort und "
        "verrichteten ihr Geschäft in der Stadt. Als sie beim Heimweg wieder auf "
        "die Heide kamen, überraschten sie den Zwerg, der auf einem reinlichen "
        "Plätzchen seinen Sack mit Edelsteinen ausgeschüttet und nicht gedacht "
        "hatte, daß so spät noch jemand daherkommen würde. Die Abendsonne schien "
        "über die glänzenden Steine, sie schimmerten und leuchteten so prächtig in "
        "allen Farben, daß die Kinder stehen blieben und sie betrachteten. »Was "
        "steht ihr da und habt Maulaffen feil!« schrie der Zwerg, und sein "
        "aschgraues Gesicht ward zinnoberrot vor Zorn. Er wollte mit seinen "
        "Scheltworten fortfahren, als sich ein lautes Brummen hören ließ und ein "
        "schwarzer Bär aus dem Walde herbeitrabte. Erschrocken sprang der Zwerg "
        "auf, aber er konnte nicht mehr zu seinem Schlupfwinkel gelangen, der Bär "
        "war schon in seiner Nähe. Da rief er in Herzensangst: »Lieber Herr Bär, "
        "verschont mich, ich will Euch alle meine Schätze geben, sehet, die "
        "schönen Edelsteine, die da liegen. Schenkt mir das Leben, was habt Ihr an "
        "mir kleinem schmächtigen Kerl? Ihr spürt mich nicht zwischen den Zähnen; "
        "da, die beiden gottlosen Mädchen packt, das sind für Euch zarte Bissen, "
        "fett wie junge Wachteln, die freßt in Gottes Namen.« Der Bär kümmerte "
        "sich um seine Worte nicht, gab dem boshaften Geschöpf einen einzigen "
        "Schlag mit der Tatze, und es regte sich nicht mehr.\n"
        "\n"
        "Die Mädchen waren fortgesprungen, aber der Bär rief ihnen nach: "
        "»Schneeweißchen und Rosenrot, fürchtet euch nicht, wartet, ich will mit "
        "euch gehen.« Da erkannten sie seine Stimme und blieben stehen, und als "
        "der Bär bei ihnen war, fiel plötzlich die Bärenhaut ab, und er stand da "
        "als ein schöner Mann und war ganz in Gold gekleidet. »Ich bin eines "
        "Königs Sohn,« sprach er, »und war von dem gottlosen Zwerg, der mir meine "
        "Schätze gestohlen hatte, verwünscht, als ein wilder Bär in dem Walde zu "
        "laufen, bis ich durch seinen Tod erlöst würde. Jetzt hat er seine "
        "wohlverdiente Strafe empfangen.«\n"
        "\n"
        "Schneeweißchen ward mit ihm vermählt und Rosenrot mit seinem Bruder, und "
        "sie teilten die großen Schätze miteinander, die der Zwerg in seine Höhle "
        "zusammengetragen hatte. Die alte Mutter lebte noch lange Jahre ruhig und "
        "glücklich bei ihren Kindern. Die zwei Rosenbäumchen aber nahm sie mit, "
        "und sie standen vor ihrem Fenster und trugen jedes Jahr die schönsten "
        "Rosen, weiß und rot.",
        "Hans hatte sieben Jahre bei seinem Herrn gedient, da sprach er zu ihm: "
        "»Herr, meine Zeit ist herum, nun wollte ich gerne wieder heim zu meiner "
        "Mutter, gebt mir meinen Lohn.« Der Herr antwortete: »Du hast mir treu und "
        "ehrlich gedient, wie der Dienst war, so soll der Lohn sein,« und gab ihm "
        "ein Stück Gold, das so groß als Hansens Kopf war. Hans zog sein Tüchlein "
        "aus der Tasche, wickelte den Klumpen hinein, setzte ihn auf die Schulter "
        "und machte sich auf den Weg nach Haus. Wie er so dahin ging und immer ein "
        "Bein vor das andere setzte, kam ihm ein Reiter in die Augen, der frisch "
        "und fröhlich auf einem muntern Pferd vorbeitrabte. »Ach,« sprach Hans "
        "ganz laut, »was ist das Reiten ein schönes Ding! Da sitzt einer wie auf "
        "einem Stuhl, stößt sich an keinen Stein, spart die Schuh und kommt fort, "
        "er weiß nicht wie.« Der Reiter, der das gehört hatte, hielt an und rief: "
        "»Ei, Hans, warum laufst du auch zu Fuß?« — »Ich muß ja wohl,« antwortete "
        "er, »da habe ich einen Klumpen heimzutragen; es ist zwar Gold, aber ich "
        "kann den Kopf dabei nicht gerad halten, auch drückt mir's auf die "
        "Schulter.« — »Weißt du was,« sagte der Reiter, »wir wollen tauschen: ich "
        "gebe dir mein Pferd und du gibst mir deinen Klumpen.« — »Von Herzen "
        "gern,« sprach Hans, »aber ich sage Euch, Ihr müßt Euch damit schleppen.« "
        "Der Reiter stieg ab, nahm das Gold und half dem Hans hinauf, gab ihm die "
        "Zügel fest in die Hände und sprach: »Wenn's nun recht geschwind soll "
        "gehen, so mußt du mit der Zunge schnalzen und hopp, hopp rufen.«\n"
        "\n"
        "Hans war seelenfroh, als er auf dem Pferde saß und so frank und frei "
        "dahinritt. Über ein Weilchen fiel's ihm ein, es sollte noch schneller "
        "gehen, und fing an mit der Zunge zu schnalzen und hopp hopp zu rufen. Das "
        "Pferd setzte sich in starken Trab, und ehe sich's Hans versah, war er "
        "abgeworfen und lag in einem Graben, der die Äcker von der Landstraße "
        "trennte. Das Pferd wäre auch durchgegangen, wenn es nicht ein Bauer "
        "aufgehalten hätte, der des Weges kam und eine Kuh vor sich her trieb. "
        "Hans suchte seine Glieder zusammen und machte sich wieder auf die Beine. "
        "Er war aber verdrießlich und sprach zu dem Bauer: »Es ist ein schlechter "
        "Spaß, das Reiten, zumal, wenn man auf so eine Mähre gerät wie diese, die "
        "stößt und einen herabwirft, daß man den Hals brechen kann; ich setze mich "
        "nun und nimmermehr wieder auf. Da lob ich mir Eure Kuh, da kann einer mit "
        "Gemächlichkeit hinterher gehen und hat obendrein seine Milch, Butter und "
        "Käse jeden Tag gewiß. Was gäb ich darum, wenn ich so eine Kuh hätte!« — "
        "»Nun,« sprach der Bauer, »geschieht Euch so ein großer Gefallen, so will "
        "ich Euch wohl die Kuh für das Pferd vertauschen.« Hans willigte mit "
        "tausend Freuden ein; der Bauer schwang sich aufs Pferd und ritt eilig "
        "davon.\n"
        "\n"
        "Hans trieb seine Kuh ruhig vor sich her und bedachte den glücklichen "
        "Handel. »Hab ich nur ein Stück Brot, und daran wird mir's doch nicht "
        "fehlen, so kann ich, so oft mir's beliebt, Butter und Käse dazu essen; "
        "hab ich Durst, so melk ich meine Kuh und trinke Milch. Herz, was "
        "verlangst du mehr?« Als er zu einem Wirtshaus kam, machte er Halt, aß in "
        "der großen Freude alles, was er bei sich hatte, sein Mittags- und "
        "Abendbrot, rein auf und ließ sich für seine letzten paar Heller ein "
        "halbes Glas Bier einschenken. Dann trieb er seine Kuh weiter, immer nach "
        "dem Dorfe seiner Mutter zu. Die Hitze ward drückender, je näher der "
        "Mittag kam, und Hans befand sich in einer Heide, die wohl noch eine "
        "Stunde dauerte. Da ward es ihm ganz heiß, so daß ihm vor Durst die Zunge "
        "am Gaumen klebte. »Dem Ding ist zu helfen,« dachte Hans, »jetzt will ich "
        "meine Kuh melken und mich an der Milch laben.« Er band sie an einen "
        "dürren Baum, und da er keinen Eimer hatte, so stellte er seine Ledermütze "
        "unter, aber wie er sich auch abmühte, es kam kein Tropfen Milch zum "
        "Vorschein. Und weil er sich ungeschickt dabei anstellte, so gab ihm das "
        "ungeduldige Tier endlich mit einem der Hinterfüße einen solchen Schlag "
        "vor den Kopf, daß er zu Boden taumelte und eine Zeitlang sich gar nicht "
        "besinnen konnte, wo er war. Glücklicherweise kam gerade ein Metzger des "
        "Weges, der auf einem Schubkarren ein junges Schwein liegen hatte. »Was "
        "sind das für Streiche!« rief er und half dem guten Hans auf. Hans "
        "erzählte, was vorgefallen war. Der Metzger reichte ihm seine Flasche und "
        "sprach: »Da trinkt einmal und erholt Euch. Die Kuh will wohl keine Milch "
        "geben, das ist ein altes Tier, das höchstens noch zum Ziehen taugt oder "
        "zum Schlachten.« — »Ei, ei,« sprach Hans und strich sich die Haare über "
        "den Kopf, »wer hätte das gedacht! Es ist freilich gut, wenn man so ein "
        "Tier ins Haus abschlachten kann, was gibt's für Fleisch! Aber ich mache "
        "mir aus dem Kuhfleisch nicht viel, es ist mir nicht saftig genug. Ja, wer "
        "so ein junges Schwein hätte! Das schmeckt anders, dabei noch die Würste.« "
        "— »Hört, Hans,« sprach da der Metzger, »Euch zuliebe will ich tauschen "
        "und will Euch das Schwein für die Kuh lassen.« — »Gott lohn Euch Eure "
        "Freundschaft,« sprach Hans, übergab ihm die Kuh, ließ sich das "
        "Schweinchen vom Karren losmachen und den Strick, woran es gebunden war, "
        "in die Hand geben.\n"
        "\n"
        "Hans zog weiter und überdachte, wie ihm doch alles nach Wunsch ginge, "
        "begegnete ihm ja eine Verdrießlichkeit, so würde sie doch gleich wieder "
        "gut gemacht. Es gesellte sich danach ein Bursch zu ihm, der trug eine "
        "schöne weiße Gans unter dem Arm. Sie boten einander die Zeit, und Hans "
        "fing an von seinem Glück zu erzählen und wie er immer so vorteilhaft "
        "getauscht hätte. Der Bursch erzählte ihm, daß er die Gans zu einem "
        "Kindtaufsschmaus brächte. »Hebt einmal,« fuhr er fort und packte sie bei "
        "den Flügeln, »wie schwer sie ist, die ist aber auch acht Wochen lang "
        "genudelt worden. Wer in den Braten beißt, muß sich das Fett von beiden "
        "Seiten abwischen.« — »Ja,« sprach Hans und wog sie mit der einen Hand, "
        "»die hat ihr Gewicht, aber mein Schwein ist auch keine Sau.« Indessen sah "
        "sich der Bursch nach allen Seiten ganz bedenklich um, schüttelte auch "
        "wohl mit dem Kopf. »Hört,« fing er darauf an, »mit Eurem Schweine mag's "
        "nicht ganz richtig sein. In dem Dorfe, durch das ich gekommen bin, ist "
        "eben dem Schulzen eins aus dem Stall gestohlen worden. Ich fürchte, ich "
        "fürchte, Ihr habt's da in der Hand. Sie haben Leute ausgeschickt, und es "
        "wäre ein schlimmer Handel, wenn sie Euch mit dem Schwein erwischten; das "
        "geringste ist, daß Ihr ins finstere Loch gesteckt werdet.« Dem guten Hans "
        "ward bang: »Ach Gott,« sprach er, »helft mir aus der Not, Ihr wißt hier "
        "herum bessern Bescheid, nehmt mein Schwein da und laßt mir Eure Gans.« — "
        "»Ich muß schon etwas aufs Spiel setzen,« antwortete der Bursche, »aber "
        "ich will doch nicht schuld sein, daß Ihr ins Unglück geratet.« Er nahm "
        "also das Seil in die Hand und trieb das Schwein schnell auf einen "
        "Seitenweg fort; der gute Hans aber ging, seiner Sorgen entledigt, mit der "
        "Gans unter dem Arme der Heimat zu. »Wenn ich's recht überlege,« sprach er "
        "mit sich selbst, »habe ich noch Vorteil bei dem Tausch: erstlich den "
        "guten Braten, hernach die Menge von Fett, die herausträufeln wird, das "
        "gibt Gänsefettbrot auf ein Vierteljahr; und endlich die schönen weißen "
        "Federn, die laß ich mir in mein Kopfkissen stopfen, und darauf will ich "
        "wohl ungewiegt einschlafen. Was wird meine Mutter eine Freude haben!«\n"
        "\n"
        "Als er durch das letzte Dorf gekommen war, stand da ein Scherenschleifer "
        "mit seinem Karren, sein Rad schnurrte, und er sang dazu:\n"
        "\n"
        "»Ich schleife die Schere und drehe geschwind, und hänge mein Mäntelchen "
        "nach dem Wind.«\n"
        "\n"
        "Hans blieb stehen und sah ihm zu; endlich redete er ihn an und sprach: "
        "»Euch geht's wohl, weil Ihr so lustig bei Eurem Schleifen seid.« — »Ja,« "
        "antwortete der Scherenschleifer, »das Handwerk hat einen güldenen Boden. "
        "Ein rechter Schleifer ist ein Mann, der, so oft er in die Tasche greift, "
        "auch Geld darin findet. Aber wo habt Ihr die schöne Gans gekauft?« — »Die "
        "hab ich nicht gekauft, sondern für mein Schwein eingetauscht.« — »Und das "
        "Schwein?« — »Das hab ich für eine Kuh gekriegt.« — »Und die Kuh?« — »Die "
        "hab ich für ein Pferd bekommen.« — »Und das Pferd?« — »Dafür hab ich "
        "einen Klumpen Gold, so groß als mein Kopf, gegeben.« — »Und das Gold?« — "
        "»Ei, das war mein Lohn für sieben Jahre Dienst.« — »Ihr habt Euch "
        "jederzeit zu helfen gewußt,« sprach der Schleifer, »könnt Ihr's nun dahin "
        "bringen, daß Ihr das Geld in der Tasche springen hört, wenn Ihr aufsteht, "
        "so habt Ihr Euer Glück gemacht.« — »Wie soll ich das anfangen?« sprach "
        "Hans. »Ihr müßt Schleifer werden, wie ich; dazu gehört eigentlich nichts, "
        "als ein Wetzstein, das andere findet sich schon von selbst. Da hab ich "
        "einen, der ist zwar ein wenig schadhaft, dafür sollt Ihr mir aber auch "
        "weiter nichts als Eure Gans geben; wollt Ihr das?« — »Wie könnt Ihr noch "
        "fragen,« antwortete Hans, »ich werde ja zum glücklichsten Menschen auf "
        "Erden; habe ich Geld, so oft ich in die Tasche greife, was brauche ich da "
        "länger zu sorgen?« reichte ihm die Gans hin und nahm den Wetzstein in "
        "Empfang. »Nun,« sprach der Schleifer und hob einen gewöhnlichen schweren "
        "Feldstein, der neben ihm lag, auf, »da habt Ihr noch einen tüchtigen "
        "Stein dazu, auf dem sich's gut schlagen läßt und Ihr Eure alten Nägel "
        "gerade klopfen könnt. Nehmt ihn und hebt ihn ordentlich auf.«\n"
        "\n"
        "Hans lud den Stein auf und ging mit vergnügtem Herzen weiter; seine Augen "
        "leuchteten vor Freude. »Ich muß in einer Glückshaut geboren sein,« rief "
        "er aus, »alles was ich wünsche trifft mir ein, wie einem Sonntagskind.« "
        "Indessen, weil er seit Tagesanbruch auf den Beinen gewesen war, begann er "
        "müde zu werden; auch plagte ihn der Hunger, da er allen Vorrat auf einmal "
        "in der Freude über die erhandelte Kuh aufgezehrt hatte. Er konnte endlich "
        "nur mit Mühe weiter gehen und mußte jeden Augenblick Halt machen; dabei "
        "drückten ihn die Steine ganz erbärmlich. Da konnte er sich des Gedankens "
        "nicht erwehren, wie gut es wäre, wenn er sie gerade jetzt nicht zu tragen "
        "brauchte. Wie eine Schnecke kam er zu einem Feldbrunnen geschlichen, "
        "wollte da ruhen und sich mit einem frischen Trunk laben; damit er aber "
        "die Steine im Niedersitzen nicht beschädigte, legte er sie bedächtig "
        "neben sich auf den Rand des Brunnens. Darauf setzte er sich nieder und "
        "wollte sich zum Trinken bücken, da versah er's, stieß ein klein wenig an, "
        "und beide Steine plumpten hinab. Hans, als er sie mit seinen Augen in die "
        "Tiefe hatte versinken sehen, sprang vor Freuden auf, kniete dann nieder "
        "und dankte Gott mit Tränen in den Augen, daß er ihm auch diese Gnade noch "
        "erwiesen und ihn auf eine so gute Art und ohne, daß er sich einen Vorwurf "
        "zu machen brauchte, von den schweren Steinen befreit hätte, die ihm "
        "allein noch hinderlich gewesen wären. »So glücklich wie ich,« rief er "
        "aus, »gibt es keinen Menschen unter der Sonne.« Mit leichtem Herzen und "
        "frei von aller Last sprang er nun fort, bis er daheim bei seiner Mutter "
        "war.",
    ],
    "Alice im Wunderland": [
        "Alice fing an sich zu langweilen; sie saß schon lange bei ihrer\n"
        "Schwester am Ufer und hatte nichts zu thun. Das Buch, das ihre "
        "Schwester\n"
        "las, gefiel ihr nicht; denn es waren weder Bilder noch Gespräche "
        "darin.\n"
        "»Und was nützen Bücher,« dachte Alice, »ohne Bilder und Gespräche?«\n"
        "\n"
        "Sie überlegte sich eben, (so gut es ging, denn sie war schläfrig "
        "und\n"
        "dumm von der Hitze,) ob es der Mühe werth sei aufzustehen und\n"
        "Gänseblümchen zu pflücken, um eine Kette damit zu machen, als "
        "plötzlich\n"
        "ein weißes Kaninchen mit rothen Augen dicht an ihr vorbeirannte.\n"
        "\n"
        "Dies war grade nicht sehr merkwürdig; Alice fand es auch nicht sehr\n"
        "außerordentlich, daß sie das Kaninchen sagen hörte: »O weh, o weh! "
        "Ich\n"
        "werde zu spät kommen!« (Als sie es später wieder überlegte, fiel "
        "ihr\n"
        "ein, daß sie sich darüber hätte wundern sollen, doch zur Zeit kam es "
        "ihr\n"
        "Alles ganz natürlich vor.) Aber als das Kaninchen seine Uhr aus der\n"
        "Westentasche zog, nach der Zeit sah und eilig fortlief, sprang "
        "Alice\n"
        "auf; denn es war ihr doch noch nie vorgekommen, ein Kaninchen mit "
        "einer\n"
        "Westentasche und eine Uhr darin zu sehen. Vor Neugierde brennend, "
        "rannte\n"
        "sie ihm nach, über den Grasplatz, und kam noch zur rechten Zeit, um "
        "es\n"
        "in ein großes Loch unter der Hecke schlüpfen zu sehen.\n"
        "\n"
        "Den nächsten Augenblick war sie ihm nach in das Loch "
        "hineingesprungen,\n"
        "ohne zu bedenken, wie in aller Welt sie wieder herauskommen könnte.\n"
        "\n"
        "Der Eingang zum Kaninchenbau lief erst geradeaus, wie ein Tunnel "
        "und\n"
        "ging dann plötzlich abwärts; ehe Alice noch den Gedanken fassen "
        "konnte\n"
        "sich schnell festzuhalten, fühlte sie schon, daß sie fiel, wie es\n"
        "schien, in einen tiefen, tiefen Brunnen.\n"
        "\n"
        "Entweder mußte der Brunnen sehr tief sein, oder sie fiel sehr "
        "langsam;\n"
        "denn sie hatte Zeit genug, sich beim Fallen umzusehen und sich zu\n"
        "wundern, was nun wohl geschehen würde. Zuerst versuchte sie hinunter "
        "zu\n"
        "sehen, um zu wissen wohin sie käme, aber es war zu dunkel etwas zu\n"
        "erkennen. Da besah sie die Wände des Brunnens und bemerkte, daß sie "
        "mit\n"
        "Küchenschränken und Bücherbrettern bedeckt waren; hier und da "
        "erblickte\n"
        "sie Landkarten und Bilder, an Haken aufgehängt. Sie nahm im "
        "Vorbeifallen\n"
        "von einem der Bretter ein Töpfchen mit der Aufschrift: »Eingemachte\n"
        "Apfelsinen«, aber zu ihrem großen Verdruß war es leer. Sie wollte "
        "es\n"
        "nicht fallen lassen, aus Furcht Jemand unter sich zu tödten; und es\n"
        "gelang ihr, es in einen andern Schrank, an dem sie vorbeikam, zu\n"
        "schieben.\n"
        "\n"
        "»Nun!« dachte Alice bei sich, »nach einem solchen Fall werde ich "
        "mir\n"
        "nichts daraus machen, wenn ich die Treppe hinunter stolpere. Wie "
        "muthig\n"
        "sie mich zu Haus finden werden! Ich würde nicht viel Redens machen, "
        "wenn\n"
        "ich selbst von der Dachspitze hinunter fiele!« (Was sehr "
        "wahrscheinlich\n"
        "war.)\n"
        "\n"
        "Hinunter, hinunter, hinunter! Wollte denn der Fall nie endigen? "
        "»Wie\n"
        "viele Meilen ich wohl jetzt gefallen bin!« sagte sie laut. »Ich muß\n"
        "ungefähr am Mittelpunkt der Erde sein. Laß sehen: das wären "
        "achthundert\n"
        "und funfzig Meilen, glaube ich --« (denn ihr müßt wissen, Alice "
        "hatte\n"
        "dergleichen in der Schule gelernt, und obgleich dies keine sehr "
        "gute\n"
        "Gelegenheit war, ihre Kenntnisse zu zeigen, da Niemand zum Zuhören "
        "da\n"
        "war, so übte sie es sich doch dabei ein) -- »ja, das ist ungefähr "
        "die\n"
        "Entfernung; aber zu welchem Länge- und Breitegrade ich wohl "
        "gekommen\n"
        "sein mag?« (Alice hatte nicht den geringsten Begriff, was weder\n"
        "Längegrad noch Breitegrad war; doch klangen ihr die Worte großartig "
        "und\n"
        "nett zu sagen.)\n"
        "\n"
        "Bald fing sie wieder an. »Ob ich wohl ganz durch die Erde fallen "
        "werde!\n"
        "Wie komisch das sein wird, bei den Leuten heraus zu kommen, die auf "
        "dem\n"
        "Kopfe gehen! die Antipathien, glaube ich.« (Diesmal war es ihr ganz\n"
        "lieb, daß Niemand zuhörte, denn das Wort klang ihr gar nicht "
        "recht.)\n"
        "»Aber natürlich werde ich sie fragen müssen, wie das Land heißt. "
        "Bitte,\n"
        "liebe Dame, ist dies Neu-Seeland oder Australien?« (Und sie "
        "versuchte\n"
        "dabei zu knixen, -- denkt doch, knixen, wenn man durch die Luft "
        "fällt!\n"
        "Könntet ihr das fertig kriegen?) »Aber sie werden mich für ein\n"
        "unwissendes kleines Mädchen halten, wenn ich frage! Nein, es geht "
        "nicht\n"
        "an zu fragen; vielleicht sehe ich es irgendwo angeschrieben.«\n"
        "\n"
        "Hinunter, hinunter, hinunter! Sie konnte nichts weiter thun, also "
        "fing\n"
        "Alice bald wieder zu sprechen an. »Dinah wird mich gewiß heut Abend\n"
        "recht suchen!« (Dinah war die Katze.) »Ich hoffe, sie werden ihren "
        "Napf\n"
        "Milch zur Theestunde nicht vergessen. Dinah! Mies! ich wollte, du "
        "wärest\n"
        "hier unten bei mir. Mir ist nur bange, es giebt keine Mäuse in der "
        "Luft;\n"
        "aber du könntest einen Spatzen fangen; die wird es hier in der Luft "
        "wohl\n"
        "geben, glaubst du nicht? Und Katzen fressen doch Spatzen?« Hier "
        "wurde\n"
        "Alice etwas schläfrig und redete halb im Traum fort. »Fressen "
        "Katzen\n"
        "gern Spatzen? Fressen Katzen gern Spatzen? Fressen Spatzen gern "
        "Katzen?«\n"
        "Und da ihr Niemand zu antworten brauchte, so kam es gar nicht "
        "darauf\n"
        "an, wie sie die Frage stellte. Sie fühlte, daß sie einschlief und "
        "hatte\n"
        "eben angefangen zu träumen, sie gehe Hand in Hand mit Dinah "
        "spazieren,\n"
        "und frage sie ganz ernsthaft: »Nun, Dinah, sage die Wahrheit, hast "
        "du je\n"
        "einen Spatzen gefressen?« da mit einem Male, plump! plump! kam sie "
        "auf\n"
        "einen Haufen trocknes Laub und Reisig zu liegen, -- und der Fall war "
        "aus.\n"
        "\n"
        "Alice hatte sich gar nicht weh gethan. Sie sprang sogleich auf und "
        "sah\n"
        "in die Höhe; aber es war dunkel über ihr. Vor ihr lag ein zweiter "
        "langer\n"
        "Gang, und sie konnte noch eben das weiße Kaninchen darin entlang "
        "laufen\n"
        "sehen. Es war kein Augenblick zu verlieren: fort rannte Alice wie "
        "der\n"
        "Wind, und hörte es gerade noch sagen, als es um eine Ecke bog: »O, "
        "Ohren\n"
        "und Schnurrbart, wie spät es ist!« Sie war dicht hinter ihm, aber "
        "als\n"
        "sie um die Ecke bog, da war das Kaninchen nicht mehr zu sehen. Sie\n"
        "befand sich in einem langen, niedrigen Corridor, der durch eine "
        "Reihe\n"
        "Lampen erleuchtet war, die von der Decke herabhingen.\n"
        "\n"
        "Zu beiden Seiten des Corridors waren Thüren; aber sie waren alle\n"
        "verschlossen. Alice versuchte jede Thür erst auf einer Seite, dann "
        "auf\n"
        "der andern; endlich ging sie traurig in der Mitte entlang, "
        "überlegend,\n"
        "wie sie je heraus kommen könnte.\n"
        "\n"
        "Plötzlich stand sie vor einem kleinen dreibeinigen Tische, ganz von\n"
        "dickem Glas. Es war nichts darauf als ein winziges goldenes\n"
        "Schlüsselchen, und Alice's erster Gedanke war, dies möchte zu einer "
        "der\n"
        "Thüren des Corridors gehören. Aber ach! entweder waren die Schlösser "
        "zu\n"
        "groß, oder der Schlüssel war zu klein; kurz, er paßte zu keiner\n"
        "einzigen. Jedoch, als sie das zweite Mal herum ging, kam sie an "
        "einen\n"
        "niedrigen Vorhang, den sie vorher nicht bemerkt hatte, und dahinter "
        "war\n"
        "eine Thür, ungefähr funfzehn Zoll hoch. Sie steckte das goldene\n"
        "Schlüsselchen in's Schlüsselloch, und zu ihrer großen Freude paßte "
        "es.\n"
        "\n"
        "Alice schloß die Thür auf und fand, daß sie zu einem kleinen Gange\n"
        "führte, nicht viel größer als ein Mäuseloch. Sie kniete nieder und "
        "sah\n"
        "durch den Gang in den reizendsten Garten, den man sich denken kann. "
        "Wie\n"
        "wünschte sie, aus dem dunkeln Corridor zu gelangen, und unter den "
        "bunten\n"
        "Blumenbeeten und kühlen Springbrunnen umher zu wandern; aber sie "
        "konnte\n"
        "kaum den Kopf durch den Eingang stecken. »Und wenn auch mein Kopf\n"
        "hindurch ginge,« dachte die arme Alice, »was würde es nützen ohne "
        "die\n"
        "Schultern. O, ich möchte mich zusammenschieben können wie ein "
        "Teleskop!\n"
        "Das geht gewiß, wenn ich nur wüßte, wie man es anfängt.« Denn es "
        "war\n"
        "kürzlich so viel Merkwürdiges mit ihr vorgegangen, daß Alice anfing "
        "zu\n"
        "glauben, es sei fast nichts unmöglich.\n"
        "\n"
        "Es schien ihr ganz unnütz, länger bei der kleinen Thür zu warten. "
        "Daher\n"
        "ging sie zum Tisch zurück, halb und halb hoffend, sie würde noch "
        "einen\n"
        "Schlüssel darauf finden, oder jedenfalls ein Buch mit Anweisungen, "
        "wie\n"
        "man sich als Teleskop zusammenschieben könne. Diesmal fand sie ein\n"
        "Fläschchen darauf. »Das gewiß vorhin nicht hier stand,« sagte Alice; "
        "und\n"
        "um den Hals des Fläschchens war ein Zettel gebunden, mit den Worten\n"
        "»Trinke mich!« wunderschön in großen Buchstaben drauf gedruckt.\n"
        "\n"
        "Es war bald gesagt, »Trinke mich«, aber die altkluge kleine Alice "
        "wollte\n"
        "sich damit nicht übereilen. »Nein, ich werde erst nachsehen,« "
        "sprach\n"
        "sie, »ob ein Todtenkopf darauf ist oder nicht.« Denn sie hatte "
        "mehre\n"
        "hübsche Geschichten gelesen von Kindern, die sich verbrannt hatten "
        "oder\n"
        "sich von wilden Thieren hatten fressen lassen, und in andere "
        "unangenehme\n"
        "Lagen gerathen waren, nur weil sie nicht an die Warnungen dachten, "
        "die\n"
        "ihre Freunde ihnen gegeben hatten; zum Beispiel, daß ein "
        "rothglühendes\n"
        "Eisen brennt, wenn man es anfaßt; und daß wenn man sich mit einem\n"
        "Messer tief in den Finger schneidet, es gewöhnlich blutet. Und sie "
        "hatte\n"
        "nicht vergessen, daß wenn man viel aus einer Flasche mit einem\n"
        "Todtenkopf darauf trinkt, es einem unfehlbar schlecht bekommt.\n"
        "\n"
        "Diese Flasche jedoch hatte keinen Todtenkopf. Daher wagte Alice zu\n"
        "kosten; und da es ihr gut schmeckte (es war eigentlich wie ein "
        "Gemisch\n"
        "von Kirschkuchen, Sahnensauce, Ananas, Putenbraten, Naute und Armen\n"
        "Rittern), so trank sie die Flasche aus.\n"
        "\n"
        "       *       *       *       *       *\n"
        "\n"
        "»Was für ein komisches Gefühl!« sagte Alice. »Ich gehe gewiß zu wie "
        "ein\n"
        "Teleskop.«\n"
        "\n"
        "Und so war es in der That: jetzt war sie nur noch zehn Zoll hoch, "
        "und\n"
        "ihr Gesicht leuchtete bei dem Gedanken, daß sie nun die rechte Höhe\n"
        "habe, um durch die kleine Thür in den schönen Garten zu gehen. Doch\n"
        "erst wartete sie einige Minuten, ob sie noch mehr einschrumpfen "
        "werde.\n"
        "Sie war einigermaßen ängstlich; »denn es könnte damit aufhören,« "
        "sagte\n"
        "Alice zu sich selbst, »daß ich ganz ausginge, wie ein Licht. Mich\n"
        "wundert, wie ich dann aussähe?« Und sie versuchte sich vorzustellen, "
        "wie\n"
        "die Flamme von einem Lichte aussieht, wenn das Licht ausgeblasen "
        "ist;\n"
        "aber sie konnte sich nicht erinnern, dies je gesehen zu haben.\n"
        "\n"
        "Nach einer Weile, als sie merkte daß weiter nichts geschah, "
        "beschloß\n"
        "sie, gleich in den Garten zu gehen. Aber, arme Alice! als sie an "
        "die\n"
        "Thür kam, hatte sie das goldene Schlüsselchen vergessen. Sie ging "
        "nach\n"
        "dem Tische zurück, es zu holen, fand aber, daß sie es unmöglich\n"
        "erreichen konnte. Sie sah es ganz deutlich durch das Glas, und sie "
        "gab\n"
        "sich alle Mühe an einem der Tischfüße hinauf zu klettern, aber er "
        "war zu\n"
        "glatt; und als sie sich ganz müde gearbeitet hatte, setzte sich das\n"
        "arme, kleine Ding hin und weinte.\n"
        "\n"
        "»Still, was nützt es so zu weinen!« sagte Alice ganz böse zu sich\n"
        "selbst; »ich rathe dir, den Augenblick aufzuhören!« Sie gab sich "
        "oft\n"
        "sehr guten Rath (obgleich sie ihn selten befolgte), und manchmal "
        "schalt\n"
        "sie sich selbst so strenge, daß sie sich zum Weinen brachte; und\n"
        "einmal, erinnerte sie sich, hatte sie versucht sich eine Ohrfeige "
        "zu\n"
        "geben, weil sie im Croquet betrogen hatte, als sie gegen sich "
        "selbst\n"
        "spielte; denn dieses eigenthümliche Kind stellte sehr gern zwei "
        "Personen\n"
        "vor. »Aber jetzt hilft es zu nichts,« dachte die arme Alice, »zu "
        "thun\n"
        "als ob ich zwei verschiedene Personen wäre. Ach! es ist ja kaum "
        "genug\n"
        "von mir übrig zu einer anständigen Person!«\n"
        "\n"
        "Bald fiel ihr Auge auf eine kleine Glasbüchse, die unter dem Tische "
        "lag;\n"
        "sie öffnete sie und fand einen sehr kleinen Kuchen darin, auf "
        "welchem\n"
        "die Worte »Iß mich!« schön in kleinen Rosinen geschrieben standen. "
        "»Gut,\n"
        "ich will ihn essen,« sagte Alice, »und wenn ich davon größer werde, "
        "so\n"
        "kann ich den Schlüssel erreichen; wenn ich aber kleiner davon werde, "
        "so\n"
        "kann ich unter der Thür durchkriechen. So, auf jeden Fall, gelange "
        "ich\n"
        "in den Garten, -- es ist mir einerlei wie.«\n"
        "\n"
        "Sie aß ein Bißchen, und sagte neugierig zu sich selbst: »Aufwärts "
        "oder\n"
        "abwärts?« Dabei hielt sie die Hand prüfend auf ihren Kopf und war "
        "ganz\n"
        "erstaunt zu bemerken, daß sie dieselbe Größe behielt. Freilich "
        "geschieht\n"
        "dies gewöhnlich, wenn man Kuchen ißt; aber Alice war schon so an\n"
        "wunderbare Dinge gewöhnt, daß es ihr ganz langweilig schien, wenn "
        "das\n"
        "Leben so natürlich fortging.\n"
        "\n"
        "Sie machte sich also daran, und verzehrte den Kuchen völlig.\n"
        "\n"
        "»Verquerer und verquerer!« rief Alice. (Sie war so überrascht, daß "
        "sie\n"
        "im Augenblick ihre eigene Sprache ganz vergaß.) »Jetzt werde ich\n"
        "auseinander geschoben wie das längste Teleskop das es je gab! Lebt "
        "wohl,\n"
        "Füße!« (Denn als sie auf ihre Füße hinabsah, konnte sie sie kaum "
        "mehr zu\n"
        "Gesicht bekommen, so weit fort waren sie schon.) »O meine armen "
        "Füßchen!\n"
        "wer euch wohl nun Schuhe und Strümpfe anziehen wird, meine Besten? "
        "denn\n"
        "ich kann es unmöglich thun! Ich bin viel zu weit ab, um mich mit "
        "euch\n"
        "abzugeben! ihr müßt sehen, wie ihr fertig werdet. Aber gut muß ich "
        "zu\n"
        "ihnen sein,« dachte Alice, »sonst gehen sie vielleicht nicht, wohin "
        "ich\n"
        "gehen möchte. Laß mal sehen: ich will ihnen jeden Weihnachten ein "
        "Paar\n"
        "neue Stiefel schenken.«\n"
        "\n"
        "Und sie dachte sich aus, wie sie das anfangen würde. »Sie müssen "
        "per\n"
        "Fracht gehen,« dachte sie; »wie drollig es sein wird, seinen eignen\n"
        "Füßen ein Geschenk zu schicken! und wie komisch die Adresse "
        "aussehen\n"
        "wird! --\n"
        "\n"
        "    An\n"
        "      Alice's rechten Fuß, Wohlgeboren,\n"
        "        Fußteppich,\n"
        "          nicht weit vom Kamin,\n"
        "            mit Alice's Grüßen.\n"
        "\n"
        "»Oh, was für Unsinn ich schwatze!«\n"
        "\n"
        "Gerade in dem Augenblick stieß sie mit dem Kopf an die Decke: sie "
        "war in\n"
        "der That über neun Fuß groß. Und sie nahm sogleich den kleinen "
        "goldenen\n"
        "Schlüssel auf und rannte nach der Gartenthür.\n"
        "\n"
        "Arme Alice! das Höchste was sie thun konnte war, auf der Seite "
        "liegend,\n"
        "mit einem Auge nach dem Garten hinunterzusehen; aber an Durchgehen "
        "war\n"
        "weniger als je zu denken. Sie setzte sich hin und fing wieder an zu\n"
        "weinen.\n"
        "\n"
        "»Du solltest dich schämen,« sagte Alice, »solch großes Mädchen« (da\n"
        "hatte sie wohl recht) »noch so zu weinen! Höre gleich auf, sage ich\n"
        "dir!« Aber sie weinte trotzdem fort, und vergoß Thränen eimerweise, "
        "bis\n"
        "sich zuletzt ein großer Pfuhl um sie bildete, ungefähr vier Zoll "
        "tief\n"
        "und den halben Corridor lang.\n"
        "\n"
        "Nach einem Weilchen hörte sie Schritte in der Entfernung und "
        "trocknete\n"
        "schnell ihre Thränen, um zu sehen wer es sei. Es war das weiße\n"
        "Kaninchen, das prachtvoll geputzt zurückkam, mit einem Paar weißen\n"
        "Handschuhen in einer Hand und einen Fächer in der andern. Es "
        "trippelte\n"
        "in großer Eile entlang vor sich hin redend: »Oh! die Herzogin, die\n"
        "Herzogin! die wird mal außer sich sein, wenn ich sie warten lasse!«\n"
        "Alice war so rathlos, daß sie Jeden um Hülfe angerufen hätte. Als "
        "das\n"
        "Kaninchen daher in ihre Nähe kam, fing sie mit leiser, schüchterner\n"
        "Stimme an: »Bitte, lieber Herr. --« Das Kaninchen fuhr zusammen, "
        "ließ\n"
        "die weißen Handschuhe und den Fächer fallen und lief davon in die "
        "Nacht\n"
        "hinein, so schnell es konnte.\n"
        "\n"
        "Alice nahm den Fächer und die Handschuhe auf, und da der Gang sehr "
        "heiß\n"
        "war, fächelte sie sich, während sie so zu sich selbst sprach:\n"
        "»Wunderbar! -- wie seltsam heute Alles ist! Und gestern war es ganz "
        "wie\n"
        "gewöhnlich. Ob ich wohl in der Nacht umgewechselt worden bin? Laß "
        "mal\n"
        "sehen: war ich dieselbe, als ich heute früh aufstand? Es kommt mir "
        "fast\n"
        "vor, als hätte ich wie eine Veränderung in mir gefühlt. Aber wenn "
        "ich\n"
        "nicht dieselbe bin, dann ist die Frage: wer in aller Welt bin ich? "
        "Ja,\n"
        "das ist das Räthsel!« So ging sie in Gedanken alle Kinder ihres "
        "Alters\n"
        "durch, die sie kannte, um zu sehen, ob sie in eins davon verwandelt\n"
        "wäre.\n"
        "\n"
        "»Ich bin sicherlich nicht Ida,« sagte sie, »denn die trägt lange "
        "Locken,\n"
        "und mein Haar ist gar nicht lockig; und bestimmt kann ich nicht "
        "Clara\n"
        "sein, denn ich weiß eine ganze Menge, und sie, oh! sie weiß so sehr\n"
        "wenig! Außerdem, sie ist sie selbst, und ich bin ich, und, o wie "
        "confus\n"
        "es Alles ist! Ich will versuchen, ob ich noch Alles weiß, was ich "
        "sonst\n"
        "wußte. Laß sehen: vier mal fünf ist zwölf, und vier mal sechs ist\n"
        "dreizehn, und vier mal sieben ist -- o weh! auf die Art komme ich "
        "nie\n"
        "bis zwanzig! Aber, das Einmaleins hat nicht so viel zu sagen; ich "
        "will\n"
        "Geographie nehmen. London ist die Hauptstadt von Paris, und Paris "
        "ist\n"
        "die Hauptstadt von Rom, und Rom -- nein, ich wette, das ist Alles\n"
        "falsch! Ich muß in Clara verwandelt sein! Ich will doch einmal "
        "sehen, ob\n"
        "ich sagen kann: »Bei einem Wirthe --« und sie faltete sie Hände, als "
        "ob\n"
        "sie ihrer Lehrerin hersagte, und fing an; aber ihre Stimme klang "
        "rauh\n"
        "und ungewohnt, und die Worte kamen nicht wie sonst: --\n"
        "\n"
        "    »Bei einem Wirthe, wunderwild,\n"
        "    Da war ich jüngst zu Gaste,\n"
        "    Ein Bienennest das war sein Schild\n"
        "    In einer braunen Tatze.\n"
        "\n"
        "    Es war der grimme Zottelbär,\n"
        "    Bei dem ich eingekehret;\n"
        "    Mit süßem Honigseim hat er\n"
        "    Sich selber wohl genähret!«\n"
        "\n"
        "»Das kommt mir gar nicht richtig vor,« sagte die arme Alice, und "
        "Thränen\n"
        "kamen ihr in die Augen, als sie weiter sprach: »Ich muß doch Clara "
        "sein,\n"
        "und ich werde in dem alten kleinen Hause wohnen müssen, und beinah "
        "keine\n"
        "Spielsachen zum Spielen haben, und ach! so viel zu lernen! Nein, "
        "das\n"
        "habe ich mir vorgenommen: wenn ich Clara bin, will ich hier unten\n"
        "bleiben! Es soll ihnen nichts helfen, wenn sie die Köpfe\n"
        "zusammenstecken und herunter rufen: »Komm wieder herauf, Herzchen!« "
        "Ich\n"
        "will nur hinauf sehen und sprechen: wer bin ich denn? Sagt mir das "
        "erst,\n"
        "und dann, wenn ich die Person gern bin, will ich kommen; wo nicht, "
        "so\n"
        "will ich hier unten bleiben, bis ich jemand Anderes bin. -- Aber o "
        "weh!«\n"
        "schluchzte Alice plötzlich auf, »ich wünschte, sie sähen herunter! "
        "Es\n"
        "ist mir so langweilig, hier ganz allein zu sein!«\n"
        "\n"
        "Als sie so sprach, sah sie auf ihre Hände hinab und bemerkte mit\n"
        "Erstaunen, daß sie beim Reden einen von den weißen "
        "Glacee-Handschuhen\n"
        "des Kaninchens angezogen hatte. »Wie habe ich das nur angefangen?«\n"
        "dachte sie. »Ich muß wieder klein geworden sein.« Sie stand auf, "
        "ging\n"
        "nach dem Tische, um sich daran zu messen, und fand, daß sie noch\n"
        "ungefähr zwei Fuß hoch sei, dabei schrumpfte sie noch zusehends ein: "
        "sie\n"
        "merkte bald, daß die Ursache davon der Fächer war, den sie hielt; "
        "sie\n"
        "warf ihn schnell hin, noch zur rechten Zeit, sich vor gänzlichem\n"
        "Verschwinden zu retten.\n"
        "\n"
        "»Das war glücklich davon gekommen!« sagte Alice, sehr erschrocken "
        "über\n"
        "die plötzliche Veränderung, aber froh, daß sie noch existirte; »und "
        "nun\n"
        "in den Garten!« und sie lief eilig nach der kleinen Thür: aber ach! "
        "die\n"
        "kleine Thür war wieder verschlossen und das goldene Schlüsselchen "
        "lag\n"
        "auf dem Glastische wie vorher. »Und es ist schlimmer als je,« dachte "
        "das\n"
        "arme Kind, »denn so klein bin ich noch nie gewesen, nein, nie! Und "
        "ich\n"
        "sage, es ist zu schlecht, ist es!«\n"
        "\n"
        "Wie sie diese Worte sprach, glitt sie aus, und den nächsten "
        "Augenblick,\n"
        "platsch! fiel sie bis an's Kinn in Salzwasser. Ihr erster Gedanke "
        "war,\n"
        "sie sei in die See gefallen, »und in dem Fall kann ich mit der "
        "Eisenbahn\n"
        "zurückreisen,« sprach sie bei sich. (Alice war einmal in ihrem Leben "
        "an\n"
        "der See gewesen und war zu dem allgemeinen Schluß gelangt, daß wo "
        "man\n"
        "auch an's Seeufer kommt, man eine Anzahl Bademaschinen im Wasser\n"
        "findet, Kinder, die den Sand mit hölzernen Spaten aufgraben, dann "
        "eine\n"
        "Reihe Wohnhäuser und dahinter eine Eisenbahn-Station); doch merkte "
        "sie\n"
        "bald, daß sie sich in dem Thränenpfuhl befand, den sie geweint "
        "hatte,\n"
        "als sie neun Fuß hoch war.\n"
        "\n"
        "»Ich wünschte, ich hätte nicht so sehr geweint!« sagte Alice, als "
        "sie\n"
        "umherschwamm und sich herauszuhelfen suchte; »jetzt werde ich wohl "
        "dafür\n"
        "bestraft werden und in meinen eigenen Thränen ertrinken! Das wird\n"
        "sonderbar sein, das! Aber Alles ist heut so sonderbar.«\n"
        "\n"
        "In dem Augenblicke hörte sie nicht weit davon etwas in dem Pfuhle\n"
        "plätschern, und sie schwamm danach, zu sehen was es sei: erst "
        "glaubte\n"
        "sie, es müsse ein Wallroß oder ein Nilpferd sein; dann aber besann "
        "sie\n"
        "sich, wie klein sie jetzt war, und merkte bald, daß es nur eine "
        "Maus\n"
        "sei, die wie sie hineingefallen war.\n"
        "\n"
        "»Würde es wohl etwas nützen,« dachte Alice, »diese Maus anzureden? "
        "Alles\n"
        "ist so wunderlich hier unten, daß ich glauben möchte, sie kann "
        "sprechen;\n"
        "auf jeden Fall habe ich das Fragen umsonst.« Demnach fing sie an: "
        "»O\n"
        "Maus, weißt du, wie man aus diesem Pfuhle gelangt, ich bin von dem\n"
        "Herumschwimmen ganz müde, o Maus!« (Alice dachte, so würde eine "
        "Maus\n"
        "richtig angeredet; sie hatte es zwar noch nie gethan, aber sie "
        "erinnerte\n"
        "sich ganz gut, in ihres Bruders lateinischer Grammatik gelesen zu "
        "haben\n"
        "»Eine Maus -- einer Maus -- einer Maus -- eine Maus -- o Maus!«) "
        "Die\n"
        "Maus sah sie etwas neugierig an und schien ihr mit dem einen Auge "
        "zu\n"
        "blinzeln, aber sie sagte nichts.\n"
        "\n"
        "»Vielleicht versteht sie nicht Englisch,« dachte Alice, »es ist\n"
        "vielleicht eine französische Maus, die mit Wilhelm dem Eroberer "
        "herüber\n"
        "gekommen ist« (denn, trotz ihrer Geschichtskenntiß hatte Alice "
        "keinen\n"
        "ganz klaren Begriff, wie lange irgend ein Ereigniß her sei). Sie "
        "fing\n"
        "also wieder an: »=Où est ma chatte?=« was der erste Satz in ihrem\n"
        "französischen Conversationsbuche war. Die Maus sprang hoch auf aus "
        "dem\n"
        "Wasser, und schien vor Angst am ganzen Leibe zu beben. »O, ich bitte "
        "um\n"
        "Verzeihung!« rief Alice schnell, erschrocken, daß sie das arme "
        "Thier\n"
        "verletzt habe. »Ich hatte ganz vergessen, daß Sie Katzen nicht "
        "mögen.«\n"
        "\n"
        "»Katzen nicht mögen!« schrie die Maus mit kreischender, wüthender\n"
        "Stimme. »Würdest du Katzen mögen, wenn du an meiner Stelle wärest?«\n"
        "\n"
        "»Nein, wohl kaum,« sagte Alice in zuredendem Tone: »sei nicht mehr "
        "böse\n"
        "darüber. Und doch möchte ich dir unsere Katze Dinah zeigen können. "
        "Ich\n"
        "glaube, du würdest Geschmack für Katzen bekommen, wenn du sie nur "
        "sehen\n"
        "könntest. Sie ist ein so liebes ruhiges Thier,« sprach Alice fort, "
        "halb\n"
        "zu sich selbst, wie sie gemüthlich im Pfuhle daherschwamm; »sie "
        "sitzt\n"
        "und spinnt so nett beim Feuer, leckt sich die Pfoten und wäscht sich "
        "das\n"
        "Schnäuzchen -- und sie ist so hübsch weich auf dem Schoß zu haben -- "
        "und\n"
        "sie ist solch famoser Mäusefänger -- oh, ich bitte um Verzeihung!« "
        "sagte\n"
        "Alice wieder, denn diesmal sträubte sich das ganze Fell der armen "
        "Maus,\n"
        "und Alice dachte, sie müßte sicherlich sehr beleidigt sein. »Wir "
        "wollen\n"
        "nicht mehr davon reden, wenn du es nicht gern hast.«\n"
        "\n"
        "»Wir, wirklich!« entgegnete die Maus, die bis zur Schwanzspitze\n"
        "zitterte. »Als ob ich je über solchen Gegenstand spräche! Unsere "
        "Familie\n"
        "hat von jeher Katzen verabscheut: häßliche, niedrige, gemeine "
        "Dinger!\n"
        "Laß mich ihren Namen nicht wieder hören!«\n"
        "\n"
        "»Nein, gewiß nicht!« sagte Alice, eifrig bemüht, einen andern "
        "Gegenstand\n"
        "der Unterhaltung zu suchen. »Magst du -- magst du gern Hunde?« Die "
        "Maus\n"
        "antwortete nicht, daher fuhr Alice eifrig fort: »Es wohnt ein so\n"
        "reizender kleiner Hund nicht weit von unserm Hause. Den möchte ich "
        "dir\n"
        "zeigen können! Ein kleiner klaräugiger Wachtelhund, weißt du, ach, "
        "mit\n"
        "solch krausem braunen Fell! Und er apportirt Alles, was man ihm\n"
        "hinwirft, und er kann aufrecht stehen und um sein Essen betteln, und "
        "so\n"
        "viel Kunststücke -- ich kann mich kaum auf die Hälfte besinnen -- "
        "und er\n"
        "gehört einem Amtmann, weißt du, und er sagt, er ist so nützlich, er "
        "ist\n"
        "ihm hundert Pfund werth! Er sagt, er vertilgt alle Ratten und -- oh "
        "wie\n"
        "dumm!« sagte Alice in reumüthigem Tone. »Ich fürchte, ich habe ihr\n"
        "wieder weh gethan!« Denn die Maus schwamm so schnell sie konnte von "
        "ihr\n"
        "fort und brachte den Pfuhl dadurch in förmliche Bewegung.\n"
        "\n"
        "Sie rief ihr daher zärtlich nach: »Liebes Mäuschen! Komm wieder "
        "zurück,\n"
        "und wir wollen weder von Katzen noch von Hunden reden, wenn du sie "
        "nicht\n"
        "gern hast!« Als die Maus das hörte, wandte sie sich um und schwamm\n"
        "langsam zu ihr zurück; ihr Gesicht war ganz blaß (vor Aerger, "
        "dachte\n"
        "Alice), und sie sagte mit leiser, zitternder Stimme: »Komm mit mir "
        "an's\n"
        "Ufer, da will ich dir meine Geschichte erzählen; dann wirst du\n"
        "begreifen, warum ich Katzen und Hunde nicht leiden kann.«\n"
        "\n"
        "Es war hohe Zeit sich fortzumachen; denn der Pfuhl begann von "
        "allerlei\n"
        "Vögeln und Gethier zu wimmeln, die hinein gefallen waren: da war "
        "eine\n"
        "Ente und ein Dodo, ein rother Papagei und ein junger Adler, und "
        "mehre\n"
        "andere merkwürdige Geschöpfe. Alice führte sie an, und die ganze\n"
        "Gesellschaft schwamm an's Ufer.\n"
        "\n"
        "Es war in der That eine wunderliche Gesellschaft, die sich am "
        "Strande\n"
        "versammelte -- die Vögel mit triefenden Federn, die übrigen Thiere "
        "mit\n"
        "fest anliegendem Fell, Alle durch und durch naß, verstimmt und\n"
        "unbehaglich. --\n"
        "\n"
        "Die erste Frage war, wie sie sich trocknen könnten: es wurde eine\n"
        "Berathung darüber gehalten, und nach wenigen Minuten kam es Alice\n"
        "ganz natürlich vor, vertraulich mit ihnen zu schwatzen, als ob sie\n"
        "sie ihr ganzes Leben gekannt hätte. Sie hatte sogar eine lange\n"
        "Auseinandersetzung mit dem Papagei, der zuletzt brummig wurde und "
        "nur\n"
        "noch sagte: »ich bin älter als du und muß es besser wissen;« dies "
        "wollte\n"
        "Alice nicht zugeben und fragte nach seinem Alter, und da der Papagei "
        "es\n"
        "durchaus nicht sagen wollte, so blieb die Sache unentschieden.\n"
        "\n"
        "Endlich rief die Maus, welche eine Person von Gewicht unter ihnen "
        "zu\n"
        "sein schien: »Setzt euch, ihr Alle, und hört mir zu! ich will euch "
        "bald\n"
        "genug trocken machen!« Alle setzten sich sogleich in einen großen "
        "Kreis\n"
        "nieder, die Maus in der Mitte. Alice hatte die Augen erwartungsvoll "
        "auf\n"
        "sie gerichtet, denn sie war überzeugt, sie werde sich entsetzlich\n"
        "erkälten, wenn sie nicht sehr bald trocken würde.\n"
        "\n"
        "»Hm!« sagte die Maus mit wichtiger Miene, »seid ihr Alle so weit? Es "
        "ist\n"
        "das Trockenste, worauf ich mich besinnen kann. Alle still, wenn ich\n"
        "bitten darf! -- Wilhelm der Eroberer, dessen Ansprüche vom Papste\n"
        "begünstigt wurden, fand bald Anhang unter den Engländern, die einen\n"
        "Anführer brauchten, und die in jener Zeit sehr an Usurpation und\n"
        "Eroberungen gewöhnt waren. Edwin und Morcar, Grafen von Mercia und\n"
        "Northumbria --«\n"
        "\n"
        "»Oooh!« gähnte der Papagei und schüttelte sich.\n"
        "\n"
        "»Bitte um Verzeihung!« sprach die Maus mit gerunzelter Stirne, aber "
        "sehr\n"
        "höflich; »bemerkten Sie etwas?«\n"
        "\n"
        "»Ich nicht!« erwiederte schnell der Papagei.\n"
        "\n"
        "»Es kam mir so vor,« sagte die Maus. -- »Ich fahre fort: Edwin und\n"
        "Morcar, Grafen von Mercia und Northumbria, erklärten sich für ihn; "
        "und\n"
        "selbst Stigand, der patriotische Erzbischof von Canterbury fand es\n"
        "rathsam --«\n"
        "\n"
        "»Fand was?« unterbrach die Ente.\n"
        "\n"
        "»Fand es,« antwortete die Maus ziemlich aufgebracht: »du wirst doch "
        "wohl\n"
        "wissen, was es bedeutet.«\n"
        "\n"
        "»Ich weiß sehr wohl, was es bedeutet, wenn ich etwas finde«, sagte "
        "die\n"
        "Ente: »es ist gewöhnlich ein Frosch oder ein Wurm. Die Frage ist, "
        "was\n"
        "fand der Erzbischof?«\n"
        "\n"
        "Die Maus beachtete die Frage nicht, sondern fuhr hastig fort: -- "
        "»fand\n"
        "es rathsam, von Edgar Atheling begleitet, Wilhelm entgegen zu gehen "
        "und\n"
        "ihm die Krone anzubieten. Wilhelms Benehmen war zuerst gemäßigt, "
        "aber\n"
        "die Unverschämtheit der Normannen -- wie steht's jetzt, Liebe?« fuhr "
        "sie\n"
        "fort, sich an Alice wendend.\n"
        "\n"
        "»Noch ganz eben so naß,« sagte Alice schwermüthig; »es scheint mich "
        "gar\n"
        "nicht trocken zu machen.«\n"
        "\n"
        "»In dem Fall,« sagte der Dodo feierlich, indem er sich erhob, "
        "»stelle\n"
        "ich den Antrag, daß die Versammlung sich vertage und zur "
        "unmittelbaren\n"
        "Anwendung von wirksameren Mitteln schreite.«\n"
        "\n"
        "»Sprich deutlich!« sagte der Adler. »Ich verstehe den Sinn von "
        "deinen\n"
        "langen Wörtern nicht, und ich wette, du auch nicht!« Und der Adler\n"
        "bückte sich, um ein Lächeln zu verbergen; einige der andern Vögel\n"
        "kicherten hörbar.\n"
        "\n"
        "»Was sich sagen wollte,« sprach der Dodo in gereiztem Tone, »war, "
        "daß\n"
        "das beste Mittel uns zu trocknen ein Caucus-Rennen wäre.«\n"
        "\n"
        "»Was ist ein Caucus-Rennen?« sagte Alice, nicht daß ihr viel daran "
        "lag\n"
        "es zu wissen; aber der Dodo hatte angehalten, als ob er eine Frage\n"
        "erwarte, und Niemand anders schien aufgelegt zu reden.\n"
        "\n"
        "»Nun,« meinte der Dodo, »die beste Art, es zu erklären, ist, es zu\n"
        "spielen.« (Und da ihr vielleicht das Spiel selbst einen\n"
        "Winter-Nachmittag versuchen möchtet, so will ich erzählen, wie der "
        "Dodo\n"
        "es anfing.)\n"
        "\n"
        "Erst bezeichnete er die Bahn, eine Art Kreis (»es kommt nicht genau "
        "auf\n"
        "die Form an,« sagte er), und dann wurde die ganze Gesellschaft hier "
        "und\n"
        "da auf der Bahn aufgestellt. Es wurde kein »eins, zwei drei, fort!«\n"
        "gezählt, sondern sie fingen an zu laufen wenn es ihnen einfiel, "
        "hörten\n"
        "auf wie es ihnen einfiel, so daß es nicht leicht zu entscheiden "
        "war,\n"
        "wann das Rennen zu Ende war. Als sie jedoch ungefähr eine halbe "
        "Stunde\n"
        "gerannt und vollständig getrocknet waren, rief der Dodo plötzlich: "
        "»Das\n"
        "Rennen ist aus!« und sie drängten sich um ihn, außer Athem, mit der\n"
        "Frage: »Aber wer hat gewonnen?«\n"
        "\n"
        "Diese Frage konnte der Dodo nicht ohne tiefes Nachdenken "
        "beantworten,\n"
        "und er saß lange mit einem Finger an die Stirn gelegt (die Stellung, "
        "in\n"
        "der ihr meistens Shakespeare in seinen Bildern seht), während die\n"
        "Uebrigen schweigend auf ihn warteten. Endlich sprach der Dodo: "
        "»Jeder\n"
        "hat gewonnen, und Alle sollen Preise haben.«\n"
        "\n"
        "»Aber wer soll die Preise geben?« fragte ein ganzer Chor von "
        "Stimmen.\n"
        "\n"
        "»Versteht sich, sie!« sagte der Dodo, mit dem Finger auf Alice "
        "zeigend,\n"
        "und sogleich umgab sie die ganze Gesellschaft, Alle durch einander\n"
        "rufend: »Preise Preise!«\n"
        "\n"
        "Alice wußte nicht im Geringsten, was da zu thun sei; in ihrer\n"
        "Verzweiflung fuhr sie mit der Hand in die Tasche und zog eine "
        "Schachtel\n"
        "Zuckerplätzchen hervor (glücklicherweise war das Salzwasser nicht "
        "hinein\n"
        "gedrungen); die vertheilte sie als Preise. Sie reichten gerade "
        "herum,\n"
        "eins für Jeden.\n"
        "\n"
        "»Aber sie selbst muß auch einen Preis bekommen, wißt ihr,« sagte "
        "die\n"
        "Maus.\n"
        "\n"
        "»Versteht sich,« entgegnete der Dodo ernst. »Was hast du noch in "
        "der\n"
        "Tasche?« fuhr er zu Alice gewandt fort.\n"
        "\n"
        "»Nur einen Fingerhut,« sagte Alice traurig.\n"
        "\n"
        "»Reiche ihn mir herüber,« versetzte der Dodo. Darauf versammelten "
        "sich\n"
        "wieder Alle um sie, während der Dodo ihr den Fingerhut feierlich\n"
        "überreichte, mit den Worten: »Wir bitten, Sie wollen uns gütigst mit "
        "der\n"
        "Annahme dieses eleganten Fingerhutes beehren;« und als er diese "
        "kurze\n"
        "Rede beendigt hatte, folgte allgemeines Beifallklatschen.\n"
        "\n"
        "Alice fand dies Alles höchst albern; aber die ganze Gesellschaft sah "
        "so\n"
        "ernst aus, daß sie sich nicht zu lachen getraute, und da ihr keine\n"
        "passende Antwort einfiel, verbeugte sie sich einfach und nahm den\n"
        "Fingerhut ganz ehrbar in Empfang.\n"
        "\n"
        "Nun mußten zunächst die Zuckerplätzchen verzehrt werden, was nicht "
        "wenig\n"
        "Lärm und Verwirrung hervorrief; die großen Vögel nämlich beklagten "
        "sich,\n"
        "daß sie nichts schmecken konnten, die kleinen aber verschluckten "
        "sich\n"
        "und mußten auf den Rücken geklopft werden. Endlich war auch dies\n"
        "vollbracht, und Alle setzten sich im Kreis herum und drangen in das\n"
        "Mäuslein, noch etwas zu erzählen.\n"
        "\n"
        "»Du hast mir deine Geschichte versprochen,« sagte Alice -- »und "
        "woher es\n"
        "kommt, daß du K. und H. nicht leiden kannst,« fügte sie leise hinzu, "
        "um\n"
        "nur das niedliche Thierchen nicht wieder böse zu machen.\n"
        "\n"
        "»Ach,« seufzte das Mäuslein, »ihr macht euch ja aus meinem Erzählen "
        "doch\n"
        "nichts; ich bin euch mit meiner Geschichte zu langschwänzig und zu\n"
        "tragisch.« Dabei sah sie Alice fragend an.\n"
        "\n"
        "»Langschwänzig! das muß wahr sein!« rief Alice und sah nun erst mit\n"
        "rechter Verwunderung auf den geringelten Schwanz der Maus hinab; "
        "»aber\n"
        "wie so tragisch? was trägst du denn?« Während sie noch darüber "
        "nachsann,\n"
        "fing die langschwänzige Erzählung schon an, folgendergestalt:\n"
        "\n"
        "                        Filax sprach zu\n"
        "                            der Maus, die\n"
        "                                    er traf\n"
        "                                      in dem\n"
        "                                        Haus:\n"
        "                                     »Geh' mit\n"
        "                                    mir vor\n"
        "                                   Gericht,\n"
        "                                 daß ich\n"
        "                               dich\n"
        "                              verklage.\n"
        "                              Komm und\n"
        "                               wehr' dich\n"
        "                                 nicht mehr;\n"
        "                                  ich muß\n"
        "                                   haben ein\n"
        "                                      Verhör,\n"
        "                                     denn ich\n"
        "                                       habe\n"
        "                                    nichts\n"
        "                                   zu thun\n"
        "                                  schon\n"
        "                                 zwei\n"
        "                                Tage.«\n"
        "                              Sprach die\n"
        "                             Maus zum\n"
        "                            Köter:\n"
        "                           »Solch\n"
        "                             Verhör,\n"
        "                              lieber Herr,\n"
        "                                 ohne\n"
        "                                Richter,\n"
        "                              ohne\n"
        "                            Zeugen\n"
        "                             thut nicht\n"
        "                                  Noth.«\n"
        "                                 »Ich bin\n"
        "                                Zeuge,\n"
        "                              ich bin\n"
        "                             Richter,«\n"
        "                            sprach\n"
        "                             er schlau\n"
        "                               und schnitt\n"
        "                                Gesichter,\n"
        "                                »das Verhör\n"
        "                                 leite ich\n"
        "                                      und\n"
        "                                 verdamme\n"
        "                                dich\n"
        "                              zum\n"
        "                                Tod!«\n"
        "\n"
        "»Du paßt nicht auf!« sagte die Maus strenge zu Alice. »Woran denkst "
        "du?«\n"
        "\n"
        "»Ich bitte um Verzeihung,« sagte Alice sehr bescheiden: »du warst "
        "bis\n"
        "zur fünften Biegung gekommen, glaube ich?«\n"
        "\n"
        "»Mit nichten!« sagte die Maus entschieden und sehr ärgerlich.\n"
        "\n"
        "»Nichten!« rief Alice, die gern neue Bekanntschaften machte, und "
        "sah\n"
        "sich neugierig überall um. »O, wo sind sie, deine Nichten? Laß mich\n"
        "gehen und sie her holen!«\n"
        "\n"
        "»Das werde ich schön bleiben lassen,« sagte die Maus, indem sie "
        "aufstand\n"
        "und fortging. »Deinen Unsinn kann ich nicht mehr mit anhören!«\n"
        "\n"
        "»Ich meinte es nicht böse!« entschuldigte sich die arme Alice. »Aber "
        "du\n"
        "bist so sehr empfindlich, du!«\n"
        "\n"
        "Das Mäuslein brummte nur als Antwort.\n"
        "\n"
        "»Bitte, komm wieder, und erzähle deine Geschichte aus!« rief Alice "
        "ihr\n"
        "nach; und die Andern wiederholten im Chor: »ja bitte!« aber das "
        "Mäuschen\n"
        "schüttelte unwillig mit dem Kopfe und ging schnell fort.\n"
        "\n"
        "»Wie schade, daß es nicht bleiben wollte!« seufzte der Papagei, "
        "sobald\n"
        "es nicht mehr zu sehen war; und eine alte Unke nahm die Gelegenheit\n"
        "wahr, zu ihrer Tochter zu sagen, »Ja, mein Kind! laß dir dies eine "
        "Lehre\n"
        "sein, niemals übler Laune zu sein!« »Halt den Mund, Mama!« sagte "
        "die\n"
        "junge Unke, etwas naseweis.\n"
        "\n"
        "»Wahrhaftig, du würdest die Geduld einer Auster erschöpfen!«\n"
        "\n"
        "»Ich wünschte, ich hätte unsere Dinah hier, das wünschte ich!« "
        "sagte\n"
        "Alice laut, ohne Jemand insbesondere anzureden. »Sie würde sie bald\n"
        "zurückholen!«\n"
        "\n"
        "»Und wer ist Dinah, wenn ich fragen darf?« sagte der Papagei.\n"
        "\n"
        "Alice antwortete eifrig, denn sie sprach gar zu gern von ihrem "
        "Liebling:\n"
        "»Dinah ist unsere Katze, und sie ist euch so geschickt im "
        "Mäusefangen,\n"
        "ihr könnt's euch gar nicht denken! Und ach, hättet ihr sie nur "
        "Vögel\n"
        "jagen sehen. Ich sage euch, sie frißt einen kleinen Vogel, so wie "
        "sie\n"
        "ihn zu Gesicht bekommt.«\n"
        "\n"
        "Diese Mittheilung verursachte große Aufregung in der Gesellschaft.\n"
        "Einige der Vögel machten sich augenblicklich davon; eine alte "
        "Elster\n"
        "fing an, sich sorgfältig einzuwickeln, indem sie bemerkte: »Ich muß\n"
        "wirklich nach Hause gehen; die Nachtluft ist nicht gut für meinen "
        "Hals!«\n"
        "und ein Canarienvogel piepte zitternd zu seinen Kleinen, »Kommt "
        "fort,\n"
        "Kinder! es ist die höchste Zeit für euch, zu Bett zu gehen!« Unter\n"
        "verschiedenen Entschuldigungen entfernten sie sich Alle, und Alice "
        "war\n"
        "bald ganz allein.\n"
        "\n"
        "»Hätte ich nur Dinah nicht erwähnt!« sprach sie bei sich mit "
        "betrübtem\n"
        "Tone. »Niemand scheint sie gern zu haben, hier unten, und dabei ist "
        "sie\n"
        "doch die beste Katze von der Welt! Oh, meine liebe Dinah! ob ich "
        "dich\n"
        "wohl je wieder sehen werde!« dabei fing die arme Alice von Neuem zu\n"
        "weinen an, denn sie fühlte sich gar zu einsam und muthlos. Nach "
        "einem\n"
        "Weilchen jedoch hörte sie wieder ein Trappeln von Schritten in der\n"
        "Entfernung und blickte aufmerksam hin, halb in der Hoffnung, daß "
        "die\n"
        "Maus sich besonnen habe und zurückkomme, ihre Geschichte "
        "auszuerzählen.\n"
        "\n"
        "Es war das weiße Kaninchen, das langsam zurückgewandert kam, indem "
        "es\n"
        "sorgfältig beim Gehen umhersah, als ob es etwas verloren hätte, und "
        "sie\n"
        "hörte wie es für sich murmelte: »die Herzogin! die Herzogin! Oh, "
        "meine\n"
        "weichen Pfoten! o mein Fell und Knebelbart! Sie wird mich hängen "
        "lassen,\n"
        "so gewiß Frettchen Frettchen sind! Wo ich sie kann haben fallen "
        "lassen,\n"
        "begreife ich nicht!« Alice errieth augenblicklich, daß es den Fächer "
        "und\n"
        "die weißen Glaceehandschuhe meinte, und gutmüthig genug fing sie "
        "an,\n"
        "danach umher zu suchen, aber sie waren nirgends zu sehen -- Alles "
        "schien\n"
        "seit ihrem Bade in dem Pfuhl verwandelt zu sein, und der große "
        "Corridor\n"
        "mit dem Glastische und der kleinen Thür war gänzlich verschwunden.\n"
        "\n"
        "Das Kaninchen erblickte Alice bald, und wie sie überall suchte, rief "
        "es\n"
        "ihr ärgerlich zu: »Was, Marianne, was hast du hier zu schaffen? "
        "Renne\n"
        "augenblicklich nach Hause, und hole mir ein Paar Handschuhe und "
        "einen\n"
        "Fächer! Schnell, vorwärts!« Alice war so erschrocken, daß sie "
        "schnell in\n"
        "der angedeuteten Richtung fortlief, ohne ihm zu erklären, daß es "
        "sich\n"
        "versehen habe.\n"
        "\n"
        "»Es hält mich für sein Hausmädchen,« sprach sie bei sich selbst und "
        "lief\n"
        "weiter. »Wie es sich wundern wird, wenn es erfährt, wer ich bin! "
        "Aber\n"
        "ich will ihm lieber seinen Fächer und seine Handschuhe bringen\n"
        "-- nämlich, wenn ich sie finden kann.« Wie sie so sprach, kam sie an "
        "ein\n"
        "nettes kleines Haus, an dessen Thür ein glänzendes Messingschild war "
        "mit\n"
        "dem Namen »W. Kaninchen« darauf. Sie ging hinein ohne anzuklopfen, "
        "lief\n"
        "die Treppe hinauf, in großer Angst, der wirklichen Marianne zu "
        "begegnen\n"
        "und zum Hause hinausgewiesen zu werden, ehe sie den Fächer und die\n"
        "Handschuhe gefunden hätte.\n"
        "\n"
        "»Wie komisch es ist,« sagte Alice bei sich, »Besorgungen für ein\n"
        "Kaninchen zu machen! Vermuthlich wird mir Dinah nächstens Aufträge\n"
        "geben!« Und sie dachte sich schon aus, wie es Alles kommen würde:\n"
        "\n"
        "»Fräulein Alice! Kommen Sie gleich, es ist Zeit zum Ausgehen für "
        "Sie!«\n"
        "»Gleich Kinderfrau! aber ich muß dieses Mäuseloch hier bewachen bis\n"
        "Dinah wiederkommt, und aufpassen, daß die Maus nicht herauskommt.« "
        "»Nur\n"
        "würde Dinah,« dachte Alice weiter, »gewiß nicht im Hause bleiben "
        "dürfen,\n"
        "wenn sie anfinge, die Leute so zu commandiren.«\n"
        "\n"
        "Mittlerweile war sie in ein sauberes kleines Zimmer gelangt, mit "
        "einem\n"
        "Tisch vor dem Fenster und darauf (wie sie gehofft hatte) ein Fächer "
        "und\n"
        "zwei oder drei Paar winziger weißer Glaceehandschuhe; sie nahm den\n"
        "Fächer und ein Paar Handschuhe und wollte eben das Zimmer verlassen, "
        "als\n"
        "ihr Blick auf ein Fläschchen fiel, das bei dem Spiegel stand. "
        "Diesmal\n"
        "war kein Zettel mit den Worten: »Trink mich« darauf, aber trotzdem "
        "zog\n"
        "sie den Pfropfen heraus und setzte es an die Lippen. »Ich weiß, "
        "etwas\n"
        "Merkwürdiges muß geschehen, sobald ich esse oder trinke; drum will "
        "ich\n"
        "versuchen, was dies Fläschchen thut. Ich hoffe, es wird mich wieder\n"
        "größer machen; denn es ist mir sehr langweilig, solch winzig "
        "kleines\n"
        "Ding zu sein!«\n"
        "\n"
        "Richtig, und zwar schneller, als sie erwartete: ehe sie das "
        "Fläschchen\n"
        "halb ausgetrunken hatte fühlte sie, wie ihr Kopf an die Decke "
        "stieß,\n"
        "und mußte sich rasch bücken, um sich nicht den Hals zu brechen. Sie\n"
        "stellte die Flasche hin, indem sie zu sich sagte: »Das ist ganz "
        "genug --\n"
        "ich hoffe, ich werde nicht weiter wachsen -- ich kann so schon nicht "
        "zur\n"
        "Thüre hinaus -- hätte ich nur nicht so viel getrunken!«\n"
        "\n"
        "O weh! es war zu spät, dies zu wünschen. Sie wuchs und wuchs, und "
        "mußte\n"
        "sehr bald auf den Fußboden niederknien; den nächsten Augenblick war\n"
        "selbst dazu nicht Platz genug, sie legte sich nun hin, mit einem\n"
        "Ellbogen gegen die Thür gestemmt und den andern Arm unter dem "
        "Kopfe.\n"
        "Immer noch wuchs sie, und als letzte Hülfsquelle streckte sie einen "
        "Arm\n"
        "zum Fenster hinaus und einen Fuß in den Kamin hinauf, und sprach zu "
        "sich\n"
        "selbst: »Nun kann ich nicht mehr thun, was auch geschehen mag. Was "
        "wird\n"
        "nur aus mir werden?«\n"
        "\n"
        "Zum Glück für Alice hatte das Zauberfläschchen nun seine volle "
        "Wirkung\n"
        "gehabt, und sie wuchs nicht weiter. Aber es war sehr unbequem, und "
        "da\n"
        "durchaus keine Aussicht war, daß sie je wieder aus dem Zimmer "
        "hinaus\n"
        "komme, so war sie natürlich sehr unglücklich.\n"
        "\n"
        "»Es war viel besser zu Hause,« dachte die arme Alice, »wo man nicht\n"
        "fortwährend größer und kleiner wurde, und sich nicht von Mäusen und\n"
        "Kaninchen commandiren zu lassen brauchte. Ich wünschte fast, ich "
        "wäre\n"
        "nicht in den Kaninchenbau hineingelaufen -- aber -- aber, es ist "
        "doch\n"
        "komisch, diese Art Leben! Ich möchte wohl wissen, was eigentlich mit "
        "mir\n"
        "vorgegangen ist! Wenn ich Märchen gelesen habe, habe ich immer "
        "gedacht,\n"
        "so etwas käme nie vor, nun bin ich mitten drin in einem! Es sollte "
        "ein\n"
        "Buch von mir geschrieben werden, und wenn ich groß bin, will ich "
        "eins\n"
        "schreiben -- aber ich bin ja jetzt groß,« sprach sie betrübt "
        "weiter,\n"
        "»wenigstens hier habe ich keinen Platz übrig, noch größer zu "
        "werden.«\n"
        "\n"
        "»Aber,« dachte Alice, »werde ich denn nie älter werden, als ich "
        "jetzt\n"
        "bin? das ist ein Trost -- nie eine alte Frau zu sein -- aber dann "
        "--\n"
        "immer Aufgaben zu lernen zu haben! Oh, das möchte ich nicht gern!«\n"
        "\n"
        "»O, du einfältige Alice,« schalt sie sich selbst. »Wie kannst du "
        "hier\n"
        "Aufgaben lernen? Sieh doch, es ist kaum Platz genug für dich, viel\n"
        "weniger für irgend ein Schulbuch!«\n"
        "\n"
        "Und so redete sie fort; erst als eine Person, dann die andere, und "
        "hatte\n"
        "so eine lange Unterhaltung mit sich selbst; aber nach einigen "
        "Minuten\n"
        "hörte sie draußen eine Stimme und schwieg still, um zu horchen.\n"
        "\n"
        "»Marianne! Marianne!« sagte die Stimme, »hole mir gleich meine\n"
        "Handschuhe!« dann kam ein Trappeln von kleinen Füßen die Treppe "
        "herauf.\n"
        "Alice wußte, daß es das Kaninchen war, das sie suchte, und sie "
        "zitterte\n"
        "so sehr, daß sie das ganze Haus erschütterte; sie hatte ganz "
        "vergessen,\n"
        "daß sie jetzt wohl tausend Mal so groß wie das Kaninchen war und "
        "keine\n"
        "Ursache hatte, sich vor ihm zu fürchten.\n"
        "\n"
        "Jetzt kam das Kaninchen an die Thür und wollte sie aufmachen; da "
        "aber\n"
        "die Thür nach innen aufging und Alice's Ellbogen fest dagegen "
        "gestemmt\n"
        "war, so war es ein vergeblicher Versuch. Alice hörte, wie es zu "
        "sich\n"
        "selbst sprach: »dann werde ich herum gehen und zum Fenster\n"
        "hineinsteigen.«\n"
        "\n"
        "»Das wirst du nicht thun,« dachte Alice, und nachdem sie gewartet "
        "hatte,\n"
        "bis sie das Kaninchen dicht unter dem Fenster zu hören glaubte, "
        "streckte\n"
        "sie mit einem Male ihre Hand aus und griff in die Luft. Sie faßte "
        "zwar\n"
        "nichts, hörte aber einen schwachen Schrei und einen Fall, dann das\n"
        "Geklirr von zerbrochenem Glase, woraus sie schloß, daß es "
        "wahrscheinlich\n"
        "in ein Gurkenbeet gefallen sei, oder etwas dergleichen.\n"
        "\n"
        "Demnächst kam eine ärgerliche Stimme -- die des Kaninchens -- »Pat! "
        "Pat!\n"
        "wo bist du?« und dann eine Stimme, die sie noch nicht gehört hatte: "
        "»Wo\n"
        "soll ich sind? ich bin hier! grabe Aepfel aus, Euer Jnaden!«\n"
        "\n"
        "»Aepfel ausgraben? so!« sagte das Kaninchen ärgerlich. »Hier! komm "
        "und\n"
        "hilf mir heraus!« (Noch mehr Geklirr von Glasscherben.)\n"
        "\n"
        "»Nun sage mir, Pat, was ist das da oben im Fenster?«\n"
        "\n"
        "»Wat soll's sind? 's is en Arm, Euer Jnaden!« (Er sprach es »Arrum«\n"
        "aus.)\n"
        "\n"
        "»Ein Arm, du Esel! Wer hat je einen so großen Arm gesehen? er nimmt "
        "ja\n"
        "das ganze Fenster ein!«\n"
        "\n"
        "»Zu dienen, des thut er, Euer Jnaden; aber en Arm is es, und en Arm\n"
        "bleebt es.«\n"
        "\n"
        "»Jedenfalls hat er da nichts zu suchen: geh' und schaffe ihn fort!«\n"
        "\n"
        "Darauf folgte eine lange Pause, während welcher Alice sie nur "
        "einzelne\n"
        "Worte flüstern hörte, wie: »Zu dienen, des scheint mer nich, Euer\n"
        "Jnaden, jar nich, jar nich!« »Thu', was ich dir sage, feige Memme!«\n"
        "zuletzt streckte sie die Hand wieder aus und that einen Griff in "
        "die\n"
        "Luft. Diesmal hörte sie ein leises Wimmern und noch mehr Geklirr "
        "von\n"
        "Glasscherben. »Wie viel Gurkenbeete da sein müssen!« dachte Alice. "
        "»Mich\n"
        "soll doch wundern, was sie nun thun werden! Mich zum Fenster hinaus\n"
        "ziehen? ja, wenn sie das nur könnten! Ich bliebe wahrlich nicht "
        "gern\n"
        "länger hier!«\n"
        "\n"
        "Sie wartete eine Zeit lang, ohne etwas zu hören; endlich kam ein "
        "Rollen\n"
        "von kleinen Leiterwagen, und ein Lärm von einer Menge Stimmen, alle\n"
        "durcheinander; sie verstand die Worte: »Wo ist die andere Leiter? -- "
        "Ich\n"
        "sollte ja nur eine bringen; Wabbel hat die andere -- Wabbel, bringe "
        "sie\n"
        "her, Junge! -- Lehnt sie hier gegen diese Ecke -- Nein, sie müssen "
        "erst\n"
        "zusammengebunden werden -- sie reichen nicht halb hinauf -- Ach, "
        "was\n"
        "werden sie nicht reichen: seid nicht so umständlich -- Hier, "
        "Wabbel!\n"
        "fange den Strick -- Wird das Dach auch tragen? -- Nimm dich mit dem "
        "losen\n"
        "Schiefer in Acht -- oh, da fällt er! Köpfe weg!« (ein lautes "
        "Krachen) --\n"
        "»Wessen Schuld war das? -- Wabbel's, glaube ich -- Wer soll in den\n"
        "Schornstein steigen? -- Ich nicht, so viel weiß ich! Ihr aber doch,\n"
        "nicht wahr? -- Nicht ich, meiner Treu! -- Wabbel kann hineinsteigen "
        "--\n"
        "Hier, Wabbel! der Herr sagt, du sollst in den Schornstein steigen!«\n"
        "\n"
        "»So, also Wabbel soll durch den Schornstein hereinkommen, "
        "wirklich?«\n"
        "sagte Alice zu sich selbst. »Sie scheinen mir Alles auf Wabbel zu\n"
        "schieben: ich möchte um Alles nicht an Wabbel's Stelle sein; der "
        "Kamin\n"
        "ist freilich eng, aber etwas werde ich doch wohl mit dem Fuße\n"
        "ausschlagen können!«\n"
        "\n"
        "Sie zog ihren Fuß so weit herunter, wie sie konnte, und wartete, bis "
        "sie\n"
        "ein kleines Thier (sie konnte nicht rathen, was für eine Art es sei) "
        "in\n"
        "dem Schornstein kratzen und klettern hörte; als es dicht über ihr "
        "war,\n"
        "sprach sie bei sich: »Dies ist Wabbel,« gab einen kräftigen Stoß in "
        "die\n"
        "Höhe, und wartete dann der Dinge, die da kommen würden.\n"
        "\n"
        "Zuerst hörte sie einen allgemeinen Chor: »Da fliegt Wabbel!« dann "
        "die\n"
        "Stimme des Kaninchens allein: -- »Fangt ihn auf, ihr da bei der "
        "Hecke!«\n"
        "darauf Stillschweigen, dann wieder verworrene Stimmen: -- »Haltet "
        "ihm\n"
        "den Kopf -- etwas Branntwein -- Ersticke ihn doch nicht -- Wie "
        "geht's,\n"
        "alter Kerl? Was ist dir denn geschehen? erzähle uns Alles!«\n"
        "\n"
        "Zuletzt kam eine kleine schwache, quiekende Stimme (»das ist "
        "Wabbel,«\n"
        "dachte Alice): »Ich weiß es ja selbst nicht -- Keinen mehr, danke! "
        "Ich\n"
        "bin schon viel besser -- aber ich bin viel zu aufgeregt, um euch zu\n"
        "erzählen -- Ich weiß nur, da kommt ein Ding in die Höhe, wie'n\n"
        "Dosen-Stehauf, und auf fliege ich wie 'ne Rackete!«\n"
        "\n"
        "»Ja, das hast du gethan, alter Kerl!« sagten die Andern.\n"
        "\n"
        "»Wir müssen das Haus niederbrennen!« rief das Kaninchen; da schrie "
        "Alice\n"
        "so laut sie konnte: »Wenn ihr das thut, werde ich Dinah über euch\n"
        "schicken!«\n"
        "\n"
        "Sogleich entstand tiefes Schweigen, und Alice dachte bei sich: »Was "
        "sie\n"
        "wohl jetzt thun werden? Wenn sie Menschenverstand hätten, würden sie "
        "das\n"
        "Dach abreißen.« Nach einer oder zwei Minuten fingen sie wieder an "
        "sich\n"
        "zu rühren, und Alice hörte das Kaninchen sagen: »Eine Karre voll ist "
        "vor\n"
        "der Hand genug.«\n"
        "\n"
        "»Eine Karre voll was?« dachte Alice; doch blieb sie nicht lange im\n"
        "Zweifel, denn den nächsten Augenblick kam ein Schauer von kleinen\n"
        "Kieseln zum Fenster herein geflogen, von denen ein Paar sie gerade "
        "in's\n"
        "Gesicht trafen. »Dem will ich ein Ende machen,« sagte sie bei sich "
        "und\n"
        "schrie hinaus: »Das laßt mir gefälligst bleiben!« worauf wieder "
        "tiefe\n"
        "Stille erfolgte.\n"
        "\n"
        "Alice bemerkte mit einigem Erstaunen, daß die Kiesel sich alle in "
        "kleine\n"
        "Kuchen verwandelten, als sie auf dem Boden lagen, und dies brachte "
        "sie\n"
        "auf einen glänzenden Gedanken. »Wenn ich einen von diesen Kuchen "
        "esse,«\n"
        "dachte sie, »wird es gewiß meine Größe verändern; und da ich "
        "unmöglich\n"
        "noch mehr wachsen kann, so wird es mich wohl kleiner machen, "
        "vermuthe\n"
        "ich.«\n"
        "\n"
        "Sie schluckte demnach einen kleinen Kuchen herunter, und merkte zu "
        "ihrem\n"
        "Entzücken, daß sie sogleich abnahm. Sobald sie klein genug war, um "
        "durch\n"
        "die Thür zu gehen, rannte sie zum Hause hinaus, und fand einen\n"
        "förmlichen Auflauf von kleinen Thieren und Vögeln davor. Die arme "
        "kleine\n"
        "Eidechse, Wabbel, war in der Mitte, von zwei Meerschweinchen\n"
        "unterstützt, die ihm etwas aus einer Flasche gaben. Es war ein\n"
        "allgemeiner Sturm auf Alice, sobald sie sich zeigte; sie lief aber "
        "so\n"
        "schnell sie konnte davon, und kam sicher in ein dichtes Gebüsch.\n"
        "\n"
        "»Das Nöthigste, was ich nun zu thun habe,« sprach Alice bei sich, "
        "wie\n"
        "sie in dem Wäldchen umher wanderte, »ist, meine richtige Größe zu\n"
        "erlangen; und das Zweite, den Weg zu dem wunderhübschen Garten zu\n"
        "finden. Ja, das wird der beste Plan sein.«\n"
        "\n"
        "Es klang freilich wie ein vortrefflicher Plan, und recht nett und\n"
        "einfach ausgedacht; die einzige Schwierigkeit war, daß sie nicht "
        "den\n"
        "geringsten Begriff hatte, wie sie ihn ausführen sollte; und während "
        "sie\n"
        "so ängstlich zwischen den Bäumen umherguckte, hörte sie plötzlich "
        "ein\n"
        "scharfes feines Bellen gerade über ihrem Kopfe und sah eilig auf.\n"
        "\n"
        "Ein ungeheuer großer junger Hund sah mit seinen hervorstehenden "
        "runden\n"
        "Augen auf sie herab und machte einen schwachen Versuch, eine Pfote\n"
        "auszustrecken und sie zu berühren. »Armes kleines Ding!« sagte Alice "
        "in\n"
        "liebkosendem Tone, und sie gab sich alle Mühe, ihm zu pfeifen; "
        "dabei\n"
        "hatte sie aber große Angst, ob er auch nicht hungrig wäre, denn "
        "dann\n"
        "würde er sie wahrscheinlich auffressen trotz allen Liebkosungen.\n"
        "\n"
        "Ohne recht zu wissen was sie that, nahm sie ein Stäbchen auf und "
        "hielt\n"
        "es ihm hin; worauf das ungeschickte Thierchen mit allen vier Füßen\n"
        "zugleich in die Höhe sprang, vor Entzücken laut aufbellte, auf das\n"
        "Stäbchen losrannte und that, als wolle es es zerreißen; da wich "
        "Alice\n"
        "ihm aus hinter eine große Distel, um nicht zertreten zu werden; und "
        "so\n"
        "wie sie auf der andern Seite hervorkam, lief der junge Hund wieder "
        "auf\n"
        "das Stäbchen los und fiel kopfüber in seiner Eile, es zu fangen. "
        "Alice,\n"
        "der es vorkam, als wenn Jemand mit einem Fuhrmannspferde Zeck "
        "spielt,\n"
        "und die jeden Augenblick fürchtete, unter seine Füße zu gerathen, "
        "lief\n"
        "wieder hinter die Distel; da machte der junge Hund eine Reihe von "
        "kurzen\n"
        "Anläufen auf das Stäbchen, wobei er jedes Mal ein klein wenig "
        "vorwärts\n"
        "und ein gutes Stück zurück rannte und sich heiser bellte, bis er "
        "sich\n"
        "zuletzt mit zum Munde heraushängender Zunge und halb geschlossenen\n"
        "Augen, ganz außer Athem hinsetzte.\n"
        "\n"
        "Dies schien Alice eine gute Gelegenheit zu sein, fortzukommen; sie\n"
        "machte sich also gleich davon, und rannte bis sie ganz müde war und\n"
        "keine Luft mehr hatte, und bis das Bellen nur noch ganz schwach in "
        "der\n"
        "Ferne zu hören war.\n"
        "\n"
        "»Und doch war es ein lieber kleiner Hund!« sagte Alice, indem sie "
        "sich\n"
        "an eine Butterblume lehnte um auszuruhen, und sich mit einem der "
        "Blätter\n"
        "fächelte. »Ich hätte ihn gern Kunststücke gelehrt, wenn -- wenn ich "
        "nur\n"
        "groß genug dazu gewesen wäre! O ja! das hätte ich beinah vergessen, "
        "ich\n"
        "muß ja machen, daß ich wieder wachse! Laß sehen -- wie fängt man es "
        "doch\n"
        "an? Ich dächte, ich sollte irgend etwas essen oder trinken; aber "
        "die\n"
        "Frage ist, was?«\n"
        "\n"
        "Das war in der That die Frage. Alice blickte um sich nach allen "
        "Blumen\n"
        "und Grashalmen; aber gar nichts sah aus, als ob es das Rechte sei, "
        "das\n"
        "sie unter den Umständen essen oder trinken müsse. In der Nähe wuchs "
        "ein\n"
        "großer Pilz, ungefähr so hoch wie sie; nachdem sie ihn sich von "
        "unten,\n"
        "von beiden Seiten, rückwärts und vorwärts betrachtet hatte, kam es "
        "ihr\n"
        "in den Sinn zu sehen, was oben darauf sei. Sie stellte sich also auf "
        "die\n"
        "Fußspitzen und guckte über den Rand des Pilzes, und sogleich "
        "begegnete\n"
        "ihr Blick dem einer großen blauen Raupe, die mit kreuzweise "
        "gelegten\n"
        "Armen da saß und ruhig aus einer großen Huhka rauchte, ohne die\n"
        "geringste Notiz von ihr noch sonst irgend Etwas zu nehmen.\n"
        "\n"
        "Die Raupe und Alice sahen sich eine Zeit lang schweigend an; "
        "endlich\n"
        "nahm die Raupe die Huhka aus dem Munde und redete sie mit "
        "schmachtender,\n"
        "langsamer Stimme an. »Wer bist du?« fragte die Raupe.\n"
        "\n"
        "Das war kein sehr ermuthigender Anfang einer Unterhaltung. Alice\n"
        "antwortete, etwas befangen: »Ich -- ich weiß nicht recht, diesen\n"
        "Augenblick -- vielmehr ich weiß, wer ich heut früh war, als ich\n"
        "aufstand; aber ich glaube, ich muß seitdem ein paar Mal verwechselt\n"
        "worden sein.«\n"
        "\n"
        "»Was meinst du damit?« sagte die Raupe strenge. »Erkläre dich\n"
        "deutlicher!«\n"
        "\n"
        "»Ich kann mich nicht deutlicher erklären, fürchte ich, Raupe,« "
        "sagte\n"
        "Alice, »weil ich nicht ich bin, sehen Sie wohl?«\n"
        "\n"
        "»Ich sehe nicht wohl,« sagte die Raupe.\n"
        "\n"
        "»Ich kann es wirklich nicht besser ausdrücken,« erwiederte Alice "
        "sehr\n"
        "höflich, »denn ich kann es selbst nicht begreifen; und wenn man an "
        "einem\n"
        "Tage so oft klein und groß wird, wird man ganz verwirrt.«\n"
        "\n"
        "»Nein, das wird man nicht,« sagte die Raupe.\n"
        "\n"
        "»Vielleicht haben Sie es noch nicht versucht,« sagte Alice, »aber "
        "wenn\n"
        "Sie sich in eine Puppe verwandeln werden, das müssen Sie über kurz "
        "oder\n"
        "lang wie Sie wissen -- und dann in einen Schmetterling, das wird "
        "sich\n"
        "doch komisch anfühlen, nicht wahr?«\n"
        "\n"
        "»Durchaus nicht,« sagte die Raupe.\n"
        "\n"
        "»Sie fühlen wahrscheinlich anders darin,« sagte Alice; »so viel "
        "weiß\n"
        "ich, daß es mir sehr komisch sein würde.«\n"
        "\n"
        "»Dir!« sagte die Raupe verächtlich. »Wer bist du denn?«\n"
        "\n"
        "Was sie wieder auf den Anfang der Unterhaltung zurückbrachte. Alice "
        "war\n"
        "etwas ärgerlich, daß die Raupe so sehr kurz angebunden war; sie warf "
        "den\n"
        "Kopf in die Höhe und sprach sehr ernst: »Ich dächte, Sie sollten "
        "mir\n"
        "erst sagen, wer Sie sind?«\n"
        "\n"
        "»Weshalb?« fragte die Raupe.\n"
        "\n"
        "Das war wieder eine schwierige Frage; und da sich Alice auf keinen "
        "guten\n"
        "Grund besinnen konnte und die Raupe sehr schlechter Laune zu sein\n"
        "schien, so ging sie ihrer Wege.\n"
        "\n"
        "»Komm zurück!« rief ihr die Raupe nach, »ich habe dir etwas "
        "Wichtiges zu\n"
        "sagen!«\n"
        "\n"
        "Das klang sehr einladend; Alice kehrte wieder um und kam zu ihr "
        "zurück.\n"
        "\n"
        "»Sei nicht empfindlich,« sagte die Raupe.\n"
        "\n"
        "»Ist das Alles?« fragte Alice, ihren Aerger so gut sie konnte\n"
        "verbergend.\n"
        "\n"
        "»Nein,« sagte die Raupe.\n"
        "\n"
        "Alice dachte, sie wollte doch warten, da sie sonst nichts zu thun "
        "habe,\n"
        "und vielleicht würde sie ihr etwas sagen, das der Mühe werth sei. "
        "Einige\n"
        "Minuten lang rauchte die Raupe fort ohne zu reden; aber zuletzt nahm "
        "sie\n"
        "die Huhka wieder aus dem Munde und sprach: »Du glaubst also, du "
        "bist\n"
        "verwandelt?«\n"
        "\n"
        "»Ich fürchte es fast, Raupe,« sagte Alice, »ich kann Sachen nicht\n"
        "behalten wie sonst, und ich werde alle zehn Minuten größer oder\n"
        "kleiner!«\n"
        "\n"
        "»Kannst welche Sachen nicht behalten?« fragte die Raupe.\n"
        "\n"
        "»Ach, ich habe versucht zu sagen: Bei einem Wirthe &c.; aber es kam "
        "ganz\n"
        "anders!« antwortete Alice in niedergeschlagenem Tone.\n"
        "\n"
        "»Sage her: Ihr seid alt, Vater Martin,« sagte die Raupe.\n"
        "\n"
        "Alice faltete die Hände und fing an: --\n"
        "\n"
        "    »Ihr seid alt, Vater Martin,« so sprach Junker Tropf,\n"
        "    »Euer Haar ist schon lange ganz weiß;\n"
        "    Doch steht ihr so gerne noch auf dem Kopf.\n"
        "    Macht Euch denn das nicht zu heiß?«\n"
        "\n"
        "    »Als ich jung war,« der Vater zur Antwort gab,\n"
        "    »Da glaubt' ich, für's Hirn sei's nicht gut;\n"
        "    Doch seit ich entdeckt, daß ich gar keines hab'\n"
        "    So thu' ich's mit fröhlichem Muth.«\n"
        "\n"
        "    »Ihr seid alt,« sprach der Sohn, »wie vorhin schon gesagt,\n"
        "    Und geworden ein gar dicker Mann;\n"
        "    Drum sprecht, wie ihr rücklings den Purzelbaum schlagt.\n"
        "    Potz tausend! wie fangt ihr's nur an?«\n"
        "\n"
        "    »Als ich jung war,« der Alte mit Kopfschütteln sagt',\n"
        "    »Da rieb ich die Glieder mir ein\n"
        "    Mit der Salbe hier, die sie geschmeidig macht.\n"
        "    Für zwei Groschen Courant ist sie dein.«\n"
        "\n"
        "    »Ihr seid alt,« sprach der Bub', »und könnt nicht recht kau'n,\n"
        "    Und solltet euch nehmen in Acht;\n"
        "    Doch aßt ihr die Ganz mit Schnabel und Klau'n;\n"
        "    Wie habt ihr das nur gemacht?«\n"
        "\n"
        "    »Ich war früher Jurist und hab' viel disputirt,\n"
        "    Besonders mit meiner Frau;\n"
        "    Das hat so mir die Kinnbacken einexercirt,\n"
        "    Daß ich jetzt noch mit Leichtigkeit kau!«\n"
        "\n"
        "    »Ihr seid alt,« sagte der Sohn, »und habt nicht viel Witz,\n"
        "    Und doch seid ihr so geschickt;\n"
        "    Balancirt einen Aal auf der Nasenspitz'!\n"
        "    Wie ist euch das nur geglückt?«\n"
        "\n"
        "    »Drei Antworten hast du, und damit genug,\n"
        "    Nun laß mich kein Wort mehr hören;\n"
        "    Du Guck in die Welt thust so überklug,\n"
        "    Ich werde dich Mores lehren!«\n"
        "\n"
        "»Das ist nicht richtig,« sagte die Raupe.\n"
        "\n"
        "»Nicht ganz richtig, glaube ich,« sagte Alice schüchtern; »manche "
        "Wörter\n"
        "sind anders gekommen.«\n"
        "\n"
        "»Es ist von Anfang bis zu Ende falsch,« sagte die Raupe mit\n"
        "Entschiedenheit, worauf eine Pause von einigen Minuten eintrat.\n"
        "\n"
        "Die Raupe sprach zuerst wieder.\n"
        "\n"
        "»Wie groß möchtest du gern sein?« fragte sie.\n"
        "\n"
        "»Oh, es kommt nicht so genau darauf an,« erwiederte Alice schnell; "
        "»nur\n"
        "das viele Wechseln ist nicht angenehm, nicht wahr?«\n"
        "\n"
        "»Nein, es ist nicht wahr!« sagte die Raupe.\n"
        "\n"
        "Alice antwortete nichts; es war ihr im Leben nicht so viel "
        "widersprochen\n"
        "worden, und sie fühlte, daß sie wieder anfing, empfindlich zu "
        "werden.\n"
        "\n"
        "»Bist du jetzt zufrieden?« sagte die Raupe.\n"
        "\n"
        "»Etwas größer, Frau Raupe, wäre ich gern, wenn ich bitten darf,« "
        "sagte\n"
        "Alice; »drei und einen halben Zoll ist gar zu winzig.«\n"
        "\n"
        "»Es ist eine sehr angenehme Größe, finde ich,« sagte die Raupe "
        "zornig\n"
        "und richtete sich dabei in die Höhe (sie war gerade drei Zoll "
        "hoch).\n"
        "\n"
        "»Aber ich bin nicht daran gewöhnt!« vertheidigte sich die arme Alice "
        "in\n"
        "weinerlichem Tone. Bei sich dachte sie: »Ich wünschte, alle diese\n"
        "Geschöpfe nähmen nicht Alles gleich übel.«\n"
        "\n"
        "»Dur wirst es mit der Zeit gewohnt werden,« sagte die Raupe, "
        "steckte\n"
        "ihre Huhka in den Mund und fing wieder an zu rauchen.\n"
        "\n"
        "Diesmal wartete Alice geduldig, bis es ihr gefällig wäre zu reden. "
        "Nach\n"
        "zwei oder drei Minuten nahm die Raupe die Huhka aus dem Munde, "
        "gähnte\n"
        "ein bis zwei Mal und schüttelte sich. Dann kam sie von dem Pilze\n"
        "herunter, kroch in's Gras hinein und bemerkte blos bei'm Weggehen: "
        "»Die\n"
        "eine Seite macht dich größer, die andere Seite macht dich kleiner.«\n"
        "\n"
        "»Eine Seite wovon? die andere Seite wovon?« dachte Alice bei sich.\n"
        "\n"
        "»Von dem Pilz,« sagte die Raupe, gerade als wenn sie laut gefragt "
        "hätte;\n"
        "und den nächsten Augenblick war sie nicht mehr zu sehen.\n"
        "\n"
        "Alice blieb ein Weilchen gedankenvoll vor dem Pilze stehen, um "
        "ausfindig\n"
        "zu machen, welches seine beiden Seiten seien; und da er vollkommen "
        "rund\n"
        "war, so fand sie die Frage schwierig zu beantworten. Zuletzt aber\n"
        "reichte sie mit beiden Armen, so weit sie herum konnte, und brach "
        "mit\n"
        "jeder Hand etwas vom Rande ab.\n"
        "\n"
        "»Nun aber, welches ist das rechte?« sprach sie zu sich, und biß ein\n"
        "wenig von dem Stück in ihrer rechten Hand ab, um die Wirkung\n"
        "auszuprobiren; den nächsten Augenblick fühlte sie einen heftigen "
        "Schmerz\n"
        "am Kinn, es hatte an ihren Fuß angestoßen!\n"
        "\n"
        "Ueber diese plötzliche Verwandlung war sie sehr erschrocken, aber da "
        "war\n"
        "keine Zeit zu verlieren, da sie sehr schnell kleiner wurde; sie "
        "machte\n"
        "sich also gleich daran, etwas von dem andern Stück zu essen. Ihr "
        "Kinn\n"
        "war so dicht an ihren Fuß gedrückt, daß ihr kaum Platz genug blieb, "
        "den\n"
        "Mund aufzumachen; endlich aber gelang es ihr, ein wenig von dem "
        "Stück in\n"
        "ihrer linken Hand herunter zu schlucken.\n"
        "\n"
        "       *       *       *       *       *\n"
        "\n"
        "»Ah! endlich ist mein Kopf frei!« rief Alice mit Entzücken, das "
        "sich\n"
        "jedoch den nächsten Augenblick in Angst verwandelte, da sie merkte, "
        "daß\n"
        "ihre Schultern nirgends zu finden waren: als sie hinunter sah, "
        "konnte\n"
        "sie weiter nichts erblicken, als einen ungeheuer langen Hals, der "
        "sich\n"
        "wie eine Stange aus einem Meer von grünen Blättern erhob, das unter "
        "ihr\n"
        "lag.\n"
        "\n"
        "»Was mag all das grüne Zeug sein?« sagte Alice. »Und wo sind meine\n"
        "Schultern nur hingekommen? Und ach, meine armen Hände, wie geht es "
        "zu,\n"
        "daß ich euch nicht sehen kann?« Sie griff bei diesen Worten um "
        "sich,\n"
        "aber es erfolgte weiter nichts, als eine kleine Bewegung in den\n"
        "entfernten grünen Blättern.\n"
        "\n"
        "Da es ihr nicht gelang, die Hände zu ihrem Kopfe zu erheben, so\n"
        "versuchte sie, den Kopf zu ihnen hinunter zu bücken, und fand zu "
        "ihrem\n"
        "Entzücken, daß sie ihren Hals in allen Richtungen biegen und wenden\n"
        "konnte, wie eine Schlange. Sie hatte ihn gerade in ein malerisches\n"
        "Zickzack gewunden und wollte eben in das Blättermeer hinunter "
        "tauchen,\n"
        "das, wie sie sah, durch die Gipfel der Bäume gebildet wurde, unter "
        "denen\n"
        "sie noch eben herumgewandert war, als ein lautes Rauschen sie "
        "plötzlich\n"
        "zurückschreckte: eine große Taube kam ihr in's Gesicht geflogen und\n"
        "schlug sie heftig mit den Flügeln.\n"
        "\n"
        "»Schlange!« kreischte die Taube.\n"
        "\n"
        "»Ich bin keine Schlange!« sagte Alice mit Entrüstung. »Laß mich in\n"
        "Ruhe!«\n"
        "\n"
        "»Schlange sage ich!« wiederholte die Taube, aber mit gedämpfter "
        "Stimme,\n"
        "und fuhr schluchzend fort: »Alles habe ich versucht, und nichts ist\n"
        "ihnen genehm!«\n"
        "\n"
        "»Ich weiß gar nicht, wovon du redest,« sagte Alice.\n"
        "\n"
        "»Baumwurzeln habe ich versucht, Flußufer habe ich versucht, Hecken "
        "habe\n"
        "ich versucht,« sprach die Taube weiter, ohne auf sie zu achten; "
        "»aber\n"
        "diese Schlangen! Nichts ist ihnen recht!«\n"
        "\n"
        "Alice verstand immer weniger; aber sie dachte, es sei unnütz etwas "
        "zu\n"
        "sagen, bis die Taube fertig wäre.\n"
        "\n"
        "»Als ob es nicht Mühe genug wäre, die Eier auszubrüten,« sagte die\n"
        "Taube, »da muß ich noch Tag und Nacht den Schlangen aufpassen! Kein "
        "Auge\n"
        "habe ich die letzten drei Wochen zugethan!«\n"
        "\n"
        "»Es thut mir sehr leid, daß du so viel Verdruß gehabt hast,« sagte\n"
        "Alice, die zu verstehen anfing, was sie meinte.\n"
        "\n"
        "»Und gerade da ich mir den höchsten Baum im Walde ausgesucht habe,« "
        "fuhr\n"
        "die Taube mit erhobener Stimme fort, »und gerade da ich dachte, ich "
        "wäre\n"
        "sie endlich los, müssen sie sich sogar noch vom Himmel "
        "herunterwinden!\n"
        "Pfui! Schlange!«\n"
        "\n"
        "»Aber ich bin keine Schlange, sage ich dir!« rief Alice, »ich bin "
        "ein --\n"
        "ich bin ein --«\n"
        "\n"
        "»Nun, was bist du denn?« fragte die Taube. »Ich merke wohl, daß du "
        "dir\n"
        "etwas ausdenken willst!«\n"
        "\n"
        "»Ich -- ich bin ein kleines Mädchen,« sagte Alice etwas unsicher, da "
        "sie\n"
        "an die vielfachen Verwandlungen dachte, die sie den Tag über schon\n"
        "durchgemacht hatte.\n"
        "\n"
        "»Eine schöne Ausrede, wahrhaftig!« sagte die Taube im Tone tiefster\n"
        "Verachtung. »Ich habe mein Lebtag genug kleine Mädchen gesehen, aber "
        "nie\n"
        "eine mit solch einem Hals! Nein, nein! du bist eine Schlange! das "
        "kannst\n"
        "du nicht abläugnen. Du wirst am Ende noch behaupten, daß du nie ein "
        "Ei\n"
        "gegessen hast.«\n"
        "\n"
        "»Ich habe Eier gegessen, freilich,« sagte Alice, die ein sehr\n"
        "wahrheitsliebendes Kind war; »aber kleine Mädchen essen Eier eben so "
        "gut\n"
        "wie Schlangen.«\n"
        "\n"
        "»Das glaube ich nicht,« sagte die Taube; »wenn sie es aber thun, "
        "nun\n"
        "dann sind sie eine Art Schlangen, so viel weiß ich.«\n"
        "\n"
        "Das war etwas so Neues für Alice, daß sie ein Paar Minuten ganz "
        "still\n"
        "schwieg; die Taube benutzte die Gelegenheit und fuhr fort: »Du "
        "suchst\n"
        "Eier, das weiß ich nur zu gut, und was kümmert es mich, ob du ein\n"
        "kleines Mädchen oder eine Schlange bist?«\n"
        "\n"
        "»Aber mich kümmert es sehr,« sagte Alice schnell; »übrigens suche "
        "ich\n"
        "zufällig nicht Eier, und wenn ich es thäte, so würde ich deine "
        "nicht\n"
        "brauchen können; ich esse sie nicht gern roh.«\n"
        "\n"
        "»Dann mach', daß du fortkommst!« sagte die Taube verdrießlich, indem "
        "sie\n"
        "sich in ihrem Nest wieder zurecht setzte. Alice duckte sich unter "
        "die\n"
        "Bäume so gut sie konnte; denn ihr Hals verwickelte sich fortwährend "
        "in\n"
        "die Zweige, und mehre Male mußte sie anhalten und ihn losmachen. "
        "Nach\n"
        "einer Weile fiel es ihr wieder ein, daß sie noch die Stückchen Pilz "
        "in\n"
        "den Händen hatte, und sie machte sich sorgfältig daran, knabberte "
        "bald\n"
        "an dem einen, bald an dem andern, und wurde abwechselnd größer und\n"
        "kleiner, bis es ihr zuletzt gelang, ihre gewöhnliche Größe zu "
        "bekommen.\n"
        "\n"
        "Es war so lange her, daß sie auch nur ungefähr ihre richtige Höhe "
        "gehabt\n"
        "hatte, daß es ihr erst ganz komisch vorkam; aber nach einigen "
        "Minuten\n"
        "hatte sie sich daran gewöhnt und sprach mit sich selbst wie "
        "gewöhnlich.\n"
        "»Schön, nun ist mein Plan halb ausgeführt! Wie verwirrt man von dem\n"
        "vielen Wechseln wird! Ich weiß nie, wie ich den nächsten Augenblick\n"
        "sein werde! Doch jetzt habe ich meine richtige Größe: nun kommt es\n"
        "darauf an, in den schönen Garten zu gelangen -- wie kann ich das\n"
        "anstellen? das möchte ich wissen!« Wie sie dies sagte, kam sie in "
        "eine\n"
        "Lichtung mit einem Häuschen in der Mitte, ungefähr vier Fuß hoch. "
        "»Wer\n"
        "auch darin wohnen mag, es geht nicht an, daß ich so groß wie ich "
        "jetzt\n"
        "bin hineingehe: sie würden vor Angst nicht wissen wohin!« Also "
        "knabberte\n"
        "sie wieder an dem Stückchen in der rechten Hand, und wagte sich "
        "nicht an\n"
        "das Häuschen heran, bis sie sich auf neun Zoll herunter gebracht "
        "hatte.\n"
        "\n"
        "Noch ein bis zwei Augenblicke stand sie und sah das Häuschen an, "
        "ohne\n"
        "recht zu wissen was sie nun thun solle, als plötzlich ein Lackei in\n"
        "Livree vom Walde her gelaufen kam -- (sie hielt ihn für einen "
        "Lackeien,\n"
        "weil er Livree trug, sonst, nach seinem Gesichte zu urtheilen, würde "
        "sie\n"
        "ihn für einen Fisch angesehen haben) -- und mit den Knöcheln laut an "
        "die\n"
        "Thür klopfte. Sie wurde von einem andern Lackeien in Livree "
        "geöffnet,\n"
        "der ein rundes Gesicht und große Augen wie ein Frosch hatte, und "
        "beide\n"
        "Lackeien hatten, wie Alice bemerkte, gepuderte Lockenperücken über "
        "den\n"
        "ganzen Kopf. Sie war sehr neugierig, was nun geschehen würde, und\n"
        "schlich sich etwas näher, um zuzuhören.\n"
        "\n"
        "Der Fisch-Lackei fing damit an, einen ungeheuren Brief, beinah so "
        "groß\n"
        "wie er selbst, unter dem Arme hervorzuziehen; diesen überreichte er "
        "dem\n"
        "anderen, in feierlichem Tone sprechend: »Für die Herzogin. Eine\n"
        "Einladung von der Königin, Croquet zu spielen.« Der Frosch-Lackei\n"
        "erwiederte in demselben feierlichen Tone, indem er nur die\n"
        "Aufeinanderfolge der Wörter etwas veränderte: »Von der Königin. "
        "Eine\n"
        "Einladung für die Herzogin, Croquet zu spielen.«\n"
        "\n"
        "Dann verbeugten sich Beide tief, und ihre Locken verwickelten sich "
        "in\n"
        "einander.\n"
        "\n"
        "Darüber lachte Alice so laut, daß sie in das Gebüsch zurücklaufen "
        "mußte,\n"
        "aus Furcht, sie möchten sie hören, und als sie wieder herausguckte, "
        "war\n"
        "der Fisch-Lackei fort, und der andere saß auf dem Boden bei der Thür "
        "und\n"
        "sah dumm in den Himmel hinauf.\n"
        "\n"
        "Alice ging furchtsam auf die Thür zu und klopfte.\n"
        "\n"
        "»Es ist durchaus unnütz, zu klopfen,« sagte der Lackei, »und das "
        "wegen\n"
        "zweier Gründe. Erstens weil ich an derselben Seite von der Thür bin "
        "wie\n"
        "du, zweitens, weil sie drinnen einen solchen Lärm machen, daß man "
        "dich\n"
        "unmöglich hören kann.« Und wirklich war ein ganz merkwürdiger Lärm\n"
        "drinnen, ein fortwährendes Heulen und Niesen, und von Zeit zu Zeit "
        "ein\n"
        "lautes Krachen, als ob eine Schüssel oder ein Kessel zerbrochen "
        "wäre.\n"
        "\n"
        "»Bitte,« sagte Alice, »wie soll ich denn hineinkommen?«\n"
        "\n"
        "»Es wäre etwas Sinn und Verstand darin, anzuklopfen,« fuhr der "
        "Lackei\n"
        "fort, ohne auf sie zu hören, »wenn wir die Thür zwischen uns hätten. "
        "Zum\n"
        "Beispiel, wenn du drinnen wärest, könntest du klopfen, und ich "
        "könnte\n"
        "dich herauslassen, nicht wahr?« Er sah die ganze Zeit über, während "
        "er\n"
        "sprach, in den Himmel hinauf, was Alice entschieden sehr unhöflich "
        "fand.\n"
        "»Aber vielleicht kann er nicht dafür,« sagte sie bei sich; »seine "
        "Augen\n"
        "sind so hoch oben auf seiner Stirn. Aber jedenfalls könnte er mir\n"
        "antworten. -- Wie soll ich denn hineinkommen?« wiederholte sie "
        "laut.\n"
        "\n"
        "»Ich werde hier sitzen,« sagte der Lackei, »bis morgen --«\n"
        "\n"
        "In diesem Augenblicke ging die Thür auf, und ein großer Teller kam\n"
        "heraus geflogen, gerade auf den Kopf des Lackeien los; er strich "
        "aber\n"
        "über seine Nase hin und brach an einem der dahinterstehenden Bäume "
        "in\n"
        "Stücke.\n"
        "\n"
        "»-- oder übermorgen, vielleicht,« sprach der Lackei in demselben "
        "Tone\n"
        "fort, als ob nichts vorgefallen wäre.\n"
        "\n"
        "»Wie soll ich denn hineinkommen?« fragte Alice wieder, lauter als\n"
        "vorher.\n"
        "\n"
        "»Sollst du überhaupt hineinkommen?« sagte der Lackei. »Das ist die "
        "erste\n"
        "Frage, nicht wahr?«\n"
        "\n"
        "Das war es allerdings; nur ließ sich Alice das nicht gern sagen. »Es "
        "ist\n"
        "wirklich schrecklich,« murmelte sie vor sich hin, »wie naseweis "
        "alle\n"
        "diese Geschöpfe sind. Es könnte Einen ganz verdreht machen!«\n"
        "\n"
        "Der Lackei schien dies für eine gute Gelegenheit anzusehen, seine\n"
        "Bemerkung zu wiederholen, und zwar mit Variationen. »Ich werde hier\n"
        "sitzen,« sagte er, »ab und an, Tage und Tage lang.«\n"
        "\n"
        "»Was soll ich aber thun?« fragte Alice.\n"
        "\n"
        "»Was dir gefällig ist,« sagte der Lackei, und fing an zu pfeifen.\n"
        "\n"
        "»Es hilft zu nichts, mit ihm zu reden,« sagte Alice außer sich, »er "
        "ist\n"
        "vollkommen blödsinnig!« Sie klinkte die Thür auf und ging hinein.\n"
        "\n"
        "Die Thür führte geradewegs in eine große Küche, welche von einem "
        "Ende\n"
        "bis zum andern voller Rauch war; in der Mitte saß auf einem "
        "dreibeinigen\n"
        "Schemel die Herzogin, mit einem Wickelkinde auf dem Schoße; die "
        "Köchin\n"
        "stand über das Feuer gebückt und rührte in einer großen Kasserole, "
        "die\n"
        "voll Suppe zu sein schien.\n"
        "\n"
        "»In der Suppe ist gewiß zu viel Pfeffer!« sprach Alice für sich, so "
        "gut\n"
        "sie vor Niesen konnte.\n"
        "\n"
        "Es war wenigstens zu viel in der Luft. Sogar die Herzogin nieste hin "
        "und\n"
        "wieder; was das Wickelkind anbelangt, so nieste und schrie es\n"
        "abwechselnd ohne die geringste Unterbrechung. Die beiden einzigen "
        "Wesen\n"
        "in der Küche, die nicht niesten, waren die Köchin und eine große "
        "Katze,\n"
        "die vor dem Herde saß und grinste, sodaß die Mundwinkel bis an die "
        "Ohren\n"
        "reichten.\n"
        "\n"
        "»Wollen Sie mir gütigst sagen,« fragte Alice etwas furchtsam, denn "
        "sie\n"
        "wußte nicht recht, ob es sich für sie schicke zuerst zu sprechen, "
        "»warum\n"
        "Ihre Katze so grinst?«\n"
        "\n"
        "»Es ist eine Grinse-Katze,« sagte die Herzogin, »darum! Ferkel!«\n"
        "\n"
        "Das letzte Wort sagte sie mit solcher Heftigkeit, daß Alice "
        "auffuhr;\n"
        "aber den nächsten Augenblick sah sie, daß es dem Wickelkinde galt, "
        "nicht\n"
        "ihr; sie faßte also Muth und redete weiter: --\n"
        "\n"
        "»Ich wußte nicht, daß Katzen manchmal grinsen; ja ich wußte nicht, "
        "daß\n"
        "Katzen überhaupt grinsen können.«\n"
        "\n"
        "»Sie können es alle,« sagte die Herzogin, »und die meisten thun "
        "es.«\n"
        "\n"
        "»Ich kenne keine, die es thut,« sagte Alice sehr höflich, da sie "
        "ganz\n"
        "froh war, eine Unterhaltung angeknüpft haben.\n"
        "\n"
        "»Du kennst noch nicht viel,« sagte die Herzogin, »und das ist die\n"
        "Wahrheit.«\n"
        "\n"
        "Alice gefiel diese Bemerkung gar nicht, und sie dachte daran, "
        "welchen\n"
        "andern Gegenstand der Unterhaltung sie einführen könnte. Während "
        "sie\n"
        "sich auf etwas Passendes besann, nahm die Köchin die Kasserole mit "
        "Suppe\n"
        "vom Feuer und fing sogleich an, Alles was sie erreichen konnte nach "
        "der\n"
        "Herzogin und dem Kinde zu werfen -- die Feuerzange kam zuerst, dann\n"
        "folgte ein Hagel von Pfannen, Tellern und Schüsseln. Die Herzogin\n"
        "beachtete sie gar nicht, auch wenn sie sie trafen; und das Kind "
        "heulte\n"
        "schon so laut, daß es unmöglich war zu wissen, ob die Stöße ihm weh\n"
        "thaten oder nicht.\n"
        "\n"
        "»Oh, bitte, nehmen Sie sich in Acht, was Sie thun!« rief Alice, die "
        "in\n"
        "wahrer Herzensangst hin und her sprang. »Oh, seine liebe kleine "
        "Nase!«\n"
        "als eine besonders große Pfanne dicht daran vorbeifuhr und sie "
        "beinah\n"
        "abstieß.\n"
        "\n"
        "»Wenn Jeder nur vor seiner Thür fegen wollte,« brummte die Herzogin "
        "mit\n"
        "heiserer Stimme, »würde die Welt sich bedeutend schneller drehen, "
        "als\n"
        "jetzt.«\n"
        "\n"
        "»Was kein Vortheil wäre,« sprach Alice, die sich über die "
        "Gelegenheit\n"
        "freute, ihre Kenntnisse zu zeigen. »Denken Sie nur, wie es Tag und "
        "Nacht\n"
        "in Unordnung bringen würde! Die Erde braucht doch jetzt vier und "
        "zwanzig\n"
        "Stunden, sich um ihre Achse zu drehen --«\n"
        "\n"
        "»Was, du redest von Axt?« sagte die Herzogin, »Hau' ihr den Kopf "
        "ab!«\n"
        "\n"
        "Alice sah sich sehr erschrocken nach der Köchin um, ob sie den Wink\n"
        "verstehen würde; aber die Köchin rührte die Suppe unverwandt und "
        "schien\n"
        "nicht zuzuhören, daher fuhr sie fort: »Vier und zwanzig Stunden, "
        "glaube\n"
        "ich; oder sind es zwölf? Ich --«\n"
        "\n"
        "»Ach, laß mich in Frieden,« sagte die Herzogin, »ich habe Zahlen "
        "nie\n"
        "ausstehen können!« Und damit fing sie an, ihr Kind zu warten und "
        "eine\n"
        "Art Wiegenlied dazu zu singen, wovon jede Reihe mit einem derben "
        "Puffe\n"
        "für das Kind endigte: --\n"
        "\n"
        "    »Schilt deinen kleinen Jungen aus,\n"
        "    Und schlag' ihn, wenn er niest;\n"
        "    Er macht es gar so bunt und kraus,\n"
        "    Nur weil es uns verdrießt.«\n"
        "\n"
        "Chor\n"
        "\n"
        "(in welchen die Köchin und das Wickelkind einfielen).\n"
        "\n"
        "    »Wau! wau! wau!«\n"
        "\n"
        "Während die Herzogin den zweiten Vers des Liedes sang, schaukelte "
        "sie\n"
        "das Kind so heftig auf und nieder, und das arme kleine Ding schrie "
        "so,\n"
        "daß Alice kaum die Worte verstehen konnte: --\n"
        "\n"
        "    »Ich schelte meinen kleinen Wicht,\n"
        "    Und schlag' ihn, wenn er niest;\n"
        "    Ich weiß, wie gern er Pfeffer riecht,\n"
        "    Wenn's ihm gefällig ist.«\n"
        "\n"
        "Chor.\n"
        "\n"
        "    »Wau! wau! wau!«\n"
        "\n"
        "»Hier! du kannst ihn ein Weilchen warten, wenn du willst!« sagte "
        "die\n"
        "Herzogin zu Alice, indem sie ihr das Kind zuwarf. »Ich muß mich "
        "zurecht\n"
        "machen, um mit der Königin Croquet zu spielen,« damit rannte sie aus "
        "dem\n"
        "Zimmer. Die Köchin warf ihr eine Bratpfanne nach; aber sie verfehlte "
        "sie\n"
        "noch eben.\n"
        "\n"
        "Alice hatte das Kind mit Mühe und Noth aufgefangen, da es ein "
        "kleines\n"
        "unförmliches Wesen war, das seine Arme und Beinchen nach allen "
        "Seiten\n"
        "ausstreckte, »gerade wie ein Seestern,« dachte Alice. Das arme "
        "kleine\n"
        "Ding stöhnte wie eine Locomotive, als sie es fing, und zog sich "
        "zusammen\n"
        "und streckte sich wieder aus, so daß sie es die ersten Paar Minuten "
        "nur\n"
        "eben halten konnte.\n"
        "\n"
        "Sobald sie aber die rechte Art entdeckt hatte, wie man es tragen "
        "mußte\n"
        "(die darin bestand, es zu einer Art Knoten zu drehen, und es dann "
        "fest\n"
        "beim rechten Ohr und linken Fuß zu fassen, damit es sich nicht "
        "wieder\n"
        "aufwickeln konnte), brachte sie es in's Freie. »Wenn ich dies Kind "
        "nicht\n"
        "mit mir nehme,« dachte Alice, »so werden sie es in wenigen Tagen\n"
        "umgebracht haben; wäre es nicht Mord, es da zu lassen?« Sie sprach "
        "die\n"
        "letzten Worte laut, und das kleine Geschöpf grunzte zur Antwort (es\n"
        "hatte mittlerweile aufgehört zu niesen). »Grunze nicht,« sagte "
        "Alice;\n"
        "»es paßt sich gar nicht für dich, dich so auszudrücken.«\n"
        "\n"
        "Der Junge grunzte wieder, so daß Alice ihm ganz ängstlich in's "
        "Gesicht\n"
        "sah, was ihm eigentlich fehle. Er hatte ohne Zweifel eine sehr\n"
        "hervorstehende Nase, eher eine Schnauze als eine wirkliche Nase; "
        "auch\n"
        "seine Augen wurden entsetzlich klein für einen kleinen Jungen: "
        "Alles\n"
        "zusammen genommen, gefiel Alice das Aussehen des Kindes gar nicht. "
        "»Aber\n"
        "vielleicht hat es nur geweint,« dachte sie und sah ihm wieder in "
        "die\n"
        "Augen, ob Thränen da seien.\n"
        "\n"
        "Nein, es waren keine Thränen da. »Wenn du ein kleines Ferkel wirst, "
        "höre\n"
        "mal,« sagte Alice sehr ernst, »so will ich nichts mehr mit dir zu\n"
        "schaffen haben, das merke dir!« Das arme kleine Ding schluchzte "
        "(oder\n"
        "grunzte, es war unmöglich, es zu unterscheiden), und dann gingen "
        "sie\n"
        "eine Weile stillschweigend weiter.\n"
        "\n"
        "Alice fing eben an, sich zu überlegen: »Nun, was soll ich mit "
        "diesem\n"
        "Geschöpf anfangen, wenn ich es mit nach Hause bringe?« als es "
        "wieder\n"
        "grunzte, so laut, daß Alice erschrocken nach ihm hinsah. Diesmal "
        "konnte\n"
        "sie sich nicht mehr irren: es war nichts mehr oder weniger als ein\n"
        "Ferkel, und sie sah, daß es höchst lächerlich für sie wäre, es noch\n"
        "weiter zu tragen.\n"
        "\n"
        "Sie setzte also das kleine Ding hin und war ganz froh, als sie es "
        "ruhig\n"
        "in den Wald traben sah. »Das wäre in einigen Jahren ein furchtbar\n"
        "häßliches Kind geworden; aber als Ferkel macht es sich recht nett, "
        "finde\n"
        "ich.« Und so dachte sie alle Kinder durch, die sie kannte, die gute\n"
        "kleine Ferkel abgeben würden, und sagte gerade für sich: »wenn man "
        "nur\n"
        "die rechten Mittel wüßte, sie zu verwandeln --« als sie einen "
        "Schreck\n"
        "bekam; die Grinse-Katze saß nämlich wenige Fuß von ihr auf einem\n"
        "Baumzweige.\n"
        "\n"
        "Die Katze grinste nur, als sie Alice sah. »Sie sieht gutmüthig "
        "aus,«\n"
        "dachte diese; aber doch hatte sie sehr lange Krallen und eine Menge\n"
        "Zähne. Alice fühlte wohl, daß sie sie rücksichtsvoll behandeln "
        "müsse.\n"
        "\n"
        "»Grinse-Mies,« fing sie etwas ängstlich an, da sie nicht wußte, ob "
        "ihr\n"
        "der Name gefallen würde: jedoch grinste sie noch etwas breiter. "
        "»Schön,\n"
        "so weit gefällt es ihr,« dachte Alice und sprach weiter: »willst du "
        "mir\n"
        "wohl sagen, wenn ich bitten darf, welchen Weg ich hier nehmen muß?«\n"
        "\n"
        "»Das hängt zum guten Theil davon ab, wohin du gehen willst,« sagte "
        "die\n"
        "Katze.\n"
        "\n"
        "»Es kommt mir nicht darauf an, wohin --« sagte Alice.\n"
        "\n"
        "»Dann kommt es auch nicht darauf an, welchen Weg du nimmst,« sagte "
        "die\n"
        "Katze.\n"
        "\n"
        "»-- wenn ich nur irgendwo hinkomme,« fügte Alice als Erklärung "
        "hinzu.\n"
        "\n"
        "»O, das wirst du ganz gewiß,« sagte die Katze, »wenn du nur lange "
        "genug\n"
        "gehest.«\n"
        "\n"
        "Alice sah, daß sie nichts dagegen einwenden konnte; sie versuchte "
        "daher\n"
        "eine andere Frage. »Was für Art Leute wohnen hier in der Nähe?«\n"
        "\n"
        "»In der Richtung,« sagte die Katze, die rechte Pfote schwenkend, "
        "»wohnt\n"
        "ein Hutmacher, und in jener Richtung,« die andere Pfote schwenkend,\n"
        "»wohnt ein Faselhase. Besuche welchen du willst: sie sind beide "
        "toll.«\n"
        "\n"
        "»Aber ich mag nicht zu tollen Leuten gehen,« bemerkte Alice.\n"
        "\n"
        "»Oh, das kannst du nicht ändern,« sagte die Katze: »wir sind alle "
        "toll\n"
        "hier. Ich bin toll. Du bist toll.«\n"
        "\n"
        "»Woher weißt du, daß ich toll bin?« fragte Alice.\n"
        "\n"
        "»Du mußt es sein,« sagte die Katze, »sonst wärest du nicht "
        "hergekommen.«\n"
        "\n"
        "Alice fand durchaus nicht, daß das ein Beweis sei; sie fragte "
        "jedoch\n"
        "weiter: »Und woher weißt du, daß du toll bist?«\n"
        "\n"
        "»Zu allererst,« sagte die Katze, »ein Hund ist nicht toll. Das "
        "giebst du\n"
        "zu?«\n"
        "\n"
        "»Zugestanden!« sagte Alice.\n"
        "\n"
        "»Nun, gut,« fuhr die Katze fort, »nicht wahr ein Hund knurrt, wenn "
        "er\n"
        "böse ist, und wedelt mit dem Schwanze, wenn er sich freut. Ich "
        "hingegen\n"
        "knurre, wenn ich mich freue, und wedle mit dem Schwanze, wenn ich\n"
        "ärgerlich bin. Daher bin ich toll.«\n"
        "\n"
        "»Ich nenne es spinnen, nicht knurren,« sagte Alice.\n"
        "\n"
        "»Nenne es, wie du willst,« sagte die Katze. »Spielst du heut Croquet "
        "mit\n"
        "der Königin?«\n"
        "\n"
        "»Ich möchte es sehr gern,« sagte Alice, »aber ich bin noch nicht\n"
        "eingeladen worden.«\n"
        "\n"
        "»Du wirst mich dort sehen,« sagte die Katze und verschwand.\n"
        "\n"
        "Alice wunderte sich nicht sehr darüber; sie war so daran gewöhnt, "
        "daß\n"
        "sonderbare Dinge geschahen. Während sie noch nach der Stelle hinsah, "
        "wo\n"
        "die Katze gesessen hatte, erschien sie plötzlich wieder.\n"
        "\n"
        "»Uebrigens, was ist aus dem Jungen geworden?« sagte die Katze. »Ich\n"
        "hätte beinah vergessen zu fragen.«\n"
        "\n"
        "»Er ist ein Ferkel geworden,« antwortete Alice sehr ruhig, gerade "
        "wie\n"
        "wenn die Katze auf gewöhnliche Weise zurückgekommen wäre.\n"
        "\n"
        "»Das dachte ich wohl,« sagte die Katze und verschwand wieder.\n"
        "\n"
        "Alice wartete noch etwas, halb und halb erwartend, sie wieder "
        "erscheinen\n"
        "zu sehen; aber sie kam nicht, und ein Paar Minuten nachher ging sie "
        "in\n"
        "der Richtung fort, wo der Faselhase wohnen sollte. »Hutmacher habe "
        "ich\n"
        "schon gesehen,« sprach sie zu sich, »der Faselhase wird viel\n"
        "interessanter sein.« Wie sie so sprach, blickte sie auf, und da saß "
        "die\n"
        "Katze wieder auf einem Baumzweige. »Sagtest du Ferkel oder Fächer?«\n"
        "fragte sie. »Ich sagte Ferkel,« antwortete Alice, »und es wäre mir "
        "sehr\n"
        "lieb, wenn du nicht immer so schnell erscheinen und verschwinden\n"
        "wolltest: du machst Einen ganz schwindlig.«\n"
        "\n"
        "»Schon gut,« sagte die Katze, und diesmal verschwand sie ganz "
        "langsam,\n"
        "wobei sie mit der Schwanzspitze anfing und mit dem Grinsen aufhörte, "
        "das\n"
        "noch einige Zeit sichtbar blieb, nachdem das Uebrige verschwunden "
        "war.\n"
        "\n"
        "»Oho, ich habe oft eine Katze ohne Grinsen gesehen,« dachte Alice, "
        "»aber\n"
        "ein Grinsen ohne Katze! so etwas Merkwürdiges habe ich in meinem "
        "Leben\n"
        "noch nicht gesehen!«\n"
        "\n"
        "Sie brauchte nicht weit zu gehen, so erblickte sie das Haus des\n"
        "Faselhasen; sie dachte, es müsse das rechte Haus sein, weil die\n"
        "Schornsteine wie Ohren geformt waren, und das Dach war mit Pelz "
        "gedeckt.\n"
        "Es war ein so großes Haus, daß, ehe sie sich näher heran wagte, sie "
        "ein\n"
        "wenig von dem Stück Pilz in ihrer linken Hand abknabberte, und sich "
        "bis\n"
        "auf zwei Fuß hoch brachte: trotzdem näherte sie sich etwas "
        "furchtsam,\n"
        "für sich sprechend: »Wenn er nur nicht ganz rasend ist! Wäre ich "
        "doch\n"
        "lieber zu dem Hutmacher gegangen!«\n"
        "\n"
        "Vor dem Hause stand ein gedeckter Theetisch, an welchem der "
        "Faselhase\n"
        "und der Hutmacher saßen; ein Murmelthier saß zwischen ihnen, fest\n"
        "eingeschlafen, und die beiden Andern benutzten es als Kissen, um "
        "ihre\n"
        "Ellbogen darauf zu stützen, und redeten über seinem Kopfe mit "
        "einander.\n"
        "»Sehr unbequem für das Murmelthier,« dachte Alice; »nun, da es "
        "schläft,\n"
        "wird es sich wohl nichts daraus machen.«\n"
        "\n"
        "Der Tisch war groß, aber die Drei saßen dicht zusammengedrängt an "
        "einer\n"
        "Ecke: »Kein Platz! Kein Platz!« riefen sie aus, sobald sie Alice "
        "kommen\n"
        "sahen. »Ueber und über genug Platz!« sagte Alice unwillig und "
        "setzte\n"
        "sich in einen großen Armstuhl am Ende des Tisches.\n"
        "\n"
        "»Ist dir etwas Wein gefällig?« nöthigte sie der Faselhase.\n"
        "\n"
        "Alice sah sich auf dem ganzen Tische um, aber es war nichts als "
        "Thee\n"
        "darauf. »Ich sehe keinen Wein,« bemerkte sie.\n"
        "\n"
        "»Es ist keiner hier,« sagte der Faselhase.\n"
        "\n"
        "»Dann war es gar nicht höflich von dir, mir welchen anzubieten,« "
        "sagte\n"
        "Alice ärgerlich.\n"
        "\n"
        "»Es war gar nicht höflich von dir, dich ungebeten herzusetzen,« "
        "sagte\n"
        "der Faselhase.\n"
        "\n"
        "»Ich wußte nicht, das es dein Tisch ist; er ist für viel mehr als "
        "drei\n"
        "gedeckt.«\n"
        "\n"
        "»Dein Haar muß verschnitten werden,« sagte der Hutmacher. Er hatte "
        "Alice\n"
        "eine Zeit lang mit großer Neugierde angesehen, und dies waren seine\n"
        "ersten Worte.\n"
        "\n"
        "»Du solltest keine persönlichen Bemerkungen machen,« sagte Alice "
        "mit\n"
        "einer gewissen Strenge, »es ist sehr grob.«\n"
        "\n"
        "Der Hutmacher riß die Augen weit auf, als er dies hörte; aber er "
        "sagte\n"
        "weiter nichts als: »Warum ist ein Rabe wie ein Reitersmann?«\n"
        "\n"
        "»Ei, jetzt wird es Spaß geben,« dachte Alice. »Ich bin so froh, daß "
        "sie\n"
        "anfangen Räthsel aufzugeben -- Ich glaube, das kann ich rathen,« "
        "fuhr sie\n"
        "laut fort.\n"
        "\n"
        "»Meinst du, daß du die Antwort dazu finden kannst?« fragte der\n"
        "Faselhase.\n"
        "\n"
        "»Ja, natürlich,« sagte Alice.\n"
        "\n"
        "»Dann solltest du sagen, was du meinst,« sprach der Hase weiter.\n"
        "\n"
        "»Das thue ich ja,« warf Alice schnell ein, »wenigstens -- "
        "wenigstens\n"
        "meine ich, was ich sage -- und das ist dasselbe.«\n"
        "\n"
        "»Nicht im Geringsten dasselbe!« sagte der Hutmacher. »Wie, du "
        "könntest\n"
        "eben so gut behaupten, daß »ich sehe, was ich esse« dasselbe ist "
        "wie\n"
        "»ich esse, was ich sehe.«\n"
        "\n"
        "»Du könntest auch behaupten,« fügte der Faselhase hinzu, »ich mag, "
        "was\n"
        "ich kriege« sei dasselbe wie »ich kriege, was ich mag!«\n"
        "\n"
        "»Du könntest eben so gut behaupten,« fiel das Murmelthier ein, das "
        "im\n"
        "Schlafe zu sprechen schien, »ich athme, wenn ich schlafe« sei "
        "dasselbe\n"
        "wie »ich schlafe, wenn ich athme!«\n"
        "\n"
        "»Es ist dasselbe bei dir,« sagte der Hutmacher, und damit endigte "
        "die\n"
        "Unterhaltung, und die Gesellschaft saß einige Minuten schweigend,\n"
        "während Alice Alles durchdachte, was sie je von Raben und "
        "Reitersmännern\n"
        "gehört hatte, und das war nicht viel.\n"
        "\n"
        "Der Hutmacher brach das Schweigen zuerst. »Den wievielsten haben "
        "wir\n"
        "heute?« sagte er, sich an Alice wendend; er hatte seine Uhr aus der\n"
        "Tasche genommen, sah sie unruhig an, schüttelte sie hin und her und\n"
        "hielt sie an's Ohr.\n"
        "\n"
        "Alice besann sich ein wenig und sagte: »Den vierten.«\n"
        "\n"
        "»Zwei Tage falsch!« seufzte der Hutmacher. »Ich sagte dir ja, daß "
        "Butter\n"
        "das Werk verderben würde,« setzte er hinzu, indem er den Hasen "
        "ärgerlich\n"
        "ansah.\n"
        "\n"
        "»Es war die beste Butter,« sagte der Faselhase demüthig.\n"
        "\n"
        "»Ja, aber es muß etwas Krume mit hinein gerathen sein,« brummte der\n"
        "Hutmacher; »du hättest sie nicht mit dem Brodmesser hinein thun "
        "sollen.«\n"
        "\n"
        "Der Faselhase nahm die Uhr und betrachtete sie trübselig; dann "
        "tunkte er\n"
        "sie in seine Tasse Thee und betrachtete sie wieder, aber es fiel "
        "ihm\n"
        "nichts Besseres ein, als seine erste Bemerkung: »Es war wirklich "
        "die\n"
        "beste Butter.«\n"
        "\n"
        "Alice hatte ihm neugierig über die Schulter gesehen. »Was für eine\n"
        "komische Uhr!« sagte sie. »Sie zeigt das Datum, und nicht wie viel "
        "Uhr\n"
        "es ist!«\n"
        "\n"
        "»Warum sollte sie?« brummte der Hase; »zeigt deine Uhr, welches Jahr "
        "es\n"
        "ist?«\n"
        "\n"
        "»Natürlich nicht,« antwortete Alice schnell, »weil es so lange\n"
        "hintereinander dasselbe Jahr bleibt.«\n"
        "\n"
        "»Und so ist es gerade mit meiner,« sagte der Hutmacher.\n"
        "\n"
        "Alice war ganz verwirrt. Die Erklärung des Hutmachers schien ihr "
        "gar\n"
        "keinen Sinn zu haben, und doch waren es deutlich gesprochne Worte. "
        "»Ich\n"
        "verstehe dich nicht ganz,« sagte sie, so höflich sie konnte.\n"
        "\n"
        "»Das Murmelthier schläft schon wieder,« sagte der Hutmacher, und goß "
        "ihm\n"
        "etwas heißen Thee auf die Nase.\n"
        "\n"
        "Das Murmelthier schüttelte ungeduldig den Kopf und sagte, ohne die "
        "Augen\n"
        "aufzuthun: »Freilich, freilich, das wollte ich eben auch bemerken.«\n"
        "\n"
        "»Hast du das Räthsel schon gerathen?« wandte sich der Hutmacher an\n"
        "Alice.\n"
        "\n"
        "»Nein, ich gebe es auf,« antwortete Alice; »was ist die Antwort?«\n"
        "\n"
        "»Davon habe ich nicht die leiseste Ahnung,« sagte der Hutmacher.\n"
        "\n"
        "»Ich auch nicht,« sagte der Faselhase.\n"
        "\n"
        "Alice seufzte verstimmt. »Ich dächte, ihr könntet die Zeit besser\n"
        "anwenden,« sagte sie, »als mit Räthseln, die keine Auflösung "
        "haben.«\n"
        "\n"
        "»Wenn du die Zeit so gut kenntest wie ich,« sagte der Hutmacher,\n"
        "»würdest du nicht davon reden, wie wir sie anwenden, sondern wie sie "
        "uns\n"
        "anwendet.«\n"
        "\n"
        "»Ich weiß nicht, was du meinst,« sagte Alice.\n"
        "\n"
        "»Natürlich kannst du das nicht wissen!« sagte der Hutmacher, indem "
        "er\n"
        "den Kopf verächtlich in die Höhe warf. »Du hast wahrscheinlich nie "
        "mit\n"
        "der Zeit gesprochen.«\n"
        "\n"
        "»Ich glaube kaum,« erwiederte Alice vorsichtig; »aber Mama sagte\n"
        "gestern, ich sollte zu meiner kleinen Schwester gehen und ihr die "
        "Zeit\n"
        "vertreiben.«\n"
        "\n"
        "»So? das wird sie dir schön übel genommen haben; sie läßt sich "
        "nicht\n"
        "gern vertreiben. Aber wenn man gut mit ihr steht, so thut sie Einem\n"
        "beinah Alles zu Gefallen mit der Uhr. Zum Beispiel, nimm den Fall, "
        "es\n"
        "wäre 9 Uhr Morgens, gerade Zeit, deine Stunden anzufangen, du "
        "brauchtest\n"
        "der Zeit nur den kleinsten Wink zu geben, schnurr! geht die Uhr "
        "herum,\n"
        "ehe du dich's versiehst! halb Zwei, Essenszeit!«\n"
        "\n"
        "(»Ich wünschte, das wäre es!« sagte der Faselhase leise für sich.)\n"
        "\n"
        "»Das wäre wirklich famos,« sagte Alice gedankenvoll, »aber dann "
        "würde\n"
        "ich nicht hungrig genug sein, nicht wahr?«\n"
        "\n"
        "»Zuerst vielleicht nicht,« antwortete der Hutmacher, »aber es würde "
        "so\n"
        "lange halb Zwei bleiben, wie du wolltest.«\n"
        "\n"
        "»So macht ihr es wohl hier?« fragte Alice.\n"
        "\n"
        "Der Hutmacher schüttelte traurig den Kopf. »Ich nicht!« sprach er. "
        "»Wir\n"
        "haben uns vorige Ostern entzweit -- kurz ehe er toll wurde, du "
        "weißt\n"
        "doch (mit seinem Theelöffel auf den Faselhasen zeigend) -- es war in "
        "dem\n"
        "großen Concert, das die Coeur-Königin gab; ich mußte singen:\n"
        "\n"
        "    »O Papagei, o Papagei!\n"
        "    Wie grün sind deine Federn!«\n"
        "\n"
        "Vielleicht kennst du das Lied?«\n"
        "\n"
        "»Ich habe etwas dergleichen gehört,« sage Alice.\n"
        "\n"
        "»Es geht weiter,« fuhr der Hutmacher fort:\n"
        "\n"
        "    »Du grünst nicht nur zur Friedenszeit,\n"
        "    Auch wenn es Teller und Töpfe schneit.\n"
        "    O Papagei, o Papagei --«\n"
        "\n"
        "Hier schüttelte sich das Murmelthier und fing an im Schlaf zu "
        "singen: »O\n"
        "Papagei, o Mamagei, o Papagei, o Mamagei --« in einem fort, so daß "
        "sie\n"
        "es zuletzt kneifen mußten, damit es nur aufhöre.\n"
        "\n"
        "»Denke dir, ich hatte kaum den ersten Vers fertig,« sagte der "
        "Hutmacher,\n"
        "»als die Königin ausrief: Abscheulich! der Mensch schlägt geradezu "
        "die\n"
        "Zeit todt mit seinem Geplärre. Aufgehängt soll er werden!«\n"
        "\n"
        "»Wie furchtbar grausam!« rief Alice.\n"
        "\n"
        "»Und seitdem,« sprach der Hutmacher traurig weiter, »hat sie mir "
        "nie\n"
        "etwas zu Gefallen thun wollen, die Zeit! Es ist nun immer sechs "
        "Uhr!«\n"
        "\n"
        "Dies brachte Alice auf einen klugen Gedanken. »Darum sind wohl so "
        "viele\n"
        "Tassen hier herumgestellt?« fragte sie.\n"
        "\n"
        "»Ja, darum,« sagte der Hutmacher mit einem Seufzer, »es ist immer\n"
        "Theestunde, und wir haben keine Zeit, die Tassen dazwischen\n"
        "aufzuwaschen.«\n"
        "\n"
        "»Dann rückt ihr wohl herum?« sagte Alice.\n"
        "\n"
        "»So ist es,« sagte der Hutmacher, »wenn die Tassen genug gebraucht\n"
        "sind.«\n"
        "\n"
        "»Aber wenn ihr wieder an den Anfang kommt?« unterstand sich Alice "
        "zu\n"
        "fragen.\n"
        "\n"
        "»Wir wollen jetzt von etwas Anderem reden,« unterbrach sie der "
        "Faselhase\n"
        "gähnend, »dieser Gegenstand ist mir nachgerade langweilig. Ich "
        "schlage\n"
        "vor, die junge Dame erzählt eine Geschichte.«\n"
        "\n"
        "»O, ich weiß leider keine,« rief Alice, ganz bestürzt über diese\n"
        "Zumuthung.\n"
        "\n"
        "»Dann soll das Murmelthier erzählen!« riefen beide; »wache auf,\n"
        "Murmelthier!« dabei kniffen sie es von beiden Seiten zugleich.\n"
        "\n"
        "Das Murmelthier machte langsam die Augen auf. »Ich habe nicht\n"
        "geschlafen,« sagte es mit heiserer, schwacher Stimme, »ich habe "
        "jedes\n"
        "Wort gehört, das ihr Jungen gesagt habt.«\n"
        "\n"
        "»Erzähle uns eine Geschichte!« sagte der Faselhase.\n"
        "\n"
        "»Ach ja, sei so gut!« bat Alice.\n"
        "\n"
        "»Und mach schnell,« fügte der Hutmacher hinzu, »sonst schläfst du "
        "ein,\n"
        "ehe sie zu Ende ist.«\n"
        "\n"
        "»Es waren einmal drei kleine Schwestern,« fing das Murmelthier eilig "
        "an,\n"
        "»die hießen Else, Lacie und Tillie, und sie lebten tief unten in "
        "einem\n"
        "Brunnen --«\n"
        "\n"
        "»Wovon lebten sie?« fragte Alice, die sich immer für Essen und "
        "Trinken\n"
        "sehr interessirte.\n"
        "\n"
        "»Sie lebten von Syrup,« versetzte das Murmelthier, nachdem es sich "
        "eine\n"
        "Minute besonnen hatte.\n"
        "\n"
        "»Das konnten sie ja aber nicht,« bemerkte Alice schüchtern, »da "
        "wären\n"
        "sie ja krank geworden.«\n"
        "\n"
        "»Das wurden sie auch,« sagte das Murmelthier, »sehr krank.«\n"
        "\n"
        "Alice versuchte es sich vorzustellen, wie eine so außergewöhnliche "
        "Art\n"
        "zu leben wohl sein möchte; aber es kam ihr zu kurios vor, sie mußte\n"
        "wieder fragen: »Aber warum lebten sie unten in dem Brunnen?«\n"
        "\n"
        "»Willst du nicht ein wenig mehr Thee?« sagte der Faselhase sehr\n"
        "ernsthaft zu Alice.\n"
        "\n"
        "»Ein wenig mehr? ich habe noch keinen gehabt,« antwortete Alice "
        "etwas\n"
        "empfindlich, »also kann ich nicht noch mehr trinken.«\n"
        "\n"
        "»Du meinst, du kannst nicht weniger trinken,« sagte der Hutmacher: "
        "»es\n"
        "ist sehr leicht, mehr als keinen zu trinken.«\n"
        "\n"
        "»Niemand hat dich um deine Meinung gefragt,« sagte Alice.\n"
        "\n"
        "»Wer macht denn nun persönliche Bemerkungen?« rief der Hutmacher\n"
        "triumphirend.\n"
        "\n"
        "Alice wußte nicht recht, was sie darauf antworten sollte; sie nahm "
        "sich\n"
        "daher etwas Thee und Butterbrot, und dann wandte sie sich an das\n"
        "Murmelthier und wiederholte ihre Frage: »Warum lebten sie in einem\n"
        "Brunnen?«\n"
        "\n"
        "Das Murmelthier besann sich einen Augenblick und sagte dann: »Es war "
        "ein\n"
        "Syrup-Brunnen.«\n"
        "\n"
        "»Den giebt es nicht!« fing Alice sehr ärgerlich an; aber der "
        "Hutmacher\n"
        "und Faselhase machten beide: »Sch, sch!« und das Murmelthier "
        "bemerkte\n"
        "brummend: »Wenn du nicht höflich sein kannst, kannst du die "
        "Geschichte\n"
        "selber auserzählen.«\n"
        "\n"
        "»Nein, bitte, erzähle weiter!« sagte Alice ganz bescheiden; »ich "
        "will\n"
        "dich nicht wieder unterbrechen. Es wird wohl einen geben.«\n"
        "\n"
        "»Einen, wirklich!« sagte das Murmelthier entrüstet. Doch ließ es "
        "sich\n"
        "zum Weitererzählen bewegen. »Also die drei kleinen Schwestern -- "
        "sie\n"
        "lernten zeichnen, müßt ihr wissen --«\n"
        "\n"
        "»Was zeichneten sie?« sagte Alice, ihr Versprechen ganz vergessend.\n"
        "\n"
        "»Syrup,« sagte das Murmelthier, diesmal ganz ohne zu überlegen.\n"
        "\n"
        "»Ich brauche eine reine Tasse,« unterbrach der Hutmacher, »wir "
        "wollen\n"
        "Alle einen Platz rücken.«\n"
        "\n"
        "Er rückte, wie er das sagte, und das Murmelthier folgte ihm; der\n"
        "Faselhase rückte an den Platz des Murmelthiers, und Alice nahm, "
        "obgleich\n"
        "etwas ungern, den Platz des Faselhasen ein. Der Hutmacher war der\n"
        "Einzige, der Vortheil von diesem Wechsel hatte, und Alice hatte es "
        "viel\n"
        "schlimmer als zuvor, da der Faselhase eben den Milchtopf über "
        "seinen\n"
        "Teller umgestoßen hatte.\n"
        "\n"
        "Alice wollte das Murmelthier nicht wieder beleidigen und fing daher "
        "sehr\n"
        "vorsichtig an: »Aber ich verstehe nicht. Wie konnten sie den Syrup\n"
        "zeichnen?«\n"
        "\n"
        "»Als ob nicht aller Syrup gezeichnet wäre, den man vom Kaufmann "
        "holt,«\n"
        "sagte der Hutmacher; »hast du nicht immer darauf gesehen: feinste\n"
        "Qualität, allerfeinste Qualität, superfeine Qualität -- oh, du "
        "kleiner\n"
        "Dummkopf?«\n"
        "\n"
        "»Wie gesagt,« fuhr das Murmelthier fort, »lernten sie zeichnen;« "
        "hier\n"
        "gähnte es und rieb sich die Augen, denn es fing an, sehr schläfrig "
        "zu\n"
        "werden; »und sie zeichneten Allerlei -- Alles was mit M. anfängt "
        "--«\n"
        "\n"
        "»Warum mit M?« fragte Alice.\n"
        "\n"
        "»Warum nicht?« sagte der Faselhase.\n"
        "\n"
        "Alice war still.\n"
        "\n"
        "Das Murmelthier hatte mittlerweile die Augen zugemacht, und war "
        "halb\n"
        "eingeschlafen; da aber der Hutmacher es zwickte, wachte es mit "
        "einem\n"
        "leisen Schrei auf und sprach weiter: -- »was mit M anfängt, wie\n"
        "Mausefallen, den Mond, Mangel und manches Mal -- ihr wißt, man sagt: "
        "ich\n"
        "habe das manches liebe Mal gethan -- hast du je manches liebe Mal\n"
        "gezeichnet gesehen?«\n"
        "\n"
        "»Wirklich, da du mich selbst fragst,« sagte Alice ganz verwirrt, "
        "»ich\n"
        "denke kaum --«\n"
        "\n"
        "»Dann solltest du auch nicht reden,« sagte der Hutmacher.\n"
        "\n"
        "Dies war nachgerade zu grob für Alice: sie stand ganz beleidigt auf "
        "und\n"
        "ging fort; das Murmelthier schlief augenblicklich wieder ein, und "
        "die\n"
        "beiden Andern beachteten ihr Fortgehen nicht, obgleich sie sich ein "
        "paar\n"
        "Mal umsah, halb in der Hoffnung, daß sie sie zurückrufen würden. Als "
        "sie\n"
        "sie zuletzt sah, versuchten sie das Murmelthier in die Theekanne zu\n"
        "stecken.\n"
        "\n"
        "»Auf keinen Fall will ich da je wieder hingehen!« sagte Alice, "
        "während\n"
        "sie sich einen Weg durch den Wald suchte. »Es ist die dümmste\n"
        "Theegesellschaft, in der ich in meinem ganzen Leben war!«\n"
        "\n"
        "Gerade wie sie so sprach, bemerkte sie, daß einer der Bäume eine "
        "kleine\n"
        "Thür hatte. »Das ist höchst komisch!« dachte sie. »Aber Alles ist "
        "heute\n"
        "komisch! Ich will lieber gleich hinein gehen.«\n"
        "\n"
        "Wie gesagt, so gethan: und sie befand sich wieder in dem langen\n"
        "Corridor, und dicht bei dem kleinen Glastische. »Diesmal will ich "
        "es\n"
        "gescheidter anfangen,« sagte sie zu sich selbst, nahm das goldne\n"
        "Schlüsselchen und schloß die Thür auf, die in den Garten führte. "
        "Sie\n"
        "machte sich daran, an dem Pilz zu knabbern (sie hatte ein Stückchen "
        "in\n"
        "ihrer Tasche behalten), bis sie ungefähr einen Fuß hoch war, dann "
        "ging\n"
        "sie den kleinen Gang hinunter; und dann -- war sie endlich in dem\n"
        "schönen Garten, unter den prunkenden Blumenbeeten und kühlen\n"
        "Springbrunnen.\n"
        "\n"
        "Ein großer hochstämmiger Rosenstrauch stand nahe bei'm Eingang; die\n"
        "Rosen, die darauf wuchsen, waren weiß, aber drei Gärtner waren "
        "damit\n"
        "beschäftigt, sie roth zu malen. Alice kam dies wunderbar vor, und da "
        "sie\n"
        "näher hinzutrat, um ihnen zuzusehen, hörte sie einen von ihnen "
        "sagen:\n"
        "»Nimm dich in Acht, Fünf! Bespritze mich nicht so mit Farbe!«\n"
        "\n"
        "»Ich konnte nicht dafür,« sagte Fünf in verdrießlichem Tone; »Sieben "
        "hat\n"
        "mich an den Ellbogen gestoßen.«\n"
        "\n"
        "Worauf Sieben aufsah und sagte: »Recht so, Fünf! Schiebe immer die\n"
        "Schuld auf andre Leute!«\n"
        "\n"
        "»Du sei nur ganz still!« sagte Fünf. »Gestern erst hörte ich die "
        "Königin\n"
        "sagen, du verdientest geköpft zu werden!«\n"
        "\n"
        "»Wofür?« fragte der, welcher zuerst gesprochen hatte.\n"
        "\n"
        "»Das geht dich nichts an, Zwei!« sagte Sieben.\n"
        "\n"
        "»Ja, es geht ihn an!« sagte Fünf, »und ich werde es ihm sagen -- "
        "dafür,\n"
        "daß er dem Koch Tulpenzwiebeln statt Küchenzwiebeln gebracht hat.«\n"
        "\n"
        "Sieben warf seinen Pinsel hin und hatte eben angefangen: »Ist je "
        "eine\n"
        "ungerechtere Anschuldigung --« als sein Auge zufällig auf Alice "
        "fiel,\n"
        "die ihnen zuhörte; er hielt plötzlich inne, die andern sahen sich "
        "auch\n"
        "um, und sie verbeugten sich Alle tief.\n"
        "\n"
        "»Wollen Sie so gut sein, mir zu sagen,« sprach Alice etwas "
        "furchtsam,\n"
        "»warum Sie diese Rosen malen?«\n"
        "\n"
        "Fünf und Sieben antworteten nichts, sahen aber Zwei an. Zwei fing "
        "mit\n"
        "leiser Stimme an: »Die Wahrheit zu gestehen, Fräulein, dies hätte "
        "hier\n"
        "ein rother Rosenstrauch sein sollen, und wir haben aus Versehen "
        "einen\n"
        "weißen gepflanzt, und wenn die Königin es gewahr würde, würden wir "
        "Alle\n"
        "geköpft werden, müssen Sie wissen. So, sehen Sie Fräulein, "
        "versuchen\n"
        "wir, so gut es geht, ehe sie kommt --« In dem Augenblick rief Fünf, "
        "der\n"
        "ängstlich tiefer in den Garten hinein gesehen hatte: »Die Königin! "
        "die\n"
        "Königin!« und die drei Gärtner warfen sich sogleich flach auf's "
        "Gesicht.\n"
        "Es entstand ein Geräusch von vielen Schritten, und Alice blickte\n"
        "neugierig hin, die Königin zu sehen.\n"
        "\n"
        "Zuerst kamen zehn Soldaten, mit Keulen bewaffnet, sie hatten alle\n"
        "dieselbe Gestalt wie die Gärtner, rechteckig und flach, und an den "
        "vier\n"
        "Ecken die Hände und Füße; danach kamen zehn Herren vom Hofe, sie "
        "waren\n"
        "über und über mit Diamanten bedeckt und gingen paarweise, wie die\n"
        "Soldaten. Nach diesen kamen die königlichen Kinder, es waren ihrer "
        "zehn,\n"
        "und die lieben Kleinen kamen lustig gesprungen Hand in Hand, "
        "paarweise,\n"
        "sie waren ganz mit Herzen geschmückt. Darauf kamen die Gäste, meist\n"
        "Könige und Königinnen, und unter ihnen erkannte Alice das weiße\n"
        "Kaninchen; es unterhielt sich in etwas eiliger und aufgeregter "
        "Weise,\n"
        "lächelte bei Allem, was gesagt wurde und ging vorbei, ohne sie zu\n"
        "bemerken. Darauf folgte der Coeur-Bube, der die königliche Krone "
        "auf\n"
        "einem rothen Sammetkissen trug, und zuletzt in diesem großartigen "
        "Zuge\n"
        "kamen der Herzenskönig und die Herzenskönigin.\n"
        "\n"
        "Alice wußte nicht recht, ob sie sich nicht flach auf's Gesicht "
        "legen\n"
        "müsse, wie die drei Gärtner; aber sie konnte sich nicht erinnern, je "
        "von\n"
        "einer solchen Sitte bei Festzügen gehört zu haben. »Und außerdem, "
        "wozu\n"
        "gäbe es überhaupt Aufzüge,« dachte sie, »wenn alle Leute flach auf "
        "dem\n"
        "Gesichte liegen müßten, so daß sie sie nicht sehen könnten?« Sie "
        "blieb\n"
        "also stehen, wo sie war, und wartete.\n"
        "\n"
        "Als der Zug bei ihr angekommen war, blieben Alle stehen und sahen "
        "sie\n"
        "an, und die Königin sagte strenge: »Wer ist das?« Sie hatte den\n"
        "Coeur-Buben gefragt, der statt aller Antwort nur lächelte und "
        "Kratzfüße\n"
        "machte.\n"
        "\n"
        "»Schafskopf!« sagte die Königin, den Kopf ungeduldig zurückwerfend; "
        "und\n"
        "zu Alice gewandt fuhr sie fort: »Wie heißt du, Kind?«\n"
        "\n"
        "»Mein Name ist Alice, Euer Majestät zu dienen!« sagte Alice sehr\n"
        "höflich; aber sie dachte bei sich: »Ach was, es ist ja nur ein Pack\n"
        "Karten. Ich brauche mich nicht vor ihnen zu fürchten!«\n"
        "\n"
        "»Und wer sind diese drei?« fuhr die Königin fort, indem sie auf die "
        "drei\n"
        "Gärtner zeigte, die um den Rosenstrauch lagen; denn natürlich, da "
        "sie\n"
        "auf dem Gesichte lagen und das Muster auf ihrer Rückseite dasselbe "
        "war\n"
        "wie für das ganze Pack, so konnte sie nicht wissen, ob es Gärtner "
        "oder\n"
        "Soldaten oder Herren vom Hofe oder drei von ihren eigenen Kindern "
        "waren.\n"
        "\n"
        "»Woher soll ich das wissen?« sagte Alice, indem sie sich selbst "
        "über\n"
        "ihren Muth wunderte. »Es ist nicht meines Amtes.«\n"
        "\n"
        "Die Königin wurde purpurroth vor Wuth, und nachdem sie sie einen\n"
        "Augenblick wie ein wildes Thier angestarrt hatte, fing sie an zu\n"
        "brüllen: »Ihren Kopf ab! ihren Kopf --«\n"
        "\n"
        "»Unsinn!« sagte Alice sehr laut und bestimmt, und die Königin war "
        "still.\n"
        "\n"
        "Der König legte seine Hand auf ihren Arm und sagte milde: »Bedenke,\n"
        "meine Liebe, es ist nur ein Kind!«\n"
        "\n"
        "Die Königin wandte sich ärgerlich von ihm ab und sagte zu dem "
        "Buben:\n"
        "»Dreh' sie um!«\n"
        "\n"
        "Der Bube that es, sehr sorgfältig, mit einem Fuße.\n"
        "\n"
        "»Steht auf!« schrie die Königin mit durchdringender Stimme, und die "
        "drei\n"
        "Gärtner sprangen sogleich auf und fingen an sich zu verneigen vor "
        "dem\n"
        "König, der Königin, den königlichen Kindern, und Jedermann.\n"
        "\n"
        "»Laßt das sein!« eiferte die Königin. »Ihr macht mich schwindlig.« "
        "Und\n"
        "dann, sich nach dem Rosenstrauch umdrehend, fuhr sie fort: »Was habt "
        "ihr\n"
        "hier gethan?«\n"
        "\n"
        "»Euer Majestät zu dienen,« sagte Zwei in sehr demüthigem Tone und "
        "sich\n"
        "auf ein Knie niederlassend, »wir haben versucht --«\n"
        "\n"
        "»Ich sehe!« sagte die Königin, die unterdessen die Rosen untersucht\n"
        "hatte. »Ihre Köpfe ab!« und der Zug bewegte sich fort, während drei "
        "von\n"
        "den Soldaten zurückblieben um die unglücklichen Gärtner zu "
        "enthaupten,\n"
        "welche zu Alice liefen und sie um Schutz baten.\n"
        "\n"
        "»Ihr sollt nicht getödtet werden!« sagte Alice, und damit steckte "
        "sie\n"
        "sie in einen großen Blumentopf, der in der Nähe stand. Die drei "
        "Soldaten\n"
        "gingen ein Weilchen hier- und dorthin, um sie zu suchen, und dann\n"
        "schlossen sie sich ruhig wieder den Andern an.\n"
        "\n"
        "»Sind ihre Köpfe gefallen?« schrie die Königin sie an.\n"
        "\n"
        "»Ihre Köpfe sind fort, zu Euer Majestät Befehl!« schrien die "
        "Soldaten\n"
        "als Antwort.\n"
        "\n"
        "»Das ist gut!« schrie die Königin. »Kannst du Croquet spielen?«\n"
        "\n"
        "Die Soldaten waren still und sahen Alice an, da die Frage\n"
        "augenscheinlich an sie gerichtet war.\n"
        "\n"
        "»Ja!« schrie Alice.\n"
        "\n"
        "»Dann komm mit!« brüllte die Königin, und Alice schloß sich dem Zuge "
        "an,\n"
        "sehr neugierig, was nun geschehen werde.\n"
        "\n"
        "»Es ist -- es ist ein sehr schöner Tag!« sagte eine schüchterne "
        "Stimme\n"
        "neben ihr. Sie ging neben dem weißen Kaninchen, das ihr ängstlich "
        "in's\n"
        "Gesicht sah.\n"
        "\n"
        "»Sehr,« sagte Alice; -- »wo ist die Herzogin?«\n"
        "\n"
        "»Still! still!« sagte das Kaninchen in einem leisen, schnellen Tone. "
        "Es\n"
        "sah dabei ängstlich über seine Schulter, stellte sich dann auf die\n"
        "Zehen, hielt den Mund dicht an Alice's Ohr und wisperte: »Sie ist "
        "zum\n"
        "Tode verurtheilt.«\n"
        "\n"
        "»Wofür?« fragte diese.\n"
        "\n"
        "»Sagtest du: wie Schade?« fragte das Kaninchen.\n"
        "\n"
        "»Nein, das sagte ich nicht,« sagte Alice, »ich finde gar nicht, daß "
        "es\n"
        "Schade ist. Ich sagte: wofür?«\n"
        "\n"
        "»Sie hat der Königin eine Ohrfeige gegeben --« fing das Kaninchen "
        "an.\n"
        "Alice lachte hörbar. »Oh still!« flüsterte das Kaninchen in sehr\n"
        "erschreckten Tone. »Die Königin wird dich hören! Sie kam nämlich "
        "etwas\n"
        "spät, und die Königin sagte --«\n"
        "\n"
        "»Macht, daß ihr an eure Plätze kommt!« donnerte die Königin, und "
        "Alle\n"
        "fingen an in allen Richtungen durcheinander zu laufen, wobei sie "
        "Einer\n"
        "über den Andern stolperten; jedoch nach ein bis zwei Minuten waren "
        "sie\n"
        "in Ordnung, und das Spiel fing an.\n"
        "\n"
        "Alice dachte bei sich, ein so merkwürdiges Croquet-Feld habe sie in\n"
        "ihrem Leben nicht gesehen; es war voller Erhöhungen und Furchen, "
        "die\n"
        "Kugeln waren lebendige Igel, und die Schlägel lebendige Flamingos, "
        "und\n"
        "die Soldaten mußten sich umbiegen und auf Händen und Füßen stehen, "
        "um\n"
        "die Bogen zu bilden.\n"
        "\n"
        "Die Hauptschwierigkeit, die Alice zuerst fand, war, den Flamingo zu\n"
        "handhaben; sie konnte zwar ziemlich bequem seinen Körper unter "
        "ihrem\n"
        "Arme festhalten, so daß die Füße herunterhingen, aber wenn sie eben\n"
        "seinen Hals schön ausgestreckt hatte, und dem Igel nun einen Schlag "
        "mit\n"
        "seinem Kopf geben wollte, so richtete er sich auf und sah ihr mit "
        "einem\n"
        "so verdutzten Ausdruck in's Gesicht, daß sie sich nicht enthalten "
        "konnte\n"
        "laut zu lachen. Wenn sie nun seinen Kopf herunter gebogen hatte und "
        "eben\n"
        "wieder anfangen wollte zu spielen, so fand sie zu ihrem großen "
        "Verdruß,\n"
        "daß der Igel sich aufgerollt hatte und eben fortkroch; außerdem war\n"
        "gewöhnlich eine Erhöhung oder eine Furche gerade da im Wege, wo sie "
        "den\n"
        "Igel hinrollen wollte, und da die umgebogenen Soldaten fortwährend\n"
        "aufstanden und an eine andere Stelle des Grasplatzes gingen, so kam\n"
        "Alice bald zu der Ueberzeugung, daß es wirklich ein sehr schweres "
        "Spiel\n"
        "sei.\n"
        "\n"
        "Die Spieler spielten Alle zugleich, ohne zu warten, bis sie an der "
        "Reihe\n"
        "waren; dabei stritten sie sich immerfort und zankten um die Igel, "
        "und in\n"
        "sehr kurzer Zeit war die Königin in der heftigsten Wuth, stampfte "
        "mit\n"
        "den Füßen und schrie: »Schlagt ihr den Kopf ab!« ungefähr ein Mal "
        "jede\n"
        "Minute.\n"
        "\n"
        "Alice fing an, sich sehr unbehaglich zu fühlen, sie hatte zwar noch\n"
        "keinen Streit mit der Königin gehabt, aber sie wußte, daß sie "
        "keinen\n"
        "Augenblick sicher davor war, »und was,« dachte sie, »würde dann aus "
        "mir\n"
        "werden? die Leute hier scheinen schrecklich gern zu köpfen; es ist "
        "das\n"
        "größte Wunder, daß überhaupt noch welche am Leben geblieben sind!« "
        "Sie\n"
        "sah sich nach einem Ausgange um und überlegte, ob sie sich wohl "
        "ohne\n"
        "gesehen zu werden, fortschleichen könne, als sie eine merkwürdige\n"
        "Erscheinung in der Luft wahrnahm: sie schien ihr zuerst ganz\n"
        "räthselhaft, aber nachdem sie sie ein Paar Minuten beobachtet "
        "hatte,\n"
        "erkannte sie, daß es ein Grinsen war, und sagte bei sich: »Es ist "
        "die\n"
        "Grinse-Katze; jetzt werde ich Jemand haben, mit dem ich sprechen "
        "kann.«\n"
        "\n"
        "»Wie geht es dir?« sagte die Katze, sobald Mund genug da war, um "
        "damit\n"
        "zu sprechen.\n"
        "\n"
        "Alice wartete, bis die Augen erschienen, und nickte ihr zu. »Es "
        "nützt\n"
        "nichts mit ihr zu reden,« dachte sie, »bis ihre Ohren gekommen "
        "sind,\n"
        "oder wenigstens eins.« Den nächsten Augenblick erschien der ganze "
        "Kopf;\n"
        "da setzte Alice ihren Flamingo nieder und fing ihren Bericht von "
        "dem\n"
        "Spiele an, sehr froh, daß sie Jemand zum Zuhören hatte. Die Katze\n"
        "schien zu glauben, daß jetzt genug von ihr sichtbar sei, und es "
        "erschien\n"
        "weiter nichts.\n"
        "\n"
        "»Ich glaube, sie spielen gar nicht gerecht,« fing Alice in etwas\n"
        "klagendem Tone an, »und sie zanken sich Alle so entsetzlich, daß "
        "man\n"
        "sein eigenes Wort nicht hören kann -- und dann haben sie gar keine\n"
        "Spielregeln, wenigstens wenn sie welche haben, so beobachtet sie "
        "Niemand\n"
        "-- und du hast keine Idee, wie es Einen verwirrt, daß alle\n"
        "Croquet-Sachen lebendig sind; zum Beispiel da ist der Bogen, durch "
        "den\n"
        "ich das nächste Mal spielen muß, und geht am andern Ende des "
        "Grasplatzes\n"
        "spazieren -- und ich hätte den Igel der Königin noch eben treffen\n"
        "können, nur daß er fortrannte, als er meinen kommen sah!«\n"
        "\n"
        "»Wie gefällt dir die Königin?« fragte die Katze leise.\n"
        "\n"
        "»Ganz und gar nicht,« sagte Alice, »sie hat so sehr viel --« da "
        "bemerkte\n"
        "sie eben, daß die Königin dicht hinter ihr war und zuhörte, also "
        "setzte\n"
        "sie hinzu: »Aussicht zu gewinnen, daß es kaum der Mühe werth ist, "
        "das\n"
        "Spiel auszuspielen.«\n"
        "\n"
        "Die Königin lächelte und ging weiter.\n"
        "\n"
        "»Mit wem redest du da?« sagte der König, indem er an Alice "
        "herantrat\n"
        "und mit großer Neugierde den Katzenkopf ansah.\n"
        "\n"
        "»Es ist einer meiner Freunde -- ein Grinse-Kater,« sagte Alice;\n"
        "»erlauben Eure Majestät, daß ich ihn Ihnen vorstelle.«\n"
        "\n"
        "»Sein Aussehen gefällt mir gar nicht,« sagte der König; »er mag mir\n"
        "jedoch die Hand küssen, wenn er will.«\n"
        "\n"
        "»O, lieber nicht!« versetzte der Kater.\n"
        "\n"
        "»Sei nicht so impertinent,« sagte der König, »und sieh mich nicht "
        "so\n"
        "an!« Er stellte sich hinter Alice, als er dies sagte.\n"
        "\n"
        "»Der Kater sieht den König an, der König sieht den Kater an,« sagte\n"
        "Alice, »das habe ich irgendwo gelesen, ich weiß nur nicht mehr wo.«\n"
        "\n"
        "»Fort muß er,« sagte der König sehr entschieden, und rief der "
        "Königin\n"
        "zu, die gerade vorbeiging: »Meine Liebe! ich wollte, du ließest "
        "diesen\n"
        "Kater fortschaffen!«\n"
        "\n"
        "Die Königin kannte nur eine Art, alle Schwierigkeiten, große und "
        "kleine,\n"
        "zu beseitigen. »Schlagt ihm den Kopf ab!« sagte sie, ohne sich "
        "einmal\n"
        "umzusehen.\n"
        "\n"
        "»Ich werde den Henker selbst holen,« sagte der König eifrig und "
        "eilte\n"
        "fort.\n"
        "\n"
        "Alice dachte, sie wollte lieber zurück gehen und sehen, wie es mit "
        "dem\n"
        "Spiele stehe, da sie in der Entfernung die Stimme der Königin hörte, "
        "die\n"
        "vor Wuth außer sich war. Sie hatte sie schon drei Spieler zum Tode\n"
        "verurtheilen hören, weil sie ihre Reihe verfehlt hatten, und der "
        "Stand\n"
        "der Dinge behagte ihr gar nicht, da das Spiel in solcher Verwirrung "
        "war,\n"
        "daß sie nie wußte, ob sie an der Reihe sei oder nicht. Sie ging "
        "also,\n"
        "sich nach ihrem Igel umzusehen.\n"
        "\n"
        "Der Igel war im Kampfe mit einem andern Igel, was Alice eine\n"
        "vortreffliche Gelegenheit schien, einen mit dem andern zu treffen; "
        "die\n"
        "einzige Schwierigkeit war, daß ihr Flamingo nach dem andern Ende "
        "des\n"
        "Gartens gegangen war, wo Alice eben sehen konnte, wie er höchst\n"
        "ungeschickt versuchte, auf einen Baum zu fliegen.\n"
        "\n"
        "Als sie den Flamingo gefangen und zurückgebracht hatte, war der "
        "Kampf\n"
        "vorüber und die beiden Igel nirgends zu sehen. »Aber es kommt nicht\n"
        "drauf an,« dachte Alice, »da alle Bogen auf dieser Seite des "
        "Grasplatzes\n"
        "fortgegangen sind.« Sie steckte also ihren Flamingo unter den Arm, "
        "damit\n"
        "er nicht wieder fortliefe, und ging zurück, um mit ihrem Freunde "
        "weiter\n"
        "zu schwatzen.\n"
        "\n"
        "Als sie zum Cheshire-Kater zurück kam, war sie sehr erstaunt, einen\n"
        "großen Auflauf um ihn versammelt zu sehen: es fand ein großer\n"
        "Wortwechsel statt zwischen dem Henker, dem Könige und der Königin,\n"
        "welche alle drei zugleich sprachen, während die Uebrigen ganz still\n"
        "waren und sehr ängstlich aussahen.\n"
        "\n"
        "Sobald Alice erschien, wurde sie von allen dreien aufgefordert, den\n"
        "streitigen Punkt zu entscheiden, und sie wiederholten ihr ihre\n"
        "Beweisgründe, obgleich, da alle zugleich sprachen, man kaum "
        "verstehen\n"
        "konnte, was jeder Einzelne sagte.\n"
        "\n"
        "Der Henker behauptete, daß man keinen Kopf abschneiden könne, wo "
        "kein\n"
        "Körper sei, von dem man ihn abschneiden könne; daß er so etwas noch "
        "nie\n"
        "gethan habe, und jetzt über die Jahre hinaus sei, wo man etwas "
        "Neues\n"
        "lerne.\n"
        "\n"
        "Der König behauptete, daß Alles, was einen Kopf habe, geköpft "
        "werden\n"
        "könne, und daß man nicht so viel Unsinn schwatzen solle.\n"
        "\n"
        "Die Königin behauptete, daß wenn nicht in weniger als keiner Frist "
        "etwas\n"
        "geschehe, sie die ganze Gesellschaft würde köpfen lassen. (Diese\n"
        "letztere Bemerkung hatte der Versammlung ein so ernstes und "
        "ängstliches\n"
        "Aussehen gegeben.)\n"
        "\n"
        'Alice wußte nichts Besseres zu sagen als: "Er gehört der Herzogin, '
        "es\n"
        'wäre am besten sie zu fragen."\n'
        "\n"
        '"Sie ist im Gefängnis," sagte die Königin zum Henker, "hole sie '
        'her."\n'
        "Und der Henker lief davon wie ein Pfeil.\n"
        "\n"
        "Da wurde der Kopf des Katers undeutlicher und undeutlicher; und "
        "gerade\n"
        "in dem Augenblicke, als der Henker mit der Herzogin zurück kam,\n"
        "verschwand er gänzlich; der König und der Henker liefen ganz wild "
        "umher,\n"
        "ihn zu suchen, während die übrige Gesellschaft zum Spiele "
        "zurückging.\n"
        "\n"
        "»Du kannst dir gar nicht denken, wie froh ich bin, dich wieder zu "
        "sehen,\n"
        "du liebes altes Herz!« sagte die Herzogin, indem sie Alice "
        "liebevoll\n"
        "unterfaßte, und beide zusammen fortspazierten.\n"
        "\n"
        "Alice war sehr froh, sie bei so guter Laune zu finden, und dachte "
        "bei\n"
        "sich, es wäre vielleicht nur der Pfeffer, der sie so böse gemacht "
        "habe,\n"
        "als sie sich zuerst in der Küche trafen. »Wenn ich Herzogin bin,« "
        "sagte\n"
        "sie für sich (doch nicht in sehr hoffnungsvollem Tone), »will ich "
        "gar\n"
        "keinen Pfeffer in meiner Küche dulden. Suppe schmeckt sehr gut ohne "
        "--\n"
        "Am Ende ist es immer Pfeffer, der die Leute heftig macht,« sprach "
        "sie\n"
        "weiter, sehr glücklich, eine neue Art Regel erfunden zu haben, »und\n"
        "Essig, der sie sauertöpfisch macht -- und Kamillenthee, der sie "
        "bitter\n"
        "macht -- und Gerstenzucker und dergleichen, was Kinder zuckersüß "
        "macht.\n"
        "Ich wünschte nur, die großen Leute wüßten das, dann würden sie nicht "
        "so\n"
        "sparsam damit sein --«\n"
        "\n"
        "Sie hatte unterdessen die Herzogin ganz vergessen und schrak "
        "förmlich\n"
        "zusammen, als sie deren Stimme dicht an ihrem Ohre hörte. »Du denkst "
        "an\n"
        "etwas, meine Liebe, und vergißt darüber zu sprechen. Ich kann dir "
        "diesen\n"
        "Augenblick nicht sagen, was die Moral davon ist, aber es wird mir "
        "gleich\n"
        "einfallen.«\n"
        "\n"
        "»Vielleicht hat es keine,« hatte Alice den Muth zu sagen.\n"
        "\n"
        "»Still, still, Kind!« sagte die Herzogin. »Alles hat seine Moral, "
        "wenn\n"
        "man sie nur finden kann.« Dabei drängte sie sich dichter an Alice "
        "heran.\n"
        "\n"
        "Alice mochte es durchaus nicht gern, daß sie ihr so nahe kam: "
        "erstens,\n"
        "weil die Herzogin sehr häßlich war, und zweitens, weil sie gerade "
        "groß\n"
        "genug war, um ihr Kinn auf Alice's Schulter zu stützen, und es war "
        "ein\n"
        "unangenehm spitzes Kinn. Da sie aber nicht gern unhöflich sein "
        "wollte,\n"
        "so ertrug sie es, so gut sie konnte.\n"
        "\n"
        "»Das Spiel ist jetzt besser im Gange,« sagte sie, um die "
        "Unterhaltung\n"
        "fortzuführen.\n"
        "\n"
        "»So ist es,« sagte die Herzogin, »und die Moral davon ist -- Mit "
        "Liebe\n"
        "und Gesange hält man die Welt im Gange!«\n"
        "\n"
        "»Wer sagte denn,« flüsterte Alice, »es geschehe dadurch, daß Jeder "
        "vor\n"
        "seiner Thüre fege.«\n"
        "\n"
        "»Ah, sehr gut, das bedeutet ungefähr dasselbe,« sagte die Herzogin, "
        "und\n"
        "indem sie ihr spitzes kleines Kinn in Alice's Schulter einbohrte, "
        "fügte\n"
        "sie hinzu »und die Moral davon ist -- So viel Köpfe, so viel "
        "Sinne.«\n"
        "\n"
        "»Wie gern sie die Moral von Allem findet!« dachte Alice bei sich.\n"
        "\n"
        "»Du wunderst dich wahrscheinlich, warum ich meinen Arm nicht um "
        "deinen\n"
        "Hals lege,« sagte die Herzogin nach einer Pause; »die Wahrheit zu\n"
        "gestehen, ich traue der Laune deines Flamingos nicht ganz. Soll ich "
        "es\n"
        "versuchen?«\n"
        "\n"
        "»Er könnte beißen,« erwiderte Alice weislich, da sie sich "
        "keineswegs\n"
        "danach sehnte, das Experiment zu versuchen.\n"
        "\n"
        "»Sehr wahr,« sagte die Herzogin, »Flamingos und Senf beißen beide. "
        "Und\n"
        "die Moral davon ist: Gleich und Gleich gesellt sich gern.«\n"
        "\n"
        "»Aber der Flamingo ist ja ein Vogel und Senf ist kein Vogel,« "
        "wandte\n"
        "Alice ein.\n"
        "\n"
        "»Ganz recht, wie immer,« sagte die Herzogin, »wie deutlich du Alles\n"
        "ausdrücken kannst.«\n"
        "\n"
        "»Es ist, glaube ich, ein Mineral,« sagte Alice.\n"
        "\n"
        "»Versteht sich,« sagte die Herzogin, die Allem, was Alice sagte,\n"
        "beizustimmen schien, »in dem großen Senf-Bergwerk hier in der "
        "Gegend\n"
        "sind ganz vorzüglich gute Minen. Und die Moral davon ist, daß wir "
        "gute\n"
        "Miene zum bösen Spiel machen müssen.«\n"
        "\n"
        "»O, ich weiß!« rief Alice aus, die die letzte Bemerkung ganz "
        "überhört\n"
        "hatte, »es ist eine Pflanze. Es sieht nicht so aus, aber es ist "
        "eine.«\n"
        "\n"
        "»Ich stimme dir vollkommen bei,« sagte die Herzogin, »und die Moral\n"
        "davon ist: Sei was du zu scheinen wünschest! -- oder einfacher\n"
        "ausgedrückt: Bilde dir nie ein verschieden von dem zu sein was "
        "Anderen\n"
        "erscheint daß was du warest oder gewesen sein möchtest nicht "
        "verschieden\n"
        "von dem war daß was du gewesen warest ihnen erschienen wäre als wäre "
        "es\n"
        "verschieden.«\n"
        "\n"
        "»Ich glaube, ich würde das besser verstehen,« sagte Alice sehr "
        "höflich,\n"
        "»wenn ich es aufgeschrieben hätte; ich kann nicht ganz folgen, wenn "
        "Sie\n"
        "es sagen.«\n"
        "\n"
        "»Das ist noch gar nichts dagegen, was ich sagen könnte, wenn ich\n"
        "wollte,« antwortete die Herzogin in selbstzufriedenem Tone.\n"
        "\n"
        "»Bitte, bemühen Sie sich nicht, es noch länger zu sagen!« sagte "
        "Alice.\n"
        "\n"
        "»O, sprich nicht von Mühe!« sagte die Herzogin, »ich will dir Alles, "
        "was\n"
        "ich bis jetzt gesagt habe, schenken.«\n"
        "\n"
        "»Eine wohlfeile Art Geschenke!« dachte Alice, »ich bin froh, daß "
        "man\n"
        "nicht solche Geburtstagsgeschenke macht!« Aber sie getraute sich "
        "nicht,\n"
        "es laut zu sagen.\n"
        "\n"
        "»Wieder in Gedanken?« fragte die Herzogin und grub ihr spitzes "
        "kleines\n"
        "Kinn tiefer ein.\n"
        "\n"
        "»Ich habe das Recht, in Gedanken zu sein, wenn ich will,« sagte "
        "Alice\n"
        "gereizt, denn die Unterhaltung fing an, ihr langweilig zu werden.\n"
        "\n"
        "»Gerade so viel Recht,« sagte die Herzogin, »wie Ferkel zum Fliegen, "
        "und\n"
        "die M --«\n"
        "\n"
        "Aber, zu Alice's großem Erstaunen stockte hier die Stimme der "
        "Herzogin,\n"
        "und zwar mitten in ihrem Lieblingsworte »Moral«, und der Arm, der in "
        "dem\n"
        "ihrigen ruhte, fing an zu zittern. Alice sah auf, und da stand die\n"
        "Königin vor ihnen, mit über der Brust gekreuzten Armen, "
        "schwarzblickend\n"
        "wie ein Gewitter.\n"
        "\n"
        "»Ein schöner Tag, Majestät!« fing die Herzogin mit leiser schwacher\n"
        "Stimme an.\n"
        "\n"
        "»Ich will Sie schön gewarnt haben,« schrie die Königin und stampfte\n"
        "dabei mit dem Fuße: »Fort augenblicklich, entweder mit Ihnen oder "
        "mit\n"
        "Ihrem Kopfe! Wählen Sie!«\n"
        "\n"
        "Die Herzogin wählte und verschwand eilig.\n"
        "\n"
        "»Wir wollen weiter spielen,« sagte die Königin zu Alice, und diese, "
        "viel\n"
        "zu erschrocken, ein Wort zu erwiedern, folgte ihr langsam nach dem\n"
        "Croquet-Felde.\n"
        "\n"
        "Die übrigen Gäste hatten die Abwesenheit der Königin benutzt, um im\n"
        "Schatten auszuruhen; sobald sie sie jedoch kommen sahen, eilten sie\n"
        "augenblicklich zum Spiele zurück, indem die Königin einfach "
        "bemerkte,\n"
        "daß eine Minute Verzug ihnen das Leben kosten würde.\n"
        "\n"
        "Die ganze Zeit, wo sie spielten, hörte die Königin nicht auf, mit "
        "den\n"
        "andern Spielern zu zanken und zu schreien: »Schlagt ihm den Kopf "
        "ab!«\n"
        "oder: »Schlagt ihr den Kopf ab!« Diejenigen, welche sie verurtheilt\n"
        "hatte, wurden von den Soldaten in Verwahrsam geführt, die natürlich "
        "dann\n"
        "aufhören mußten, die Bogen zu bilden, so daß nach ungefähr einer "
        "halben\n"
        "Stunde keine Bogen mehr übrig waren, und alle Spieler, außer dem "
        "Könige,\n"
        "der Königin und Alice, in Verwahrsam und zum Tode verurtheilt "
        "waren.\n"
        "\n"
        "Da hörte die Königin, ganz außer Athem, auf, und sagte zu Alice: "
        "»Hast\n"
        "du die Falsche Schildkröte schon gesehen?«\n"
        "\n"
        "»Nein,« sagte Alice. »Ich weiß nicht einmal, was eine Falsche\n"
        "Schildkröte ist.«\n"
        "\n"
        "»Es ist das, woraus falsche Schildkrötensuppe gemacht wird,« sagte "
        "die\n"
        "Königin.\n"
        "\n"
        "»Ich habe weder eine gesehen, noch von einer gehört,« sagte Alice.\n"
        "\n"
        "»Komm schnell,« sagte die Königin, »sie soll dir ihre Geschichte\n"
        "erzählen.«\n"
        "\n"
        "Als sie mit einander fortgingen, hörte Alice den König leise zu der\n"
        "ganzen Versammlung sagen: »Ihr seid Alle begnadigt!« »Ach, das ist "
        "ein\n"
        "Glück!« sagte sie für sich, denn sie war über die vielen "
        "Enthauptungen,\n"
        "welche die Königin angeordnet hatte, ganz außer sich gewesen.\n"
        "\n"
        "Sie kamen bald zu einem Greifen, der in der Sonne lag und schlief. "
        "(Wenn\n"
        "ihr nicht wißt, was ein Greif ist, seht euch das Bild an.) »Auf, du\n"
        "Faulpelz,« sagte die Königin, »und bringe dies kleine Fräulein zu "
        "der\n"
        "falschen Schildkröte, sie möchte gern ihre Geschichte hören. Ich "
        "muß\n"
        "zurück und nach einigen Hinrichtungen sehen, die ich angeordnet "
        "habe;«\n"
        "damit ging sie fort und ließ Alice mit dem Greifen allein. Der "
        "Anblick\n"
        "des Thieres gefiel Alice nicht recht; aber im Ganzen genommen, "
        "dachte\n"
        "sie, würde es eben so sicher sein, bei ihm zu bleiben, als dieser\n"
        "grausamen Königin zu folgen, sie wartete also.\n"
        "\n"
        "Der Greif richtete sich auf und rieb sich die Augen: darauf sah er "
        "der\n"
        "Königin nach, bis sie verschwunden war; dann schüttelte er sich. "
        "»Ein\n"
        "köstlicher Spaß!« sagte der Greif, halb zu sich selbst, halb zu "
        "Alice.\n"
        "\n"
        "»Was ist ein Spaß?« fragte Alice.\n"
        "\n"
        "»Sie,« sagte der Greif. »Es ist Alles ihre Einbildung, das: Niemand "
        "wird\n"
        "niemals nicht hingerichtet. Komm schnell.«\n"
        "\n"
        "»Jeder sagte hier, komm schnell,« dachte Alice, indem sie ihm "
        "langsam\n"
        "nachging, »so viel bin ich in meinem Leben nicht hin und her "
        "kommandirt\n"
        "worden, nein, in meinem ganzen Leben nicht!«\n"
        "\n"
        "Sie brauchten nicht weit zu gehen, als sie schon die falsche "
        "Schildkröte\n"
        "in der Entfernung sahen, wie sie einsam und traurig auf einem\n"
        "Felsenriffe saß; und als sie näher kamen, hörte Alice sie seufzen, "
        "als\n"
        "ob ihr das Herz brechen wollte. Sie bedauerte sie herzlich. »Was "
        "für\n"
        "einen Kummer hat sie?« fragte sie den Greifen, und der Greif "
        "antwortete,\n"
        "fast in denselben Worten wie zuvor: »Es ist Alles ihre Einbildung, "
        "das;\n"
        "sie hat keinen Kummer nicht. Komm schnell.«\n"
        "\n"
        "Sie gingen also an die falsche Schildkröte heran, die sie mit\n"
        "thränenschweren Augen anblickte, aber nichts sagte.\n"
        "\n"
        "»Die kleine Mamsell hier,« sprach der Greif, »sie sagt, sie möchte "
        "gern\n"
        "deine Geschichte wissen, sagt sie.«\n"
        "\n"
        "»Ich will sie ihr erzählen,« sprach die falsche Schildkröte mit "
        "tiefer,\n"
        "hohler Stimme; »setzt euch beide her und sprecht kein Wort, bis ich\n"
        "fertig bin.«\n"
        "\n"
        "Gut, sie setzten sich hin und Keiner sprach mehre Minuten lang. "
        "Alice\n"
        "dachte bei sich: »Ich begreife nicht, wie sie je fertig werden "
        "kann,\n"
        "wenn sie nicht anfängt.« Aber sie wartete geduldig.\n"
        "\n"
        "»Einst,« sagte die falsche Schildkröte endlich mit einem tiefen "
        "Seufzer,\n"
        "»war ich eine wirkliche Schildkröte.«\n"
        "\n"
        "Auf diese Worte folgte ein sehr langes Schweigen, nur hin und "
        "wieder\n"
        "unterbrochen durch den Ausruf des Greifen »Hjckrrh!« und durch das\n"
        "heftige Schluchzen der falschen Schildkröte. Alice wäre beinah\n"
        "aufgestanden und hätte gesagt: »Danke sehr für die interessante\n"
        "Geschichte!« aber sie konnte nicht umhin zu denken, daß doch noch "
        "etwas\n"
        "kommen müsse; daher blieb sie sitzen und sagte nichts.\n"
        "\n"
        "»Als wir klein waren,« sprach die falsche Schildkröte endlich "
        "weiter,\n"
        "und zwar ruhiger, obgleich sie noch hin und wieder schluchzte, "
        "»gingen\n"
        "wir zur Schule in der See. Die Lehrerin war eine alte Schildkröte -- "
        "wir\n"
        "nannten sie Mamsell Schalthier --«\n"
        "\n"
        "»Warum nanntet ihr sie Mamsell Schalthier?« fragte Alice.\n"
        "\n"
        "»Sie schalt hier oder sie schalt da alle Tage, darum,« sagte die "
        "falsche\n"
        "Schildkröte ärgerlich; »du bist wirklich sehr dumm.«\n"
        "\n"
        "»Du solltest dich schämen, eine so dumme Frage zu thun,« setzte der\n"
        "Greif hinzu, und dann saßen beide und sahen schweigend die arme "
        "Alice\n"
        "an, die in die Erde hätte sinken mögen. Endlich sagte der Greif zu "
        "der\n"
        "falschen Schildkröte: »Fahr' zu, alte Kutsche! Laß uns nicht den "
        "ganzen\n"
        "Tag warten!« Und sie fuhr in folgenden Worten fort:\n"
        "\n"
        "»Ja, wir gingen zur Schule, in der See, ob ihr es glaubt oder nicht "
        "--«\n"
        "\n"
        "»Ich habe nicht gesagt, daß ich es nicht glaubte,« unterbrach sie "
        "Alice.\n"
        "\n"
        "»Ja, das hast du,« sagte die falsche Schildkröte.\n"
        "\n"
        "»Halt' den Mund!« fügte der Greif hinzu, ehe Alice antworten konnte. "
        "Die\n"
        "falsche Schildkröte fuhr fort.\n"
        "\n"
        "»Wir gingen in die allerbeste Schule; wir hatten vier und zwanzig\n"
        "Stunden regelmäßig jeden Tag.«\n"
        "\n"
        "»Das haben wir auf dem Lande auch,« sagte Alice, »darauf brauchst du "
        "dir\n"
        "nicht so viel einzubilden.«\n"
        "\n"
        "»Habt ihr auch Privatstunden außerdem?« fragte die falsche "
        "Schildkröte\n"
        "etwas kleinlaut.\n"
        "\n"
        "»Ja,« sagte Alice, »Französisch und Klavier.«\n"
        "\n"
        "»Und Wäsche?« sagte die falsche Schildkröte.\n"
        "\n"
        "»Ich dächte gar!« sagte Alice entrüstet.\n"
        "\n"
        "»Ah! dann gehst du in keine wirklich gute Schule,« sagte die "
        "falsche\n"
        "Schildkröte sehr beruhigt. »In unserer Schule stand immer am Ende "
        "der\n"
        "Rechnung, »Französisch, Klavierspielen, Wäsche -- extra.«\n"
        "\n"
        "»Das könnt ihr nicht sehr nöthig gehabt haben,« sagte Alice, »wenn "
        "ihr\n"
        "auf dem Grunde des Meeres wohntet.«\n"
        "\n"
        "»Ich konnte keine Privatstunden bezahlen,« sagte die falsche\n"
        "Schildkröte mit einem Seufzer. »Ich nahm nur den regelmäßigen\n"
        "Unterricht.«\n"
        "\n"
        "»Und was war das?« fragte Alice.\n"
        "\n"
        "»Legen und Treiben, natürlich, zu allererst,« erwiederte die "
        "falsche\n"
        "Schildkröte; »und dann die vier Abtheilungen vom Rechnen: Zusehen,\n"
        "Abziehen, Vervielfraßen und Stehlen.«\n"
        "\n"
        "»Ich habe nie von Vervielfraßen gehört,« warf Alice ein. »Was ist "
        "das?«\n"
        "\n"
        "Der Greif erhob beide Klauen voller Verwunderung. »Nie von "
        "Vervielfraßen\n"
        "gehört!« rief er aus. »Du weißt, was Verhungern ist? vermuthe ich.«\n"
        "\n"
        "»Ja,« sagte Alice unsicher, »es heißt -- nichts -- essen -- und "
        "davon --\n"
        "sterben.«\n"
        "\n"
        "»Nun,« fuhr der Greif fort, »wenn du nicht verstehst, was "
        "Vervielfraßen\n"
        "ist, dann bist du ein Pinsel.«\n"
        "\n"
        "Alice hatte allen Muth verloren, sich weiter danach zu erkundigen, "
        "und\n"
        "wandte sich daher an die falsche Schildkröte mit der Frage: »Was "
        "hattet\n"
        "ihr sonst noch zu lernen?«\n"
        "\n"
        "»Nun, erstens Gewichte,« erwiederte die falsche Schildkröte, indem "
        "sie\n"
        "die Gegenstände an den Pfoten aufzählte, »Gewichte, alte und neue, "
        "mit\n"
        "Seeographie; dann Springen -- der Springelehrer war ein alter\n"
        "Stockfisch, der ein Mal wöchentlich zu kommen pflegte, er lehrte "
        "uns\n"
        "Pfoten Reiben und Unarten, meerschwimmig Springen, Schillern und\n"
        "Imponiren.«\n"
        "\n"
        "»Wie war denn das?« fragte Alice.\n"
        "\n"
        "»Ich kann es dir nicht selbst zeigen,« sagte die falsche "
        "Schildkröte,\n"
        "»ich bin zu steif. Und der Greif hat es nicht gelernt.«\n"
        "\n"
        "»Hatte keine Zeit,« sagte der Greif; »ich hatte aber Stunden bei "
        "dem\n"
        "Lehrer der alten Sprachen. Das war ein alter Barsch, ja, das war "
        "er.«\n"
        "\n"
        "»Bei dem bin ich nicht gewesen,« sagte die falsche Schildkröte mit "
        "einem\n"
        "Seufzer, »er lehrte Zebräisch und Greifisch, sagten sie immer.«\n"
        "\n"
        "»Das that er auch, das that er auch, und besonders Laßsein,« sagte "
        "der\n"
        "Greif, indem er ebenfalls seufzte, worauf beide Thiere sich das "
        "Gesicht\n"
        "mit den Pfoten bedeckten.\n"
        "\n"
        "»Und wie viel Schüler wart ihr denn in einer Klasse?« sagte Alice, "
        "die\n"
        "schnell auf einen andern Gegenstand kommen wollte.\n"
        "\n"
        "»Zehn den ersten Tag,« sagte die falsche Schildkröte, »neun den\n"
        "nächsten, und so fort.«\n"
        "\n"
        "»Was für eine merkwürdige Einrichtung!« rief Alice aus.\n"
        "\n"
        "»Das ist der Grund, warum man Lehrer hält, weil sie die Klasse von "
        "Tag\n"
        "zu Tag leeren.«\n"
        "\n"
        "Dies war ein ganz neuer Gedanke für Alice, welchen sie gründlich\n"
        "überlegte, ehe sie wieder eine Bemerkung machte. »Den elften Tag "
        "müssen\n"
        "dann Alle frei gehabt haben?«\n"
        "\n"
        "»Natürlich!« sagte die falsche Schildkröte.\n"
        "\n"
        "»Und wie wurde es den zwölften Tag gemacht?« fuhr Alice eifrig "
        "fort.\n"
        "\n"
        "»Das ist genug von Stunden,« unterbrach der Greif sehr bestimmt:\n"
        "»erzähle ihr jetzt etwas von den Spielen.«\n"
        "\n"
        "Die falsche Schildkröte seufzte tief auf und wischte sich mit dem "
        "Rücken\n"
        "ihrer Pfote die Augen. Sie sah Alice an und versuchte zu sprechen, "
        "aber\n"
        "ein bis zwei Minuten lang erstickte lautes Schluchzen ihre Stimme.\n"
        "»Sieht aus, als ob sie einen Knochen in der Kehle hätt',« sagte der\n"
        "Greif und machte sich daran, sie zu schütteln und auf den Rücken zu\n"
        "klopfen. Endlich erhielt die falsche Schildkröte den Gebrauch ihrer\n"
        "Stimme wieder, und während Thränen ihre Wangen herabflossen, "
        "erzählte\n"
        "sie weiter.\n"
        "\n"
        "»Vielleicht hast du nicht viel unter dem Wasser gelebt --« (»Nein,«\n"
        "sagte Alice) -- »und vielleicht hast du nie die Bekanntschaft eines\n"
        "Hummers gemacht --« (Alice wollte eben sagen: »ich kostete einmal,« "
        "aber\n"
        "sie hielt schnell ein und sagte: »Nein, niemals«) -- »du kannst dir "
        "also\n"
        "nicht vorstellen, wie reizend ein Hummerballet ist.«\n"
        "\n"
        "»Nein, in der That nicht,« sagte Alice, »was für eine Art Tanz ist "
        "es?«\n"
        "\n"
        "»Nun,« sagte der Greif, »erst stellt man sich in einer Reihe am "
        "Strand\n"
        "auf --«\n"
        "\n"
        "»In zwei Reihen!« rief die falsche Schildkröte. »Seehunde, "
        "Schildkröten,\n"
        "Lachse, und so weiter; dann, wenn alle Seesterne aus dem Wege "
        "geräumt\n"
        "sind --«\n"
        "\n"
        "»Was gewöhnlich einige Zeit dauert,« unterbrach der Greif.\n"
        "\n"
        "»-- geht man zwei Mal vorwärts --«\n"
        "\n"
        "»Jeder einen Hummer zum Tanze führend!« rief der Greif.\n"
        "\n"
        "»Natürlich,« sagte die falsche Schildkröte: »zwei Mal vorwärts, "
        "wieder\n"
        "paarweis gestellt --«\n"
        "\n"
        "»-- wechselt die Hummer, und geht in derselben Ordnung zurück,« fuhr "
        "der\n"
        "Greif fort.\n"
        "\n"
        "»Dann, mußt du wissen,« fiel die falsche Schildkröte ein, »wirft man "
        "die\n"
        "--«\n"
        "\n"
        "»Die Hummer!« schrie der Greif mit einem Luftsprunge.\n"
        "\n"
        "»-- so weit in's Meer, als man kann --«\n"
        "\n"
        "»Schwimmt ihnen nach!« kreischte der Greif.\n"
        "\n"
        "»Schlägt einen Purzelbaum im Wasser!« rief die falsche Schildkröte,\n"
        "indem sie unbändig umhersprang.\n"
        "\n"
        "»Wechselt die Hummer wieder!« heulte der Greif mit erhobener "
        "Stimme.\n"
        "\n"
        "»Zurück an's Land, und -- das ist die ganze erste Figur,« sagte die\n"
        "falsche Schildkröte, indem ihre Stimme plötzlich sank; und beide "
        "Thiere,\n"
        "die bis dahin wie toll umhergesprungen waren, setzten sich sehr "
        "betrübt\n"
        "und still nieder und sahen Alice an.\n"
        "\n"
        "»Es muß ein sehr hübscher Tanz sein,« sagte Alice ängstlich.\n"
        "\n"
        "»Möchtest du eine kleine Probe sehen?« fragte die falsche "
        "Schildkröte.\n"
        "\n"
        "»Sehr gern,« sagte Alice.\n"
        "\n"
        "»Komm, laß uns die erste Figur versuchen!« sagte die falsche "
        "Schildkröte\n"
        "zum Greifen. »Wir können es ohne Hummer, glaube ich. Wer soll "
        "singen?«\n"
        "\n"
        "»Oh, singe du!« sagte der Greif. »Ich habe die Worte vergessen.«\n"
        "\n"
        "So fingen sie denn an, feierlich im Kreise um Alice zu tanzen; "
        "zuweilen\n"
        "traten sie ihr auf die Füße, wenn sie ihr zu nahe kamen; die "
        "falsche\n"
        "Schildkröte sang dazu, sehr langsam und traurig, Folgendes: --\n"
        "\n"
        "    Zu der Schnecke sprach ein Weißfisch: »Kannst du denn nicht\n"
        "        schneller gehn?\n"
        "    Siehst du denn nicht die Schildkröten und die Hummer\n"
        "        alle stehn?\n"
        "    Hinter uns da kommt ein Meerschwein, und es tritt mir auf\n"
        "        den Schwanz;\n"
        "    Und sie warten an dem Strande, daß wir kommen zu\n"
        "        dem Tanz.\n"
        "    Willst du denn nicht, willst du denn nicht, willst du kommen\n"
        "        zu dem Tanz?\n"
        "    Willst du denn nicht, willst du denn nicht, willst du kommen\n"
        "        zu dem Tanz?«\n"
        "\n"
        "    »Nein, du kannst es nicht ermessen, wie so herrlich es wird "
        "sein,\n"
        "    Nehmen sie uns mit den Hummern, werfen uns in's Meer hinein!«\n"
        "    Doch die Schnecke thät nicht trauen. »Das gefällt mir doch nicht "
        "ganz!\n"
        "    Viel zu weit, zu weit! ich danke -- gehe nicht mit euch zum "
        "Tanz!\n"
        "    Nein, ich kann, ich mag, ich will nicht, kann nicht kommen zu "
        "dem Tanz!\n"
        "    Nein, ich kann, ich mag, ich will nicht, mag nicht kommen zu dem "
        "Tanz!«\n"
        "\n"
        "    Und der Weißfisch sprach dagegen: »'s kommt ja nicht drauf an, "
        "wie\n"
        "        weit!\n"
        "    Ist doch wohl ein andres Ufer, drüben auf der andern Seit'!\n"
        "    Und noch viele schöne Küsten giebt es außer Engelland's;\n"
        "    Nur nicht blöde, liebe Schnecke, komm' geschwind mit mir zum "
        "Tanz!\n"
        "    Willst du denn nicht, willst du denn nicht, willst du kommen zu "
        "dem\n"
        "        Tanz?\n"
        "    Willst du denn nicht, willst du denn nicht, willst nicht kommen "
        "zu dem\n"
        "        Tanz?«\n"
        "\n"
        "»Danke sehr, es ist sehr, sehr interessant, diesem Tanze "
        "zuzusehen,«\n"
        "sagte Alice, obgleich sie sich freute, daß er endlich vorüber war; "
        "»und\n"
        "das komische Lied von dem Weißfisch gefällt mir so!«\n"
        "\n"
        "»Oh, was die Weißfische anbelangt,« sagte die falsche Schildkröte, "
        "»die\n"
        "-- du hast sie doch gesehen?«\n"
        "\n"
        "»Ja,« sagte Alice, »ich habe sie oft gesehen, bei'm Mitt --« sie "
        "hielt\n"
        "schnell inne.\n"
        "\n"
        "»Ich weiß nicht, wer Mitt sein mag,« sagte die falsche Schildkröte,\n"
        "»aber da du sie so oft gesehen hast, so weißt du natürlich, wie sie\n"
        "aussehen?«\n"
        "\n"
        "»Ja, ich glaube,« sagte Alice nachdenklich, »sie haben den Schwanz "
        "im\n"
        "Maule, -- und sind ganz mit geriebener Semmel bestreut.«\n"
        "\n"
        "»Die geriebene Semmel ist ein Irrthum,« sagte die falsche "
        "Schildkröte;\n"
        "»sie würde in der See bald abgespült werden. Aber den Schwanz haben "
        "sie\n"
        "im Maule, und der Grund ist« -- hier gähnte die falsche Schildkröte "
        "und\n"
        "machte die Augen zu. -- »Sage ihr Alles das von dem Grunde,« sprach "
        "sie\n"
        "zum Greifen.\n"
        "\n"
        "»Der Grund ist,« sagte der Greif, »daß sie durchaus im Hummerballet\n"
        "mittanzen wollten. So wurden sie denn in die See hinein geworfen. "
        "So\n"
        "mußten sie denn sehr weit fallen. So kamen ihnen denn die Schwänze "
        "in\n"
        "die Mäuler. So konnten sie sie denn nicht wieder heraus bekommen. So "
        "ist\n"
        "es.«\n"
        "\n"
        "»Danke dir,« sagte Alice, »es ist sehr interessant. Ich habe nie so "
        "viel\n"
        "vom Weißfisch zu hören bekommen.«\n"
        "\n"
        "»Ich kann dir noch mehr über ihn sagen, wenn du willst,« sagte der\n"
        "Greif, »weißt du, warum er Weißfisch heißt?«\n"
        "\n"
        "»Ich habe darüber noch nicht nachgedacht,« sagte Alice. »Warum?«\n"
        "\n"
        "»Darum eben,« sagte der Greif mit tiefer, feierlicher Stimme, »weil "
        "man\n"
        "so wenig von ihm weiß. Nun aber mußt du uns auch etwas von deinen\n"
        "Abenteuern erzählen.«\n"
        "\n"
        "»Ich könnte euch meine Erlebnisse von heute früh an erzählen,« "
        "sagte\n"
        "Alice verschämt, »aber bis gestern zurück zu gehen, wäre ganz "
        "unnütz,\n"
        "weil ich da jemand Anderes war.«\n"
        "\n"
        "»Erkläre das deutlich,« sagte die falsche Schildkröte.\n"
        "\n"
        "»Nein, die Erlebnisse erst,« sagte der Greif in ungeduldigem Tone,\n"
        "»Erklärungen nehmen so schrecklich viel Zeit fort.«\n"
        "\n"
        "Alice fing also an, ihnen ihre Abenteuer von da an zu erzählen, wo "
        "sie\n"
        "das weiße Kaninchen zuerst gesehen hatte. Im Anfange war sie etwas\n"
        "ängstlich, die beiden Thiere kamen ihr so nah, eins auf jeder "
        "Seite,\n"
        "und sperrten Augen und Mund so weit auf; aber nach und nach wurde "
        "sie\n"
        "dreister. Ihre Zuhörer waren ganz ruhig, bis sie an die Stelle kam, "
        "wo\n"
        "sie der Raupe 'Ihr seid alt, Vater Martin' hergesagt hatte, und wo\n"
        "lauter andere Worte gekommen waren, da holte die falsche "
        "Schildkröte\n"
        "tief Athem und sagte: »das ist sehr merkwürdig.«\n"
        "\n"
        "»Es ist Alles so merkwürdig, wie nur möglich,« sagte der Greif.\n"
        "\n"
        "»Es kam ganz verschieden!« wiederholte die falsche Schildkröte\n"
        "gedankenvoll. »Ich möchte sie wohl etwas hersagen hören. Sage ihr, "
        "daß\n"
        "sie anfangen soll.« Sie sah den Greifen an, als ob sie dächte, daß "
        "er\n"
        "einigen Einfluß auf Alice habe.\n"
        "\n"
        "»Steh' auf und sage her: 'Preisend mit viel schönen Reden',« sagte "
        "der\n"
        "Greif.\n"
        "\n"
        "»Wie die Geschöpfe alle Einen kommandiren und Gedichte aufsagen "
        "lassen!«\n"
        "dachte Alice, »dafür könnte ich auch lieber gleich in der Schule "
        "sein.«\n"
        "Sie stand jedoch auf und fing an, das Gedicht herzusagen; aber ihr "
        "Kopf\n"
        "war so voll von dem Hummerballet, daß sie kaum wußte, was sie sagte, "
        "und\n"
        "die Worte kamen sehr sonderbar: --\n"
        "\n"
        "    »Preisend mit viel schönen Kniffen seiner Scheeren Werth und "
        "Zahl,\n"
        "    Stand der Hummer vor dem Spiegel in der schönen rothen Schal'!\n"
        "    »Herrlich,« sprach der Fürst der Krebse, »steht mir dieser lange "
        "Bart!«\n"
        "    Rückt die Füße mit der Nase auswärts, als er dieses sagt.«\n"
        "\n"
        "»Das ist anders, als ich's als Kind gesagt habe,« sagte der Greif.\n"
        "\n"
        "»Ich habe es zwar noch niemals gehört,« sagte die falsche "
        "Schildkröte;\n"
        "»aber es klingt wie blühender Unsinn.«\n"
        "\n"
        "Alice erwiederte nichts; sie setzte sich, bedeckte das Gesicht mit\n"
        "beiden Händen und überlegte, ob wohl je wieder irgend etwas "
        "natürlich\n"
        "sein würde.\n"
        "\n"
        "»Ich möchte es gern erklärt haben,« sagte die falsche Schildkröte.\n"
        "\n"
        "»Sie kann's nicht erklären,« warf der Greif schnell ein. »Sage den\n"
        "nächsten Vers.«\n"
        "\n"
        "»Aber das von den Füßen?« fragte die falsche Schildkröte wieder. "
        "»Wie\n"
        "kann er sie mit der Nase auswärts rücken?«\n"
        "\n"
        "»Es ist die erste Position bei'm Tanzen,« sagte Alice; aber sie war "
        "über\n"
        "Alles dies entsetzlich verwirrt und hätte am liebsten aufgehört.\n"
        "\n"
        "»Sage den nächsten Vers!« wiederholte der Greif ungeduldig, »er "
        "fängt\n"
        "an: 'Seht mein Land!'«\n"
        "\n"
        "Alice wagte nicht, es abzuschlagen, obgleich sie überzeugt war, es "
        "würde\n"
        "Alles falsch kommen, sie fuhr also mit zitternder Stimme fort: --\n"
        "\n"
        "    »Seht mein Land und grüne Fluten,« sprach ein fetter Lachs vom "
        "Rhein;\n"
        "    Goldne Schuppen meine Rüstung, und mit Austern trink' ich "
        "Wein.«\n"
        "\n"
        "»Wozu sollen wir das dumme Zeug mit anhören,« unterbrach sie die "
        "falsche\n"
        "Schildkröte, »wenn sie es nicht auch erklären kann? Es ist das\n"
        "verworrenste Zeug, das ich je gehört habe!«\n"
        "\n"
        "»Ja, ich glaube auch, es ist besser du hörst auf,« sagte der Greif, "
        "und\n"
        "Alice gehorchte nur zu gern.\n"
        "\n"
        "»Sollen wir noch eine Figur von dem Hummerballet versuchen?« fuhr "
        "der\n"
        "Greif fort. »Oder möchtest du lieber, daß die falsche Schildkröte "
        "dir\n"
        "ein Lied vorsingt?«\n"
        "\n"
        "»Oh, ein Lied! bitte, wenn die falsche Schildkröte so gut sein "
        "will,«\n"
        "antwortete Alice mit solchem Eifer, daß der Greif etwas beleidigt "
        "sagte:\n"
        "»Hm! der Geschmack ist verschieden! Singe ihr vor "
        "'Schildkrötensuppe',\n"
        "hörst du, alte Tante?«\n"
        "\n"
        "Die falsche Schildkröte seufzte tief auf und fing an, mit halb von\n"
        "Schluchzen erstickter Stimme, so zu singen: --\n"
        "\n"
        "    »Schöne Suppe, so schwer und so grün,\n"
        "    Dampfend in der heißen Terrin'!\n"
        "    Wem nach einem so schönen Gericht\n"
        "    Wässerte denn der Mund wohl nicht?\n"
        "    Kön'gin der Suppen, du schönste Supp'!\n"
        "    Kön'gin der Suppen, du schönste Supp'!\n"
        "      Wu -- underschöne Su -- uppe!\n"
        "      Wu -- underschöne Su -- uppe!\n"
        "      Kö -- önigin der Su -- uppen,\n"
        "      Wunder-wunderschöne Supp'!\n"
        "\n"
        "    Schöne Suppe, wer fragt noch nach Fisch,\n"
        "    Wildpret oder was sonst auf dem Tisch?\n"
        "    Alles lassen wir stehen zu p\n"
        "    Reisen allein die wunderschöne Supp',\n"
        "    Preisen allein die wunderschöne Supp'!\n"
        "      Wu -- underschöne Su -- uppe!\n"
        "      Wu -- underschöne Su -- uppe!\n"
        "      Kö -- önigin der Su -- uppen,\n"
        "      Wunder-wunderschöne Supp'!\n"
        "\n"
        "»Den Chor noch einmal!« rief der Greif, und die falsche Schildkröte\n"
        "hatte ihn eben wieder angefangen, als ein Ruf: »Das Verhör fängt "
        "an!« in\n"
        "der Ferne erscholl.\n"
        "\n"
        "»Komm schnell!« rief der Greif, und Alice bei der Hand nehmend lief "
        "er\n"
        "fort, ohne auf das Ende des Gesanges zu warten.\n"
        "\n"
        "»Was für ein Verhör?« keuchte Alice bei'm Rennen; aber der Greif\n"
        "antwortete nichts als: »Komm schnell!« und rannte weiter, während\n"
        "schwächer und schwächer, vom Winde getragen, die Worte ihnen "
        "folgten: --\n"
        "\n"
        "    »Kö -- önigin der Su -- uppen,\n"
        "    Wunder-wunderschöne Supp'!«\n"
        "\n"
        "Der König und die Königin der Herzen saßen auf ihrem Throne, als "
        "sie\n"
        "ankamen, und eine große Menge war um sie versammelt -- allerlei "
        "kleine\n"
        "Vögel und Thiere, außerdem das ganze Pack Karten: der Bube stand "
        "vor\n"
        "ihnen, in Ketten, einen Soldaten an jeder Seite, um ihn zu "
        "bewachen;\n"
        "dicht bei dem Könige befand sich das weiße Kaninchen, eine Trompete "
        "in\n"
        "einer Hand, in der andern eine Pergamentrolle. Im Mittelpunkte des\n"
        "Gerichtshofes stand ein Tisch mit einer Schüssel voll Torten: sie "
        "sahen\n"
        "so appetitlich aus, daß der bloße Anblick Alice ganz hungrig darauf\n"
        "machte. -- »Ich wünschte, sie machten schnell mit dem Verhör und\n"
        "reichten die Erfrischungen herum.« Aber dazu schien wenig Aussicht "
        "zu\n"
        "sein, so daß sie anfing, Alles genau in Augenschein zu nehmen, um "
        "sich\n"
        "die Zeit zu vertreiben.\n"
        "\n"
        "Alice war noch nie in einem Gerichtshofe gewesen, aber sie hatte in\n"
        "ihren Büchern davon gelesen und bildete sich etwas Rechtes darauf "
        "ein,\n"
        "daß sie Alles, was sie dort sah, bei Namen zu nennen wußte. »Das ist "
        "der\n"
        "Richter,« sagte sie für sich, »wegen seiner großen Perrücke.«\n"
        "\n"
        "Der Richter war übrigens der König, und er trug die Krone über der\n"
        "Perücke (seht euch das Titelbild an, wenn ihr wissen wollt, wie), es "
        "sah\n"
        "nicht aus, als sei es ihm bequem, und sicherlich stand es ihm nicht "
        "gut.\n"
        "\n"
        "»Und jene zwölf kleinen Thiere da sind vermuthlich die "
        "Geschwornen,«\n"
        "dachte Alice. Sie wiederholte sich selbst dies Wort zwei bis drei "
        "Mal,\n"
        "weil sie so stolz darauf war; denn sie glaubte, und das mit Recht, "
        "daß\n"
        "wenig kleine Mädchen ihres Alters überhaupt etwas von diesen Sachen\n"
        "wissen würden.\n"
        "\n"
        "Die zwölf Geschwornen schrieben alle sehr eifrig auf "
        "Schiefertafeln.\n"
        "»Was thun sie?« fragte Alice den Greifen in's Ohr. »Sie können ja "
        "noch\n"
        "nichts aufzuschreiben haben, ehe das Verhör beginnt.«\n"
        "\n"
        "»Sie schreiben ihre Namen auf,« sagte ihr der Greif in's Ohr, »weil "
        "sie\n"
        "bange sind, sie zu vergessen, ehe das Verhör zu Ende ist.«\n"
        "\n"
        "»Dumme Dinger!« fing Alice entrüstet ganz laut an; aber sie hielt\n"
        "augenblicklich inne, denn das weiße Kaninchen rief aus: »Ruhe im "
        "Saal!«\n"
        "und der König setzte seine Brille auf und blickte spähend umher, um "
        "zu\n"
        "sehen, wer da gesprochen habe.\n"
        "\n"
        "Alice konnte ganz deutlich sehen, daß alle Geschworne »dumme "
        "Dinger!«\n"
        "auf ihre Tafeln schrieben, und sie merkte auch, daß Einer von ihnen\n"
        "nicht wußte, wie es geschrieben wird, und seinen Nachbar fragen "
        "mußte.\n"
        "»Die Tafeln werden in einem schönen Zustande sein, wenn das Verhör\n"
        "vorüber ist!« dachte Alice.\n"
        "\n"
        "Einer der Geschwornen hatte einen Tafelstein, der quiekste. Das "
        "konnte\n"
        "Alice natürlich nicht aushalten, sie ging auf die andere Seite des\n"
        "Saales, gelangte dicht hinter ihn und fand sehr bald eine "
        "Gelegenheit,\n"
        "den Tafelstein fortzunehmen. Sie hatte es so schnell gethan, daß "
        "der\n"
        "arme kleine Geschworne (es war Wabbel), durchaus nicht begreifen "
        "konnte,\n"
        "wo sein Griffel hingekommen war; nachdem er ihn also überall "
        "gesucht\n"
        "hatte, mußte er sich endlich entschließen, mit einem Finger zu\n"
        "schreiben, und das war von sehr geringem Nutzen, da es keine Spuren "
        "auf\n"
        "der Tafel zurückließ.\n"
        "\n"
        "»Herold, verlies die Anklage!« sagte der König.\n"
        "\n"
        "Da blies das weiße Kaninchen drei Mal in die Trompete, entfaltete "
        "darauf\n"
        "die Pergamentrolle und las wie folgt: --\n"
        "\n"
        "    »Coeur-Königin, sie buk Kuchen,\n"
        "    Juchheisasah, juchhe!\n"
        "    Coeur-Bube kam, die Kuchen nahm.\n"
        "    Wo sind sie nun? O weh!«\n"
        "\n"
        "»Gebt euer Urtheil ab!« sprach der König zu den Geschwornen.\n"
        "\n"
        "»Noch nicht, noch nicht!« unterbrach ihn das Kaninchen schnell. »Da\n"
        "kommt noch Vielerlei erst.«\n"
        "\n"
        "»Laßt den ersten Zeugen eintreten!« sagte der König, worauf das\n"
        "Kaninchen drei Mal in die Trompete blies und ausrief: »Erster "
        "Zeuge!«\n"
        "\n"
        "Der erste Zeuge war der Hutmacher. Er kam herein, eine Tasse in "
        "einer\n"
        "Hand und in der andern ein Stück Butterbrot haltend. »Ich bitte um\n"
        "Verzeihung, Eure Majestät, daß ich das mitbringe; aber ich war "
        "nicht\n"
        "ganz fertig mit meinem Thee, als nach mir geschickt wurde.«\n"
        "\n"
        "»Du hättest aber damit fertig sein sollen,« sagte der König. »Wann "
        "hast\n"
        "du damit angefangen?«\n"
        "\n"
        "Der Hutmacher sah den Faselhasen an, der ihm in den Gerichtssaal "
        "gefolgt\n"
        "war, Arm in Arm mit dem Murmelthier. »Vierzehnten März, glaube ich "
        "war\n"
        "es,« sagte er.\n"
        "\n"
        "»Funfzehnten,« sagte der Faselhase.\n"
        "\n"
        "»Sechzehnten,« fügte das Murmelthier hinzu.\n"
        "\n"
        "»Nehmt das zu Protokoll,« sagte der König zu den Geschwornen, und "
        "die\n"
        "Geschwornen schrieben eifrig die drei Daten auf ihre Tafeln, "
        "addirten\n"
        "sie dann und machten die Summe zu Groschen und Pfennigen.\n"
        "\n"
        "»Nimm deinen Hut ab,« sagte der König zum Hutmacher.\n"
        "\n"
        "»Es ist nicht meiner,« sagte der Hutmacher.\n"
        "\n"
        "»Gestohlen!« rief der König zu den Geschwornen gewendet aus, welche\n"
        "sogleich die Thatsache notirten.\n"
        "\n"
        "»Ich halte sie zum Verkauf,« fügte der Hutmacher als Erklärung "
        "hinzu,\n"
        "»ich habe keinen eigenen. Ich bin ein Hutmacher.«\n"
        "\n"
        "Da setzte sich die Königin die Brille auf und fing an, den "
        "Hutmacher\n"
        "scharf zu beobachten, was ihn sehr blaß und unruhig machte.\n"
        "\n"
        "»Gieb du deine Aussage,« sprach der König, »und sei nicht "
        "ängstlich,\n"
        "oder ich lasse dich auf der Stelle hängen.«\n"
        "\n"
        "Dies beruhigte den Zeugen augenscheinlich nicht; er stand "
        "abwechselnd\n"
        "auf dem linken und rechten Fuße, sah die Königin mit großem "
        "Unbehagen\n"
        "an, und in seiner Befangenheit biß er ein großes Stück aus seiner\n"
        "Theetasse statt aus seinem Butterbrot.\n"
        "\n"
        "Gerade in diesem Augenblick spürte Alice eine seltsame Empfindung, "
        "die\n"
        "sie sich durchaus nicht erklären konnte, bis sie endlich merkte, was "
        "es\n"
        "war: sie fing wieder an zu wachsen, und sie wollte sogleich "
        "aufstehen\n"
        "und den Gerichtshof verlassen; aber nach weiterer Ueberlegung "
        "beschloß\n"
        "sie zu bleiben, wo sie war, so lange sie Platz genug hatte.\n"
        "\n"
        "»Du brauchtest mich wirklich nicht so zu drängen,« sagte das\n"
        "Murmelthier, welches neben ihr saß. »Ich kann kaum athmen.«\n"
        "\n"
        "»Ich kann nicht dafür,« sagte Alice bescheiden, »ich wachse.«\n"
        "\n"
        "»Du hast kein Recht dazu, hier zu wachsen,« sagte das Murmelthier.\n"
        "\n"
        "»Rede nicht solchen Unsinn,« sagte Alice dreister; »du weißt recht "
        "gut,\n"
        "daß du auch wächst.«\n"
        "\n"
        "»Ja, aber ich wachse in vernünftigem Maßstabe,« sagte das "
        "Murmelthier,\n"
        "»nicht auf so lächerliche Art.« Dabei stand es verdrießlich auf und "
        "ging\n"
        "an die andere Seite des Saales.\n"
        "\n"
        "Die ganze Zeit über hatte die Königin unablässig den Hutmacher\n"
        "angestarrt, und gerade als das Murmelthier durch den Saal ging, "
        "sprach\n"
        "sie zu einem der Gerichtsbeamten: »Bringe mir die Liste der Sänger "
        "im\n"
        "letzten Concerte!« worauf der unglückliche Hutmacher so zitterte, "
        "daß\n"
        "ihm beide Schuhe abflogen.\n"
        "\n"
        "»Gieb deine Aussage,« wiederholte der König ärgerlich, »oder ich "
        "werde\n"
        "dich hinrichten lassen, ob du dich ängstigst oder nicht.«\n"
        "\n"
        "»Ich bin ein armer Mann, Eure Majestät,« begann der Hutmacher mit\n"
        "zitternder Stimme, »und ich hatte eben erst meinen Thee angefangen "
        "--\n"
        "nicht länger als eine Woche ungefähr -- und da die Butterbrote so "
        "dünn\n"
        "wurden -- und es Teller und Töpfe in den Thee schneite.«\n"
        "\n"
        "»Teller und Töpfe -- was?« fragte der König.\n"
        "\n"
        "»Es fing mit dem Thee an,« erwiederte der Hutmacher.\n"
        "\n"
        "»Natürlich fangen Teller und Töpfe mit einem T an. Hältst du mich "
        "für\n"
        "einen Esel? Rede weiter!«\n"
        "\n"
        "»Ich bin ein armer Mann,« fuhr der Hutmacher fort, »und seitdem "
        "schneite\n"
        "Alles -- der Faselhase sagte nur --«\n"
        "\n"
        "»Nein, ich hab's nicht gesagt!« unterbrach ihn der Faselhase "
        "schnell.\n"
        "\n"
        "»Du hast's wohl gesagt!« rief der Hutmacher.\n"
        "\n"
        "»Ich läugne es!« sagte der Faselhase.\n"
        "\n"
        "»Er läugnet es!« sagte der König: »laßt den Theil der Aussage "
        "fort.«\n"
        "\n"
        "»Gut, auf jeden Fall hat's das Murmelthier gesagt --« fuhr der "
        "Hutmacher\n"
        "fort, indem er sich ängstlich umsah, ob es auch läugnen würde; aber "
        "das\n"
        "Murmelthier läugnete nichts, denn es war fest eingeschlafen. "
        "»Dann,«\n"
        "sprach der Hutmacher weiter, »schnitt ich noch etwas Butterbrot --«\n"
        "\n"
        "»Aber was hat das Murmelthier gesagt?« fragte einer der "
        "Geschwornen.\n"
        "\n"
        "»Das ist mir ganz entfallen,« sagte der Hutmacher.\n"
        "\n"
        "»Aber es muß dir wieder einfallen,« sagte der König, »sonst lasse "
        "ich\n"
        "dich köpfen.«\n"
        "\n"
        "Der unglückliche Hutmacher ließ Tasse und Butterbrot fallen und "
        "ließ\n"
        "sich auf ein Knie nieder. »Ich bin ein armseliger Mann, Eure "
        "Majestät,«\n"
        "fing er an.\n"
        "\n"
        "»Du bist ein sehr armseliger Redner,« sagte der König.\n"
        "\n"
        "Hier klatschte eins der Meerschweinchen Beifall, was sofort von den\n"
        "Gerichtsdienern unterdrückt wurde. (Da dies ein etwas schweres Wort "
        "ist,\n"
        "so will ich beschreiben, wie es gemacht wurde. Es war ein großer\n"
        "Leinwandsack bei der Hand, mit Schnüren zum Zusammenziehen: da "
        "hinein\n"
        "wurde das Meerschweinchen gesteckt, den Kopf nach unten, und dann "
        "saßen\n"
        "sie darauf.)\n"
        "\n"
        "»Es ist mir lieb, daß ich das gesehen habe,« dachte Alice, »ich habe "
        "so\n"
        "oft in der Zeitung am Ende eines Verhörs gelesen: 'Das Publikum fing "
        "an,\n"
        "Beifall zu klatschen, was aber sofort von den Gerichtsdienern\n"
        "unterdrückt wurde,' und ich konnte bis jetzt nie verstehen, was es\n"
        "bedeutete.«\n"
        "\n"
        "»Wenn dies Alles ist, was du zu sagen weißt, so kannst du "
        "abtreten,«\n"
        "fuhr der König fort.\n"
        "\n"
        "»Ich kann nichts mehr abtreten,« sagte der Hutmacher: »ich stehe so\n"
        "schon auf den Strümpfen.«\n"
        "\n"
        "»Dann kannst du abwarten, bis du wieder gefragt wirst,« erwiederte "
        "der\n"
        "König.\n"
        "\n"
        "Hier klatschte das zweite Meerschweinchen und wurde unterdrückt.\n"
        "\n"
        "»Ha, nun sind die Meerschweinchen besorgt,« dachte Alice, »nun wird "
        "es\n"
        "besser vorwärts gehen.«\n"
        "\n"
        "»Ich möchte lieber zu meinem Thee zurückgehen,« sagte der Hutmacher "
        "mit\n"
        "einem ängstlichen Blicke auf die Königin, welche die Liste der "
        "Sänger\n"
        "durchlas.\n"
        "\n"
        "»Du kannst gehen,« sagte der König, worauf der Hutmacher eilig den\n"
        "Gerichtssaal verließ, ohne sich einmal Zeit zu nehmen, seine Schuhe\n"
        "anzuziehen.\n"
        "\n"
        "»-- und draußen schneidet ihm doch den Kopf ab,« fügte die Königin "
        "zu\n"
        "einem der Beamten gewandt hinzu; aber der Hutmacher war nicht mehr "
        "zu\n"
        "sehen, als der Beamte die Thür erreichte.\n"
        "\n"
        "»Ruft den nächsten Zeugen!« sagte der König.\n"
        "\n"
        "Der nächste Zeuge war die Köchin der Herzogin. Sie trug die\n"
        "Pfefferbüchse in der Hand, und Alice errieth, schon ehe sie in den "
        "Saal\n"
        "trat, wer es sei, weil alle Leute in der Nähe der Thür mit einem "
        "Male\n"
        "anfingen zu niesen.\n"
        "\n"
        "»Gieb deine Aussage,« sagte der König.\n"
        "\n"
        "»Ne!« antwortete die Köchin.\n"
        "\n"
        "Der König sah ängstlich das weiße Kaninchen an, welches leise "
        "sprach:\n"
        "»Eure Majestät müssen diesen Zeugen einem Kreuzverhör unterwerfen.«\n"
        "\n"
        "»Wohl, wenn ich muß, muß ich,« sagte der König trübsinnig, und "
        "nachdem\n"
        "er die Arme gekreuzt und die Augenbraunen so fest zusammengezogen "
        "hatte,\n"
        "daß seine Augen kaum mehr zu sehen waren, sagte er mit tiefer "
        "Stimme:\n"
        "»Wovon macht man kleine Kuchen?«\n"
        "\n"
        "»Pfeffer, hauptsächlich,« sagte die Köchin.\n"
        "\n"
        "»Syrup,« sagte eine schläfrige Stimme hinter ihr.\n"
        "\n"
        "»Nehmt dieses Murmelthier fest!« heulte die Königin. »Köpft dieses\n"
        "Murmelthier! Schafft dieses Murmelthier aus dem Saale! Unterdrückt "
        "es!\n"
        "Kneift es! Brennt ihm den Bart ab!«\n"
        "\n"
        "Einige Minuten lang war das ganze Gericht in Bewegung, um das\n"
        "Murmelthier fortzuschaffen; und als endlich Alles wieder zur Ruhe\n"
        "gekommen war, war die Köchin verschwunden.\n"
        "\n"
        "»Schadet nichts!« sagte der König und sah aus, als falle ihm ein "
        "Stein\n"
        "vom Herzen. »Ruft den nächsten Zeugen.« Und zu der Königin gewandt,\n"
        "fügte er leise hinzu: »Wirklich, meine Liebe, du mußt das nächste\n"
        "Kreuzverhör übernehmen, meine Arme sind schon ganz lahm.«\n"
        "\n"
        "Alice beobachtete das weiße Kaninchen, das die Liste durchsuchte, da "
        "sie\n"
        "sehr neugierig war, wer wohl der nächste Zeuge sein möchte, -- »denn "
        "sie\n"
        "haben noch nicht viel Beweise,« sagte sie für sich. Denkt euch ihre\n"
        "Ueberraschung, als das weiße Kaninchen mit seiner höchsten "
        "Kopfstimme\n"
        "vorlas: »Alice!«\n"
        "\n"
        "»Hier!« rief Alice, in der augenblicklichen Erregung ganz "
        "vergessend,\n"
        "wie sehr sie die letzten Minuten gewachsen war; sie sprang in "
        "solcher\n"
        "Eile auf, daß sie mit ihrem Rock das Pult vor sich umstieß, so daß "
        "alle\n"
        "Geschworne auf die Köpfe der darunter sitzenden Versammlung fielen. "
        "Da\n"
        "lagen sie unbehülflich umher und erinnerten sie sehr an ein Glas "
        "mit\n"
        "Goldfischen, das sie die Woche vorher aus Versehen umgestoßen "
        "hatte.\n"
        "\n"
        "»Oh, ich bitte um Verzeihung,« rief sie mit sehr bestürztem Tone, "
        "und\n"
        "fing an, sie so schnell wie möglich aufzunehmen; denn der Unfall mit "
        "den\n"
        "Goldfischen lag ihr noch im Sinne, und sie hatte eine unbestimmte "
        "Art\n"
        "Vorstellung, als ob sie gleich gesammelt und wieder in ihr Pult "
        "gethan\n"
        "werden müßten, sonst würden sie sterben.\n"
        "\n"
        "»Das Verhör kann nicht fortgesetzt werden,« sagte der König sehr "
        "ernst,\n"
        "»bis alle Geschworne wieder an ihrem rechten Platze sind -- alle,«\n"
        "wiederholte er mit großem Nachdrucke, und sah dabei Alice fest an.\n"
        "\n"
        "Alice sah sich nach dem Pulte um und bemerkte, daß sie in der Eile "
        "die\n"
        "Eidechse kopfunten hineingestellt hatte, und das arme kleine Ding\n"
        "bewegte den Schwanz trübselig hin und her, da es sich übrigens "
        "nicht\n"
        "rühren konnte. Sie zog es schnell wieder heraus und stellte es "
        "richtig\n"
        "hinein. »Es hat zwar nichts zu bedeuten,« sagte sie für sich, »ich\n"
        "glaube, es würde für das Verhör ganz eben so nützlich sein kopfoben "
        "wie\n"
        "kopfunten.«\n"
        "\n"
        "Sobald sich die Geschwornen etwas von dem Schreck erholt hatten,\n"
        "umgeworfen worden zu sein, und nachdem ihre Tafeln und Tafelsteine\n"
        "gefunden und ihnen zurückgegeben worden waren, machten sie sich "
        "eifrig\n"
        "daran, die Geschichte ihres Unfalles aufzuschreiben, alle außer der\n"
        "Eidechse, welche zu angegriffen war, um etwas zu thun; sie saß nur "
        "mit\n"
        "offnem Maule da und starrte die Saaldecke an.\n"
        "\n"
        "»Was weißt du von dieser Angelegenheit?« fragte der König Alice.\n"
        "\n"
        "»Nichts!« sagte Alice.\n"
        "\n"
        "»Durchaus nichts?« drang der König in sie.\n"
        "\n"
        "»Durchaus nichts!« sagte Alice.\n"
        "\n"
        "»Daß ist sehr wichtig,« sagte der König, indem er sich an die\n"
        "Geschwornen wandte. Sie wollten dies eben auf ihre Tafeln "
        "schreiben,\n"
        "als das weiße Kaninchen ihn unterbrach. »Unwichtig, meinten Eure\n"
        "Majestät natürlich!« sagte es in sehr ehrfurchtsvollem Tone, wobei "
        "es\n"
        "ihn aber mit Stirnrunzeln und verdrießlichem Gesichte ansah.\n"
        "\n"
        "»Unwichtig, natürlich, meinte ich,« bestätigte der König eilig, und "
        "fuhr\n"
        "mit halblauter Stimme für sich fort: »wichtig -- unwichtig -- "
        "unwichtig\n"
        "-- wichtig --« als ob er versuchte, welches Wort am besten klänge.\n"
        "\n"
        "Einige der Geschwornen schrieben auf »wichtig«, und einige "
        "»unwichtig.«\n"
        "Alice konnte dies sehen, da sie nahe genug war, um ihre Tafeln zu\n"
        "überblicken; »aber es kommt nicht das Geringste darauf an,« dachte "
        "sie\n"
        "bei sich.\n"
        "\n"
        "In diesem Augenblick rief der König, der eifrig in seinem Notizbuch\n"
        "geschrieben hatte, plötzlich aus: »Still!« und las dann aus seinem "
        "Buche\n"
        "vor: »Zweiundvierzigstes Gesetz. Alle Personen, die mehr als eine "
        "Meile\n"
        "hoch sind, haben den Gerichtshof zu verlassen.\n"
        "\n"
        "Alle sahen Alice an.\n"
        "\n"
        "»Ich bin keine Meile groß,« sagte Alice.\n"
        "\n"
        "»Das bist du wohl,« sagte der König.\n"
        "\n"
        "»Beinahe zwei Meilen groß,« fügte die Königin hinzu.\n"
        "\n"
        "»Auf jeden Fall werde ich nicht fortgehen,« sagte Alice, »übrigens "
        "ist\n"
        "das kein regelmäßiges Gesetz; Sie haben es sich eben erst "
        "ausgedacht.«\n"
        "\n"
        "»Es ist das älteste Gesetz in dem Buche,« sagte der König.\n"
        "\n"
        "»Dann müßte es Nummer Eins sein,« sagte Alice.\n"
        "\n"
        "Der König erbleichte und machte sein Notizbuch schnell zu. »Gebt "
        "euer\n"
        "Urtheil ab!« sagte er leise und mit zitternder Stimme zu den\n"
        "Geschwornen.\n"
        "\n"
        "»Majestät halten zu Gnaden, es sind noch mehr Beweise aufzunehmen,«\n"
        "sagte das weiße Kaninchen, indem es eilig aufsprang; »dieses Papier "
        "ist\n"
        "soeben gefunden worden.«\n"
        "\n"
        "»Was enthält es?« fragte die Königin.\n"
        "\n"
        "»Ich habe es noch nicht geöffnet,« sagte das weiße Kaninchen, »aber "
        "es\n"
        "scheint ein Brief von dem Gefangenen an -- an Jemand zu sein.«\n"
        "\n"
        "»Ja, das wird es wohl sein,« sagte der König, »wenn es nicht an "
        "Niemand\n"
        "ist, was, wie bekannt nicht oft vorkommt.«\n"
        "\n"
        "»An wen ist es adressirt?« fragte einer der Geschwornen.\n"
        "\n"
        "»Es ist gar nicht adressirt,« sagte das weiße Kaninchen; »überhaupt\n"
        "steht auf der Außenseite gar nichts.« Es faltete bei diesen Worten "
        "das\n"
        "Papier auseinander und sprach weiter: »Es ist übrigens gar kein "
        "Brief,\n"
        "es sind Verse.«\n"
        "\n"
        "»Sind sie in der Handschrift des Gefangenen?« fragte ein anderer\n"
        "Geschworner.\n"
        "\n"
        "»Nein, das sind sie nicht,« sagte das weiße Kaninchen, »und das ist "
        "das\n"
        "Merkwürdigste dabei.« (Die Geschwornen sahen alle ganz verdutzt "
        "aus.)\n"
        "\n"
        "»Er muß eines Andern Handschrift nachgeahmt haben,« sagte der "
        "König.\n"
        "(Die Gesichter der Geschwornen klärten sich auf.)\n"
        "\n"
        "»Eure Majestät halten zu Gnaden,« sagte der Bube, »ich habe es "
        "nicht\n"
        "geschrieben, und Niemand kann beweisen, daß ich es geschrieben "
        "haben, es\n"
        "ist keine Unterschrift darunter.«\n"
        "\n"
        "»Wenn du es nicht unterschrieben hast,« sagte der König, »so macht "
        "das\n"
        "die Sache nur schlimmer. Du mußt schlechte Absichten dabei gehabt "
        "haben,\n"
        "sonst hättest du wie ein ehrlicher Mann deinen Namen darunter "
        "gesetzt.«\n"
        "\n"
        "Hierauf folgte allgemeines Beifallklatschen; es war der erste "
        "wirklich\n"
        "kluge Ausspruch, den der König an dem Tage gethan hatte.\n"
        "\n"
        "»Das beweist seine Schuld,« sagte die Königin.\n"
        "\n"
        "»Es beweist durchaus gar nichts!« sagte Alice, »Ihr wißt ja noch "
        "nicht\n"
        "einmal, worüber die Verse sind!«\n"
        "\n"
        "»Lies sie!« sagte der König.\n"
        "\n"
        "Das weiße Kaninchen setzte seine Brille auf. »Wo befehlen Eure "
        "Majestät,\n"
        "daß ich anfangen soll?« fragte es.\n"
        "\n"
        "»Fange beim Anfang an,« sagte der König ernsthaft, »und lies bis du "
        "an's\n"
        "Ende kommst, dann halte an.«\n"
        "\n"
        "Dies waren die Verse, welche das weiße Kaninchen vorlas: --\n"
        "\n"
        "    »Ich höre ja du warst bei ihr,\n"
        "    Und daß er mir es gönnt;\n"
        "    Sie sprach, sie hielte viel von mir,\n"
        "    Wenn ich nur schwimmen könnt'!\n"
        "\n"
        "    Er schrieb an sie, ich ginge nicht\n"
        "    (Nur wußten wir es gleich):\n"
        "    Wenn ihr viel an der Sache liegt,\n"
        "    Was würde dann aus euch?\n"
        "\n"
        "    Ich gab ihr eins, sie gab ihm zwei,\n"
        "    Ihr gabt uns drei Mal vier;\n"
        "    Jetzt sind sie hier, er steht dabei;\n"
        "    Doch alle gehörten erst mir.\n"
        "\n"
        "    Würd' ich und sie vielleicht darein\n"
        "    Verwickelt und verfahren,\n"
        "    Vertraut er dir, sie zu befrei'n,\n"
        "    Gerade wie wir waren.\n"
        "\n"
        "    Ich dachte schon in meinem Sinn,\n"
        "    Eh' sie den Anfall hätt',\n"
        "    Ihr wär't derjenige, der ihn,\n"
        "    Es und uns hindertet.\n"
        "\n"
        "    Sag' ihm um keinen Preis, daß ihr\n"
        "    Die Andern lieber war'n;\n"
        "    Denn keine Seele außer dir\n"
        "    Und mir darf dies erfahr'n.«\n"
        "\n"
        "»Das ist das wichtigste Beweisstück, das wir bis jetzt gehört "
        "haben,«\n"
        "sagte der König, indem er sich die Hände rieb; »laßt also die\n"
        "Geschwornen --«\n"
        "\n"
        "»Wenn es Einer von ihnen erklären kann,« sagte Alice (sie war die\n"
        "letzten Paar Minuten so sehr gewachsen, daß sie sich gar nicht\n"
        "fürchtete, ihn zu unterbrechen), »so will ich ihm sechs Dreier "
        "schenken.\n"
        "Ich finde, daß auch keine Spur von Sinn darin ist.«\n"
        "\n"
        "Die Geschwornen schrieben Alle auf ihre Tafeln: »Sie findet, daß "
        "auch\n"
        "keine Spur von Sinn darin ist;« aber keiner von ihnen versuchte, "
        "das\n"
        "Schriftstück zu erklären.\n"
        "\n"
        "»Wenn kein Sinn darin ist,« sagte der König, »das spart uns ja "
        "ungeheuer\n"
        "viel Arbeit; dann haben wir nicht nöthig, ihn zu suchen. Und "
        "dennoch\n"
        "weiß ich nicht,« fuhr er fort, indem er das Papier auf dem Knie\n"
        "ausbreitete und es prüfend beäugelte, »es kommt mir vor, als könnte "
        "ich\n"
        "etwas Sinn darin finden. '-- wenn ich nur schwimmen könnt'!' du "
        "kannst\n"
        "nicht schwimmen, nicht wahr?« wandte er sich an den Buben.\n"
        "\n"
        "Der Bube schüttelte traurig das Haupt. »Seh' ich etwa danach aus?« "
        "(was\n"
        "freilich nicht der Fall war, da er gänzlich aus Papier bestand.)\n"
        "\n"
        "»Das trifft zu, so weit,« sagte der König und fuhr fort, die Verse "
        "leise\n"
        "durchzulesen. »'Nur wußten wir es gleich' -- das sind die "
        "Geschwornen,\n"
        "natürlich -- 'Ich gab ihr eins, sie gab ihm zwei --' ja wohl, so "
        "hat\n"
        "er's mit den Kuchen gemacht, versteht sich --«\n"
        "\n"
        "»Aber es geht weiter: 'Jetzt sind sie hier,'« sagte Alice.\n"
        "\n"
        "»Freilich, da sind sie ja! er steht dabei!« sagte der König "
        "triumphirend\n"
        "und wies dabei nach den Kuchen auf dem Tische und nach dem Buben;\n"
        "»nichts kann klarer sein. Dann wieder -- 'Eh sie den Anfall hätt'' "
        "-- du\n"
        "hast nie einen Anfall gehabt, Liebe, glaube ich,« sagte er zu der\n"
        "Königin.\n"
        "\n"
        "»Niemals,« rief die Königin wüthend und warf dabei der Eidechse ein\n"
        "Tintenfaß an den Kopf. (Der unglückliche kleine Wabbel hatte "
        "aufgehört,\n"
        "mit dem Finger auf seiner Tafel zu schreiben, da er merkte, daß es "
        "keine\n"
        "Spuren hinterließ; doch nun fing er eilig wieder an, indem er die "
        "Tinte\n"
        "benutzte, die von seinem Gesichte herabträufelte, so lange dies\n"
        "vorhielt.)\n"
        "\n"
        "»Dann ist dies nicht dein Fall,« sagte der König und blickte "
        "lächelnd in\n"
        "dem ganzen Saale herum. Alles blieb todtenstill.\n"
        "\n"
        "»-- 's ist ja 'n Witz!« fügte der König in ärgerlichem Tone hinzu "
        "--\n"
        "sogleich lachte Jedermann. »Die Geschwornen sollen ihren Ausspruch\n"
        "thun,« sagte der König wohl zum zwanzigsten Male.\n"
        "\n"
        "»Nein, nein!« sagte die Königin. »Erst das Urtheil, der Ausspruch "
        "der\n"
        "Geschwornen nachher.«\n"
        "\n"
        "»Dummer Unsinn!« sagte Alice laut. »Was für ein Einfall, erst das\n"
        "Urtheil haben zu wollen!«\n"
        "\n"
        "»Halt den Mund!« sagte die Königin, indem sie purpurroth wurde.\n"
        "\n"
        "»Ich will nicht!« sagte Alice.\n"
        "\n"
        "»Schlagt ihr den Kopf ab!« brüllte die Königin so laut sie konnte.\n"
        "Niemand rührte sich.\n"
        "\n"
        "»Wer fragt nach euch?« sagte Alice (unterdessen hatte sie ihre "
        "volle\n"
        "Größe erreicht). »Ihr seid nichts weiter als ein Spiel Karten!«\n"
        "\n"
        "Bei diesen Worten erhob sich das ganze Spiel in die Luft und flog "
        "auf\n"
        "sie herab; sie schrie auf, halb vor Furcht, halb vor Aerger, "
        "versuchte\n"
        "sie sich abzuwehren und merkte, daß sie am Ufer lag, den Kopf auf "
        "dem\n"
        "Schoße ihrer Schwester, welche leise einige welke Blätter fortnahm, "
        "die\n"
        "ihr von den Bäumen herunter auf's Gesicht gefallen waren.\n"
        "\n"
        "»Wach auf, liebe Alice!« sagte ihre Schwester; »du hast mal lange\n"
        "geschlafen!«\n"
        "\n"
        "»O, und ich habe einen so merkwürdigen Traum gehabt!« sagte Alice, "
        "und\n"
        "sie erzählte ihrer Schwester, so gut sie sich errinnern konnte, alle "
        "die\n"
        "seltsamen Abenteuer, welche ihr eben gelesen habt. Als sie fertig "
        "war,\n"
        "gab ihre Schwester ihr einen Kuß und sagte: »Es war ein sonderbarer\n"
        "Traum, das ist gewiß; aber nun lauf hinein zum Thee, es wird spät.« "
        "Da\n"
        "stand Alice auf und rannte fort, und dachte dabei, und zwar mit "
        "Recht,\n"
        "daß es doch ein wunderschöner Traum gewesen sei.\n"
        "\n"
        "       *       *       *       *       *\n"
        "\n"
        "Aber ihre Schwester blieb sitzen, wie sie sie verlassen hatte, den "
        "Kopf\n"
        "auf die Hand gestützt, blickte in die untergehende Sonne und dachte "
        "an\n"
        "die kleine Alice und ihre wunderbaren Abenteuer, bis auch sie auf "
        "ihre\n"
        "Weise zu träumen anfing, und dies war ihr Traum:\n"
        "\n"
        "Zuerst träumte sie von der kleinen Alice selbst: wieder sah sie die\n"
        "kleinen Händchen zusammengefaltet auf ihrem Knie, und die klaren\n"
        "sprechenden Augen, die zu ihr aufblickten -- sie konnte selbst den "
        "Ton\n"
        "ihrer Stimme hören und das komische Zurückwerfen des kleinen "
        "Köpfchens\n"
        "sehen, womit sie die einzelnen Haare abschüttelte, die ihr immer "
        "wieder\n"
        "in die Augen kamen -- und jemehr sie zuhörte oder zuzuhören meinte, "
        "desto\n"
        "mehr belebte sich der ganze Platz um sie herum mit den seltsamen\n"
        "Geschöpfen aus ihrer kleinen Schwester Traum.\n"
        "\n"
        "Das lange Gras zu ihren Füßen rauschte, da das weiße Kaninchen\n"
        "vorbeihuschte -- die erschrockene Maus plätscherte durch den nahen "
        "Teich\n"
        "-- sie konnte das Klappern der Theetassen hören, wo der Faselhase "
        "und\n"
        "seine Freunde ihre immerwährende Mahlzeit hielten, und die gellende\n"
        "Stimme der Königin, die ihre unglücklichen Gäste zur Hinrichtung\n"
        "abschickte -- wieder nieste das Ferkel-Kind auf dem Schoße der "
        "Herzogin,\n"
        "während Pfannen und Schüsseln rund herum in Scherben brachen -- "
        "wieder\n"
        "erfüllten der Schrei des Greifen, das Quieken von dem Tafelstein "
        "der\n"
        "Eidechse und das Stöhnen des unterdrückten Meerschweinchens die Luft "
        "und\n"
        "vermischten sich mit dem Schluchzen der unglücklichen falschen\n"
        "Schildkröte in der Entfernung.\n"
        "\n"
        "So saß sie da, mit geschlossenen Augen, und glaubte fast, sie sei "
        "im\n"
        "Wunderlande, obgleich sie ja wußte, daß sobald sie die Augen "
        "öffnete,\n"
        "Alles wieder zur alltäglichen Wirklichkeit werden würde; das Gras "
        "würde\n"
        "dann nur im Winde rauschen, der Teich mit seinem Rieseln das Wogen "
        "des\n"
        "Rohres begleiten; das Klappern der Theetassen würde sich in "
        "klingende\n"
        "Heerdenglocken verwandeln und die gellende Stimme der Königin in "
        "die\n"
        "Rufe des Hirtenknaben -- und das Niesen des Kindes, das Geschrei "
        "des\n"
        "Greifen und all die andern außerordentlichen Töne würden sich (das "
        "wußte\n"
        "sie) in das verworrene Getöse des geschäftigen Gutshofes verwandeln "
        "--\n"
        "während sie statt des schwermüthigen Schluchzens der falschen\n"
        "Schildkröte in der Ferne das wohlbekannte Brüllen des Rindviehes "
        "hören\n"
        "würde.\n"
        "\n"
        "Endlich malte sie sich aus, wie ihre kleine Schwester Alice in "
        "späterer\n"
        "Zeit selbst erwachsen sein werde; und wie sie durch alle reiferen "
        "Jahre\n"
        "hindurch das einfache liebevolle Herz ihrer Kindheit bewahren, und "
        "wie\n"
        "sie andere kleine Kinder um sich versammeln und deren Blicke "
        "neugierig\n"
        "und gespannt machen werde mit manch einer wunderbaren Erzählung,\n"
        "vielleicht sogar mit dem Traume vom Wunderlande aus alten Zeiten; "
        "und\n"
        "wie sie alle ihre kleinen Sorgen nachfühlen, sich über alle ihre "
        "kleinen\n"
        "Freuden mitfreuen werde in der Erinnerung an ihr eigenes Kindesleben "
        "und\n"
        "die glücklichen Sommertage."
    ],
    "Die Verwandlung": [
        "Als Gregor Samsa eines Morgens aus unruhigen Träumen erwachte, fand er\n"
        "sich in seinem Bett zu einem ungeheueren Ungeziefer verwandelt. Er lag\n"
        "auf seinem panzerartig harten Rücken und sah, wenn er den Kopf ein "
        "wenig\n"
        "hob, seinen gewölbten, braunen, von bogenförmigen Versteifungen\n"
        "geteilten Bauch, auf dessen Höhe sich die Bettdecke, zum gänzlichen\n"
        "Niedergleiten bereit, kaum noch erhalten konnte. Seine vielen, im\n"
        "Vergleich zu seinem sonstigen Umfang kläglich dünnen Beine flimmerten\n"
        "ihm hilflos vor den Augen.\n"
        "\n"
        "»Was ist mit mir geschehen?« dachte er. Es war kein Traum. Sein Zimmer,\n"
        "ein richtiges, nur etwas zu kleines Menschenzimmer, lag ruhig zwischen\n"
        "den vier wohlbekannten Wänden. Über dem Tisch, auf dem eine\n"
        "auseinandergepackte Musterkollektion von Tuchwaren ausgebreitet war --\n"
        "Samsa war Reisender --, hing das Bild, das er vor kurzem aus einer\n"
        "illustrierten Zeitschrift ausgeschnitten und in einem hübschen,\n"
        "vergoldeten Rahmen untergebracht hatte. Es stellte eine Dame dar, die,\n"
        "mit einem Pelzhut und einer Pelzboa versehen, aufrecht dasaß und einen\n"
        "schweren Pelzmuff, in dem ihr ganzer Unterarm verschwunden war, dem\n"
        "Beschauer entgegenhob.\n"
        "\n"
        "Gregors Blick richtete sich dann zum Fenster, und das trübe Wetter --\n"
        "man hörte Regentropfen auf das Fensterblech aufschlagen -- machte ihn\n"
        "ganz melancholisch. »Wie wäre es, wenn ich noch ein wenig\n"
        "weiterschliefe und alle Narrheiten vergäße,« dachte er, aber das war\n"
        "gänzlich undurchführbar, denn er war gewöhnt, auf der rechten Seite zu\n"
        "schlafen, konnte sich aber in seinem gegenwärtigen Zustand nicht in\n"
        "diese Lage bringen. Mit welcher Kraft er sich auch auf die rechte Seite\n"
        "warf, immer wieder schaukelte er in die Rückenlage zurück. Er versuchte\n"
        "es wohl hundertmal, schloß die Augen, um die zappelnden Beine nicht\n"
        "sehen zu müssen, und ließ erst ab, als er in der Seite einen noch nie\n"
        "gefühlten, leichten, dumpfen Schmerz zu fühlen begann.\n"
        "\n"
        "»Ach Gott,« dachte er, »was für einen anstrengenden Beruf habe ich\n"
        "gewählt! Tag aus, Tag ein auf der Reise. Die geschäftlichen Aufregungen\n"
        "sind viel größer, als im eigentlichen Geschäft zu Hause, und außerdem\n"
        "ist mir noch diese Plage des Reisens auferlegt, die Sorgen um die\n"
        "Zuganschlüsse, das unregelmäßige, schlechte Essen, ein immer\n"
        "wechselnder, nie andauernder, nie herzlich werdender menschlicher\n"
        "Verkehr. Der Teufel soll das alles holen!« Er fühlte ein leichtes "
        "Jucken\n"
        "oben auf dem Bauch; schob sich auf dem Rücken langsam näher zum\n"
        "Bettpfosten, um den Kopf besser heben zu können; fand die juckende\n"
        "Stelle, die mit lauter kleinen weißen Pünktchen besetzt war, die er\n"
        "nicht zu beurteilen verstand; und wollte mit einem Bein die Stelle\n"
        "betasten, zog es aber gleich zurück, denn bei der Berührung umwehten "
        "ihn\n"
        "Kälteschauer.\n"
        "\n"
        "Er glitt wieder in seine frühere Lage zurück. »Dies frühzeitige\n"
        "Aufstehen«, dachte er, »macht einen ganz blödsinnig. Der Mensch muß\n"
        "seinen Schlaf haben. Andere Reisende leben wie Haremsfrauen. Wenn ich\n"
        "zum Beispiel im Laufe des Vormittags ins Gasthaus zurückgehe, um die\n"
        "erlangten Aufträge zu überschreiben, sitzen diese Herren erst beim\n"
        "Frühstück. Das sollte ich bei meinem Chef versuchen; ich würde auf der\n"
        "Stelle hinausfliegen. Wer weiß übrigens, ob das nicht sehr gut für mich\n"
        "wäre. Wenn ich mich nicht wegen meiner Eltern zurückhielte, ich hätte\n"
        "längst gekündigt, ich wäre vor den Chef hingetreten und hätte ihm meine\n"
        "Meinung von Grund des Herzens aus gesagt. Vom Pult hätte er fallen\n"
        "müssen! Es ist auch eine sonderbare Art, sich auf das Pult zu setzen "
        "und\n"
        "von der Höhe herab mit dem Angestellten zu reden, der überdies wegen "
        "der\n"
        "Schwerhörigkeit des Chefs ganz nahe herantreten muß. Nun, die Hoffnung\n"
        "ist noch nicht gänzlich aufgegeben, habe ich einmal das Geld beisammen,\n"
        "um die Schuld der Eltern an ihn abzuzahlen -- es dürfte noch fünf bis\n"
        "sechs Jahre dauern --, mache ich die Sache unbedingt. Dann wird der\n"
        "große Schnitt gemacht. Vorläufig allerdings muß ich aufstehen, denn "
        "mein\n"
        "Zug fährt um fünf.«\n"
        "\n"
        "Und er sah zur Weckuhr hinüber, die auf dem Kasten tickte. »Himmlischer\n"
        "Vater!« dachte er, Es war halb sieben Uhr, und die Zeiger gingen ruhig\n"
        "vorwärts, es war sogar halb vorüber, es näherte sich schon dreiviertel.\n"
        "Sollte der Wecker nicht geläutet haben? Man sah vom Bett aus, daß er "
        "auf\n"
        "vier Uhr richtig eingestellt war; gewiß hatte er auch geläutet. Ja, "
        "aber\n"
        "war es möglich, dieses möbelerschütternde Läuten ruhig zu verschlafen?\n"
        "Nun, ruhig hatte er ja nicht geschlafen, aber wahrscheinlich desto\n"
        "fester. Was aber sollte er jetzt tun? Der nächste Zug ging um sieben\n"
        "Uhr; um den einzuholen, hätte er sich unsinnig beeilen müssen, und die\n"
        "Kollektion war noch nicht eingepackt, und er selbst fühlte sich "
        "durchaus\n"
        "nicht besonders frisch und beweglich. Und selbst wenn er den Zug\n"
        "einholte, ein Donnerwetter des Chefs war nicht zu vermeiden, denn der\n"
        "Geschäftsdiener hatte beim Fünfuhrzug gewartet und die Meldung von\n"
        "seiner Versäumnis längst erstattet. Es war eine Kreatur des Chefs, ohne\n"
        "Rückgrat und Verstand. Wie nun, wenn er sich krank meldete? Das wäre\n"
        "aber äußerst peinlich und verdächtig, denn Gregor war während seines\n"
        "fünfjährigen Dienstes noch nicht einmal krank gewesen. Gewiß würde der\n"
        "Chef mit dem Krankenkassenarzt kommen, würde den Eltern wegen des "
        "faulen\n"
        "Sohnes Vorwürfe machen und alle Einwände durch den Hinweis auf den\n"
        "Krankenkassenarzt abschneiden, für den es ja überhaupt nur ganz "
        "gesunde,\n"
        "aber arbeitsscheue Menschen gibt. Und hätte er übrigens in diesem Falle\n"
        "so ganz unrecht? Gregor fühlte sich tatsächlich, abgesehen von einer\n"
        "nach dem langen Schlaf wirklich überflüssigen Schläfrigkeit, ganz wohl\n"
        "und hatte sogar einen besonders kräftigen Hunger.\n"
        "\n"
        "Als er dies alles in größter Eile überlegte, ohne sich entschließen zu\n"
        "können, das Bett zu verlassen -- gerade schlug der Wecker dreiviertel\n"
        "sieben -- klopfte es vorsichtig an die Tür am Kopfende seines Bettes.\n"
        "»Gregor,« rief es -- es war die Mutter --, »es ist dreiviertel sieben.\n"
        "Wolltest du nicht wegfahren?« Die sanfte Stimme! Gregor erschrak, als "
        "er\n"
        "seine antwortende Stimme hörte, die wohl unverkennbar seine frühere "
        "war,\n"
        "in die sich aber, wie von unten her, ein nicht zu unterdrückendes,\n"
        "schmerzliches Piepsen mischte, das die Worte förmlich nur im ersten\n"
        "Augenblick in ihrer Deutlichkeit beließ, um sie im Nachklang derart zu\n"
        "zerstören, daß man nicht wußte, ob man recht gehört hatte. Gregor hatte\n"
        "ausführlich antworten und alles erklären wollen, beschränkte sich aber\n"
        "bei diesen Umständen darauf, zu sagen: »Ja, ja, danke, Mutter, ich "
        "stehe\n"
        "schon auf.« Infolge der Holztür war die Veränderung in Gregors Stimme\n"
        "draußen wohl nicht zu merken, denn die Mutter beruhigte sich mit dieser\n"
        "Erklärung und schlürfte davon. Aber durch das kleine Gespräch waren die\n"
        "anderen Familienmitglieder darauf aufmerksam geworden, daß Gregor wider\n"
        "Erwarten noch zu Hause war, und schon klopfte an der einen Seitentür "
        "der\n"
        "Vater, schwach, aber mit der Faust. »Gregor, Gregor,« rief er, »was ist\n"
        "denn?« Und nach einer kleinen Weile mahnte er nochmals mit tieferer\n"
        "Stimme: »Gregor! Gregor!« An der anderen Seitentür aber klagte leise "
        "die\n"
        "Schwester: »Gregor? Ist dir nicht wohl? Brauchst du etwas?« Nach beiden\n"
        "Seiten hin antwortete Gregor: »Bin schon fertig,« und bemühte sich,\n"
        "durch die sorgfältigste Aussprache und durch Einschaltung von langen\n"
        "Pausen zwischen den einzelnen Worten seiner Stimme alles Auffallende zu\n"
        "nehmen. Der Vater kehrte auch zu seinem Frühstück zurück, die Schwester\n"
        "aber flüsterte: »Gregor, mach auf, ich beschwöre dich.« Gregor aber\n"
        "dachte gar nicht daran aufzumachen, sondern lobte die vom Reisen her\n"
        "übernommene Vorsicht, auch zu Hause alle Türen während der Nacht zu\n"
        "versperren.\n"
        "\n"
        "Zunächst wollte er ruhig und ungestört aufstehen, sich anziehen und vor\n"
        "allem frühstücken, und dann erst das Weitere überlegen, denn, das "
        "merkte\n"
        "er wohl, im Bett würde er mit dem Nachdenken zu keinem vernünftigen "
        "Ende\n"
        "kommen. Er erinnerte sich, schon öfters im Bett irgendeinen vielleicht\n"
        "durch ungeschicktes Liegen erzeugten, leichten Schmerz empfunden zu\n"
        "haben, der sich dann beim Aufstehen als reine Einbildung herausstellte,\n"
        "und er war gespannt, wie sich seine heutigen Vorstellungen allmählich\n"
        "auflösen würden. Daß die Veränderung der Stimme nichts anderes war als\n"
        "der Vorbote einer tüchtigen Verkühlung, einer Berufskrankheit der\n"
        "Reisenden, daran zweifelte er nicht im geringsten.\n"
        "\n"
        "Die Decke abzuwerfen war ganz einfach; er brauchte sich nur ein wenig\n"
        "aufzublasen und sie fiel von selbst. Aber weiterhin wurde es schwierig,\n"
        "besonders weil er so ungemein breit war. Er hätte Arme und Hände\n"
        "gebraucht, um sich aufzurichten; statt dessen aber hatte er nur die\n"
        "vielen Beinchen, die ununterbrochen in der verschiedensten Bewegung\n"
        "waren und die er überdies nicht beherrschen konnte. Wollte er eines\n"
        "einmal einknicken, so war es das erste, daß er sich streckte; und "
        "gelang\n"
        "es ihm endlich, mit diesem Bein das auszuführen, was er wollte, so\n"
        "arbeiteten inzwischen alle anderen, wie freigelassen, in höchster,\n"
        "schmerzlicher Aufregung. »Nur sich nicht im Bett unnütz aufhalten,«\n"
        "sagte sich Gregor.\n"
        "\n"
        "Zuerst wollte er mit dem unteren Teil seines Körpers aus dem Bett\n"
        "hinauskommen, aber dieser untere Teil, den er übrigens noch nicht\n"
        "gesehen hatte und von dem er sich auch keine rechte Vorstellung machen\n"
        "konnte, erwies sich als zu schwer beweglich; es ging so langsam; und "
        "als\n"
        "er schließlich, fast wild geworden, mit gesammelter Kraft, ohne\n"
        "Rücksicht sich vorwärtsstieß, hatte er die Richtung falsch gewählt,\n"
        "schlug an den unteren Bettpfosten heftig an, und der brennende Schmerz,\n"
        "den er empfand, belehrte ihn, daß gerade der untere Teil seines Körpers\n"
        "augenblicklich vielleicht der empfindlichste war.\n"
        "\n"
        "Er versuchte es daher, zuerst den Oberkörper aus dem Bett zu bekommen,\n"
        "und drehte vorsichtig den Kopf dem Bettrand zu. Dies gelang auch "
        "leicht,\n"
        "und trotz ihrer Breite und Schwere folgte schließlich die Körpermasse\n"
        "langsam der Wendung des Kopfes. Aber als er den Kopf endlich außerhalb\n"
        "des Bettes in der freien Luft hielt, bekam er Angst, weiter auf diese\n"
        "Weise vorzurücken, denn wenn er sich schließlich so fallen ließ, mußte\n"
        "geradezu ein Wunder geschehen wenn der Kopf nicht verletzt werden\n"
        "sollte. Und die Besinnung durfte er gerade jetzt um keinen Preis\n"
        "verlieren; lieber wollte er im Bett bleiben.\n"
        "\n"
        "Aber als er wieder nach gleicher Mühe aufseufzend so dalag wie früher,\n"
        "und wieder seine Beinchen womöglich noch ärger gegeneinander kämpfen "
        "sah\n"
        "und keine Möglichkeit fand, in diese Willkür Ruhe und Ordnung zu\n"
        "bringen, sagte er sich wieder, daß er unmöglich im Bett bleiben könne\n"
        "und daß es das Vernünftigste sei, alles zu opfern, wenn auch nur die\n"
        "kleinste Hoffnung bestünde, sich dadurch vom Bett zu befreien.\n"
        "Gleichzeitig aber vergaß er nicht, sich zwischendurch daran zu "
        "erinnern,\n"
        "daß viel besser als verzweifelte Entschlüsse ruhige und ruhigste\n"
        "Überlegung sei. In solchen Augenblicken richtete er die Augen möglichst\n"
        "scharf auf das Fenster, aber leider war aus dem Anblick des\n"
        "Morgennebels, der sogar die andere Seite der engen Straße verhüllte,\n"
        "wenig Zuversicht und Munterkeit zu holen. »Schon sieben Uhr,« sagte er\n"
        "sich beim neuerlichen Schlagen des Weckers, »schon sieben Uhr und noch\n"
        "immer ein solcher Nebel.« Und ein Weilchen lang lag er ruhig mit\n"
        "schwachem Atem, als erwarte er vielleicht von der völligen Stille die\n"
        "Wiederkehr der wirklichen und selbstverständlichen Verhältnisse.\n"
        "\n"
        "Dann aber sagte er sich: »Ehe es einviertel acht schlägt, muß ich\n"
        "unbedingt das Bett vollständig verlassen haben. Im übrigen wird auch "
        "bis\n"
        "dahin jemand aus dem Geschäft kommen, um nach mir zu fragen, denn das\n"
        "Geschäft wird vor sieben Uhr geöffnet.« Und er machte sich nun daran,\n"
        "den Körper in seiner ganzen Länge vollständig gleichmäßig aus dem Bett\n"
        "hinauszuschaukeln. Wenn er sich auf diese Weise aus dem Bett fallen\n"
        "ließ, blieb der Kopf, den er beim Fall scharf heben wollte,\n"
        "voraussichtlich unverletzt. Der Rücken schien hart zu sein; dem würde\n"
        "wohl bei dem Fall auf den Teppich nichts geschehen. Das größte Bedenken\n"
        "machte ihm die Rücksicht auf den lauten Krach, den es geben müßte und\n"
        "der wahrscheinlich hinter allen Türen wenn nicht Schrecken, so doch\n"
        "Besorgnisse erregen würde. Das mußte aber gewagt werden.\n"
        "\n"
        "Als Gregor schon zur Hälfte aus dem Bette ragte -- die neue Methode war\n"
        "mehr ein Spiel als eine Anstrengung, er brauchte immer nur ruckweise zu\n"
        "schaukeln --, fiel ihm ein, wie einfach alles wäre, wenn man ihm zu\n"
        "Hilfe käme. Zwei starke Leute -- er dachte an seinen Vater und das\n"
        "Dienstmädchen -- hätten vollständig genügt; sie hätten ihre Arme nur\n"
        "unter seinen gewölbten Rücken schieben, ihn so aus dem Bett schälen,\n"
        "sich mit der Last niederbeugen und dann bloß vorsichtig dulden müssen,\n"
        "daß er den Überschwung auf dem Fußboden vollzog, wo dann die Beinchen\n"
        "hoffentlich einen Sinn bekommen würden. Nun, ganz abgesehen davon, daß\n"
        "die Türen versperrt waren, hätte er wirklich um Hilfe rufen sollen?\n"
        "Trotz aller Not konnte er bei diesem Gedanken ein Lächeln nicht\n"
        "unterdrücken.\n"
        "\n"
        "Schon war er so weit, daß er bei stärkerem Schaukeln kaum das\n"
        "Gleichgewicht noch erhielt, und sehr bald mußte er sich nun endgültig\n"
        "entscheiden, denn es war in fünf Minuten einviertel acht, -- als es an\n"
        "der Wohnungstür läutete. »Das ist jemand aus dem Geschäft,« sagte er\n"
        "sich und erstarrte fast, während seine Beinchen nur desto eiliger\n"
        "tanzten. Einen Augenblick blieb alles still. »Sie öffnen nicht,« sagte\n"
        "sich Gregor, befangen in irgendeiner unsinnigen Hoffnung. Aber dann "
        "ging\n"
        "natürlich wie immer das Dienstmädchen festen Schrittes zur Tür und\n"
        "öffnete. Gregor brauchte nur das erste Grußwort des Besuchers zu hören\n"
        "und wußte schon, wer es war -- der Prokurist selbst. Warum war nur\n"
        "Gregor dazu verurteilt, bei einer Firma zu dienen, wo man bei der\n"
        "kleinsten Versäumnis gleich den größten Verdacht faßte? Waren denn alle\n"
        "Angestellten samt und sonders Lumpen, gab es denn unter ihnen keinen\n"
        "treuen ergebenen Menschen, den, wenn er auch nur ein paar Morgenstunden\n"
        "für das Geschäft nicht ausgenützt hatte, vor Gewissensbissen närrisch\n"
        "wurde und geradezu nicht imstande war, das Bett zu verlassen? Genügte "
        "es\n"
        "wirklich nicht, einen Lehrjungen nachfragen zu lassen -- wenn überhaupt\n"
        "diese Fragerei nötig war --, mußte da der Prokurist selbst kommen, und\n"
        "mußte dadurch der ganzen unschuldigen Familie gezeigt werden, daß die\n"
        "Untersuchung dieser verdächtigen Angelegenheit nur dem Verstand des\n"
        "Prokuristen anvertraut werden konnte? Und mehr infolge der Erregung, in\n"
        "welche Gregor durch diese Überlegungen versetzt wurde, als infolge "
        "eines\n"
        "richtigen Entschlusses, schwang er sich mit aller Macht aus dem Bett. "
        "Es\n"
        "gab einen lauten Schlag, aber ein eigentlicher Krach war es nicht. Ein\n"
        "wenig wurde der Fall durch den Teppich abgeschwächt, auch war der "
        "Rücken\n"
        "elastischer, als Gregor gedacht hatte, daher kam der nicht gar so\n"
        "auffallende dumpfe Klang. Nur den Kopf hatte er nicht vorsichtig genug\n"
        "gehalten und ihn angeschlagen; er drehte ihn und rieb ihn an dem "
        "Teppich\n"
        "vor Ärger und Schmerz.\n"
        "\n"
        "»Da drin ist etwas gefallen,« sagte der Prokurist im Nebenzimmer links.\n"
        "Gregor suchte sich vorzustellen, ob nicht auch einmal dem Prokuristen\n"
        "etwas Ähnliches passieren könnte, wie heute ihm; die Möglichkeit dessen\n"
        "mußte man doch eigentlich zugeben. Aber wie zur rohen Antwort auf diese\n"
        "Frage machte jetzt der Prokurist im Nebenzimmer ein paar bestimmte\n"
        "Schritte und ließ seine Lackstiefel knarren. Aus dem Nebenzimmer rechts\n"
        "flüsterte die Schwester, um Gregor zu verständigen: »Gregor, der\n"
        "Prokurist ist da.« »Ich weiß,« sagte Gregor vor sich hin; aber so laut,\n"
        "daß es die Schwester hätte hören können, wagte er die Stimme nicht zu\n"
        "erheben.\n"
        "\n"
        "»Gregor,« sagte nun der Vater aus dem Nebenzimmer links, »der Herr\n"
        "Prokurist ist gekommen und erkundigt sich, warum du nicht mit dem\n"
        "Frühzug weggefahren bist. Wir wissen nicht, was wir ihm sagen sollen.\n"
        "Übrigens will er auch mit dir persönlich sprechen. Also bitte mach die\n"
        "Tür auf. Er wird die Unordnung im Zimmer zu entschuldigen schon die "
        "Güte\n"
        "haben.« »Guten Morgen, Herr Samsa,« rief der Prokurist freundlich\n"
        "dazwischen. »Ihm ist nicht wohl,« sagte die Mutter zum Prokuristen,\n"
        "während der Vater noch an der Tür redete, »ihm ist nicht wohl, glauben\n"
        "Sie mir, Herr Prokurist. Wie würde denn Gregor sonst einen Zug\n"
        "versäumen! Der Junge hat ja nichts im Kopf als das Geschäft. Ich ärgere\n"
        "mich schon fast, daß er abends niemals ausgeht; jetzt war er doch acht\n"
        "Tage in der Stadt, aber jeden Abend war er zu Hause. Da sitzt er bei "
        "uns\n"
        "am Tisch und liest still die Zeitung oder studiert Fahrpläne. Es ist\n"
        "schon eine Zerstreuung für ihn, wenn er sich mit Laubsägearbeiten\n"
        "beschäftigt. Da hat er zum Beispiel im Laufe von zwei, drei Abenden\n"
        "einen kleinen Rahmen geschnitzt; Sie werden staunen, wie hübsch er ist;\n"
        "er hängt drin im Zimmer; Sie werden ihn gleich sehen, wenn Gregor\n"
        "aufmacht. Ich bin übrigens glücklich, daß Sie da sind, Herr Prokurist;\n"
        "wir allein hätten Gregor nicht dazu gebracht, die Tür zu öffnen; er ist\n"
        "so hartnäckig; und bestimmt ist ihm nicht wohl, trotzdem er es am "
        "Morgen\n"
        "geleugnet hat.« »Ich komme gleich,« sagte Gregor langsam und bedächtig\n"
        "und rührte sich nicht, um kein Wort der Gespräche zu verlieren. "
        "»Anders,\n"
        "gnädige Frau, kann ich es mir auch nicht erklären,« sagte der "
        "Prokurist,\n"
        "»hoffentlich ist es nichts Ernstes. Wenn ich auch andererseits sagen\n"
        "muß, daß wir Geschäftsleute -- wie man will, leider oder\n"
        "glücklicherweise -- ein leichtes Unwohlsein sehr oft aus geschäftlichen\n"
        "Rücksichten einfach überwinden müssen.« »Also kann der Herr Prokurist\n"
        "schon zu dir hinein?« fragte der ungeduldige Vater und klopfte wiederum\n"
        "an die Tür. »Nein,« sagte Gregor. Im Nebenzimmer links trat eine\n"
        "peinliche Stille ein, im Nebenzimmer rechts begann die Schwester zu\n"
        "schluchzen.\n"
        "\n"
        "Warum ging denn die Schwester nicht zu den anderen? Sie war wohl erst\n"
        "jetzt aus dem Bett aufgestanden und hatte noch gar nicht angefangen "
        "sich\n"
        "anzuziehen. Und warum weinte sie denn? Weil er nicht aufstand und den\n"
        "Prokuristen nicht hereinließ, weil er in Gefahr war, den Posten zu\n"
        "verlieren und weil dann der Chef die Eltern mit den alten Forderungen\n"
        "wieder verfolgen würde? Das waren doch vorläufig wohl unnötige Sorgen.\n"
        "Noch war Gregor hier und dachte nicht im geringsten daran, seine "
        "Familie\n"
        "zu verlassen. Augenblicklich lag er wohl da auf dem Teppich, und\n"
        "niemand, der seinen Zustand gekannt hätte, hätte im Ernst von ihm\n"
        "verlangt, daß er den Prokuristen hereinlasse. Aber wegen dieser kleinen\n"
        "Unhöflichkeit, für die sich ja später leicht eine passende Ausrede\n"
        "finden würde, konnte Gregor doch nicht gut sofort weggeschickt werden.\n"
        "Und Gregor schien es, daß es viel vernünftiger wäre, ihn jetzt in Ruhe\n"
        "zu lassen, statt ihn mit Weinen und Zureden zu stören. Aber es war eben\n"
        "die Ungewißheit, welche die anderen bedrängte und ihr Benehmen\n"
        "entschuldigte.\n"
        "\n"
        "»Herr Samsa,« rief nun der Prokurist mit erhobener Stimme, »was ist "
        "denn\n"
        "los? Sie verbarrikadieren sich da in Ihrem Zimmer, antworten bloß mit "
        "ja\n"
        "und nein, machen Ihren Eltern schwere, unnötige Sorgen und versäumen --\n"
        "dies nur nebenbei erwähnt -- Ihre geschäftlichen Pflichten in einer\n"
        "eigentlich unerhörten Weise. Ich spreche hier im Namen Ihrer Eltern und\n"
        "Ihres Chefs und bitte Sie ganz ernsthaft um eine augenblickliche,\n"
        "deutliche Erklärung. Ich staune, ich staune. Ich glaubte Sie als einen\n"
        "ruhigen, vernünftigen Menschen zu kennen, und nun scheinen Sie "
        "plötzlich\n"
        "anfangen zu wollen, mit sonderbaren Launen zu paradieren. Der Chef\n"
        "deutete mir zwar heute früh eine mögliche Erklärung für Ihre Versäumnis\n"
        "an -- sie betraf das Ihnen seit kurzem anvertraute Inkasso --, aber ich\n"
        "legte wahrhaftig fast mein Ehrenwort dafür ein, daß diese Erklärung\n"
        "nicht zutreffen könne. Nun aber sehe ich hier Ihren unbegreiflichen\n"
        "Starrsinn und verliere ganz und gar jede Lust, mich auch nur im\n"
        "geringsten für Sie einzusetzen. Und Ihre Stellung ist durchaus nicht "
        "die\n"
        "festeste. Ich hatte ursprünglich die Absicht, Ihnen das alles unter "
        "vier\n"
        "Augen zu sagen, aber da Sie mich hier nutzlos meine Zeit versäumen\n"
        "lassen, weiß ich nicht, warum es nicht auch Ihre Herren Eltern erfahren\n"
        "sollen. Ihre Leistungen in der letzten Zeit waren also sehr\n"
        "unbefriedigend; es ist zwar nicht die Jahreszeit, um besondere "
        "Geschäfte\n"
        "zu machen, das erkennen wir an; aber eine Jahreszeit, um keine "
        "Geschäfte\n"
        "zu machen, gibt es überhaupt nicht, Herr Samsa, darf es nicht geben.«\n"
        "\n"
        "»Aber Herr Prokurist,« rief Gregor außer sich und vergaß in der\n"
        "Aufregung alles andere, »ich mache ja sofort, augenblicklich auf. Ein\n"
        "leichtes Unwohlsein, ein Schwindelanfall, haben mich verhindert\n"
        "aufzustehen. Ich liege noch jetzt im Bett. Jetzt bin ich aber schon\n"
        "wieder ganz frisch. Eben steige ich aus dem Bett. Nur einen kleinen\n"
        "Augenblick Geduld! Es geht noch nicht so gut, wie ich dachte. Es ist "
        "mir\n"
        "aber schon wohl. Wie das nur einen Menschen so überfallen kann! Noch\n"
        "gestern abend war mir ganz gut, meine Eltern wissen es ja, oder besser,\n"
        "schon gestern abend hatte ich eine kleine Vorahnung. Man hätte es mir\n"
        "ansehen müssen. Warum habe ich es nur im Geschäfte nicht gemeldet! Aber\n"
        "man denkt eben immer, daß man die Krankheit ohne Zuhausebleiben\n"
        "überstehen wird. Herr Prokurist! Schonen Sie meine Eltern! Für alle die\n"
        "Vorwürfe, die Sie mir jetzt machen, ist ja kein Grund; man hat mir ja\n"
        "davon auch kein Wort gesagt. Sie haben vielleicht die letzten Aufträge,\n"
        "die ich geschickt habe, nicht gelesen. Übrigens, noch mit dem "
        "Achtuhrzug\n"
        "fahre ich auf die Reise, die paar Stunden Ruhe haben mich gekräftigt.\n"
        "Halten Sie sich nur nicht auf, Herr Prokurist; ich bin gleich selbst im\n"
        "Geschäft, und haben Sie die Güte, das zu sagen und mich dem Herrn Chef\n"
        "zu empfehlen!«\n"
        "\n"
        "Und während Gregor dies alles hastig ausstieß und kaum wußte, was er\n"
        "sprach, hatte er sich leicht, wohl infolge der im Bett bereits "
        "erlangten\n"
        "Übung, dem Kasten genähert und versuchte nun, an ihm sich aufzurichten.\n"
        "Er wollte tatsächlich die Tür aufmachen, tatsächlich sich sehen lassen\n"
        "und mit dem Prokuristen sprechen; er war begierig zu erfahren, was die\n"
        "anderen, die jetzt so nach ihm verlangten, bei seinem Anblick sagen\n"
        "würden. Würden sie erschrecken, dann hatte Gregor keine Verantwortung\n"
        "mehr und konnte ruhig sein. Würden sie aber alles ruhig hinnehmen, dann\n"
        "hatte auch er keinen Grund sich aufzuregen, und konnte, wenn er sich\n"
        "beeilte, um acht Uhr tatsächlich auf dem Bahnhof sein. Zuerst glitt er\n"
        "nun einigemale von dem glatten Kasten ab, aber endlich gab er sich\n"
        "einen letzten Schwung und stand aufrecht da; auf die Schmerzen im\n"
        "Unterleib achtete er gar nicht mehr, so sehr sie auch brannten. Nun "
        "ließ\n"
        "er sich gegen die Rücklehne eines nahen Stuhles fallen, an deren "
        "Rändern\n"
        "er sich mit seinen Beinchen festhielt. Damit hatte er aber auch die\n"
        "Herrschaft über sich erlangt und verstummte, denn nun konnte er den\n"
        "Prokuristen anhören.\n"
        "\n"
        "»Haben Sie auch nur ein Wort verstanden?« fragte der Prokurist die\n"
        "Eltern, »er macht sich doch wohl nicht einen Narren aus uns?« »Um "
        "Gottes\n"
        "willen,« rief die Mutter schon unter Weinen, »er ist vielleicht schwer\n"
        "krank, und wir quälen ihn. Grete! Grete!« schrie sie dann. »Mutter?«\n"
        "rief die Schwester von der anderen Seite. Sie verständigten sich durch\n"
        "Gregors Zimmer. »Du mußt augenblicklich zum Arzt. Gregor ist krank.\n"
        "Rasch um den Arzt. Hast du Gregor jetzt reden hören?« »Das war eine\n"
        "Tierstimme,« sagte der Prokurist, auffallend leise gegenüber dem\n"
        "Schreien der Mutter. »Anna! Anna!« rief der Vater durch das Vorzimmer "
        "in\n"
        "die Küche und klatschte in die Hände, »sofort einen Schlosser holen!«\n"
        "Und schon liefen die zwei Mädchen mit rauschenden Röcken durch das\n"
        "Vorzimmer -- wie hatte sich die Schwester denn so schnell angezogen? --\n"
        "und rissen die Wohnungstüre auf. Man hörte gar nicht die Türe\n"
        "zuschlagen; sie hatten sie wohl offen gelassen, wie es in Wohnungen zu\n"
        "sein pflegt, in denen ein großes Unglück geschehen ist.\n"
        "\n"
        "Gregor war aber viel ruhiger geworden. Man verstand zwar also seine\n"
        "Worte nicht mehr, trotzdem sie ihm genug klar, klarer als früher,\n"
        "vorgekommen waren, vielleicht infolge der Gewöhnung des Ohres. Aber\n"
        "immerhin glaubte man nun schon daran, daß es mit ihm nicht ganz in\n"
        "Ordnung war, und war bereit, ihm zu helfen. Die Zuversicht und\n"
        "Sicherheit, womit die ersten Anordnungen getroffen worden waren, taten\n"
        "ihm wohl. Er fühlte sich wieder einbezogen in den menschlichen Kreis "
        "und\n"
        "erhoffte von beiden, vom Arzt und vom Schlosser, ohne sie eigentlich\n"
        "genau zu scheiden, großartige und überraschende Leistungen. Um für die\n"
        "sich nähernden entscheidenden Besprechungen eine möglichst klare Stimme\n"
        "zu bekommen, hustete er ein wenig ab, allerdings bemüht, dies ganz\n"
        "gedämpft zu tun, da möglicherweise auch schon dieses Geräusch anders "
        "als\n"
        "menschlicher Husten klang, was er selbst zu entscheiden sich nicht mehr\n"
        "getraute. Im Nebenzimmer war es inzwischen ganz still geworden.\n"
        "Vielleicht saßen die Eltern mit dem Prokuristen beim Tisch und\n"
        "tuschelten, vielleicht lehnten alle an der Türe und horchten.\n"
        "\n"
        "Gregor schob sich langsam mit dem Sessel zur Tür hin, ließ ihn dort "
        "los,\n"
        "warf sich gegen die Tür, hielt sich an ihr aufrecht -- die Ballen "
        "seiner\n"
        "Beinchen hatten ein wenig Klebstoff -- und ruhte sich dort einen\n"
        "Augenblick lang von der Anstrengung aus. Dann aber machte er sich "
        "daran,\n"
        "mit dem Mund den Schlüssel im Schloß umzudrehen. Es schien leider, daß\n"
        "er keine eigentlichen Zähne hatte, -- womit sollte er gleich den\n"
        "Schlüssel fassen? -- aber dafür waren die Kiefer freilich sehr stark,\n"
        "mit ihrer Hilfe brachte er auch wirklich den Schlüssel in Bewegung und\n"
        "achtete nicht darauf, daß er sich zweifellos irgendeinen Schaden\n"
        "zufügte, denn eine braune Flüssigkeit kam ihm aus dem Mund, floß über\n"
        "den Schlüssel und tropfte auf den Boden. »Hören Sie nur,« sagte der\n"
        "Prokurist im Nebenzimmer, »er dreht den Schlüssel um.« Das war für\n"
        "Gregor eine große Aufmunterung; aber alle hätten ihm zurufen sollen,\n"
        "auch der Vater und die Mutter: »Frisch, Gregor,« hätten sie rufen\n"
        "sollen, »immer nur heran, fest an das Schloß heran!« Und in der\n"
        "Vorstellung, daß alle seine Bemühungen mit Spannung verfolgten, verbiß\n"
        "er sich mit allem, was er an Kraft aufbringen konnte, besinnungslos in\n"
        "den Schlüssel. Je nach dem Fortschreiten der Drehung des Schlüssels\n"
        "umtanzte er das Schloß, hielt sich jetzt nur noch mit dem Munde\n"
        "aufrecht, und je nach Bedarf hing er sich an den Schlüssel oder drückte\n"
        "ihn dann wieder nieder mit der ganzen Last seines Körpers. Der hellere\n"
        "Klang des endlich zurückschnappenden Schlosses erweckte Gregor "
        "förmlich.\n"
        "Aufatmend sagte er sich: »Ich habe also den Schlosser nicht gebraucht,«\n"
        "und legte den Kopf auf die Klinke, um die Türe gänzlich zu öffnen.\n"
        "\n"
        "Da er die Türe auf diese Weise öffnen mußte, war sie eigentlich schon\n"
        "recht weit geöffnet, und er selbst noch nicht zu sehen. Er mußte sich\n"
        "erst langsam um den einen Türflügel herumdrehen, und zwar sehr\n"
        "vorsichtig, wenn er nicht gerade vor dem Eintritt ins Zimmer plump auf\n"
        "den Rücken fallen wollte. Er war noch mit jener schwierigen Bewegung\n"
        "beschäftigt und hatte nicht Zeit, auf anderes zu achten, da hörte er\n"
        "schon den Prokuristen ein lautes »Oh!« ausstoßen -- es klang, wie wenn\n"
        "der Wind saust -- und nun sah er ihn auch, wie er, der der Nächste an\n"
        "der Türe war, die Hand gegen den offenen Mund drückte und langsam\n"
        "zurückwich, als vertreibe ihn eine unsichtbare, gleichmäßig "
        "fortwirkende\n"
        "Kraft. Die Mutter -- sie stand hier trotz der Anwesenheit des\n"
        "Prokuristen mit von der Nacht her noch aufgelösten, hoch sich\n"
        "sträubenden Haaren -- sah zuerst mit gefalteten Händen den Vater an,\n"
        "ging dann zwei Schritte zu Gregor hin und fiel inmitten ihrer rings um\n"
        "sie herum sich ausbreitenden Röcke nieder, das Gesicht ganz "
        "unauffindbar\n"
        "zu ihrer Brust gesenkt. Der Vater ballte mit feindseligem Ausdruck die\n"
        "Faust, als wolle er Gregor in sein Zimmer zurückstoßen, sah sich dann\n"
        "unsicher im Wohnzimmer um, beschattete dann mit den Händen die Augen "
        "und\n"
        "weinte, daß sich seine mächtige Brust schüttelte.\n"
        "\n"
        "Gregor trat nun gar nicht in das Zimmer, sondern lehnte sich von innen\n"
        "an den festgeriegelten Türflügel, so daß sein Leib nur zur Hälfte und\n"
        "darüber der seitlich geneigte Kopf zu sehen war, mit dem er zu den\n"
        "anderen hinüberlugte. Es war inzwischen viel heller geworden; klar "
        "stand\n"
        "auf der anderen Straßenseite ein Ausschnitt des gegenüberliegenden,\n"
        "endlosen, grauschwarzen Hauses -- es war ein Krankenhaus -- mit seinen\n"
        "hart die Front durchbrechenden regelmäßigen Fenstern; der Regen fiel\n"
        "noch nieder, aber nur mit großen, einzeln sichtbaren und förmlich auch\n"
        "einzelnweise auf die Erde hinuntergeworfenen Tropfen. Das\n"
        "Frühstücksgeschirr stand in überreicher Zahl auf dem Tisch, denn für "
        "den\n"
        "Vater war das Frühstück die wichtigste Mahlzeit des Tages, die er bei\n"
        "der Lektüre verschiedener Zeitungen stundenlang hinzog. Gerade an der\n"
        "gegenüberliegenden Wand hing eine Photographie Gregors aus seiner\n"
        "Militärzeit, die ihn als Leutnant darstellte, wie er, die Hand am "
        "Degen,\n"
        "sorglos lächelnd, Respekt für seine Haltung und Uniform verlangte. Die\n"
        "Tür zum Vorzimmer war geöffnet, und man sah, da auch die Wohnungstür\n"
        "offen war, auf den Vorplatz der Wohnung hinaus und auf den Beginn der\n"
        "abwärts führenden Treppe.\n"
        "\n"
        "»Nun,« sagte Gregor und war sich dessen wohl bewußt, daß er der einzige\n"
        "war, der die Ruhe bewahrt hatte, »ich werde mich gleich anziehen, die\n"
        "Kollektion zusammenpacken und wegfahren. Wollt ihr, wollt ihr mich\n"
        "wegfahren lassen? Nun, Herr Prokurist, Sie sehen, ich bin nicht\n"
        "starrköpfig und ich arbeite gern; das Reisen ist beschwerlich, aber ich\n"
        "könnte ohne das Reisen nicht leben. Wohin gehen Sie denn, Herr\n"
        "Prokurist? Ins Geschäft? Ja? Werden Sie alles wahrheitsgetreu "
        "berichten?\n"
        "Man kann im Augenblick unfähig sein zu arbeiten, aber dann ist gerade\n"
        "der richtige Zeitpunkt, sich an die früheren Leistungen zu erinnern und\n"
        "zu bedenken, daß man später, nach Beseitigung des Hindernisses, gewiß\n"
        "desto fleißiger und gesammelter arbeiten wird. Ich bin ja dem Herrn "
        "Chef\n"
        "so sehr verpflichtet, das wissen Sie doch recht gut. Andererseits habe\n"
        "ich die Sorge um meine Eltern und die Schwester. Ich bin in der Klemme,\n"
        "ich werde mich aber auch wieder herausarbeiten. Machen Sie es mir aber\n"
        "nicht schwieriger, als es schon ist. Halten Sie im Geschäft meine\n"
        "Partei! Man liebt den Reisenden nicht, ich weiß. Man denkt, er verdient\n"
        "ein Heidengeld und führt dabei ein schönes Leben. Man hat eben keine\n"
        "besondere Veranlassung, dieses Vorurteil besser zu durchdenken. Sie\n"
        "aber, Herr Prokurist, Sie haben einen besseren Überblick über die\n"
        "Verhältnisse, als das sonstige Personal, ja sogar, ganz im Vertrauen\n"
        "gesagt, einen besseren Überblick, als der Herr Chef selbst, der in\n"
        "seiner Eigenschaft als Unternehmer sich in seinem Urteil leicht\n"
        "zuungunsten eines Angestellten beirren läßt. Sie wissen auch sehr wohl,\n"
        "daß der Reisende, der fast das ganze Jahr außerhalb des Geschäftes ist,\n"
        "so leicht ein Opfer von Klatschereien, Zufälligkeiten und grundlosen\n"
        "Beschwerden werden kann, gegen die sich zu wehren ihm ganz unmöglich\n"
        "ist, da er von ihnen meistens gar nichts erfährt und nur dann, wenn er\n"
        "erschöpft eine Reise beendet hat, zu Hause die schlimmen, auf ihre\n"
        "Ursachen hin nicht mehr zu durchschauenden Folgen am eigenen Leibe zu\n"
        "spüren bekommt. Herr Prokurist, gehen Sie nicht weg, ohne mir ein Wort\n"
        "gesagt zu haben, das mir zeigt, daß Sie mir wenigstens zu einem kleinen\n"
        "Teil recht geben!«\n"
        "\n"
        "Aber der Prokurist hatte sich schon bei den ersten Worten Gregors\n"
        "abgewendet, und nur über die zuckende Schulter hinweg sah er mit\n"
        "aufgeworfenen Lippen nach Gregor zurück. Und während Gregors Rede stand\n"
        "er keinen Augenblick still, sondern verzog sich, ohne Gregor aus den\n"
        "Augen zu lassen, gegen die Tür, aber ganz allmählich, als bestehe ein\n"
        "geheimes Verbot, das Zimmer zu verlassen. Schon war er im Vorzimmer, "
        "und\n"
        "nach der plötzlichen Bewegung, mit der er zum letztenmal den Fuß aus "
        "dem\n"
        "Wohnzimmer zog, hätte man glauben können, er habe sich soeben die Sohle\n"
        "verbrannt. Im Vorzimmer aber streckte er die rechte Hand weit von sich\n"
        "zur Treppe hin, als warte dort auf ihn eine geradezu überirdische\n"
        "Erlösung.\n"
        "\n"
        "Gregor sah ein, daß er den Prokuristen in dieser Stimmung auf keinen\n"
        "Fall weggehen lassen dürfe, wenn dadurch seine Stellung im Geschäft\n"
        "nicht aufs äußerste gefährdet werden sollte. Die Eltern verstanden das\n"
        "alles nicht so gut; sie hatten sich in den langen Jahren die "
        "Überzeugung\n"
        "gebildet, daß Gregor in diesem Geschäft für sein Leben versorgt war, "
        "und\n"
        "hatten außerdem jetzt mit den augenblicklichen Sorgen so viel zu tun,\n"
        "daß ihnen jede Voraussicht abhanden gekommen war. Aber Gregor hatte\n"
        "diese Voraussicht. Der Prokurist mußte gehalten, beruhigt, überzeugt "
        "und\n"
        "schließlich gewonnen werden; die Zukunft Gregors und seiner Familie "
        "hing\n"
        "doch davon ab! Wäre doch die Schwester hier gewesen! Sie war klug; sie\n"
        "hatte schon geweint, als Gregor noch ruhig auf dem Rücken lag. Und "
        "gewiß\n"
        "hätte der Prokurist, dieser Damenfreund, sich von ihr lenken lassen;\n"
        "sie hätte die Wohnungstür zugemacht und ihm im Vorzimmer den Schrecken\n"
        "ausgeredet. Aber die Schwester war eben nicht da, Gregor selbst mußte\n"
        "handeln. Und ohne daran zu denken, daß er seine gegenwärtigen\n"
        "Fähigkeiten, sich zu bewegen, noch gar nicht kannte, ohne auch daran zu\n"
        "denken, daß seine Rede möglicher- ja wahrscheinlicherweise wieder nicht\n"
        "verstanden worden war, verließ er den Türflügel; schob sich durch die\n"
        "Öffnung; wollte zum Prokuristen hingehen, der sich schon am Geländer "
        "des\n"
        "Vorplatzes lächerlicherweise mit beiden Händen festhielt; fiel aber\n"
        "sofort, nach einem Halt suchend, mit einem kleinen Schrei auf seine\n"
        "vielen Beinchen nieder. Kaum war das geschehen, fühlte er zum erstenmal\n"
        "an diesem Morgen ein körperliches Wohlbehagen; die Beinchen hatten\n"
        "festen Boden unter sich; sie gehorchten vollkommen, wie er zu seiner\n"
        "Freude merkte; strebten sogar darnach, ihn fortzutragen, wohin er\n"
        "wollte; und schon glaubte er, die endgültige Besserung alles Leidens\n"
        "stehe unmittelbar bevor. Aber im gleichen Augenblick, als er da\n"
        "schaukelnd vor verhaltener Bewegung, gar nicht weit von seiner Mutter\n"
        "entfernt, ihr gerade gegenüber auf dem Boden lag, sprang diese, die "
        "doch\n"
        "so ganz in sich versunken schien, mit einemmale in die Höhe, die Arme\n"
        "weit ausgestreckt, die Finger gespreizt, rief: »Hilfe, um Gottes willen\n"
        "Hilfe!«, hielt den Kopf geneigt, als wolle sie Gregor besser sehen, "
        "lief\n"
        "aber, im Widerspruch dazu, sinnlos zurück; hatte vergessen, daß hinter\n"
        "ihr der gedeckte Tisch stand; setzte sich, als sie bei ihm angekommen\n"
        "war, wie in Zerstreutheit, eilig auf ihn, und schien gar nicht zu\n"
        "merken, daß neben ihr aus der umgeworfenen großen Kanne der Kaffee in\n"
        "vollem Strome auf den Teppich sich ergoß.\n"
        "\n"
        "»Mutter, Mutter,« sagte Gregor leise und sah zu ihr hinauf. Der\n"
        "Prokurist war ihm für einen Augenblick ganz aus dem Sinn gekommen;\n"
        "dagegen konnte er sich nicht versagen, im Anblick des fließenden "
        "Kaffees\n"
        "mehrmals mit den Kiefern ins Leere zu schnappen. Darüber schrie die\n"
        "Mutter neuerdings auf, flüchtete vom Tisch und fiel dem ihr\n"
        "entgegeneilenden Vater in die Arme. Aber Gregor hatte jetzt keine Zeit\n"
        "für seine Eltern; der Prokurist war schon auf der Treppe; das Kinn auf\n"
        "dem Geländer, sah er noch zum letzten Male zurück. Gregor nahm einen\n"
        "Anlauf, um ihn möglichst sicher einzuholen; der Prokurist mußte etwas\n"
        "ahnen, denn er machte einen Sprung über mehrere Stufen und verschwand;\n"
        "»Huh!« aber schrie er noch, es klang durchs ganze Treppenhaus. Leider\n"
        "schien nun auch diese Flucht des Prokuristen den Vater, der bisher\n"
        "verhältnismäßig gefaßt gewesen war, völlig zu verwirren, denn statt\n"
        "selbst dem Prokuristen nachzulaufen oder wenigstens Gregor in der\n"
        "Verfolgung nicht zu hindern, packte er mit der Rechten den Stock des\n"
        "Prokuristen, den dieser mit Hut und Überzieher auf einem Sessel\n"
        "zurückgelassen hatte, holte mit der Linken eine große Zeitung vom Tisch\n"
        "und machte sich unter Füßestampfen daran, Gregor durch Schwenken des\n"
        "Stockes und der Zeitung in sein Zimmer zurückzutreiben. Kein Bitten\n"
        "Gregors half, kein Bitten wurde auch verstanden, er mochte den Kopf "
        "noch\n"
        "so demütig drehen, der Vater stampfte nur stärker mit den Füßen. Drüben\n"
        "hatte die Mutter trotz des kühlen Wetters ein Fenster aufgerissen, und\n"
        "hinausgelehnt drückte sie ihr Gesicht weit außerhalb des Fensters in\n"
        "ihre Hände. Zwischen Gasse und Treppenhaus entstand eine starke "
        "Zugluft,\n"
        "die Fenstervorhänge flogen auf, die Zeitungen auf dem Tische rauschten,\n"
        "einzelne Blätter wehten über den Boden hin. Unerbittlich drängte der\n"
        "Vater und stieß Zischlaute aus, wie ein Wilder. Nun hatte aber Gregor\n"
        "noch gar keine Übung im Rückwärtsgehen, es ging wirklich sehr langsam.\n"
        "Wenn sich Gregor nur hätte umdrehen dürfen, er wäre gleich in seinem\n"
        "Zimmer gewesen, aber er fürchtete sich, den Vater durch die "
        "zeitraubende\n"
        "Umdrehung ungeduldig zu machen, und jeden Augenblick drohte ihm doch "
        "von\n"
        "dem Stock in des Vaters Hand der tödliche Schlag auf den Rücken oder "
        "auf\n"
        "den Kopf. Endlich aber blieb Gregor doch nichts anderes übrig, denn er\n"
        "merkte mit Entsetzen, daß er im Rückwärtsgehen nicht einmal die "
        "Richtung\n"
        "einzuhalten verstand; und so begann er, unter unaufhörlichen "
        "ängstlichen\n"
        "Seitenblicken nach dem Vater, sich nach Möglichkeit rasch, in\n"
        "Wirklichkeit aber doch nur sehr langsam umzudrehen. Vielleicht merkte\n"
        "der Vater seinen guten Willen, denn er störte ihn hierbei nicht, "
        "sondern\n"
        "dirigierte sogar hie und da die Drehbewegung von der Ferne mit der\n"
        "Spitze seines Stockes. Wenn nur nicht dieses unerträgliche Zischen des\n"
        "Vaters gewesen wäre! Gregor verlor darüber ganz den Kopf. Er war schon\n"
        "fast ganz umgedreht, als er sich, immer auf dieses Zischen horchend,\n"
        "sogar irrte und sich wieder ein Stück zurückdrehte. Als er aber endlich\n"
        "glücklich mit dem Kopf vor der Türöffnung war, zeigte es sich, daß sein\n"
        "Körper zu breit war, um ohne weiteres durchzukommen. Dem Vater fiel es\n"
        "natürlich in seiner gegenwärtigen Verfassung auch nicht entfernt ein,\n"
        "etwa den anderen Türflügel zu öffnen, um für Gregor einen genügenden\n"
        "Durchgang zu schaffen. Seine fixe Idee war bloß, daß Gregor so rasch "
        "als\n"
        "möglich in sein Zimmer müsse. Niemals hätte er auch die umständlichen\n"
        "Vorbereitungen gestattet, die Gregor brauchte, um sich aufzurichten und\n"
        "vielleicht auf diese Weise durch die Tür zu kommen. Vielleicht trieb "
        "er,\n"
        "als gäbe es kein Hindernis, Gregor jetzt unter besonderem Lärm\n"
        "vorwärts; es klang schon hinter Gregor gar nicht mehr wie die Stimme\n"
        "bloß eines einzigen Vaters; nun gab es wirklich keinen Spaß mehr, und\n"
        "Gregor drängte sich -- geschehe was wolle -- in die Tür. Die eine Seite\n"
        "seines Körpers hob sich, er lag schief in der Türöffnung, seine eine\n"
        "Flanke war ganz wundgerieben, an der weißen Tür blieben häßliche "
        "Flecke,\n"
        "bald steckte er fest und hätte sich allein nicht mehr rühren können, "
        "die\n"
        "Beinchen auf der einen Seite hingen zitternd oben in der Luft, die auf\n"
        "der anderen waren schmerzhaft zu Boden gedrückt -- da gab ihm der Vater\n"
        "von hinten einen jetzt wahrhaftig erlösenden starken Stoß, und er flog,\n"
        "heftig blutend, weit in sein Zimmer hinein. Die Tür wurde noch mit dem\n"
        "Stock zugeschlagen, dann war es endlich still.\n"
        "\n"
        "Erst in der Abenddämmerung erwachte Gregor aus seinem schweren\n"
        "ohnmachtähnlichen Schlaf. Er wäre gewiß nicht viel später auch ohne\n"
        "Störung erwacht, denn er fühlte sich genügend ausgeruht und\n"
        "ausgeschlafen, doch schien es ihm, als hätte ihn ein flüchtiger Schritt\n"
        "und ein vorsichtiges Schließen der zum Vorzimmer führenden Tür geweckt.\n"
        "Der Schein der elektrischen Straßenbahn lag bleich hier und da auf der\n"
        "Zimmerdecke und auf den höheren Teilen der Möbel, aber unten bei Gregor\n"
        "war es finster. Langsam schob er sich, noch ungeschickt mit seinen\n"
        "Fühlern tastend, die er jetzt erst schätzen lernte, zur Türe hin, um\n"
        "nachzusehen, was dort geschehen war. Seine linke Seite schien eine\n"
        "einzige lange, unangenehm spannende Narbe, und er mußte auf seinen zwei\n"
        "Beinreihen regelrecht hinken. Ein Beinchen war übrigens im Laufe der\n"
        "vormittägigen Vorfälle schwer verletzt worden -- es war fast ein\n"
        "Wunder, daß nur eines verletzt worden war -- und schleppte leblos nach.\n"
        "\n"
        "Erst bei der Tür merkte er, was ihn dorthin eigentlich gelockt hatte; "
        "es\n"
        "war der Geruch von etwas Eßbarem gewesen. Denn dort stand ein Napf mit\n"
        "süßer Milch gefüllt, in der kleine Schnitte von Weißbrot schwammen. "
        "Fast\n"
        "hätte er vor Freude gelacht, denn er hatte noch größeren Hunger als am\n"
        "Morgen, und gleich tauchte er seinen Kopf fast bis über die Augen in "
        "die\n"
        "Milch hinein. Aber bald zog er ihn enttäuscht wieder zurück; nicht nur,\n"
        "daß ihm das Essen wegen seiner heiklen linken Seite Schwierigkeiten\n"
        "machte -- und er konnte nur essen, wenn der ganze Körper schnaufend\n"
        "mitarbeitete --, so schmeckte ihm überdies die Milch, die sonst sein\n"
        "Lieblingsgetränk war und die ihm gewiß die Schwester deshalb\n"
        "hereingestellt hatte, gar nicht, ja er wandte sich fast mit Widerwillen\n"
        "von dem Napf ab und kroch in die Zimmermitte zurück.\n"
        "\n"
        "Im Wohnzimmer war, wie Gregor durch die Türspalte sah, das Gas\n"
        "angezündet, aber während sonst zu dieser Tageszeit der Vater seine\n"
        "nachmittags erscheinende Zeitung der Mutter und manchmal auch der\n"
        "Schwester mit erhobener Stimme vorzulesen pflegte, hörte man jetzt\n"
        "keinen Laut. Nun vielleicht war dieses Vorlesen, von dem ihm die\n"
        "Schwester immer erzählte und schrieb, in der letzten Zeit überhaupt aus\n"
        "der Übung gekommen. Aber auch ringsherum war es so still, trotzdem doch\n"
        "gewiß die Wohnung nicht leer war. »Was für ein stilles Leben die "
        "Familie\n"
        "doch führte,« sagte sich Gregor und fühlte, während er starr vor sich\n"
        "ins Dunkle sah, einen großen Stolz darüber, daß er seinen Eltern und\n"
        "seiner Schwester ein solches Leben in einer so schönen Wohnung hatte\n"
        "verschaffen können. Wie aber, wenn jetzt alle Ruhe, aller Wohlstand,\n"
        "alle Zufriedenheit ein Ende mit Schrecken nehmen sollte? Um sich nicht\n"
        "in solche Gedanken zu verlieren, setzte sich Gregor lieber in Bewegung\n"
        "und kroch im Zimmer auf und ab.\n"
        "\n"
        "Einmal während des langen Abends wurde die eine Seitentüre und einmal\n"
        "die andere bis zu einer kleinen Spalte geöffnet und rasch wieder\n"
        "geschlossen; jemand hatte wohl das Bedürfnis hereinzukommen, aber auch\n"
        "wieder zu viele Bedenken. Gregor machte nun unmittelbar bei der\n"
        "Wohnzimmertür Halt, entschlossen, den zögernden Besucher doch irgendwie\n"
        "hereinzubringen oder doch wenigstens zu erfahren, wer es sei; aber nun\n"
        "wurde die Tür nicht mehr geöffnet und Gregor wartete vergebens. Früh,\n"
        "als die Türen versperrt waren, hatten alle zu ihm hereinkommen wollen,\n"
        "jetzt, da er die eine Tür geöffnet hatte und die anderen offenbar\n"
        "während des Tages geöffnet worden waren, kam keiner mehr, und die\n"
        "Schlüssel steckten nun auch von außen.\n"
        "\n"
        "Spät erst in der Nacht wurde das Licht im Wohnzimmer ausgelöscht, und\n"
        "nun war leicht festzustellen, daß die Eltern und die Schwester so lange\n"
        "wachgeblieben waren, denn wie man genau hören konnte, entfernten sich\n"
        "jetzt alle drei auf den Fußspitzen. Nun kam gewiß bis zum Morgen "
        "niemand\n"
        "mehr zu Gregor herein; er hatte also eine lange Zeit, um ungestört zu\n"
        "überlegen, wie er sein Leben jetzt neu ordnen sollte. Aber das hohe\n"
        "freie Zimmer, in dem er gezwungen war, flach auf dem Boden zu liegen,\n"
        "ängstigte ihn, ohne daß er die Ursache herausfinden konnte, denn es war\n"
        "ja sein seit fünf Jahren von ihm bewohntes Zimmer -- und mit einer halb\n"
        "unbewußten Wendung und nicht ohne eine leichte Scham eilte er unter das\n"
        "Kanapee, wo er sich, trotzdem sein Rücken ein wenig gedrückt wurde und\n"
        "trotzdem er den Kopf nicht mehr erheben konnte, gleich sehr behaglich\n"
        "fühlte und nur bedauerte, daß sein Körper zu breit war, um vollständig\n"
        "unter dem Kanapee untergebracht zu werden.\n"
        "\n"
        "Dort blieb er die ganze Nacht, die er zum Teil im Halbschlaf, aus dem\n"
        "ihn der Hunger immer wieder aufschreckte, verbrachte, zum Teil aber in\n"
        "Sorgen und undeutlichen Hoffnungen, die aber alle zu dem Schlusse\n"
        "führten, daß er sich vorläufig ruhig verhalten und durch Geduld und\n"
        "größte Rücksichtnahme der Familie die Unannehmlichkeiten erträglich\n"
        "machen müsse, die er ihr in seinem gegenwärtigen Zustand nun einmal zu\n"
        "verursachen gezwungen war.\n"
        "\n"
        "Schon am frühen Morgen, es war fast noch Nacht, hatte Gregor\n"
        "Gelegenheit, die Kraft seiner eben gefaßten Entschlüsse zu prüfen, denn\n"
        "vom Vorzimmer her öffnete die Schwester, fast völlig angezogen, die Tür\n"
        "und sah mit Spannung herein. Sie fand ihn nicht gleich, aber als sie "
        "ihn\n"
        "unter dem Kanapee bemerkte -- Gott, er mußte doch irgendwo sein, er\n"
        "hatte doch nicht wegfliegen können -- erschrak sie so sehr, daß sie,\n"
        "ohne sich beherrschen zu können, die Tür von außen wieder zuschlug. "
        "Aber\n"
        "als bereue sie ihr Benehmen, öffnete sie die Tür sofort wieder und "
        "trat,\n"
        "als sei sie bei einem Schwerkranken oder gar bei einem Fremden, auf den\n"
        "Fußspitzen herein. Gregor hatte den Kopf bis knapp zum Rande des\n"
        "Kanapees vorgeschoben und beobachtete sie. Ob sie wohl bemerken würde,\n"
        "daß er die Milch stehen gelassen hatte, und zwar keineswegs aus Mangel\n"
        "an Hunger, und ob sie eine andere Speise hereinbringen würde, die ihm\n"
        "besser entsprach? Täte sie es nicht von selbst, er wollte lieber\n"
        "verhungern, als sie darauf aufmerksam machen, trotzdem es ihn "
        "eigentlich\n"
        "ungeheuer drängte, unterm Kanapee vorzuschießen, sich der Schwester zu\n"
        "Füßen zu werfen und sie um irgend etwas Gutes zum Essen zu bitten. Aber\n"
        "die Schwester bemerkte sofort mit Verwunderung den noch vollen Napf, "
        "aus\n"
        "dem nur ein wenig Milch ringsherum verschüttet war, sie hob ihn gleich\n"
        "auf, zwar nicht mit den bloßen Händen, sondern mit einem Fetzen, und\n"
        "trug ihn hinaus. Gregor war äußerst neugierig, was sie zum Ersatze\n"
        "bringen würde, und er machte sich die verschiedensten Gedanken darüber.\n"
        "Niemals aber hätte er erraten können, was die Schwester in ihrer Güte\n"
        "wirklich tat. Sie brachte ihm, um seinen Geschmack zu prüfen, eine "
        "ganze\n"
        "Auswahl, alles auf einer alten Zeitung ausgebreitet. Da war altes\n"
        "halbverfaultes Gemüse; Knochen vom Nachtmahl her, die von "
        "festgewordener\n"
        "weißer Sauce umgeben waren; ein paar Rosinen und Mandeln; ein Käse, den\n"
        "Gregor vor zwei Tagen für ungenießbar erklärt hatte; ein trockenes "
        "Brot,\n"
        "ein mit Butter beschmiertes Brot und ein mit Butter beschmiertes und\n"
        "gesalzenes Brot. Außerdem stellte sie zu dem allen noch den\n"
        "wahrscheinlich ein für allemal für Gregor bestimmten Napf, in den sie\n"
        "Wasser gegossen hatte. Und aus Zartgefühl, da sie wußte, daß Gregor vor\n"
        "ihr nicht essen würde, entfernte sie sich eiligst und drehte sogar den\n"
        "Schlüssel um, damit nur Gregor merken könne, daß er es sich so "
        "behaglich\n"
        "machen dürfe, wie er wolle. Gregors Beinchen schwirrten, als es jetzt\n"
        "zum Essen ging. Seine Wunden mußten übrigens auch schon vollständig\n"
        "geheilt sein, er fühlte keine Behinderung mehr, er staunte darüber und\n"
        "dachte daran, wie er vor mehr als einem Monat sich mit dem Messer ganz\n"
        "wenig in den Finger geschnitten, und wie ihm diese Wunde noch "
        "vorgestern\n"
        "genug wehgetan hatte. »Sollte ich jetzt weniger Feingefühl haben?«\n"
        "dachte er und saugte schon gierig an dem Käse, zu dem es ihn vor allen\n"
        "anderen Speisen sofort und nachdrücklich gezogen hatte. Rasch\n"
        "hintereinander und mit vor Befriedigung tränenden Augen verzehrte er "
        "den\n"
        "Käse, das Gemüse und die Sauce; die frischen Speisen dagegen schmeckten\n"
        "ihm nicht, er konnte nicht einmal ihren Geruch vertragen und schleppte\n"
        "sogar die Sachen, die er essen wollte, ein Stückchen weiter weg. Er war\n"
        "schon längst mit allem fertig und lag nur noch faul auf der gleichen\n"
        "Stelle, als die Schwester zum Zeichen, daß er sich zurückziehen solle,\n"
        "langsam den Schlüssel umdrehte. Das schreckte ihn sofort auf, trotzdem\n"
        "er schon fast schlummerte, und er eilte wieder unter das Kanapee. Aber\n"
        "es kostete ihn große Selbstüberwindung, auch nur die kurze Zeit, "
        "während\n"
        "welcher die Schwester im Zimmer war, unter dem Kanapee zu bleiben, denn\n"
        "von dem reichlichen Essen hatte sich sein Leib ein wenig gerundet, und\n"
        "er konnte dort in der Enge kaum atmen. Unter kleinen "
        "Erstickungsanfällen\n"
        "sah er mit etwas hervorgequollenen Augen zu, wie die nichtsahnende\n"
        "Schwester mit einem Besen nicht nur die Überbleibsel zusammenkehrte,\n"
        "sondern selbst die von Gregor gar nicht berührten Speisen, als seien\n"
        "also auch diese nicht mehr zu gebrauchen, und wie sie alles hastig in\n"
        "einen Kübel schüttete, den sie mit einem Holzdeckel schloß, worauf sie\n"
        "alles hinaustrug. Kaum hatte sie sich umgedreht, zog sich schon Gregor\n"
        "unter dem Kanapee hervor und streckte und blähte sich.\n"
        "\n"
        "Auf diese Weise bekam nun Gregor täglich sein Essen, einmal am Morgen,\n"
        "wenn die Eltern und das Dienstmädchen noch schliefen, das zweitemal "
        "nach\n"
        "dem allgemeinen Mittagessen, denn dann schliefen die Eltern gleichfalls\n"
        "noch ein Weilchen, und das Dienstmädchen wurde von der Schwester mit\n"
        "irgendeiner Besorgung weggeschickt. Gewiß wollten auch sie nicht, daß\n"
        "Gregor verhungere, aber vielleicht hätten sie es nicht ertragen können,\n"
        "von seinem Essen mehr als durch Hörensagen zu erfahren, vielleicht\n"
        "wollte die Schwester ihnen auch eine möglicherweise nur kleine Trauer\n"
        "ersparen, denn tatsächlich litten sie ja gerade genug.\n"
        "\n"
        "Mit welchen Ausreden man an jenem ersten Vormittag den Arzt und den\n"
        "Schlosser wieder aus der Wohnung geschafft hatte, konnte Gregor gar\n"
        "nicht erfahren, denn da er nicht verstanden wurde, dachte niemand "
        "daran,\n"
        "auch die Schwester nicht, daß er die anderen verstehen könne, und so\n"
        "mußte er sich, wenn die Schwester in seinem Zimmer war, damit begnügen,\n"
        "nur hier und da ihre Seufzer und Anrufe der Heiligen zu hören. Erst\n"
        "später, als sie sich ein wenig an alles gewöhnt hatte -- von\n"
        "vollständiger Gewöhnung konnte natürlich niemals die Rede sein --,\n"
        "erhaschte Gregor manchmal eine Bemerkung, die freundlich gemeint war\n"
        "oder so gedeutet werden konnte. »Heute hat es ihm aber geschmeckt,«\n"
        "sagte sie, wenn Gregor unter dem Essen tüchtig aufgeräumt hatte, "
        "während\n"
        "sie im gegenteiligen Fall, der sich allmählich immer häufiger\n"
        "wiederholte, fast traurig zu sagen pflegte: »Nun ist wieder alles\n"
        "stehengeblieben.«\n"
        "\n"
        "Während aber Gregor unmittelbar keine Neuigkeit erfahren konnte,\n"
        "erhorchte er manches aus den Nebenzimmern, und wo er nun einmal Stimmen\n"
        "hörte, lief er gleich zu der betreffenden Tür und drückte sich mit\n"
        "ganzem Leib an sie. Besonders in der ersten Zeit gab es kein Gespräch,\n"
        "das nicht irgendwie wenn auch nur im geheimen, von ihm handelte. Zwei\n"
        "Tage lang waren bei allen Mahlzeiten Beratungen darüber zu hören, wie\n"
        "man sich jetzt verhalten solle; aber auch zwischen den Mahlzeiten "
        "sprach\n"
        "man über das gleiche Thema, denn immer waren zumindest zwei\n"
        "Familienmitglieder zu Hause, da wohl niemand allein zu Hause bleiben\n"
        "wollte und man die Wohnung doch auf keinen Fall gänzlich verlassen\n"
        "konnte. Auch hatte das Dienstmädchen gleich am ersten Tag -- es war\n"
        "nicht ganz klar, was und wieviel sie von dem Vorgefallenen wußte --\n"
        "kniefällig die Mutter gebeten, sie sofort zu entlassen, und als sie "
        "sich\n"
        "eine Viertelstunde danach verabschiedete, dankte sie für die Entlassung\n"
        "unter Tränen, wie für die größte Wohltat, die man ihr hier erwiesen\n"
        "hatte, und gab, ohne daß man es von ihr verlangte, einen fürchterlichen\n"
        "Schwur ab, niemandem auch nur das geringste zu verraten.\n"
        "\n"
        "Nun mußte die Schwester im Verein mit der Mutter auch kochen; "
        "allerdings\n"
        "machte das nicht viel Mühe, denn man aß fast nichts. Immer wieder hörte\n"
        "Gregor, wie der eine den anderen vergebens zum Essen aufforderte und\n"
        "keine andere Antwort bekam, als: »Danke ich habe genug« oder etwas\n"
        "Ähnliches. Getrunken wurde vielleicht auch nichts. Öfters fragte die\n"
        "Schwester den Vater, ob er Bier haben wolle, und herzlich erbot sie\n"
        "sich, es selbst zu holen, und als der Vater schwieg, sagte sie, um ihm\n"
        "jedes Bedenken zu nehmen, sie könne auch die Hausmeisterin darum\n"
        "schicken, aber dann sagte der Vater schließlich ein großes »Nein«, und\n"
        "es wurde nicht mehr davon gesprochen.\n"
        "\n"
        "Schon im Laufe des ersten Tages legte der Vater die ganzen\n"
        "Vermögensverhältnisse und Aussichten sowohl der Mutter als auch der\n"
        "Schwester dar. Hie und da stand er vom Tische auf und holte aus seiner\n"
        "kleinen Wertheimkassa, die er aus dem vor fünf Jahren erfolgten\n"
        "Zusammenbruch seines Geschäftes gerettet hatte, irgendeinen Beleg oder\n"
        "irgendein Vormerkbuch. Man hörte, wie er das komplizierte Schloß\n"
        "aufsperrte und nach Entnahme des Gesuchten wieder verschloß. Diese\n"
        "Erklärungen des Vaters waren zum Teil das erste Erfreuliche, was Gregor\n"
        "seit seiner Gefangenschaft zu hören bekam. Er war der Meinung gewesen,\n"
        "daß dem Vater von jenem Geschäft her nicht das Geringste übriggeblieben\n"
        "war, zumindest hatte ihm der Vater nichts Gegenteiliges gesagt, und\n"
        "Gregor allerdings hatte ihn auch nicht darum gefragt. Gregors Sorge war\n"
        "damals nur gewesen, alles daranzusetzen, um die Familie das\n"
        "geschäftliche Unglück, das alle in eine vollständige Hoffnungslosigkeit\n"
        "gebracht hatte, möglichst rasch vergessen zu lassen. Und so hatte er\n"
        "damals mit ganz besonderem Feuer zu arbeiten angefangen und war fast\n"
        "über Nacht aus einem kleinen Kommis ein Reisender geworden, der\n"
        "natürlich ganz andere Möglichkeiten des Geldverdienens hatte, und "
        "dessen\n"
        "Arbeitserfolge sich sofort in Form der Provision zu Bargeld\n"
        "verwandelten, das der erstaunten und beglückten Familie zu Hause auf "
        "den\n"
        "Tisch gelegt werden konnte. Es waren schöne Zeiten gewesen, und niemals\n"
        "nachher hatten sie sich, wenigstens in diesem Glanze, wiederholt,\n"
        "trotzdem Gregor später so viel Geld verdiente, daß er den Aufwand der\n"
        "ganzen Familie zu tragen imstande war und auch trug. Man hatte sich "
        "eben\n"
        "daran gewöhnt, sowohl die Familie, als auch Gregor, man nahm das Geld\n"
        "dankbar an, er lieferte es gern ab, aber eine besondere Wärme wollte\n"
        "sich nicht mehr ergeben. Nur die Schwester war Gregor doch noch nahe\n"
        "geblieben, und es war sein geheimer Plan, sie, die zum Unterschied von\n"
        "Gregor Musik sehr liebte und rührend Violine zu spielen verstand,\n"
        "nächstes Jahr, ohne Rücksicht auf die großen Kosten, die das "
        "verursachen\n"
        "mußte, und die man schon auf andere Weise hereinbringen würde, auf das\n"
        "Konservatorium zu schicken. Öfters während der kurzen Aufenthalte\n"
        "Gregors in der Stadt wurde in den Gesprächen mit der Schwester das\n"
        "Konservatorium erwähnt, aber immer nur als schöner Traum, an dessen\n"
        "Verwirklichung nicht zu denken war, und die Eltern hörten nicht einmal\n"
        "diese unschuldigen Erwähnungen gern; aber Gregor dachte sehr bestimmt\n"
        "daran und beabsichtigte, es am Weihnachtsabend feierlich zu erklären.\n"
        "\n"
        "Solche in seinem gegenwärtigen Zustand ganz nutzlose Gedanken gingen "
        "ihm\n"
        "durch den Kopf, während er dort aufrecht an der Türe klebte und "
        "horchte.\n"
        "Manchmal konnte er vor allgemeiner Müdigkeit gar nicht mehr zuhören und\n"
        "ließ den Kopf nachlässig gegen die Tür schlagen, hielt ihn aber sofort\n"
        "wieder fest, denn selbst das kleine Geräusch, das er damit verursacht\n"
        "hatte, war nebenan gehört worden und hatte alle verstummen lassen. »Was\n"
        "er nur wieder treibt,« sagte der Vater nach einer Weile, offenbar zur\n"
        "Türe hingewendet, und dann erst wurde das unterbrochene Gespräch\n"
        "allmählich wieder aufgenommen.\n"
        "\n"
        "Gregor erfuhr nun zur Genüge -- denn der Vater pflegte sich in seinen\n"
        "Erklärungen öfters zu wiederholen, teils, weil er selbst sich mit "
        "diesen\n"
        "Dingen schon lange nicht beschäftigt hatte, teils auch, weil die Mutter\n"
        "nicht alles gleich beim erstenmal verstand --, daß trotz allen Unglücks\n"
        "ein allerdings ganz kleines Vermögen aus der alten Zeit noch vorhanden\n"
        "war, das die nicht angerührten Zinsen in der Zwischenzeit ein wenig\n"
        "hatten anwachsen lassen. Außerdem aber war das Geld, das Gregor\n"
        "allmonatlich nach Hause gebracht hatte -- er selbst hatte nur ein paar\n"
        "Gulden für sich behalten --, nicht vollständig aufgebraucht worden und\n"
        "hatte sich zu einem kleinen Kapital angesammelt. Gregor, hinter seiner\n"
        "Türe, nickte eifrig, erfreut über diese unerwartete Vorsicht und\n"
        "Sparsamkeit. Eigentlich hätte er ja mit diesen überschüssigen Geldern\n"
        "die Schuld des Vaters gegenüber dem Chef weiter abgetragen haben "
        "können,\n"
        "und jener Tag, an dem er diesen Posten hätte loswerden können, wäre "
        "weit\n"
        "näher gewesen, aber jetzt war es zweifellos besser so, wie es der Vater\n"
        "eingerichtet hatte.\n"
        "\n"
        "Nun genügte dieses Geld aber ganz und gar nicht, um die Familie etwa "
        "von\n"
        "den Zinsen leben zu lassen; es genügte vielleicht, um die Familie ein,\n"
        "höchstens zwei Jahre zu erhalten, mehr war es nicht. Es war also bloß\n"
        "eine Summe, die man eigentlich nicht angreifen durfte, und die für den\n"
        "Notfall zurückgelegt werden mußte; das Geld zum Leben aber mußte man\n"
        "verdienen. Nun war aber der Vater ein zwar gesunder, aber alter Mann,\n"
        "der schon fünf Jahre nichts gearbeitet hatte und sich jedenfalls nicht\n"
        "viel zutrauen durfte; er hatte in diesen fünf Jahren, welche die ersten\n"
        "Ferien seines mühevollen und doch erfolglosen Lebens waren, viel Fett\n"
        "angesetzt und war dadurch recht schwerfällig geworden. Und die alte\n"
        "Mutter sollte nun vielleicht Geld verdienen, die an Asthma litt, der\n"
        "eine Wanderung durch die Wohnung schon Anstrengung verursachte, und die\n"
        "jeden zweiten Tag in Atembeschwerden auf dem Sofa beim offenen Fenster\n"
        "verbrachte? Und die Schwester sollte Geld verdienen, die noch ein Kind\n"
        "war mit ihren siebzehn Jahren, und der ihre bisherige Lebensweise so\n"
        "sehr zu gönnen war, die daraus bestanden hatte, sich nett zu kleiden,\n"
        "lange zu schlafen, in der Wirtschaft mitzuhelfen, an ein paar\n"
        "bescheidenen Vergnügungen sich zu beteiligen und vor allem Violine zu\n"
        "spielen? Wenn die Rede auf diese Notwendigkeit des Geldverdienens kam,\n"
        "ließ zuerst immer Gregor die Türe los und warf sich auf das neben der\n"
        "Tür befindliche kühle Ledersofa, denn ihm war ganz heiß vor Beschämung\n"
        "und Trauer.\n"
        "\n"
        "Oft lag er dort die ganzen langen Nächte über, schlief keinen "
        "Augenblick\n"
        "und scharrte nur stundenlang auf dem Leder. Oder er scheute nicht die\n"
        "große Mühe, einen Sessel zum Fenster zu schieben, dann die\n"
        "Fensterbrüstung hinaufzukriechen und, in den Sessel gestemmt, sich ans\n"
        "Fenster zu lehnen, offenbar nur in irgendeiner Erinnerung an das\n"
        "Befreiende, das früher für ihn darin gelegen war, aus dem Fenster zu\n"
        "schauen. Denn tatsächlich sah er von Tag zu Tag die auch nur ein wenig\n"
        "entfernten Dinge immer undeutlicher; das gegenüberliegende Krankenhaus,\n"
        "dessen nur allzu häufigen Anblick er früher verflucht hatte, bekam er\n"
        "überhaupt nicht mehr zu Gesicht, und wenn er nicht genau gewußt hätte,\n"
        "daß er in der stillen, aber völlig städtischen Charlottenstraße wohnte,\n"
        "hätte er glauben können, von seinem Fenster aus in eine Einöde zu\n"
        "schauen in welcher der graue Himmel und die graue Erde ununterscheidbar\n"
        "sich vereinigten. Nur zweimal hatte die aufmerksame Schwester sehen\n"
        "müssen, daß der Sessel beim Fenster stand, als sie schon jedesmal,\n"
        "nachdem sie das Zimmer aufgeräumt hatte, den Sessel wieder genau zum\n"
        "Fenster hinschob, ja sogar von nun ab den inneren Fensterflügel offen\n"
        "ließ.\n"
        "\n"
        "Hätte Gregor nur mit der Schwester sprechen und ihr für alles danken\n"
        "können, was sie für ihn machen mußte, er hätte ihre Dienste leichter\n"
        "ertragen; so aber litt er darunter. Die Schwester suchte freilich die\n"
        "Peinlichkeit des Ganzen möglichst zu verwischen, und je längere Zeit\n"
        "verging, desto besser gelang es ihr natürlich auch, aber auch Gregor\n"
        "durchschaute mit der Zeit alles viel genauer. Schon ihr Eintritt war "
        "für\n"
        "ihn schrecklich. Kaum war sie eingetreten, lief sie, ohne sich Zeit zu\n"
        "nehmen, die Türe zu schließen, so sehr sie sonst darauf achtete, jedem\n"
        "den Anblick von Gregors Zimmer zu ersparen, geradewegs zum Fenster und\n"
        "riß es, als ersticke sie fast, mit hastigen Händen auf, blieb auch,\n"
        "selbst wenn es noch so kalt war, ein Weilchen beim Fenster und atmete\n"
        "tief. Mit diesem Laufen und Lärmen erschreckte sie Gregor täglich\n"
        "zweimal; die ganze Zeit über zitterte er unter dem Kanapee und wußte\n"
        "doch sehr gut, daß sie ihn gewiß gerne damit verschont hätte, wenn es\n"
        "ihr nur möglich gewesen wäre, sich in einem Zimmer, in dem sich Gregor\n"
        "befand, bei geschlossenem Fenster aufzuhalten.\n"
        "\n"
        "Einmal, es war wohl schon ein Monat seit Gregors Verwandlung vergangen,\n"
        "und es war doch schon für die Schwester kein besonderer Grund mehr, "
        "über\n"
        "Gregors Aussehen in Erstaunen zu geraten, kam sie ein wenig früher als\n"
        "sonst und traf Gregor noch an, wie er, unbeweglich und so recht zum\n"
        "Erschrecken aufgestellt, aus dem Fenster schaute. Es wäre für Gregor\n"
        "nicht unerwartet gewesen, wenn sie nicht eingetreten wäre, da er sie\n"
        "durch seine Stellung verhinderte, sofort das Fenster zu öffnen, aber "
        "sie\n"
        "trat nicht nur nicht ein, sie fuhr sogar zurück und schloß die Tür; ein\n"
        "Fremder hätte geradezu denken können, Gregor habe ihr aufgelauert und\n"
        "habe sie beißen wollen. Gregor versteckte sich natürlich sofort unter\n"
        "dem Kanapee, aber er mußte bis zum Mittag warten, ehe die Schwester\n"
        "wiederkam, und sie schien viel unruhiger als sonst. Er erkannte daraus,\n"
        "daß ihr sein Anblick noch immer unerträglich war und ihr auch weiterhin\n"
        "unerträglich bleiben müsse, und daß sie sich wohl sehr überwinden "
        "mußte,\n"
        "vor dem Anblick auch nur der kleinen Partie seines Körpers nicht\n"
        "davonzulaufen, mit der er unter dem Kanapee hervorragte. Um ihr auch\n"
        "diesen Anblick zu ersparen, trug er eines Tages auf seinem Rücken -- er\n"
        "brauchte zu dieser Arbeit vier Stunden -- das Leintuch auf das Kanapee\n"
        "und ordnete es in einer solchen Weise an, daß er nun gänzlich verdeckt\n"
        "war, und daß die Schwester, selbst wenn sie sich bückte, ihn nicht "
        "sehen\n"
        "konnte. Wäre dieses Leintuch ihrer Meinung nach nicht nötig gewesen,\n"
        "dann hätte sie es ja entfernen können, denn daß es nicht zum Vergnügen\n"
        "Gregors gehören konnte, sich so ganz und gar abzusperren, war doch klar\n"
        "genug, aber sie ließ das Leintuch, so wie es war, und Gregor glaubte\n"
        "sogar einen dankbaren Blick erhascht zu haben, als er einmal mit dem\n"
        "Kopf vorsichtig das Leintuch ein wenig lüftete, um nachzusehen, wie die\n"
        "Schwester die neue Einrichtung aufnahm.\n"
        "\n"
        "In den ersten vierzehn Tagen konnten es die Eltern nicht über sich\n"
        "bringen, zu ihm hereinzukommen, und er hörte oft, wie sie die jetzige\n"
        "Arbeit der Schwester völlig anerkannten, während sie sich bisher häufig\n"
        "über die Schwester geärgert hatten, weil sie ihnen als ein etwas\n"
        "nutzloses Mädchen erschienen war. Nun aber warteten oft beide, der "
        "Vater\n"
        "und die Mutter, vor Gregors Zimmer, während die Schwester dort\n"
        "aufräumte, und kaum war sie herausgekommen, mußte sie ganz genau\n"
        "erzählen, wie es in dem Zimmer aussah, was Gregor gegessen hatte, wie "
        "er\n"
        "sich diesmal benommen hatte, und ob vielleicht eine kleine Besserung zu\n"
        "bemerken war. Die Mutter übrigens wollte verhältnismäßig bald Gregor\n"
        "besuchen, aber der Vater und die Schwester hielten sie zuerst mit\n"
        "Vernunftgründen zurück, denen Gregor sehr aufmerksam zuhörte, und die "
        "er\n"
        "vollständig billigte. Später aber mußte man sie mit Gewalt "
        "zurückhalten,\n"
        "und wenn sie dann rief: »Laßt mich doch zu Gregor, er ist ja mein\n"
        "unglücklicher Sohn! Begreift ihr es denn nicht, daß ich zu ihm muß?«,\n"
        "dann dachte Gregor, daß es vielleicht doch gut wäre, wenn die Mutter\n"
        "hereinkäme, nicht jeden Tag natürlich, aber vielleicht einmal in der\n"
        "Woche; sie verstand doch alles viel besser als die Schwester, die trotz\n"
        "all ihrem Mute doch nur ein Kind war und im letzten Grunde vielleicht\n"
        "nur aus kindlichem Leichtsinn eine so schwere Aufgabe übernommen hatte.\n"
        "\n"
        "Der Wunsch Gregors, die Mutter zu sehen, ging bald in Erfüllung. "
        "Während\n"
        "des Tages wollte Gregor schon aus Rücksicht auf seine Eltern sich nicht\n"
        "beim Fenster zeigen, kriechen konnte er aber auf den paar Quadratmetern\n"
        "des Fußbodens auch nicht viel, das ruhige Liegen ertrug er schon "
        "während\n"
        "der Nacht schwer, das Essen machte ihm bald nicht mehr das geringste\n"
        "Vergnügen, und so nahm er zur Zerstreuung die Gewohnheit an, kreuz und\n"
        "quer über Wände und Plafond zu kriechen. Besonders oben an der Decke\n"
        "hing er gern; es war ganz anders, als das Liegen auf dem Fußboden; man\n"
        "atmete freier; ein leichtes Schwingen ging durch den Körper, und in der\n"
        "fast glücklichen Zerstreutheit, in der sich Gregor dort oben befand,\n"
        "konnte es geschehen, daß er zu seiner eigenen Überraschung sich losließ\n"
        "und auf den Boden klatschte. Aber nun hatte er natürlich seinen Körper\n"
        "ganz anders in der Gewalt als früher und beschädigte sich selbst bei\n"
        "einem so großen Falle nicht. Die Schwester nun bemerkte sofort die neue\n"
        "Unterhaltung, die Gregor für sich gefunden hatte -- er hinterließ ja\n"
        "auch beim Kriechen hie und da Spuren seines Klebstoffes --, und da\n"
        "setzte sie es sich in den Kopf, Gregor das Kriechen in größtem Ausmaße\n"
        "zu ermöglichen und die Möbel, die es verhinderten, also vor allem den\n"
        "Kasten und den Schreibtisch, wegzuschaffen. Nun war sie aber nicht\n"
        "imstande, dies allein zu tun; den Vater wagte sie nicht um Hilfe zu\n"
        "bitten; das Dienstmädchen hätte ihr ganz gewiß nicht geholfen, denn\n"
        "dieses etwa sechzehnjährige Mädchen harrte zwar tapfer seit Entlassung\n"
        "der früheren Köchin aus, hatte aber um die Vergünstigung gebeten, die\n"
        "Küche unaufhörlich versperrt halten zu dürfen und nur auf besonderen\n"
        "Anruf öffnen zu müssen; so blieb der Schwester also nichts übrig, als\n"
        "einmal in Abwesenheit des Vaters die Mutter zu holen. Mit Ausrufen\n"
        "erregter Freude kam die Mutter auch heran, verstummte aber an der Tür\n"
        "vor Gregors Zimmer. Zuerst sah natürlich die Schwester nach, ob alles "
        "im\n"
        "Zimmer in Ordnung war; dann erst ließ sie die Mutter eintreten. Gregor\n"
        "hatte in größter Eile das Leintuch noch tiefer und mehr in Falten\n"
        "gezogen, das Ganze sah wirklich nur wie ein zufällig über das Kanapee\n"
        "geworfenes Leintuch aus. Gregor unterließ auch diesmal, unter dem\n"
        "Leintuch zu spionieren; er verzichtete darauf, die Mutter schon diesmal\n"
        "zu sehen, und war nur froh, daß sie nun doch gekommen war. »Komm nur,\n"
        "man sieht ihn nicht,« sagte die Schwester, und offenbar führte sie die\n"
        "Mutter an der Hand. Gregor hörte nun, wie die zwei schwachen Frauen den\n"
        "immerhin schweren alten Kasten von seinem Platze rückten, und wie die\n"
        "Schwester immerfort den größten Teil der Arbeit für sich beanspruchte,\n"
        "ohne auf die Warnungen der Mutter zu hören, welche fürchtete, daß sie\n"
        "sich überanstrengen werde. Es dauerte sehr lange. Wohl nach schon\n"
        "viertelstündiger Arbeit sagte die Mutter, man solle den Kasten doch\n"
        "lieber hier lassen, denn erstens sei er zu schwer, sie würden vor\n"
        "Ankunft des Vaters nicht fertig werden und mit dem Kasten in der Mitte\n"
        "des Zimmers Gregor jeden Weg verrammeln, zweitens aber sei es doch gar\n"
        "nicht sicher, daß Gregor mit der Entfernung der Möbel ein Gefallen\n"
        "geschehe. Ihr scheine das Gegenteil der Fall zu sein; ihr bedrücke der\n"
        "Anblick der leeren Wand geradezu das Herz; und warum solle nicht auch\n"
        "Gregor diese Empfindung haben, da er doch an die Zimmermöbel längst\n"
        "gewöhnt sei und sich deshalb im leeren Zimmer verlassen fühlen werde.\n"
        "»Und ist es dann nicht so,« schloß die Mutter ganz leise, wie sie\n"
        "überhaupt fast flüsterte, als wolle sie vermeiden, daß Gregor, dessen\n"
        "genauen Aufenthalt sie ja nicht kannte, auch nur den Klang der Stimme\n"
        "höre, denn daß er die Worte nicht verstand, davon war sie überzeugt,\n"
        "»und ist es nicht so, als ob wir durch die Entfernung der Möbel "
        "zeigten,\n"
        "daß wir jede Hoffnung auf Besserung aufgeben und ihn rücksichtslos sich\n"
        "selbst überlassen? Ich glaube, es wäre das beste, wir suchen das Zimmer\n"
        "genau in dem Zustand zu erhalten, in dem es früher war, damit Gregor,\n"
        "wenn er wieder zu uns zurückkommt, alles unverändert findet und um so\n"
        "leichter die Zwischenzeit vergessen kann.«\n"
        "\n"
        "Beim Anhören dieser Worte der Mutter erkannte Gregor, daß der Mangel\n"
        "jeder unmittelbaren menschlichen Ansprache, verbunden mit dem\n"
        "einförmigen Leben inmitten der Familie, im Laufe dieser zwei Monate\n"
        "seinen Verstand hatte verwirren müssen, denn anders konnte er es sich\n"
        "nicht erklären, daß er ernsthaft darnach hatte verlangen können, daß\n"
        "sein Zimmer ausgeleert würde. Hatte er wirklich Lust, das warme, mit\n"
        "ererbten Möbeln gemütlich ausgestattete Zimmer in eine Höhle verwandeln\n"
        "zu lassen, in der er dann freilich nach allen Richtungen ungestört "
        "würde\n"
        "kriechen können, jedoch auch unter gleichzeitigem, schnellen, "
        "gänzlichen\n"
        "Vergessen seiner menschlichen Vergangenheit? War er doch jetzt schon\n"
        "nahe daran, zu vergessen, und nur die seit langem nicht gehörte Stimme\n"
        "der Mutter hatte ihn aufgerüttelt. Nichts sollte entfernt werden, alles\n"
        "mußte bleiben, die guten Einwirkungen der Möbel auf seinen Zustand\n"
        "konnte er nicht entbehren; und wenn die Möbel ihn hinderten, das\n"
        "sinnlose Herumkriechen zu betreiben, so war es kein Schaden, sondern "
        "ein\n"
        "großer Vorteil.\n"
        "\n"
        "Aber die Schwester war leider anderer Meinung; sie hatte sich,\n"
        "allerdings nicht ganz unberechtigt, angewöhnt, bei Besprechung der\n"
        "Angelegenheiten Gregors als besonders Sachverständige gegenüber den\n"
        "Eltern aufzutreten, und so war auch jetzt der Rat der Mutter für die\n"
        "Schwester Grund genug, auf der Entfernung nicht nur des Kastens und des\n"
        "Schreibtisches, an die sie zuerst allein gedacht hatte, sondern auf der\n"
        "Entfernung sämtlicher Möbel, mit Ausnahme des unentbehrlichen Kanapees,\n"
        "zu bestehen. Es war natürlich nicht nur kindlicher Trotz und das in der\n"
        "letzten Zeit so unerwartet und schwer erworbene Selbstvertrauen, das "
        "sie\n"
        "zu dieser Forderung bestimmte; sie hatte doch auch tatsächlich\n"
        "beobachtet, daß Gregor viel Raum zum Kriechen brauchte, dagegen die\n"
        "Möbel, soweit man sehen konnte, nicht im geringsten benützte. "
        "Vielleicht\n"
        "aber spielte auch der schwärmerische Sinn der Mädchen ihres Alters mit,\n"
        "der bei jeder Gelegenheit seine Befriedigung sucht, und durch den Grete\n"
        "jetzt sich dazu verlocken ließ, die Lage Gregors noch\n"
        "schreckenerregender machen zu wollen, um dann noch mehr als bis jetzt\n"
        "für ihn leisten zu können. Denn in einem Raum, in dem Gregor ganz "
        "allein\n"
        "die leeren Wände beherrschte, würde wohl kein Mensch außer Grete jemals\n"
        "einzutreten sich getrauen.\n"
        "\n"
        "Und so ließ sie sich von ihrem Entschlusse durch die Mutter nicht\n"
        "abbringen, die auch in diesem Zimmer vor lauter Unruhe unsicher schien,\n"
        "bald verstummte und der Schwester nach Kräften beim Hinausschaffen des\n"
        "Kastens half. Nun, den Kasten konnte Gregor im Notfall noch entbehren,\n"
        "aber schon der Schreibtisch mußte bleiben. Und kaum hatten die Frauen\n"
        "mit dem Kasten, an dem sie sich ächzend drückten, das Zimmer verlassen,\n"
        "als Gregor den Kopf unter dem Kanapee hervorstieß, um zu sehen, wie er\n"
        "vorsichtig und möglichst rücksichtsvoll eingreifen könnte. Aber zum\n"
        "Unglück war es gerade die Mutter, welche zuerst zurückkehrte, während\n"
        "Grete im Nebenzimmer den Kasten umfangen hielt und ihn allein hin und\n"
        "her schwang, ohne ihn natürlich von der Stelle zu bringen. Die Mutter\n"
        "aber war Gregors Anblick nicht gewöhnt, er hätte sie krank machen\n"
        "können, und so eilte Gregor erschrocken im Rückwärtslauf bis an das\n"
        "andere Ende des Kanapees, konnte es aber nicht mehr verhindern, daß das\n"
        "Leintuch vorne ein wenig sich bewegte. Das genügte, um die Mutter\n"
        "aufmerksam zu machen. Sie stockte, stand einen Augenblick still und "
        "ging\n"
        "dann zu Grete zurück.\n"
        "\n"
        "Trotzdem sich Gregor immer wieder sagte, daß ja nichts "
        "Außergewöhnliches\n"
        "geschehe, sondern nur ein paar Möbel umgestellt würden, wirkte doch, "
        "wie\n"
        "er sich bald eingestehen mußte, dieses Hin- und Hergehen der Frauen,\n"
        "ihre kleinen Zurufe, das Kratzen der Möbel auf dem Boden, wie ein\n"
        "großer, von allen Seiten genährter Trubel auf ihn, und er mußte sich, "
        "so\n"
        "fest er Kopf und Beine an sich zog und den Leib bis an den Boden\n"
        "drückte, unweigerlich sagen, daß er das Ganze nicht lange aushalten\n"
        "werde. Sie räumten ihm sein Zimmer aus; nahmen ihm alles, was ihm lieb\n"
        "war; den Kasten, in dem die Laubsäge und andere Werkzeuge lagen, hatten\n"
        "sie schon hinausgetragen; lockerten jetzt den schon im Boden fest\n"
        "eingegrabenen Schreibtisch, an dem er als Handelsakademiker, als\n"
        "Bürgerschüler, ja sogar schon als Volksschüler seine Aufgaben\n"
        "geschrieben hatte, -- da hatte er wirklich keine Zeit mehr, die guten\n"
        "Absichten zu prüfen, welche die zwei Frauen hatten, deren Existenz er\n"
        "übrigens fast vergessen hatte, denn vor Erschöpfung arbeiteten sie "
        "schon\n"
        "stumm, und man hörte nur das schwere Tappen ihrer Füße.\n"
        "\n"
        "Und so brach er denn hervor -- die Frauen stützten sich gerade im\n"
        "Nebenzimmer an den Schreibtisch, um ein wenig zu verschnaufen --,\n"
        "wechselte viermal die Richtung des Laufes, er wußte wirklich nicht, was\n"
        "er zuerst retten sollte, da sah er an der im übrigen schon leeren Wand\n"
        "auffallend das Bild der in lauter Pelzwerk gekleideten Dame hängen,\n"
        "kroch eilends hinauf und preßte sich an das Glas, das ihn festhielt und\n"
        "seinem heißen Bauch wohltat. Dieses Bild wenigstens, das Gregor jetzt\n"
        "ganz verdeckte, würde nun gewiß niemand wegnehmen. Er verdrehte den "
        "Kopf\n"
        "nach der Tür des Wohnzimmers, um die Frauen bei ihrer Rückkehr zu\n"
        "beobachten.\n"
        "\n"
        "Sie hatten sich nicht viel Ruhe gegönnt und kamen schon wieder; Grete\n"
        "hatte den Arm um die Mutter gelegt und trug sie fast. »Also was nehmen\n"
        "wir jetzt?« sagte Grete und sah sich um, Da kreuzten sich ihre Blicke\n"
        "mit denen Gregors an der Wand. Wohl nur infolge der Gegenwart der "
        "Mutter\n"
        "behielt sie ihre Fassung, beugte ihr Gesicht zur Mutter, um diese vom\n"
        "Herumschauen abzuhalten, und sagte, allerdings zitternd und unüberlegt:\n"
        "»Komm, wollen wir nicht lieber auf einen Augenblick noch ins Wohnzimmer\n"
        "zurückgehen?« Die Absicht Gretes war für Gregor klar, sie wollte die\n"
        "Mutter in Sicherheit bringen und dann ihn von der Wand hinunterjagen.\n"
        "Nun, sie konnte es ja immerhin versuchen! Er saß auf seinem Bild und\n"
        "gab es nicht her. Lieber würde er Grete ins Gesicht springen.\n"
        "\n"
        "Aber Gretes Worte hatten die Mutter erst recht beunruhigt, sie trat zur\n"
        "Seite, erblickte den riesigen braunen Fleck auf der geblümten Tapete,\n"
        "rief, ehe ihr eigentlich zum Bewußtsein kam, daß das Gregor war, was "
        "sie\n"
        "sah, mit schreiender, rauher Stimme: »Ach Gott, ach Gott!« und fiel mit\n"
        "ausgebreiteten Armen, als gebe sie alles auf, über das Kanapee hin und\n"
        "rührte sich nicht. »Du, Gregor!« rief die Schwester mit erhobener Faust\n"
        "und eindringlichen Blicken. Es waren seit der Verwandlung die ersten\n"
        "Worte, die sie unmittelbar an ihn gerichtet hatte. Sie lief ins\n"
        "Nebenzimmer, um irgendeine Essenz zu holen, mit der sie die Mutter aus\n"
        "ihrer Ohnmacht wecken könnte; Gregor wollte auch helfen -- zur Rettung\n"
        "des Bildes war noch Zeit --; er klebte aber fest an dem Glas und mußte\n"
        "sich mit Gewalt losreißen; er lief dann auch ins Nebenzimmer, als könne\n"
        "er der Schwester irgendeinen Rat geben, wie in früherer Zeit; mußte "
        "aber\n"
        "dann untätig hinter ihr stehen; während sie in verschiedenen Fläschchen\n"
        "kramte, erschreckte sie noch, als sie sich umdrehte; eine Flasche fiel\n"
        "auf den Boden und zerbrach; ein Splitter verletzte Gregor im Gesicht,\n"
        "irgendeine ätzende Medizin umfloß ihn; Grete nahm nun, ohne sich länger\n"
        "aufzuhalten, so viele Fläschchen, als sie nur halten konnte, und rannte\n"
        "mit ihnen zur Mutter hinein; die Tür schlug sie mit dem Fuße zu. Gregor\n"
        "war nun von der Mutter abgeschlossen, die durch seine Schuld vielleicht\n"
        "dem Tode nahe war; die Tür durfte er nicht öffnen, wollte er die\n"
        "Schwester, die bei der Mutter bleiben mußte, nicht verjagen; er hatte\n"
        "jetzt nichts zu tun, als zu warten; und von Selbstvorwürfen und\n"
        "Besorgnis bedrängt, begann er zu kriechen, überkroch alles, Wände,\n"
        "Möbel und Zimmerdecke und fiel endlich in seiner Verzweiflung, als sich\n"
        "das ganze Zimmer schon um ihn zu drehen anfing, mitten auf den großen\n"
        "Tisch.\n"
        "\n"
        "Es verging eine kleine Weile, Gregor lag matt da, ringsherum war es\n"
        "still, vielleicht war das ein gutes Zeichen. Da läutete es. Das Mädchen\n"
        "war natürlich in ihrer Küche eingesperrt und Grete mußte daher öffnen\n"
        "gehen. Der Vater war gekommen. »Was ist geschehen?« waren seine ersten\n"
        "Worte; Gretes Aussehen hatte ihm wohl alles verraten. Grete antwortete\n"
        "mit dumpfer Stimme, offenbar drückte sie ihr Gesicht an des Vaters\n"
        "Brust: »Die Mutter war ohnmächtig, aber es geht ihr schon besser. "
        "Gregor\n"
        "ist ausgebrochen.« »Ich habe es ja erwartet,« sagte der Vater, »ich "
        "habe\n"
        "es euch ja immer gesagt, aber ihr Frauen wollt nicht hören.« Gregor war\n"
        "es klar, daß der Vater Gretes allzukurze Mitteilung schlecht gedeutet\n"
        "hatte und annahm, daß Gregor sich irgendeine Gewalttat habe zuschulden\n"
        "kommen lassen. Deshalb mußte Gregor den Vater jetzt zu besänftigen\n"
        "suchen, denn ihn aufzuklären hatte er weder Zeit noch Möglichkeit. Und\n"
        "so flüchtete er sich zur Tür seines Zimmers und drückte sich an sie,\n"
        "damit der Vater beim Eintritt vom Vorzimmer her gleich sehen könne, daß\n"
        "Gregor die beste Absicht habe, sofort in sein Zimmer zurückzukehren, "
        "und\n"
        "daß es nicht nötig sei, ihn zurückzutreiben, sondern daß man nur die "
        "Tür\n"
        "zu öffnen brauchte, und gleich werde er verschwinden.\n"
        "\n"
        "Aber der Vater war nicht in der Stimmung, solche Feinheiten zu "
        "bemerken.\n"
        "»Ah!« rief er gleich beim Eintritt in einem Tone, als sei er\n"
        "gleichzeitig wütend und froh. Gregor zog den Kopf von der Tür zurück "
        "und\n"
        "hob ihn gegen den Vater. So hatte er sich den Vater wirklich nicht\n"
        "vorgestellt, wie er jetzt dastand; allerdings hatte er in der letzten\n"
        "Zeit über dem neuartigen Herumkriechen versäumt, sich so wie früher um\n"
        "die Vorgänge in der übrigen Wohnung zu kümmern, und hätte eigentlich\n"
        "darauf gefaßt sein müssen, veränderte Verhältnisse anzutreffen.\n"
        "Trotzdem, trotzdem, war das noch der Vater? Der gleiche Mann, der müde\n"
        "im Bett vergraben lag, wenn früher Gregor zu einer Geschäftsreise\n"
        "ausgerückt war; der ihn an Abenden der Heimkehr im Schlafrock im\n"
        "Lehnstuhl empfangen hatte; gar nicht recht imstande war, aufzustehen,\n"
        "sondern zum Zeichen der Freude nur die Arme gehoben hatte, und der bei\n"
        "den seltenen gemeinsamen Spaziergängen an ein paar Sonntagen im Jahr "
        "und\n"
        "an den höchsten Feiertagen zwischen Gregor und der Mutter, die schon an\n"
        "und für sich langsam gingen, immer noch ein wenig langsamer, in seinen\n"
        "alten Mantel eingepackt, mit stets vorsichtig aufgesetztem Krückstock\n"
        "sich vorwärts arbeitete und, wenn er etwas sagen wollte, fast immer\n"
        "stillstand und seine Begleitung um sich versammelte? Nun aber war er\n"
        "doch gut aufgerichtet; in eine straffe blaue Uniform mit Goldknöpfen\n"
        "gekleidet, wie sie Diener der Bankinstitute tragen; über dem hohen\n"
        "steifen Kragen des Rockes entwickelte sich sein starkes Doppelkinn;\n"
        "unter den buschigen Augenbrauen drang der Blick der schwarzen Augen\n"
        "frisch und aufmerksam hervor; das sonst zerzauste weiße Haar war zu\n"
        "einer peinlich genauen, leuchtenden Scheitelfrisur niedergekämmt. Er\n"
        "warf seine Mütze, auf der ein Goldmonogramm, wahrscheinlich das einer\n"
        "Bank, angebracht war, über das ganze Zimmer im Bogen auf das Kanapee "
        "hin\n"
        "und ging, die Enden seines langen Uniformrockes zurückgeschlagen, die\n"
        "Hände in den Hosentaschen, mit verbissenem Gesicht auf Gregor zu. Er\n"
        "wußte wohl selbst nicht, was er vorhatte; immerhin hob er die Füße\n"
        "ungewöhnlich hoch, und Gregor staunte über die Riesengröße seiner\n"
        "Stiefelsohlen. Doch hielt er sich dabei nicht auf, er wußte ja noch vom\n"
        "ersten Tage seines neuen Lebens her, daß der Vater ihm gegenüber nur "
        "die\n"
        "größte Strenge für angebracht ansah. Und so lief er vor dem Vater her,\n"
        "stockte, wenn der Vater stehen blieb, und eilte schon wieder vorwärts,\n"
        "wenn sich der Vater nur rührte. So machten sie mehrmals die Runde um "
        "das\n"
        "Zimmer, ohne daß sich etwas Entscheidendes ereignete, ja ohne daß das\n"
        "Ganze infolge seines langsamen Tempos den Anschein einer Verfolgung\n"
        "gehabt hätte. Deshalb blieb auch Gregor vorläufig auf dem Fußboden,\n"
        "zumal er fürchtete, der Vater könnte eine Flucht auf die Wände oder den\n"
        "Plafond für besondere Bosheit halten. Allerdings mußte sich Gregor\n"
        "sagen, daß er sogar dieses Laufen nicht lange aushalten würde, denn\n"
        "während der Vater einen Schritt machte, mußte er eine Unzahl von\n"
        "Bewegungen ausführen. Atemnot begann sich schon bemerkbar zu machen, "
        "wie\n"
        "er ja auch in seiner früheren Zeit keine ganz vertrauenswürdige Lunge\n"
        "besessen hatte. Als er nun so dahintorkelte, um alle Kräfte für den "
        "Lauf\n"
        "zu sammeln, kaum die Augen offenhielt; in seiner Stumpfheit an eine\n"
        "andere Rettung als durch Laufen gar nicht dachte; und fast schon\n"
        "vergessen hatte, daß ihm die Wände freistanden, die hier allerdings mit\n"
        "sorgfältig geschnitzten Möbeln voll Zacken und Spitzen verstellt waren\n"
        "-- da flog knapp neben ihm, leicht geschleudert, irgend etwas nieder "
        "und\n"
        "rollte vor ihm her. Es war ein Apfel; gleich flog ihm ein zweiter nach;\n"
        "Gregor blieb vor Schrecken stehen; ein Weiterlaufen war nutzlos, denn\n"
        "der Vater hatte sich entschlossen, ihn zu bombardieren. Aus der\n"
        "Obstschale auf der Kredenz hatte er sich die Taschen gefüllt und warf\n"
        "nun, ohne vorläufig scharf zu zielen, Apfel für Apfel. Diese kleinen\n"
        "roten Äpfel rollten wie elektrisiert auf dem Boden herum und stießen\n"
        "aneinander. Ein schwach geworfener Apfel streifte Gregors Rücken, glitt\n"
        "aber unschädlich ab. Ein ihm sofort nachfliegender drang dagegen\n"
        "förmlich in Gregors Rücken ein; Gregor wollte sich weiterschleppen, als\n"
        "könne der überraschende unglaubliche Schmerz mit dem Ortswechsel\n"
        "vergehen; doch fühlte er sich wie festgenagelt und streckte sich in\n"
        "vollständiger Verwirrung aller Sinne. Nur mit dem letzten Blick sah er\n"
        "noch, wie die Tür seines Zimmers aufgerissen wurde, und vor der\n"
        "schreienden Schwester die Mutter hervoreilte, im Hemd, denn die\n"
        "Schwester hatte sie entkleidet, um ihr in der Ohnmacht Atemfreiheit zu\n"
        "verschaffen, wie dann die Mutter auf den Vater zulief und ihr auf dem\n"
        "Weg die aufgebundenen Röcke einer nach dem anderen zu Boden glitten, "
        "und\n"
        "wie sie stolpernd über die Röcke auf den Vater eindrang und ihn\n"
        "umarmend, in gänzlicher Vereinigung mit ihm -- nun versagte aber "
        "Gregors\n"
        "Sehkraft schon -- die Hände an des Vaters Hinterkopf um Schonung von\n"
        "Gregors Leben bat.\n"
        "\n"
        "Die schwere Verwundung Gregors, an der er über einen Monat litt -- der\n"
        "Apfel blieb, da ihn niemand zu entfernen wagte, als sichtbares Andenken\n"
        "im Fleische sitzen --, schien selbst den Vater daran erinnert zu haben,\n"
        "daß Gregor trotz seiner gegenwärtigen traurigen und ekelhaften Gestalt\n"
        "ein Familienglied war, das man nicht wie einen Feind behandeln durfte,\n"
        "sondern dem gegenüber es das Gebot der Familienpflicht war, den\n"
        "Widerwillen hinunterzuschlucken und zu dulden, nichts als dulden.\n"
        "\n"
        "Und wenn nun auch Gregor durch seine Wunde an Beweglichkeit\n"
        "wahrscheinlich für immer verloren hatte und vorläufig zur Durchquerung\n"
        "seines Zimmers wie ein alter Invalide lange, lange Minuten brauchte --\n"
        "an das Kriechen in der Höhe war nicht zu denken --, so bekam er für\n"
        "diese Verschlimmerung seines Zustandes einen seiner Meinung nach\n"
        "vollständig genügenden Ersatz dadurch, daß immer gegen Abend die\n"
        "Wohnzimmertür, die er schon ein bis zwei Stunden vorher scharf zu\n"
        "beobachten pflegte, geöffnet wurde, so daß er, im Dunkel seines Zimmers\n"
        "liegend, vom Wohnzimmer aus unsichtbar, die ganze Familie beim\n"
        "beleuchteten Tische sehen und ihre Reden, gewissermaßen mit allgemeiner\n"
        "Erlaubnis, also ganz anders als früher, anhören durfte.\n"
        "\n"
        "Freilich waren es nicht mehr die lebhaften Unterhaltungen der früheren\n"
        "Zeiten, an die Gregor in den kleinen Hotelzimmern stets mit einigem\n"
        "Verlangen gedacht hatte, wenn er sich müde in das feuchte Bettzeug "
        "hatte\n"
        "werfen müssen. Es ging jetzt meist nur sehr still zu. Der Vater schlief\n"
        "bald nach dem Nachtessen in seinem Sessel ein; die Mutter und Schwester\n"
        "ermahnten einander zur Stille; die Mutter nähte, weit über das Licht\n"
        "vorgebeugt, feine Wäsche für ein Modengeschäft; die Schwester, die eine\n"
        "Stellung als Verkäuferin angenommen hatte, lernte am Abend Stenographie\n"
        "und Französisch, um vielleicht später einmal einen besseren Posten zu\n"
        "erreichen. Manchmal wachte der Vater auf, und als wisse er gar nicht,\n"
        "daß er geschlafen habe, sagte er zur Mutter: »Wie lange du heute schon\n"
        "wieder nähst!« und schlief sofort wieder ein, während Mutter und\n"
        "Schwester einander müde zulächelten.\n"
        "\n"
        "Mit einer Art Eigensinn weigerte sich der Vater, auch zu Hause seine\n"
        "Dieneruniform abzulegen; und während der Schlafrock nutzlos am\n"
        "Kleiderhaken hing, schlummerte der Vater vollständig angezogen auf\n"
        "seinem Platz, als sei er immer zu seinem Dienste bereit und warte auch\n"
        "hier auf die Stimme des Vorgesetzten. Infolgedessen verlor die gleich\n"
        "anfangs nicht neue Uniform trotz aller Sorgfalt von Mutter und "
        "Schwester\n"
        "an Reinlichkeit, und Gregor sah oft ganze Abende lang auf dieses über\n"
        "und über fleckige, mit seinen stets geputzten Goldknöpfen leuchtende\n"
        "Kleid, in dem der alte Mann höchst unbequem und doch ruhig schlief.\n"
        "\n"
        "Sobald die Uhr zehn schlug, suchte die Mutter durch leise Zusprache den\n"
        "Vater zu wecken und dann zu überreden, ins Bett zu gehen, denn hier war\n"
        "es doch kein richtiger Schlaf und diesen hatte der Vater, der um sechs\n"
        "Uhr seinen Dienst antreten mußte, äußerst nötig. Aber in dem Eigensinn,\n"
        "der ihn, seitdem er Diener war, ergriffen hatte, bestand er immer\n"
        "darauf, noch länger bei Tisch zu bleiben, trotzdem er regelmäßig\n"
        "einschlief, und war dann überdies nur mit der größten Mühe zu bewegen,\n"
        "den Sessel mit dem Bett zu vertauschen. Da mochten Mutter und Schwester\n"
        "mit kleinen Ermahnungen noch so sehr auf ihn eindringen,\n"
        "viertelstundenlang schüttelte er langsam den Kopf, hielt die Augen\n"
        "geschlossen und stand nicht auf. Die Mutter zupfte ihn am Ärmel, sagte\n"
        "ihm Schmeichelworte ins Ohr, die Schwester verließ ihre Aufgabe, um der\n"
        "Mutter zu helfen, aber beim Vater verfing das nicht. Er versank nur "
        "noch\n"
        "tiefer in seinen Sessel. Erst bis ihn die Frauen unter den Achseln\n"
        "faßten, schlug er die Augen auf, sah abwechselnd die Mutter und die\n"
        "Schwester an und pflegte zu sagen: »Das ist ein Leben. Das ist die Ruhe\n"
        "meiner alten Tage.« Und auf die beiden Frauen gestützt, erhob er sich,\n"
        "umständlich, als sei er für sich selbst die größte Last, ließ sich von\n"
        "den Frauen bis zur Türe führen, winkte ihnen dort ab und ging nun\n"
        "selbständig weiter, während die Mutter ihr Nähzeug, die Schwester ihre\n"
        "Feder eiligst hinwarfen, um hinter dem Vater zu laufen und ihm weiter\n"
        "behilflich zu sein.\n"
        "\n"
        "Wer hatte in dieser abgearbeiteten und übermüdeten Familie Zeit, sich "
        "um\n"
        "Gregor mehr zu kümmern, als unbedingt nötig war? Der Haushalt wurde\n"
        "immer mehr eingeschränkt; das Dienstmädchen wurde nun doch entlassen;\n"
        "eine riesige knochige Bedienerin mit weißem, den Kopf umflatterndem "
        "Haar\n"
        "kam des Morgens und des Abends, um die schwerste Arbeit zu leisten;\n"
        "alles andere besorgte die Mutter neben ihrer vielen Näharbeit. Es\n"
        "geschah sogar, daß verschiedene Familienschmuckstücke, welche früher "
        "die\n"
        "Mutter und die Schwester überglücklich bei Unterhaltungen und\n"
        "Feierlichkeiten getragen hatten, verkauft wurden, wie Gregor am Abend\n"
        "aus der allgemeinen Besprechung der erzielten Preise erfuhr. Die größte\n"
        "Klage war aber stets, daß man diese für die gegenwärtigen Verhältnisse\n"
        "allzugroße Wohnung nicht verlassen konnte, da es nicht auszudenken war,\n"
        "wie man Gregor übersiedeln sollte. Aber Gregor sah wohl ein, daß es\n"
        "nicht nur die Rücksicht auf ihn war, welche eine Übersiedlung\n"
        "verhinderte, denn ihn hätte man doch in einer passenden Kiste mit ein\n"
        "paar Luftlöchern leicht transportieren können; was die Familie\n"
        "hauptsächlich vom Wohnungswechsel abhielt, war vielmehr die völlige\n"
        "Hoffnungslosigkeit und der Gedanke daran, daß sie mit einem Unglück\n"
        "geschlagen war, wie niemand sonst im ganzen Verwandten- und\n"
        "Bekanntenkreis. Was die Welt von armen Leuten verlangt, erfüllten sie\n"
        "bis zum äußersten, der Vater holte den kleinen Bankbeamten das\n"
        "Frühstück, die Mutter opferte sich für die Wäsche fremder Leute, die\n"
        "Schwester lief nach dem Befehl der Kunden hinter dem Pulte hin und her,\n"
        "aber weiter reichten die Kräfte der Familie schon nicht. Und die Wunde\n"
        "im Rücken fing Gregor wie neu zu schmerzen an, wenn Mutter und\n"
        "Schwester, nachdem sie den Vater zu Bett gebracht hatten, nun\n"
        "zurückkehrten, die Arbeit liegen ließen, nahe zusammenrückten, schon\n"
        "Wange an Wange saßen; wenn jetzt die Mutter, auf Gregors Zimmer "
        "zeigend,\n"
        "sagte: »Mach' dort die Tür zu, Grete,« und wenn nun Gregor wieder im\n"
        "Dunkel war, während nebenan die Frauen ihre Tränen vermischten oder gar\n"
        "tränenlos den Tisch anstarrten.\n"
        "\n"
        "Die Nächte und Tage verbrachte Gregor fast ganz ohne Schlaf. Manchmal\n"
        "dachte er daran, beim nächsten Öffnen der Tür die Angelegenheiten der\n"
        "Familie ganz so wie früher wieder in die Hand zu nehmen; in seinen\n"
        "Gedanken erschienen wieder nach langer Zeit der Chef und der Prokurist,\n"
        "die Kommis und die Lehrjungen, der so begriffsstützige Hausknecht, zwei\n"
        "drei Freunde aus anderen Geschäften, ein Stubenmädchen aus einem Hotel\n"
        "in der Provinz, eine liebe, flüchtige Erinnerung, eine Kassiererin aus\n"
        "einem Hutgeschäft, um die er sich ernsthaft, aber zu langsam beworben\n"
        "hatte -- sie alle erschienen untermischt mit Fremden oder schon\n"
        "Vergessenen, aber statt ihm und seiner Familie zu helfen, waren sie\n"
        "sämtlich unzugänglich, und er war froh, wenn sie verschwanden. Dann "
        "aber\n"
        "war er wieder gar nicht in der Laune, sich um seine Familie zu sorgen,\n"
        "bloß Wut über die schlechte Wartung erfüllte ihn, und trotzdem er sich\n"
        "nichts vorstellen konnte, worauf er Appetit gehabt hätte, machte er "
        "doch\n"
        "Pläne, wie er in die Speisekammer gelangen könnte, um dort zu nehmen,\n"
        "was ihm, auch wenn er keinen Hunger hatte, immerhin gebührte. Ohne "
        "jetzt\n"
        "mehr nachzudenken, womit man Gregor einen besonderen Gefallen machen\n"
        "könnte, schob die Schwester eiligst, ehe sie morgens und mittags ins\n"
        "Geschäft lief, mit dem Fuß irgendeine beliebige Speise in Gregors "
        "Zimmer\n"
        "hinein, um sie am Abend, gleichgültig dagegen, ob die Speise vielleicht\n"
        "nur gekostet oder -- der häufigste Fall -- gänzlich unberührt war, mit\n"
        "einem Schwenken des Besens hinauszukehren. Das Aufräumen des Zimmers,\n"
        "das sie nun immer abends besorgte, konnte gar nicht mehr schneller "
        "getan\n"
        "sein. Schmutzstreifen zogen sich die Wände entlang, hie und da lagen\n"
        "Knäuel von Staub und Unrat. In der ersten Zeit stellte sich Gregor bei\n"
        "der Ankunft der Schwester in derartige besonders bezeichnende Winkel, "
        "um\n"
        "ihr durch diese Stellung gewissermaßen einen Vorwurf zu machen. Aber er\n"
        "hätte wohl wochenlang dort bleiben können, ohne daß sich die Schwester\n"
        "gebessert hätte; sie sah ja den Schmutz genau so wie er, aber sie hatte\n"
        "sich eben entschlossen, ihn zu lassen. Dabei wachte sie mit einer an "
        "ihr\n"
        "ganz neuen Empfindlichkeit, die überhaupt die ganze Familie ergriffen\n"
        "hatte, darüber, daß das Aufräumen von Gregors Zimmer ihr vorbehalten\n"
        "blieb. Einmal hatte die Mutter Gregors Zimmer einer großen Reinigung\n"
        "unterzogen, die ihr nur nach Verbrauch einiger Kübel Wasser gelungen "
        "war\n"
        "-- die viele Feuchtigkeit kränkte allerdings Gregor auch und er lag\n"
        "breit, verbittert und unbeweglich auf dem Kanapee --, aber die Strafe\n"
        "blieb für die Mutter nicht aus. Denn kaum hatte am Abend die Schwester\n"
        "die Veränderung in Gregors Zimmer bemerkt, als sie, aufs höchste\n"
        "beleidigt, ins Wohnzimmer lief und, trotz der beschwörend erhobenen\n"
        "Hände der Mutter, in einen Weinkrampf ausbrach, dem die Eltern -- der\n"
        "Vater war natürlich aus seinem Sessel aufgeschreckt worden -- zuerst\n"
        "erstaunt und hilflos zusahen; bis auch sie sich zu rühren anfingen; der\n"
        "Vater rechts der Mutter Vorwürfe machte, daß sie Gregors Zimmer nicht\n"
        "der Schwester zur Reinigung überließ; links dagegen die Schwester\n"
        "anschrie, sie werde niemals mehr Gregors Zimmer reinigen dürfen; "
        "während\n"
        "die Mutter den Vater, der sich vor Erregung nicht mehr kannte, ins\n"
        "Schlafzimmer zu schleppen suchte; die Schwester, von Schluchzen\n"
        "geschüttelt, mit ihren kleinen Fäusten den Tisch bearbeitete; und "
        "Gregor\n"
        "laut vor Wut darüber zischte, daß es keinem einfiel, die Tür zu\n"
        "schließen und ihm diesen Anblick und Lärm zu ersparen.\n"
        "\n"
        "Aber selbst wenn die Schwester, erschöpft von ihrer Berufsarbeit, "
        "dessen\n"
        "überdrüssig geworden war, für Gregor, wie früher, zu sorgen, so hätte\n"
        "noch keineswegs die Mutter für sie eintreten müssen und Gregor hätte\n"
        "doch nicht vernachlässigt zu werden brauchen. Denn nun war die\n"
        "Bedienerin da. Diese alte Witwe, die in ihrem langen Leben mit Hilfe\n"
        "ihres starken Knochenbaues das Ärgste überstanden haben mochte, hatte\n"
        "keinen eigentlichen Abscheu vor Gregor. Ohne irgendwie neugierig zu\n"
        "sein, hatte sie zufällig einmal die Tür von Gregors Zimmer aufgemacht\n"
        "und war im Anblick Gregors, der, gänzlich überrascht, trotzdem ihn\n"
        "niemand jagte, hin- und herzulaufen begann, die Hände im Schoß gefaltet\n"
        "staunend stehen geblieben. Seitdem versäumte sie nicht, stets flüchtig\n"
        "morgens und abends die Tür ein wenig zu öffnen und zu Gregor\n"
        "hineinzuschauen. Anfangs rief sie ihn auch zu sich herbei, mit Worten,\n"
        "die sie wahrscheinlich für freundlich hielt, wie »Komm mal herüber,\n"
        "alter Mistkäfer!« oder »Seht mal den alten Mistkäfer!« Auf solche\n"
        "Ansprachen antwortete Gregor mit nichts, sondern blieb unbeweglich auf\n"
        "seinem Platz, als sei die Tür gar nicht geöffnet worden. Hätte man doch\n"
        "dieser Bedienerin, statt sie nach ihrer Laune ihn nutzlos stören zu\n"
        "lassen, lieber den Befehl gegeben, sein Zimmer täglich zu reinigen!\n"
        "Einmal am frühen Morgen -- ein heftiger Regen, vielleicht schon ein\n"
        "Zeichen des kommenden Frühjahrs, schlug an die Scheiben -- war Gregor,\n"
        "als die Bedienerin mit ihren Redensarten wieder begann, derartig\n"
        "erbittert, daß er, wie zum Angriff, allerdings langsam und hinfällig,\n"
        "sich gegen sie wendete. Die Bedienerin aber, statt sich zu fürchten, "
        "hob\n"
        "bloß einen in der Nähe der Tür befindlichen Stuhl hoch empor, und wie\n"
        "sie mit groß geöffnetem Munde dastand, war ihre Absicht klar, den Mund\n"
        "erst zu schließen, wenn der Sessel in ihrer Hand auf Gregors Rücken\n"
        "niederschlagen würde. »Also weiter geht es nicht?« fragte sie, als\n"
        "Gregor sich wieder umdrehte, und stellte den Sessel ruhig in die Ecke\n"
        "zurück.\n"
        "\n"
        "Gregor aß nun fast gar nichts mehr. Nur wenn er zufällig an der\n"
        "vorbereiteten Speise vorüberkam, nahm er zum Spiel einen Bissen in den\n"
        "Mund, hielt ihn dort stundenlang und spie ihn dann meist wieder aus.\n"
        "Zuerst dachte er, es sei die Trauer über den Zustand seines Zimmers, "
        "die\n"
        "ihn vom Essen abhalte, aber gerade mit den Veränderungen des Zimmers\n"
        "söhnte er sich sehr bald aus. Man hatte sich angewöhnt, Dinge, die man\n"
        "anderswo nicht unterbringen konnte, in dieses Zimmer hineinzustellen,\n"
        "und solcher Dinge gab es nun viele, da man ein Zimmer der Wohnung an\n"
        "drei Zimmerherren vermietet hatte. Diese ernsten Herren, -- alle drei\n"
        "hatten Vollbärte, wie Gregor einmal durch eine Türspalte feststellte --\n"
        "waren peinlich auf Ordnung, nicht nur in ihrem Zimmer, sondern, da sie\n"
        "sich nun einmal hier eingemietet hatten, in der ganzen Wirtschaft, also\n"
        "insbesondere in der Küche, bedacht. Unnützen oder gar schmutzigen Kram\n"
        "ertrugen sie nicht. Überdies hatten sie zum größten Teil ihre eigenen\n"
        "Einrichtungsstücke mitgebracht. Aus diesem Grunde waren viele Dinge\n"
        "überflüssig geworden, die zwar nicht verkäuflich waren, die man aber\n"
        "auch nicht wegwerfen wollte. Alle diese wanderten in Gregors Zimmer.\n"
        "Ebenso auch die Aschenkiste und die Abfallkiste aus der Küche. Was nur\n"
        "im Augenblick unbrauchbar war, schleuderte die Bedienerin, die es immer\n"
        "sehr eilig hatte, einfach in Gregors Zimmer; Gregor sah "
        "glücklicherweise\n"
        "meist nur den betreffenden Gegenstand und die Hand, die ihn hielt. Die\n"
        "Bedienerin hatte vielleicht die Absicht, bei Zeit und Gelegenheit die\n"
        "Dinge wieder zu holen oder alle insgesamt mit einemmal hinauszuwerfen,\n"
        "tatsächlich aber blieben sie dort liegen, wohin sie durch den ersten\n"
        "Wurf gekommen waren, wenn nicht Gregor sich durch das Rumpelzeug wand\n"
        "und es in Bewegung brachte, zuerst gezwungen, weil kein sonstiger Platz\n"
        "zum Kriechen frei war, später aber mit wachsendem Vergnügen, obwohl er\n"
        "nach solchen Wanderungen, zum Sterben müde und traurig, wieder\n"
        "stundenlang sich nicht rührte.\n"
        "\n"
        "Da die Zimmerherren manchmal auch ihr Abendessen zu Hause im "
        "gemeinsamen\n"
        "Wohnzimmer einnahmen, blieb die Wohnzimmertür an manchen Abenden\n"
        "geschlossen, aber Gregor verzichtete ganz leicht auf das Öffnen der "
        "Tür,\n"
        "hatte er doch schon manche Abende, an denen sie geöffnet war, nicht\n"
        "ausgenützt, sondern war, ohne daß es die Familie merkte, im dunkelsten\n"
        "Winkel seines Zimmers gelegen. Einmal aber hatte die Bedienerin die Tür\n"
        "zum Wohnzimmer ein wenig offen gelassen, und sie blieb so offen, auch\n"
        "als die Zimmerherren am Abend eintraten und Licht gemacht wurde. Sie\n"
        "setzten sich oben an den Tisch, wo in früheren Zeiten der Vater, die\n"
        "Mutter und Gregor gesessen hatten, entfalteten die Servietten und "
        "nahmen\n"
        "Messer und Gabel in die Hand. Sofort erschien in der Tür die Mutter mit\n"
        "einer Schüssel Fleisch und knapp hinter ihr die Schwester mit einer\n"
        "Schüssel hochgeschichteter Kartoffeln. Das Essen dampfte mit starkem\n"
        "Rauch. Die Zimmerherren beugten sich über die vor sie hingestellten\n"
        "Schüsseln, als wollten sie sie vor dem Essen prüfen, und tatsächlich\n"
        "zerschnitt der, welcher in der Mitte saß und den anderen zwei als\n"
        "Autorität zu gelten schien, ein Stück Fleisch noch auf der Schüssel,\n"
        "offenbar um festzustellen, ob es mürbe genug sei und ob es nicht etwa "
        "in\n"
        "die Küche zurückgeschickt werden solle. Er war befriedigt, und Mutter\n"
        "und Schwester, die gespannt zugesehen hatten, begannen aufatmend zu\n"
        "lächeln.\n"
        "\n"
        "Die Familie selbst aß in der Küche. Trotzdem kam der Vater, ehe er in\n"
        "die Küche ging, in dieses Zimmer herein und machte mit einer einzigen\n"
        "Verbeugung, die Kappe in der Hand, einen Rundgang um den Tisch. Die\n"
        "Zimmerherren erhoben sich sämtlich und murmelten etwas in ihre Bärte.\n"
        "Als sie dann allein waren, aßen sie fast unter vollkommenem\n"
        "Stillschweigen. Sonderbar schien es Gregor, daß man aus allen\n"
        "mannigfachen Geräuschen des Essens immer wieder ihre kauenden Zähne\n"
        "heraushörte, als ob damit Gregor gezeigt werden sollte, daß man Zähne\n"
        "brauche, um zu essen, und daß man auch mit den schönsten zahnlosen\n"
        "Kiefern nichts ausrichten könne. »Ich habe ja Appetit,« sagte sich\n"
        "Gregor sorgenvoll, »aber nicht auf diese Dinge. Wie sich diese\n"
        "Zimmerherren nähren, und ich komme um!«\n"
        "\n"
        "Gerade an diesem Abend -- Gregor erinnerte sich nicht, während der\n"
        "ganzen Zeit die Violine gehört zu haben -- ertönte sie von der Küche\n"
        "her. Die Zimmerherren hatten schon ihr Nachtmahl beendet, der mittlere\n"
        "hatte eine Zeitung hervorgezogen, den zwei anderen je ein Blatt "
        "gegeben,\n"
        "und nun lasen sie zurückgelehnt und rauchten. Als die Violine zu "
        "spielen\n"
        "begann, wurden sie aufmerksam, erhoben sich und gingen auf den\n"
        "Fußspitzen zur Vorzimmertür, in der sie aneinandergedrängt stehen\n"
        "blieben. Man mußte sie von der Küche aus gehört haben, denn der Vater\n"
        "rief: »Ist den Herren das Spiel vielleicht unangenehm? Es kann sofort\n"
        "eingestellt werden.« »Im Gegenteil,« sagte der mittlere der Herren,\n"
        "»möchte das Fräulein nicht zu uns hereinkommen und hier im Zimmer\n"
        "spielen, wo es doch viel bequemer und gemütlicher ist?« »O bitte,« rief\n"
        "der Vater, als sei er der Violinspieler. Die Herren traten ins Zimmer\n"
        "zurück und warteten. Bald kam der Vater mit dem Notenpult, die Mutter\n"
        "mit den Noten und die Schwester mit der Violine. Die Schwester "
        "bereitete\n"
        "alles ruhig zum Spiele vor; die Eltern, die niemals früher Zimmer\n"
        "vermietet hatten und deshalb die Höflichkeit gegen die Zimmerherren\n"
        "übertrieben, wagten gar nicht, sich auf ihre eigenen Sessel zu setzen;\n"
        "der Vater lehnte an der Tür, die rechte Hand zwischen zwei Knöpfe des\n"
        "geschlossenen Livreerockes gesteckt; die Mutter aber erhielt von einem\n"
        "Herrn einen Sessel angeboten und saß, da sie den Sessel dort ließ, "
        "wohin\n"
        "ihn der Herr zufällig gestellt hatte, abseits in einem Winkel.\n"
        "\n"
        "Die Schwester begann zu spielen; Vater und Mutter verfolgten, jeder von\n"
        "seiner Seite, aufmerksam die Bewegungen ihrer Hände. Gregor hatte, von\n"
        "dem Spiele angezogen, sich ein wenig weiter vorgewagt und war schon mit\n"
        "dem Kopf im Wohnzimmer. Er wunderte sich kaum darüber, daß er in "
        "letzter\n"
        "Zeit so wenig Rücksicht auf die andern nahm; früher war diese\n"
        "Rücksichtnahme sein Stolz gewesen. Und dabei hätte er gerade jetzt mehr\n"
        "Grund gehabt, sich zu verstecken, denn infolge des Staubes, der in\n"
        "seinem Zimmer überall lag und bei der kleinsten Bewegung umherflog, war\n"
        "auch er ganz staubbedeckt; Fäden, Haare, Speiseüberreste schleppte er\n"
        "auf seinem Rücken und an den Seiten mit sich herum; seine\n"
        "Gleichgültigkeit gegen alles war viel zu groß, als daß er sich, wie\n"
        "früher mehrmals während des Tages, auf den Rücken gelegt und am Teppich\n"
        "gescheuert hätte. Und trotz dieses Zustandes hatte er keine Scheu, ein\n"
        "Stück auf dem makellosen Fußboden des Wohnzimmers vorzurücken.\n"
        "\n"
        "Allerdings achtete auch niemand auf ihn. Die Familie war gänzlich vom\n"
        "Violinspiel in Anspruch genommen; die Zimmerherren dagegen, die\n"
        "zunächst, die Hände in den Hosentaschen, viel zu nahe hinter dem\n"
        "Notenpult der Schwester sich aufgestellt hatten, so daß sie alle in die\n"
        "Noten hätte sehen können, was sicher die Schwester stören mußte, zogen\n"
        "sich bald unter halblauten Gesprächen mit gesenkten Köpfen zum Fenster\n"
        "zurück, wo sie, vom Vater besorgt beobachtet, auch blieben. Es hatte "
        "nun\n"
        "wirklich den überdeutlichen Anschein, als wären sie in ihrer Annahme,\n"
        "ein schönes oder unterhaltendes Violinspiel zu hören, enttäuscht, "
        "hätten\n"
        "die ganze Vorführung satt und ließen sich nur aus Höflichkeit noch in\n"
        "ihrer Ruhe stören. Besonders die Art, wie sie alle aus Nase und Mund "
        "den\n"
        "Rauch ihrer Zigarren in die Höhe bliesen, ließ auf große Nervosität\n"
        "schließen. Und doch spielte die Schwester so schön. Ihr Gesicht war zur\n"
        "Seite geneigt, prüfend und traurig folgten ihre Blicke den Notenzeilen.\n"
        "Gregor kroch noch ein Stück vorwärts und hielt den Kopf eng an den\n"
        "Boden, um möglicherweise ihren Blicken begegnen zu können. War er ein\n"
        "Tier, da ihn Musik so ergriff? Ihm war, als zeige sich ihm der Weg zu\n"
        "der ersehnten unbekannten Nahrung. Er war entschlossen, bis zur\n"
        "Schwester vorzudringen, sie am Rock zu zupfen und ihr dadurch\n"
        "anzudeuten, sie möge doch mit ihrer Violine in sein Zimmer kommen, denn\n"
        "niemand lohnte hier das Spiel so, wie er es lohnen wollte. Er wollte "
        "sie\n"
        "nicht mehr aus seinem Zimmer lassen, wenigstens nicht, solange er "
        "lebte;\n"
        "seine Schreckgestalt sollte ihm zum erstenmal nützlich werden; an allen\n"
        "Türen seines Zimmers wollte er gleichzeitig sein und den Angreifern\n"
        "entgegenfauchen; die Schwester aber sollte nicht gezwungen, sondern\n"
        "freiwillig bei ihm bleiben; sie sollte neben ihm auf dem Kanapee "
        "sitzen,\n"
        "das Ohr zu ihm herunterneigen, und er wollte ihr dann anvertrauen, daß\n"
        "er die feste Absicht gehabt habe, sie auf das Konservatorium zu\n"
        "schicken, und daß er dies, wenn nicht das Unglück dazwischen gekommen\n"
        "wäre, vergangene Weihnachten -- Weihnachten war doch wohl schon "
        "vorüber?\n"
        "-- allen gesagt hätte, ohne sich um irgendwelche Widerreden zu kümmern.\n"
        "Nach dieser Erklärung würde die Schwester in Tränen der Rührung\n"
        "ausbrechen, und Gregor würde sich bis zu ihrer Achsel erheben und ihren\n"
        "Hals küssen, den sie, seitdem sie ins Geschäft ging, frei ohne Band "
        "oder\n"
        "Kragen trug.\n"
        "\n"
        "»Herr Samsa!« rief der mittlere Herr dem Vater zu und zeigte, ohne ein\n"
        "weiteres Wort zu verlieren, mit dem Zeigefinger auf den langsam sich\n"
        "vorwärtsbewegenden Gregor. Die Violine verstummte, der mittlere\n"
        "Zimmerherr lächelte erst einmal kopfschüttelnd seinen Freunden zu und\n"
        "sah dann wieder auf Gregor hin. Der Vater schien es für nötiger zu\n"
        "halten, statt Gregor zu vertreiben, vorerst die Zimmerherren zu\n"
        "beruhigen, trotzdem diese gar nicht aufgeregt waren und Gregor sie mehr\n"
        "als das Violinspiel zu unterhalten schien. Er eilte zu ihnen und suchte\n"
        "sie mit ausgebreiteten Armen in ihr Zimmer zu drängen und gleichzeitig\n"
        "mit seinem Körper ihnen den Ausblick auf Gregor zu nehmen. Sie wurden\n"
        "nun tatsächlich ein wenig böse, man wußte nicht mehr, ob über das\n"
        "Benehmen des Vaters oder über die ihnen jetzt aufgehende Erkenntnis,\n"
        "ohne es zu wissen, einen solchen Zimmernachbar wie Gregor besessen zu\n"
        "haben. Sie verlangten vom Vater Erklärungen, hoben ihrerseits die Arme,\n"
        "zupften unruhig an ihren Bärten und wichen nur langsam gegen ihr Zimmer\n"
        "zurück. Inzwischen hatte die Schwester die Verlorenheit, in die sie "
        "nach\n"
        "dem plötzlich abgebrochenen Spiel verfallen war, überwunden, hatte "
        "sich,\n"
        "nachdem sie eine Zeitlang in den lässig hängenden Händen Violine und\n"
        "Bogen gehalten und weiter, als spiele sie noch, in die Noten gesehen\n"
        "hatte, mit einem Male aufgerafft, hatte das Instrument auf den Schoß "
        "der\n"
        "Mutter gelegt, die in Atembeschwerden mit heftig arbeitenden Lungen "
        "noch\n"
        "auf ihrem Sessel saß, und war in das Nebenzimmer gelaufen, dem sich die\n"
        "Zimmerherren unter dem Drängen des Vaters schon schneller näherten. Man\n"
        "sah, wie unter den geübten Händen der Schwester die Decken und Polster\n"
        "in den Betten in die Höhe flogen und sich ordneten. Noch ehe die Herren\n"
        "das Zimmer erreicht hatten, war sie mit dem Aufbetten fertig und\n"
        "schlüpfte heraus. Der Vater schien wieder von seinem Eigensinn derartig\n"
        "ergriffen, daß er jeden Respekt vergaß, den er seinen Mietern immerhin\n"
        "schuldete. Er drängte nur und drängte, bis schon in der Tür des Zimmers\n"
        "der mittlere der Herren donnernd mit dem Fuß aufstampfte und dadurch "
        "den\n"
        "Vater zum Stehen brachte. »Ich erkläre hiermit,« sagte er, hob die Hand\n"
        "und suchte mit den Blicken auch die Mutter und die Schwester, »daß ich\n"
        "mit Rücksicht auf die in dieser Wohnung und Familie herrschenden\n"
        "widerlichen Verhältnisse« -- hierbei spie er kurz entschlossen auf den\n"
        "Boden -- »mein Zimmer augenblicklich kündige. Ich werde natürlich auch\n"
        "für die Tage, die ich hier gewohnt habe, nicht das Geringste bezahlen,\n"
        "dagegen werde ich es mir noch überlegen, ob ich nicht mit irgendwelchen\n"
        "-- glauben Sie mir -- sehr leicht zu begründenden Forderungen gegen Sie\n"
        "auftreten werde.« Er schwieg und sah gerade vor sich hin, als erwarte "
        "er\n"
        "etwas. Tatsächlich fielen sofort seine zwei Freunde mit den Worten ein:\n"
        "»Auch wir kündigen augenblicklich.« Darauf faßte er die Türklinke und\n"
        "schloß mit einem Krach die Tür.\n"
        "\n"
        "Der Vater wankte mit tastenden Händen zu seinem Sessel und ließ sich\n"
        "hineinfallen; es sah aus, als strecke er sich zu seinem gewöhnlichen\n"
        "Abendschläfchen, aber das starke Nicken seines wie haltlosen Kopfes\n"
        "zeigte, daß er ganz und gar nicht schlief. Gregor war die ganze Zeit\n"
        "still auf dem Platz gelegen, auf dem ihn die Zimmerherren ertappt\n"
        "hatten. Die Enttäuschung über das Mißlingen seines Planes, vielleicht\n"
        "aber auch die durch das viele Hungern verursachte Schwäche machten es\n"
        "ihm unmöglich, sich zu bewegen. Er fürchtete mit einer gewissen\n"
        "Bestimmtheit schon für den nächsten Augenblick einen allgemeinen über\n"
        "ihn sich entladenden Zusammensturz und wartete. Nicht einmal die "
        "Violine\n"
        "schreckte ihn auf, die, unter den zitternden Fingern der Mutter hervor,\n"
        "ihr vom Schoße fiel und einen hallenden Ton von sich gab.\n"
        "\n"
        "»Liebe Eltern,« sagte die Schwester und schlug zur Einleitung mit der\n"
        "Hand auf den Tisch, »so geht es nicht weiter. Wenn ihr das vielleicht\n"
        "nicht einsehet, ich sehe es ein. Ich will vor diesem Untier nicht den\n"
        "Namen meines Bruders aussprechen und sage daher bloß: wir müssen\n"
        "versuchen es loszuwerden. Wir haben das Menschenmögliche versucht, es "
        "zu\n"
        "pflegen und zu dulden, ich glaube, es kann uns niemand den geringsten\n"
        "Vorwurf machen.«\n"
        "\n"
        "»Sie hat tausendmal recht,« sagte der Vater für sich. Die Mutter, die\n"
        "noch immer nicht genug Atem finden konnte, fing mit einem irrsinnigen\n"
        "Ausdruck der Augen dumpf in die vorgehaltene Hand zu husten an.\n"
        "\n"
        "Die Schwester eilte zur Mutter und hielt ihr die Stirn. Der Vater "
        "schien\n"
        "durch die Worte der Schwester auf bestimmtere Gedanken gebracht zu "
        "sein,\n"
        "hatte sich aufrecht gesetzt, spielte mit seiner Dienermütze zwischen "
        "den\n"
        "Tellern, die noch vom Nachtmahl der Zimmerherren her auf dem Tische\n"
        "standen, und sah bisweilen auf den stillen Gregor hin.\n"
        "\n"
        "»Wir müssen es loszuwerden suchen,« sagte die Schwester nun\n"
        "ausschließlich zum Vater, denn die Mutter hörte in ihrem Husten nichts,\n"
        "»es bringt euch noch beide um, ich sehe es kommen. Wenn man schon so\n"
        "schwer arbeiten muß, wie wir alle, kann man nicht noch zu Hause diese\n"
        "ewige Quälerei ertragen. Ich kann es auch nicht mehr.« Und sie brach so\n"
        "heftig in Weinen aus, daß ihre Tränen auf das Gesicht der Mutter\n"
        "niederflossen, von dem sie sie mit mechanischen Handbewegungen wischte.\n"
        "\n"
        "»Kind,« sagte der Vater mitleidig und mit auffallendem Verständnis, "
        "»was\n"
        "sollen wir aber tun?«\n"
        "\n"
        "Die Schwester zuckte nur die Achseln zum Zeichen der Ratlosigkeit, die\n"
        "sie nun während des Weinens im Gegensatz zu ihrer früheren Sicherheit\n"
        "ergriffen hatte.\n"
        "\n"
        "»Wenn er uns verstünde,« sagte der Vater halb fragend; die Schwester\n"
        "schüttelte aus dem Weinen heraus heftig die Hand zum Zeichen, daß daran\n"
        "nicht zu denken sei.\n"
        "\n"
        "»Wenn er uns verstünde,« wiederholte der Vater und nahm durch Schließen\n"
        "der Augen die Überzeugung der Schwester von der Unmöglichkeit dessen in\n"
        "sich auf, »dann wäre vielleicht ein Übereinkommen mit ihm möglich. Aber\n"
        "so --«\n"
        "\n"
        "»Weg muß es,« rief die Schwester, »das ist das einzige Mittel, Vater. "
        "Du\n"
        "mußt bloß den Gedanken loszuwerden suchen, daß es Gregor ist. Daß wir "
        "es\n"
        "so lange geglaubt haben, das ist ja unser eigentliches Unglück. Aber "
        "wie\n"
        "kann es denn Gregor sein? Wenn es Gregor wäre, er hätte längst\n"
        "eingesehen, daß ein Zusammenleben von Menschen mit einem solchen Tier\n"
        "nicht möglich ist, und wäre freiwillig fortgegangen. Wir hätten dann\n"
        "keinen Bruder, aber könnten weiter leben und sein Andenken in Ehren\n"
        "halten. So aber verfolgt uns dieses Tier, vertreibt die Zimmerherren,\n"
        "will offenbar die ganze Wohnung einnehmen und uns auf der Gasse\n"
        "übernachten lassen. Sieh nur, Vater,« schrie sie plötzlich auf, »er\n"
        "fängt schon wieder an!« Und in einem für Gregor gänzlich\n"
        "unverständlichen Schrecken verließ die Schwester sogar die Mutter, "
        "stieß\n"
        "sich förmlich von ihrem Sessel ab, als wollte sie lieber die Mutter\n"
        "opfern, als in Gregors Nähe bleiben, und eilte hinter den Vater, der,\n"
        "lediglich durch ihr Benehmen erregt, auch aufstand und die Arme wie zum\n"
        "Schutze der Schwester vor ihr halb erhob.\n"
        "\n"
        "Aber Gregor fiel es doch gar nicht ein, irgend jemandem und gar seiner\n"
        "Schwester Angst machen zu wollen. Er hatte bloß angefangen sich\n"
        "umzudrehen, um in sein Zimmer zurückzuwandern, und das nahm sich\n"
        "allerdings auffallend aus, da er infolge seines leidenden Zustandes bei\n"
        "den schwierigen Umdrehungen mit seinem Kopfe nachhelfen mußte, den er\n"
        "hierbei viele Male hob und gegen den Boden schlug. Er hielt inne und "
        "sah\n"
        "sich um. Seine gute Absicht schien erkannt worden zu sein; es war nur\n"
        "ein augenblicklicher Schrecken gewesen. Nun sahen ihn alle schweigend\n"
        "und traurig an. Die Mutter lag, die Beine ausgestreckt und\n"
        "aneinandergedrückt, in ihrem Sessel, die Augen fielen ihr vor Ermattung\n"
        "fast zu; der Vater und die Schwester saßen nebeneinander, die Schwester\n"
        "hatte ihre Hand um des Vaters Hals gelegt.\n"
        "\n"
        "»Nun darf ich mich schon vielleicht umdrehen,« dachte Gregor und begann\n"
        "seine Arbeit wieder. Er konnte das Schnaufen der Anstrengung nicht\n"
        "unterdrücken und mußte auch hie und da ausruhen. Im übrigen drängte ihn\n"
        "auch niemand, es war alles ihm selbst überlassen. Als er die Umdrehung\n"
        "vollendet hatte, fing er sofort an, geradeaus zurückzuwandern. Er\n"
        "staunte über die große Entfernung, die ihn von seinem Zimmer trennte,\n"
        "und begriff gar nicht, wie er bei seiner Schwäche vor kurzer Zeit den\n"
        "gleichen Weg, fast ohne es zu merken, zurückgelegt hatte. Immerfort nur\n"
        "auf rasches Kriechen bedacht, achtete er kaum darauf, daß kein Wort,\n"
        "kein Ausruf seiner Familie ihn störte. Erst als er schon in der Tür "
        "war,\n"
        "wendete er den Kopf, nicht, vollständig, denn er fühlte den Hals steif\n"
        "werden, immerhin sah er noch, daß sich hinter ihm nichts verändert\n"
        "hatte, nur die Schwester war aufgestanden. Sein letzter Blick streifte\n"
        "die Mutter, die nun völlig eingeschlafen war.\n"
        "\n"
        "Kaum war er innerhalb seines Zimmers, wurde die Tür eiligst zugedrückt,\n"
        "festgeriegelt und versperrt. Über den plötzlichen Lärm hinter sich\n"
        "erschrak Gregor so, daß ihm die Beinchen einknickten. Es war die\n"
        "Schwester, die sich so beeilt hatte. Aufrecht war sie schon da\n"
        "gestanden und hatte gewartet, leichtfüßig war sie dann\n"
        "vorwärtsgesprungen, Gregor hatte sie gar nicht kommen hören, und ein\n"
        "»Endlich!« rief sie den Eltern zu, während sie den Schlüssel im Schloß\n"
        "umdrehte.\n"
        "\n"
        "»Und jetzt?« fragte sich Gregor und sah sich im Dunkeln um. Er machte\n"
        "bald die Entdeckung, daß er sich nun überhaupt nicht mehr rühren "
        "konnte.\n"
        "Er wunderte sich darüber nicht, eher kam es ihm unnatürlich vor, daß er\n"
        "sich bis jetzt tatsächlich mit diesen dünnen Beinchen hatte fortbewegen\n"
        "können. Im übrigen fühlte er sich verhältnismäßig behaglich. Er hatte\n"
        "zwar Schmerzen im ganzen Leib, aber ihm war, als würden sie allmählich\n"
        "schwächer und schwächer und würden schließlich ganz vergehen. Den\n"
        "verfaulten Apfel in seinem Rücken und die entzündete Umgebung, die ganz\n"
        "von weichem Staub bedeckt war, spürte er schon kaum. An seine Familie\n"
        "dachte er mit Rührung und Liebe zurück. Seine Meinung darüber, daß er\n"
        "verschwinden müsse, war womöglich noch entschiedener, als die seiner\n"
        "Schwester. In diesem Zustand leeren und friedlichen Nachdenkens blieb\n"
        "er, bis die Turmuhr die dritte Morgenstunde schlug. Den Anfang des\n"
        "allgemeinen Hellerwerdens draußen vor dem Fenster erlebte er noch. Dann\n"
        "sank sein Kopf ohne seinen Willen gänzlich nieder, und aus seinen\n"
        "Nüstern strömte sein letzter Atem schwach hervor.\n"
        "\n"
        "Als am frühen Morgen die Bedienerin kam -- vor lauter Kraft und Eile\n"
        "schlug sie, wie oft man sie auch schon gebeten hatte, das zu vermeiden,\n"
        "alle Türen derartig zu, daß in der ganzen Wohnung von ihrem Kommen an\n"
        "kein ruhiger Schlaf mehr möglich war --, fand sie bei ihrem "
        "gewöhnlichen\n"
        "kurzen Besuch bei Gregor zuerst nichts Besonderes. Sie dachte, er liege\n"
        "absichtlich so unbeweglich da und spiele den Beleidigten; sie traute\n"
        "ihm allen möglichen Verstand zu. Weil sie zufällig den langen Besen in\n"
        "der Hand hielt, suchte sie mit ihm Gregor von der Tür aus zu kitzeln.\n"
        "Als sich auch da kein Erfolg zeigte, wurde sie ärgerlich und stieß ein\n"
        "wenig in Gregor hinein, und erst als sie ihn ohne jeden Widerstand von\n"
        "seinem Platze geschoben hatte, wurde sie aufmerksam. Als sie bald den\n"
        "wahren Sachverhalt erkannte, machte sie große Augen, pfiff vor sich "
        "hin,\n"
        "hielt sich aber nicht lange auf, sondern riß die Tür des Schlafzimmers\n"
        "auf und rief mit lauter Stimme in das Dunkel hinein: »Sehen Sie nur mal\n"
        "an, es ist krepiert; da liegt es, ganz und gar krepiert!«\n"
        "\n"
        "Das Ehepaar Samsa saß im Ehebett aufrecht da und hatte zu tun, den\n"
        "Schrecken über die Bedienerin zu verwinden, ehe es dazu kam, ihre\n"
        "Meldung aufzufassen. Dann aber stiegen Herr und Frau Samsa, jeder auf\n"
        "seiner Seite, eiligst aus dem Bett, Herr Samsa warf die Decke über "
        "seine\n"
        "Schultern, Frau Samsa kam nur im Nachthemd hervor; so traten sie in\n"
        "Gregors Zimmer. Inzwischen hatte sich auch die Tür des Wohnzimmers\n"
        "geöffnet, in dem Grete seit dem Einzug der Zimmerherren schlief; sie "
        "war\n"
        "völlig angezogen, als hätte sie gar nicht geschlafen, auch ihr bleiches\n"
        "Gesicht schien das zu beweisen. »Tot?« sagte Frau Samsa und sah fragend\n"
        "zur Bedienerin auf, trotzdem sie doch alles selbst prüfen und sogar "
        "ohne\n"
        "Prüfung erkennen konnte. »Das will ich meinen,« sagte die Bedienerin "
        "und\n"
        "stieß zum Beweis Gregors Leiche mit dem Besen noch ein großes Stück\n"
        "seitwärts. Frau Samsa machte eine Bewegung, als wolle sie den Besen\n"
        "zurückhalten, tat es aber nicht. »Nun,« sagte Herr Samsa, »jetzt können\n"
        "wir Gott danken.« Er bekreuzte sich, und die drei Frauen folgten seinem\n"
        "Beispiel. Grete, die kein Auge von der Leiche wendete, sagte: »Seht\n"
        "nur, wie mager er war. Er hat ja auch schon so lange Zeit nichts\n"
        "gegessen. So wie die Speisen hereinkamen, sind sie wieder\n"
        "hinausgekommen.« Tatsächlich war Gregors Körper vollständig flach und\n"
        "trocken, man erkannte das eigentlich erst jetzt, da er nicht mehr von\n"
        "den Beinchen gehoben war und auch sonst nichts den Blick ablenkte.\n"
        "\n"
        "»Komm, Grete, auf ein Weilchen zu uns herein,« sagte Frau Samsa mit\n"
        "einem wehmütigen Lächeln, und Grete ging, nicht ohne nach der Leiche\n"
        "zurückzusehen, hinter den Eltern in das Schlafzimmer. Die Bedienerin\n"
        "schloß die Tür und öffnete gänzlich das Fenster. Trotz des frühen\n"
        "Morgens war der frischen Luft schon etwas Lauigkeit beigemischt. Es war\n"
        "eben schon Ende März.\n"
        "\n"
        "Aus ihrem Zimmer traten die drei Zimmerherren und sahen sich erstaunt\n"
        "nach ihrem Frühstück um; man hatte sie vergessen. »Wo ist das\n"
        "Frühstück?« fragte der mittlere der Herren mürrisch die Bedienerin.\n"
        "Diese aber legte den Finger an den Mund und winkte dann hastig und\n"
        "schweigend den Herren zu, sie möchten in Gregors Zimmer kommen. Sie\n"
        "kamen auch und standen dann, die Hände in den Taschen ihrer etwas\n"
        "abgenützten Röckchen, in dem nun schon ganz hellen Zimmer um Gregors\n"
        "Leiche herum.\n"
        "\n"
        "Da öffnete sich die Tür des Schlafzimmers, und Herr Samsa erschien in\n"
        "seiner Livree, an einem Arm seine Frau, am anderen seine Tochter. Alle\n"
        "waren ein wenig verweint; Grete drückte bisweilen ihr Gesicht an den "
        "Arm\n"
        "des Vaters.\n"
        "\n"
        "»Verlassen Sie sofort meine Wohnung!« sagte Herr Samsa und zeigte auf\n"
        "die Tür, ohne die Frauen von sich zu lassen. »Wie meinen Sie das?« "
        "sagte\n"
        "der mittlere der Herren etwas bestürzt und lächelte süßlich. Die zwei\n"
        "anderen hielten die Hände auf dem Rücken und rieben sie ununterbrochen\n"
        "aneinander, wie in freudiger Erwartung eines großen Streites, der aber\n"
        "für sie günstig ausfallen mußte. »Ich meine es genau so, wie ich es\n"
        "sage,« antwortete Herr Samsa und ging in einer Linie mit seinen zwei\n"
        "Begleiterinnen auf den Zimmerherrn zu. Dieser stand zuerst still da und\n"
        "sah zu Boden, als ob sich die Dinge in seinem Kopf zu einer neuen\n"
        "Ordnung zusammenstellten. »Dann gehen wir also,« sagte er dann und sah\n"
        "zu Herrn Samsa auf, als verlange er in einer plötzlich ihn "
        "überkommenden\n"
        "Demut sogar für diesen Entschluß eine neue Genehmigung. Herr Samsa\n"
        "nickte ihm bloß mehrmals kurz mit großen Augen zu. Daraufhin ging der\n"
        "Herr tatsächlich sofort mit langen Schritten ins Vorzimmer; seine "
        "beiden\n"
        "Freunde hatten schon ein Weilchen lang mit ganz ruhigen Händen\n"
        "aufgehorcht und hüpften ihm jetzt geradezu nach, wie in Angst, Herr\n"
        "Samsa könnte vor ihnen ins Vorzimmer eintreten und die Verbindung mit\n"
        "ihrem Führer stören. Im Vorzimmer nahmen alle drei die Hüte vom\n"
        "Kleiderrechen, zogen ihre Stöcke aus dem Stockbehälter, verbeugten sich\n"
        "stumm und verließen die Wohnung. In einem, wie sich zeigte, gänzlich\n"
        "unbegründeten Mißtrauen trat Herr Samsa mit den zwei Frauen auf den\n"
        "Vorplatz hinaus; an das Geländer gelehnt, sahen sie zu, wie die drei\n"
        "Herren zwar langsam, aber ständig die lange Treppe hinunterstiegen, in\n"
        "jedem Stockwerk in einer bestimmten Biegung des Treppenhauses\n"
        "verschwanden und nach ein paar Augenblicken wieder hervorkamen; je\n"
        "tiefer sie gelangten, desto mehr verlor sich das Interesse der Familie\n"
        "Samsa für sie, und als ihnen entgegen und dann hoch über sie hinweg ein\n"
        "Fleischergeselle mit der Trage auf dem Kopf in stolzer Haltung\n"
        "heraufstieg, verließ bald Herr Samsa mit den Frauen das Geländer, und\n"
        "alle kehrten, wie erleichtert, in ihre Wohnung zurück.\n"
        "\n"
        "Sie beschlossen, den heutigen Tag zum Ausruhen und Spazierengehen zu\n"
        "verwenden; sie hatten diese Arbeitsunterbrechung nicht nur verdient, "
        "sie\n"
        "brauchten sie sogar unbedingt. Und so setzten sie sich zum Tisch und\n"
        "schrieben drei Entschuldigungsbriefe, Herr Samsa an seine Direktion,\n"
        "Frau Samsa an ihren Auftraggeber, und Grete an ihren Prinzipal. Während\n"
        "des Schreibens kam die Bedienerin herein, um zu sagen, daß sie "
        "fortgehe,\n"
        "denn ihre Morgenarbeit war beendet. Die drei Schreibenden nickten "
        "zuerst\n"
        "bloß, ohne aufzuschauen, erst als die Bedienerin sich immer noch nicht\n"
        "entfernen wollte, sah man ärgerlich auf. »Nun?« fragte Herr Samsa. Die\n"
        "Bedienerin stand lächelnd in der Tür, als habe sie der Familie ein\n"
        "großes Glück zu melden, werde es aber nur dann tun, wenn sie gründlich\n"
        "ausgefragt werde. Die fast aufrechte kleine Straußfeder auf ihrem Hut,\n"
        "über die sich Herr Samsa schon während ihrer ganzen Dienstzeit ärgerte,\n"
        "schwankte leicht nach allen Richtungen. »Also was wollen Sie\n"
        "eigentlich?« fragte Frau Samsa, vor welcher die Bedienerin noch am\n"
        "meisten Respekt hatte. »Ja,« antwortete die Bedienerin und konnte vor\n"
        "freundlichem Lachen nicht gleich weiter reden, »also darüber, wie das\n"
        "Zeug von nebenan weggeschafft werden soll, müssen Sie sich keine Sorge\n"
        "machen. Es ist schon in Ordnung.« Frau Samsa und Grete beugten sich zu\n"
        "ihren Briefen nieder, als wollten sie weiterschreiben; Herr Samsa,\n"
        "welcher merkte, daß die Bedienerin nun alles ausführlich zu beschreiben\n"
        "anfangen wollte, wehrte dies mit ausgestreckter Hand entschieden ab. Da\n"
        "sie aber nicht erzählen durfte, erinnerte sie sich an die große Eile,\n"
        "die sie hatte, rief offenbar beleidigt: »Adjes allseits,« drehte sich\n"
        "wild um und verließ unter fürchterlichem Türezuschlagen die Wohnung.\n"
        "\n"
        "»Abends wird sie entlassen,« sagte Herr Samsa, bekam aber weder von\n"
        "seiner Frau noch von seiner Tochter eine Antwort, denn die Bedienerin\n"
        "schien ihre kaum gewonnene Ruhe wieder gestört zu haben. Sie erhoben\n"
        "sich, gingen zum Fenster und blieben dort, sich umschlungen haltend.\n"
        "Herr Samsa drehte sich in seinem Sessel nach ihnen um und beobachtete\n"
        "sie still ein Weilchen. Dann rief er: »Also kommt doch her. Laßt schon\n"
        "endlich die alten Sachen. Und nehmt auch ein wenig Rücksicht auf mich.«\n"
        "Gleich folgten ihm die Frauen, eilten zu ihm, liebkosten ihn und\n"
        "beendeten rasch ihre Briefe.\n"
        "\n"
        "Dann verließen alle drei gemeinschaftlich die Wohnung, was sie schon\n"
        "seit Monaten nicht getan hatten, und fuhren mit der Elektrischen ins\n"
        "Freie vor die Stadt. Der Wagen, in dem sie allein saßen, war ganz von\n"
        "warmer Sonne durchschienen. Sie besprachen, bequem auf ihren Sitzen\n"
        "zurückgelehnt, die Aussichten für die Zukunft, und es fand sich, daß\n"
        "diese bei näherer Betrachtung durchaus nicht schlecht waren, denn aller\n"
        "drei Anstellungen waren, worüber sie einander eigentlich noch gar nicht\n"
        "ausgefragt hatten, überaus günstig und besonders für später\n"
        "vielversprechend. Die größte augenblickliche Besserung der Lage mußte\n"
        "sich natürlich leicht durch einen Wohnungswechsel ergeben; sie wollten\n"
        "nun eine kleinere und billigere, aber besser gelegene und überhaupt\n"
        "praktischere Wohnung nehmen, als es die jetzige, noch von Gregor\n"
        "ausgesuchte war. Während sie sich so unterhielten, fiel es Herrn und\n"
        "Frau Samsa im Anblick ihrer immer lebhafter werdenden Tochter fast\n"
        "gleichzeitig ein, wie sie in der letzten Zeit trotz aller Pflege, die\n"
        "ihre Wangen bleich gemacht hatte, zu einem schönen und üppigen Mädchen\n"
        "aufgeblüht war. Stiller werdend und fast unbewußt durch Blicke sich\n"
        "verständigend, dachten sie daran, daß es nun Zeit sein werde, auch "
        "einen\n"
        "braven Mann für sie zu suchen. Und es war ihnen wie eine Bestätigung\n"
        "ihrer neuen Träume und guten Absichten, als am Ziele ihrer Fahrt die\n"
        "Tochter als erste sich erhob und ihren jungen Körper dehnte."
    ],
    "Max und Moritz": [
        "Ach, was muß man oft von bösen\n"
        "Kindern hören oder lesen!\n"
        "Wie zum Beispiel hier von diesen,\n"
        "\n"
        "Welche Max und Moritz hießen.\n"
        "Die, anstatt durch weise Lehren\n"
        "Sich zum Guten zu bekehren,\n"
        "Oftmals noch darüber lachten\n"
        "Und sich heimlich lustig machten. --\n"
        "-- Ja, zur Übeltätigkeit,\n"
        "Ja, dazu ist man bereit! --\n"
        "-- Menschen necken, Tiere quälen,\n"
        "Äpfel, Birnen, Zwetschgen stehlen --\n"
        "Das ist freilich angenehmer\n"
        "Und dazu auch viel bequemer,\n"
        "Als in Kirche oder Schule\n"
        "Festzusitzen auf dem Stuhle. --\n"
        "-- Aber wehe, wehe, wehe!\n"
        "Wenn ich auf das Ende sehe!! --\n"
        "-- Ach, das war ein schlimmes Ding,\n"
        "Wie es Max und Moritz ging.\n"
        "-- Drum ist hier, was sie getrieben,\n"
        "Abgemalt und aufgeschrieben.\n"
        "\n"
        "Mancher gibt sich viele Müh'\n"
        "Mit dem lieben Federvieh;\n"
        "Einesteils der Eier wegen,\n"
        "Welche diese Vögel legen,\n"
        "Zweitens: Weil man dann und wann\n"
        "Einen Braten essen kann;\n"
        "Drittens aber nimmt man auch\n"
        "Ihre Federn zum Gebrauch\n"
        "In die Kissen und die Pfühle,\n"
        "Denn man liegt nicht gerne kühle. --\n"
        "\n"
        "Seht, da ist die Witwe Bolte,\n"
        "Die das auch nicht gerne wollte.\n"
        "\n"
        "Ihrer Hühner waren drei\n"
        "Und ein stolzer Hahn dabei. --\n"
        "Max und Moritz dachten nun:\n"
        "Was ist hier jetzt wohl zu tun? --\n"
        "-- Ganz geschwinde, eins, zwei, drei\n"
        "Schneiden sie sich Brot entzwei,\n"
        "In vier Teile jedes Stück\n"
        "Wie ein kleiner Finger dick.\n"
        "Diese binden sie an Fäden,\n"
        "Übers Kreuz, ein Stück an jeden,\n"
        "\n"
        "Und verlegen sie genau\n"
        "In den Hof der guten Frau. --\n"
        "\n"
        "Kaum hat dies der Hahn gesehen,\n"
        "Fängt er auch schon an zu krähen:\n"
        "Kikeriki! Kikikerikih!!\n"
        "Tak, tak, tak! -- da kommen sie.\n"
        "\n"
        "Hahn und Hühner schlucken munter\n"
        "Jedes ein Stück Brot hinunter;\n"
        "\n"
        "Aber als sie sich besinnen,\n"
        "Konnte keines recht von hinnen.\n"
        "\n"
        "In die Kreuz und in die Quer\n"
        "Reißen sie sich hin und her,\n"
        "\n"
        "Flattern auf und in die Höh',\n"
        "Ach herje, herjemineh!\n"
        "\n"
        "Ach, sie bleiben an dem langen,\n"
        "Dürren Ast des Baumes hangen. --\n"
        "-- Und ihr Hals wird lang und länger,\n"
        "Ihr Gesang wird bang und bänger.\n"
        "\n"
        "Jedes legt noch schnell ein Ei,\n"
        "Und dann kommt der Tod herbei. --\n"
        "\n"
        "Witwe Bolte in der Kammer\n"
        "Hört im Bette diesen Jammer:\n"
        "\n"
        "Ahnungsvoll tritt sie heraus,\n"
        "Ach, was war das für ein Graus!\n"
        "\n"
        "»Fließet aus dem Aug', ihr Tränen!\n"
        "All' mein Hoffen, all' mein Sehnen,\n"
        "Meines Lebens schönster Traum\n"
        "Hängt an diesem Apfelbaum!«\n"
        "\n"
        "Tiefbetrübt und sorgenschwer\n"
        "Kriegt sie jetzt das Messer her,\n"
        "Nimmt die Toten von den Strängen,\n"
        "Daß sie so nicht länger hängen,\n"
        "\n"
        "Und mit stummem Trauerblick\n"
        "Kehrt sie in ihr Haus zurück.\n"
        "\n"
        "Dieses war der erste Streich,\n"
        "Doch der zweite folgt sogleich.\n"
        "\n"
        "Als die gute Witwe Bolte\n"
        "Sich von ihrem Schmerz erholte,\n"
        "Dachte sie so hin und her,\n"
        "Daß es wohl das beste wär',\n"
        "Die Verstorb'nen, die hienieden\n"
        "Schon so frühe abgeschieden,\n"
        "Ganz im stillen und in Ehren\n"
        "Gut gebraten zu verzehren. --\n"
        "-- Freilich war die Trauer groß,\n"
        "Als sie nun so nackt und bloß\n"
        "Abgerupft am Herde lagen,\n"
        "Sie, die einst in schönen Tagen\n"
        "Bald im Hofe, bald im Garten\n"
        "Lebensfroh im Sande scharrten. --\n"
        "\n"
        "Ach, Frau Bolte weint aufs neu,\n"
        "Und der Spitz steht auch dabei.\n"
        "Max und Moritz rochen dieses;\n"
        "»Schnell aufs Dach gekrochen!« hieß es.\n"
        "\n"
        "Durch den Schornstein mit Vergnügen\n"
        "Sehen sie die Hühner liegen,\n"
        "Die schon ohne Kopf und Gurgeln\n"
        "Lieblich in der Pfanne schmurgeln. --\n"
        "\n"
        "Eben geht mit einem Teller\n"
        "Witwe Bolte in den Keller,\n"
        "\n"
        "Daß sie von dem Sauerkohle\n"
        "Eine Portion sich hole,\n"
        "Wofür sie besonders schwärmt,\n"
        "Wenn er wieder aufgewärmt. --\n"
        "-- Unterdessen auf dem Dache\n"
        "Ist man tätig bei der Sache.\n"
        "Max hat schon mit Vorbedacht\n"
        "Eine Angel mitgebracht.\n"
        "\n"
        "Schnupdiwup! da wird nach oben\n"
        "Schon ein Huhn heraufgehoben;\n"
        "Schnupdiwup! Jetzt Numro zwei;\n"
        "Schnupdiwup! Jetzt Numro drei;\n"
        "Und jetzt kommt noch Numro vier:\n"
        "Schnupdiwup! Dich haben wir! --\n"
        "-- Zwar der Spitz sah es genau,\n"
        "Und er bellt: Rawau! Rawau!\n"
        "\n"
        "Aber schon sind sie ganz munter\n"
        "Fort und von dem Dach herunter. --\n"
        "-- Na! Das wird Spektakel geben,\n"
        "Denn Frau Bolte kommt soeben; --\n"
        "-- Angewurzelt stand sie da,\n"
        "Als sie nach der Pfanne sah.\n"
        "\n"
        "Alle Hühner waren fort,\n"
        "»Spitz!« -- Das war ihr erstes Wort.\n"
        "\n"
        "»Oh, du Spitz, du Ungetüm!\n"
        "Aber wart! ich komme ihm!«\n"
        "\n"
        "Mit dem Löffel, groß und schwer,\n"
        "Geht es über Spitzen her;\n"
        "Laut ertönt sein Wehgeschrei,\n"
        "Denn er fühlt sich schuldenfrei.\n"
        "\n"
        "Max und Moritz im Verstecke\n"
        "Schnarchen aber an der Hecke,\n"
        "Und vom ganzen Hühnerschmaus\n"
        "Guckt nur noch ein Bein heraus.\n"
        "\n"
        "Dieses war der zweite Streich,\n"
        "Doch der dritte folgt sogleich.\n"
        "\n"
        "Jedermann im Dorfe kannte\n"
        "Einen, der sich Böck benannte.\n"
        "\n"
        "Alltagsröcke, Sonntagsröcke,\n"
        "Lange Hosen, spitze Fräcke,\n"
        "Westen mit bequemen Taschen,\n"
        "Warme Mäntel und Gamaschen --\n"
        "Alle diese Kleidungssachen\n"
        "Wußte Schneider Böck zu machen. --\n"
        "Oder wäre was zu flicken,\n"
        "Abzuschneiden, anzustücken,\n"
        "Oder gar ein Knopf der Hose\n"
        "Abgerissen oder lose --\n"
        "Wie und wo und wann es sei,\n"
        "Hinten, vorne, einerlei --\n"
        "Alles macht der Meister Böck,\n"
        "Denn das ist sein Lebenszweck.\n"
        "D'rum so hat in der Gemeinde\n"
        "Jedermann ihn gern zum Freunde. --\n"
        "-- Aber Max und Moritz dachten,\n"
        "Wie sie ihn verdrießlich machten.\n"
        "Nämlich vor des Meisters Hause\n"
        "Floß ein Wasser mit Gebrause.\n"
        "\n"
        "Übers Wasser führt ein Steg\n"
        "Und darüber geht der Weg.\n"
        "\n"
        "Max und Moritz, gar nicht träge,\n"
        "Sägen heimlich mit der Säge,\n"
        "Ritzeratze! voller Tücke,\n"
        "In die Brücke eine Lücke.\n"
        "Als nun diese Tat vorbei,\n"
        "Hört man plötzlich ein Geschrei:\n"
        "\n"
        "»He, heraus! du Ziegen-Böck!\n"
        "Schneider, Schneider, meck, meck, meck!« --\n"
        "-- Alles konnte Böck ertragen,\n"
        "Ohne nur ein Wort zu sagen;\n"
        "Aber, wenn er dies erfuhr,\n"
        "Ging's ihm wider die Natur.\n"
        "\n"
        "Schnelle springt er mit der Elle\n"
        "Über seines Hauses Schwelle,\n"
        "Denn schon wieder ihm zum Schreck\n"
        "Tönt ein lautes: »Meck, meck, meck!«\n"
        "\n"
        "Und schon ist er auf der Brücke,\n"
        "Kracks! Die Brücke bricht in Stücke;\n"
        "\n"
        "Wieder tönt es: »Meck, meck, meck!«\n"
        "Plumps! Da ist der Schneider weg!\n"
        "G'rad als dieses vorgekommen,\n"
        "Kommt ein Gänsepaar geschwommen,\n"
        "\n"
        "Welches Böck in Todeshast\n"
        "Krampfhaft bei den Beinen faßt.\n"
        "\n"
        "Beide Gänse in der Hand,\n"
        "Flattert er auf trocknes Land.\n"
        "\n"
        "Übrigens bei alle dem\n"
        "Ist so etwas nicht bequem!\n"
        "\n"
        "Wie denn Böck von der Geschichte\n"
        "Auch das Magendrücken kriegte.\n"
        "\n"
        "Hoch ist hier Frau Böck zu preisen!\n"
        "Denn ein heißes Bügeleisen,\n"
        "Auf den kalten Leib gebracht,\n"
        "Hat es wieder gut gemacht.\n"
        "\n"
        "-- Bald im Dorf hinauf, hinunter,\n"
        "Hieß es, Böck ist wieder munter.\n"
        "\n"
        "Dieses war der dritte Streich,\n"
        "Doch der vierte folgt sogleich.\n"
        "\n"
        "Also lautet ein Beschluß:\n"
        "Daß der Mensch was lernen muß. --\n"
        "Nicht allein das A-B-C\n"
        "Bringt den Menschen in die Höh';\n"
        "Nicht allein im Schreiben, Lesen\n"
        "Übt sich ein vernünftig Wesen;\n"
        "Nicht allein in Rechnungssachen\n"
        "Soll der Mensch sich Mühe machen;\n"
        "Sondern auch der Weisheit Lehren\n"
        "Muß man mit Vergnügen hören.\n"
        "\n"
        "Daß dies mit Verstand geschah,\n"
        "War Herr Lehrer Lämpel da. --\n"
        "-- Max und Moritz, diese beiden,\n"
        "Mochten ihn darum nicht leiden;\n"
        "Denn wer böse Streiche macht,\n"
        "Gibt nicht auf den Lehrer acht.\n"
        "Nun war dieser brave Lehrer\n"
        "Von dem Tobak ein Verehrer,\n"
        "Was man ohne alle Frage\n"
        "Nach des Tages Müh und Plage\n"
        "Einem guten, alten Mann\n"
        "Auch von Herzen gönnen kann. --\n"
        "-- Max und Moritz, unverdrossen,\n"
        "Sinnen aber schon auf Possen,\n"
        "Ob vermittelst seiner Pfeifen\n"
        "Dieser Mann nicht anzugreifen. --\n"
        "-- Einstens, als es Sonntag wieder\n"
        "Und Herr Lämpel brav und bieder\n"
        "\n"
        "In der Kirche mit Gefühle\n"
        "Saß vor seinem Orgelspiele,\n"
        "Schlichen sich die bösen Buben\n"
        "In sein Haus und seine Stuben,\n"
        "Wo die Meerschaumpfeife stand;\n"
        "Max hält sie in seiner Hand;\n"
        "\n"
        "Aber Moritz aus der Tasche\n"
        "Zieht die Flintenpulverflasche,\n"
        "Und geschwinde, stopf, stopf, stopf!\n"
        "Pulver in den Pfeifenkopf. --\n"
        "Jetzt nur still und schnell nach Haus,\n"
        "Denn schon ist die Kirche aus. --\n"
        "\n"
        "Eben schließt in sanfter Ruh'\n"
        "Lämpel seine Kirche zu;\n"
        "Und mit Buch und Notenheften,\n"
        "Nach besorgten Amtsgeschäften,\n"
        "\n"
        "Lenkt er freudig seine Schritte\n"
        "Zu der heimatlichen Hütte,\n"
        "\n"
        "Und voll Dankbarkeit sodann,\n"
        "Zündet er sein Pfeifchen an.\n"
        "\n"
        "»Ach!« -- spricht er -- »die größte Freud'\n"
        "Ist doch die Zufriedenheit!«\n"
        "\n"
        "Rums! Da geht die Pfeife los\n"
        "Mit Getöse, schrecklich groß.\n"
        "Kaffeetopf und Wasserglas,\n"
        "Tabaksdose, Tintenfaß,\n"
        "Ofen, Tisch und Sorgensitz --\n"
        "Alles fliegt in [im] Pulverblitz.\n"
        "\n"
        "Als der Dampf sich nun erhob,\n"
        "Sieht man Lämpel, der gottlob!\n"
        "Lebend auf dem Rücken liegt;\n"
        "Doch er hat was abgekriegt.\n"
        "\n"
        "Nase, Hand, Gesicht und Ohren\n"
        "Sind so schwarz als wie die Mohren,\n"
        "Und des Haares letzter Schopf\n"
        "Ist verbrannt bis auf den Kopf.\n"
        "Wer soll nun die Kinder lehren\n"
        "Und die Wissenschaft vermehren?\n"
        "Wer soll nun für Lämpel leiten\n"
        "Seine Amtestätigkeiten?\n"
        "Woraus soll der Lehrer rauchen,\n"
        "Wenn die Pfeife nicht zu brauchen?\n"
        "\n"
        "Mit der Zeit wird alles heil,\n"
        "Nur die Pfeife hat ihr Teil.\n"
        "\n"
        "Dieses war der vierte Streich,\n"
        "Doch der fünfte folgt sogleich.\n"
        "\n"
        "Wer im Dorfe oder Stadt\n"
        "Einen Onkel wohnen hat,\n"
        "Der sei höflich und bescheiden,\n"
        "Denn das mag der Onkel leiden. --\n"
        "-- Morgens sagt man: »Guten Morgen!\n"
        "Haben Sie was zu besorgen?«\n"
        "Bringt ihm, was er haben muß:\n"
        "Zeitung, Pfeife, Fidibus. --\n"
        "Oder sollt' es wo im Rücken\n"
        "Drücken, beißen oder zwicken,\n"
        "Gleich ist man mit Freudigkeit\n"
        "Dienstbeflissen und bereit. --\n"
        "Oder sei's nach einer Prise,\n"
        "Daß der Onkel heftig niese,\n"
        "Ruft man: »Prosit!« allsogleich,\n"
        "»Danke, wohl bekomm' es euch!« --\n"
        "[»Danke,« - »wohl bekomm' es euch!« --]\n"
        "Oder kommt er spät nach Haus,\n"
        "Zieht man ihm die Stiefel aus,\n"
        "Holt Pantoffel, Schlafrock, Mütze,\n"
        "Daß er nicht im Kalten sitze, --\n"
        "Kurz, man ist darauf bedacht,\n"
        "Was dem Onkel Freude macht. --\n"
        "-- Max und Moritz ihrerseits\n"
        "Fanden darin keinen Reiz. --\n"
        "-- Denkt euch nur, welch' schlechten Witz\n"
        "Machten sie mit Onkel Fritz!\n"
        "Jeder weiß, was so ein Mai-\n"
        "Käfer für ein Vogel sei.\n"
        "\n"
        "In den Bäumen hin und her\n"
        "Fliegt und kriecht und krabbelt er.\n"
        "\n"
        "Max und Moritz, immer munter,\n"
        "Schütteln sie vom Baum herunter.\n"
        "\n"
        "In die Düte von Papiere\n"
        "Sperren sie die Krabbeltiere.\n"
        "\n"
        "Fort damit und in die Ecke\n"
        "Unter Onkel Fritzens Decke!\n"
        "\n"
        "Bald zu Bett geht Onkel Fritze\n"
        "In der spitzen Zippelmütze;\n"
        "Seine Augen macht er zu,\n"
        "Hüllt sich ein und schläft in Ruh.\n"
        "\n"
        "Doch die Käfer, kritze, kratze!\n"
        "Kommen schnell aus der Matratze.\n"
        "\n"
        "Schon faßt einer, der voran,\n"
        "Onkel Fritzens Nase an.\n"
        "\n"
        "»Bau!« schreit er -- »Was ist das hier?«\n"
        "Und erfaßt das Ungetier.\n"
        "\n"
        "Und den Onkel voller Grausen\n"
        "Sieht man aus dem Bette sausen.\n"
        "\n"
        "»Autsch!« -- Schon wieder hat er einen\n"
        "Im Genicke, an den Beinen;\n"
        "\n"
        "Hin und her und rund herum\n"
        "Kriecht es, fliegt es mit Gebrumm.\n"
        "\n"
        "Onkel Fritz, in dieser Not,\n"
        "Haut und trampelt alles tot.\n"
        "\n"
        "Guckste wohl! Jetzt ist's vorbei\n"
        "Mit der Käferkrabbelei!\n"
        "\n"
        "Onkel Fritz hat wieder Ruh'\n"
        "Und macht seine Augen zu.\n"
        "\n"
        "Dieses war der fünfte Streich,\n"
        "Doch der sechste folgt sogleich.\n"
        "\n"
        "In der schönen Osterzeit,\n"
        "Wenn die frommen Bäckersleut'\n"
        "Viele süße Zuckersachen\n"
        "Backen und zurechte machen,\n"
        "Wünschten Max und Moritz auch\n"
        "Sich so etwas zum Gebrauch.\n"
        "\n"
        "Doch der Bäcker, mit Bedacht,\n"
        "Hat das Backhaus zugemacht.\n"
        "\n"
        "Also will hier einer stehlen,\n"
        "Muß er durch den Schlot sich quälen.\n"
        "\n"
        "Ratsch! Da kommen die zwei Knaben\n"
        "Durch den Schornstein, schwarz wie Raben.\n"
        "\n"
        "Puff! Sie fallen in die Kist',\n"
        "Wo das Mehl darinnen ist.\n"
        "\n"
        "Da! Nun sind sie alle beide,\n"
        "Rund herum so weiß wie Kreide.\n"
        "\n"
        "Aber schon mit viel Vergnügen\n"
        "Sehen sie die Brezeln liegen.\n"
        "\n"
        "Knacks! -- Da bricht der Stuhl entzwei;\n"
        "\n"
        "Schwapp! -- Da liegen sie im Brei.\n"
        "\n"
        "Ganz von Kuchenteig umhüllt,\n"
        "Steh'n sie da als Jammerbild. --\n"
        "\n"
        "Gleich erscheint der Meister Bäcker\n"
        "Und bemerkt die Zuckerlecker.\n"
        "\n"
        "Eins, zwei, drei! -- eh' man's gedacht,\n"
        "Sind zwei Brote d'raus gemacht.\n"
        "\n"
        "In dem Ofen glüht es noch --\n"
        "Ruff! -- damit ins Ofenloch!\n"
        "\n"
        "Ruff! man zieht sie aus der Glut;\n"
        "Denn nun sind sie braun und gut. --\n"
        "\n"
        "Jeder denkt, die sind perdü!\n"
        "Aber nein -- noch leben sie.\n"
        "\n"
        "Knusper, Knasper! -- wie zwei Mäuse\n"
        "Fressen sie durch das Gehäuse;\n"
        "\n"
        "Und der Meister Bäcker schrie:\n"
        "»Ach herrjeh! da laufen sie!«\n"
        "\n"
        "Dieses war der sechste Streich,\n"
        "Doch der letzte folgt sogleich.\n"
        "\n"
        "Max und Moritz, wehe euch!\n"
        "Jetzt kommt euer letzter Streich!\n"
        "\n"
        "Wozu müssen auch die beiden\n"
        "Löcher in die Säcke schneiden?\n"
        "\n"
        "Seht, da trägt der Bauer Mecke\n"
        "Einen seiner Maltersäcke.\n"
        "\n"
        "Aber kaum, daß er von hinnen,\n"
        "Fängt das Korn schon an zu rinnen.\n"
        "\n"
        "Und verwundert steht und spricht er:\n"
        "»Zapperment! dat Ding werd lichter!«\n"
        "\n"
        "Hei! Da sieht er voller Freude\n"
        "Max und Moritz im Getreide.\n"
        "\n"
        "Rabs! -- in seinen großen Sack\n"
        "Schaufelt er das Lumpenpack.\n"
        "\n"
        "Max und Moritz wird es schwüle,\n"
        "Denn nun geht es nach der Mühle. --\n"
        "\n"
        "»Meister Müller, he, heran!\n"
        "Mahl er das, so schnell er kann!«\n"
        "\n"
        "»Her damit!« Und in den Trichter\n"
        "Schüttelt er die Bösewichter. --\n"
        "\n"
        "Rickeracke! Rickeracke!\n"
        "Geht die Mühle mit Geknacke.\n"
        "\n"
        "Hier kann man sie noch erblicken\n"
        "Fein geschroten und in Stücken.\n"
        "\n"
        "Doch sogleich verzehret sie\n"
        "\n"
        "Meister Müllers Federvieh.\n"
        "\n"
        "Als man dies im Dorf erfuhr,\n"
        "War von Trauer keine Spur. --\n"
        "-- Witwe Bolte, mild und weich,\n"
        "Sprach: »Sieh' da, ich dacht' es gleich!« --\n"
        "-- »Ja, ja, ja!« rief Meister Böck,\n"
        "»Bosheit ist kein Lebenszweck!«\n"
        "-- Drauf so sprach Herr Lehrer Lämpel:\n"
        "»Dies ist wieder ein Exempel!« --\n"
        "-- »Freilich!« meint der Zuckerbäcker,\n"
        "»Warum ist der Mensch so lecker!« --\n"
        "-- Selbst der gute Onkel Fritze\n"
        "Sprach: »Das kommt von dumme Witze!« --\n"
        "-- Doch der brave Bauersmann\n"
        "Dachte: »Wat geiht meck dat an!« --\n"
        "-- Kurz im ganzen Ort herum\n"
        "Ging ein freudiges Gebrumm:\n"
        "»Gott sei Dank! Nun ist's vorbei\n"
        "Mit der Übeltäterei!!«"
    ],
    "Hänsel und Gretel": [
        "Vor einem großen Walde wohnte ein armer Holzhacker mit seiner Frau und "
        "seinen zwei Kindern; das Bübchen hieß Hänsel und das Mädchen Gretel. "
        "Er hatte wenig zu beißen und zu brechen, und einmal, als große "
        "Teuerung ins Land kam, konnte er auch das täglich Brot nicht mehr "
        "schaffen. Wie er sich nun Abends im Bett Gedanken machte und sich vor "
        "Sorgen herum wälzte, seufzte er und sprach zu seiner Frau 'was soll "
        "aus uns werden? wie können wir unsere armen Kinder ernähren, da wir "
        "für uns selbst nichts mehr haben?' 'Weißt du was, Mann,' antwortete "
        "die Frau, 'wir wollen Morgen in aller Frühe die Kinder hinaus in den "
        "Wald führen, wo er am dicksten ist: da machen wir ihnen ein Feuer an "
        "und geben jedem noch ein Stückchen Brot, dann gehen wir an unsere "
        "Arbeit und lassen sie allein. Sie finden den Weg nicht wieder nach "
        "Haus und wir sind sie los.' 'Nein, Frau,' sagte der Mann, 'das thue "
        "ich nicht; wie sollt ichs übers Herz bringen meine Kinder im Walde "
        "allein zu lassen, die wilden Thiere würden bald kommen und sie "
        "zerreißen.' 'O du Narr,' sagte sie, 'dann müssen wir alle viere "
        "Hungers sterben, du kannst nur die Bretter für die Särge hobelen,' und "
        "ließ ihm keine Ruhe bis er einwilligte. 'Aber die armen Kinder dauern "
        "mich doch' sagte der Mann.\n"
        "Die zwei Kinder hatten vor Hunger auch nicht einschlafen können und "
        "hatten gehört was die Stiefmutter zum Vater gesagt hatte. Grethel "
        "weinte bittere Thränen und sprach zu Hansel 'nun ists um uns "
        "geschehen.' 'Still, Grethel,' sprach Hänsel, 'gräme dich nicht, ich "
        "will uns schon helfen.' Und als die Alten eingeschlafen waren, stand "
        "er auf, zog sein Röcklein an, machte die Unterthüre auf und schlich "
        "sich hinaus. Da schien der Mond ganz helle, und die weißen "
        "Kieselsteine, die vor dem Haus lagen, glänzten wie lauter Batzen. "
        "Hänsel bückte sich und steckte so viel in sein Rocktäschlein, als nur "
        "hinein wollten. Dann gieng er wieder zurück, sprach zu Grethel 'sei "
        "getrost, liebes Schwesterchen und schlaf nur ruhig ein, Gott wird uns "
        "nicht verlassen,' und legte sich wieder in sein Bett.\n"
        "Als der Tag anbrach, noch ehe die Sonne aufgegangen war, kam schon die "
        "Frau und weckte die beiden Kinder, 'steht auf, ihr Faullenzer, wir "
        "wollen in den Wald gehen und Holz holen.' Dann gab sie jedem ein "
        "Stückchen Brot und sprach 'da habt ihr etwas für den Mittag, aber eßts "
        "nicht vorher auf, weiter kriegt ihr nichts.' Grethel nahm das Brot "
        "unter die Schürze, weil Hänsel die Steine in der Tasche hatte. Danach "
        "machten sie sich alle zusammen auf den Weg nach dem Wald. Als sie ein "
        "Weilchen gegangen waren, stand Hänsel still und guckte nach dem Haus "
        "zurück und that das wieder und immer wieder. Der Vater sprach 'Hänsel, "
        "was guckst du da und bleibst zurück, hab Acht und vergiß deine Beine "
        "nicht.' 'Ach, Vater,' sagte Hänsel, 'ich sehe nach meinem weißen "
        "Kätzchen, das sitzt oben auf dem Dach und will mir Ade sagen.' Die "
        "Frau sprach 'Narr, das ist dein Kätzchen nicht, das ist die "
        "Morgensonne, die auf den Schornstein scheint.' Hänsel aber hatte nicht "
        "nach dem Kätzchen gesehen, sondern immer einen von den blanken "
        "Kieselsteinen aus seiner Tasche auf den Weg geworfen.\n"
        "Als sie mitten in den Wald gekommen waren, sprach der Vater 'nun "
        "sammelt Holz, ihr Kinder, ich will ein Feuer anmachen, damit ihr nicht "
        "friert. Hänsel und Grethel trugen Reisig zusammen, einen kleinen Berg "
        "hoch. Das Reisig ward angezündet, und als die Flamme recht hoch "
        "brannte, sagte die Frau 'nun legt euch ans Feuer, ihr Kinder und ruht "
        "euch aus, wir gehen in den Wald und hauen Holz. Wenn wir fertig sind, "
        "kommen wir wieder und holen euch ab.' Hänsel und Grethel saßen am "
        "Feuer, und als der Mittag kam, aß jedes sein Stücklein Brot. Und weil "
        "sie die Schläge der Holzaxt, hörten, so glaubten sie ihr Vater wäre in "
        "der Nähe. Es war aber nicht die Holzaxt, es war ein Ast, den er an "
        "einen dürren Baum gebunden hatte und den der Wind hin und her schlug. "
        "Und als sie so lange gesessen hatten, fielen ihnen die Augen vor "
        "Müdigkeit zu, und sie schliefen fest ein. Als sie endlich erwachten, "
        "war es schon finstere Nacht. Grethel fieng an zu weinen und sprach "
        "'wie sollen wir nun aus dem Wald kommen!' Hänsel aber tröstete sie, "
        "'wart nur ein Weilchen, bis der Mond aufgegangen ist, dann wollen wir "
        "den Weg schon finden.' Und als der volle Mond aufgestiegen war, so "
        "nahm Hänsel sein Schwesterchen an der Hand und gieng den Kieselsteinen "
        "nach, die schimmerten wie neu geschlagene Batzen und zeigten ihnen den "
        "Weg. Sie giengen die ganze Nacht hindurch und kamen bei anbrechendem "
        "Tag wieder zu ihres Vaters Haus. Sie klopften an die Thür, und als die "
        "Frau aufmachte und sah daß es Hänsel und Grethel war, sprach sie 'ihr "
        "bösen Kinder, was habt ihr so lange im Walde geschlafen, wir haben "
        "geglaubt ihr wolltet gar nicht wieder kommen.' Der Vater aber freute "
        "sich, denn es war ihm zu Herzen gegangen daß er sie so allein zurück "
        "gelassen hatte.\n"
        "Nicht lange danach war wieder Noth in allen Ecken, und die Kinder "
        "hörten wie die Mutter Nachts im Bette zu dem Vater sprach 'alles ist "
        "wieder aufgezehrt, wir haben noch einen halben Laib Brot, hernach hat "
        "das Lied ein Ende. Die Kinder müssen fort, wir wollen sie tiefer in "
        "den Wald hineinführen, damit sie den Weg nicht wieder heraus finden; "
        "es ist sonst keine Rettung für uns.' Dem Mann fiels schwer aufs Herz "
        "und er dachte 'es wäre besser, daß du den letzten Bissen mit deinen "
        "Kindern theiltest.' Aber die Frau hörte auf nichts, was er sagte, "
        "schalt ihn und machte ihm Vorwürfe. Wer A sagt muß auch B sagen, und "
        "weil er das erste Mal nachgegeben hatte, so mußte er es auch zum "
        "zweiten Mal.\n"
        "Die Kinder waren aber noch wach gewesen und hatten das Gespräch mit "
        "angehört. Als die Alten schliefen, stand Hänsel wieder auf, wollte "
        "hinaus und Kieselsteine auflesen, wie das vorigemal, aber die Frau "
        "hatte die Thür verschlossen, und Hänsel konnte nicht heraus. Aber er "
        "tröstete sein Schwesterchen und sprach 'weine nicht, Grethel, und "
        "schlaf nur ruhig, der liebe Gott wird uns schon helfen.' Am frühen "
        "Morgen kam die Frau und holte die Kinder aus dem Bette. Sie erhielten "
        "ihr Stückchen Brot, das war aber noch kleiner als das vorigemal. Auf "
        "dem Wege nach dem Wald bröckelte es Hänsel in der Tasche, stand oft "
        "still und warf ein Bröcklein auf die Erde. 'Hänsel, was stehst du und "
        "guckst dich um,' sagte der Vater, 'geh deiner Wege.' 'Ich sehe nach "
        "meinem Täubchen, das sitzt auf dem Dache und will mir Ade sagen,' "
        "antwortete Hänsel. 'Narr,' sagte die Frau, 'das ist dein Täubchen "
        "nicht, das ist die Morgensonne, die auf den Schornstein oben scheint.' "
        "Hänsel aber warf nach und nach alle Bröcklein auf den Weg.\n"
        "Die Frau führte die Kinder noch tiefer in den Wald, wo sie ihr Lebtag "
        "noch nicht gewesen waren. Da ward wieder ein großes Feuer angemacht, "
        "und die Mutter sagte 'bleibt nur da sitzen, ihr Kinder, und wenn ihr "
        "müde seid, könnt ihr ein wenig schlafen: wir gehen in den Wald und "
        "hauen Holz, und Abends, wenn wir fertig sind, kommen wir und holen "
        "euch ab.' Als es Mittag war, theilte Grethel ihr Brot mit Hänsel, der "
        "sein Stück auf den Weg gestreut hatte. Dann schliefen sie ein, und der "
        "Abend vergieng, aber niemand kam zu den armen Kindern. Sie erwachten "
        "erst in der finstern Nacht, und Hänsel tröstete sein Schwesterchen und "
        "sagte, 'wart nur, Grethel, bis der Mond aufgeht, dann werden wir die "
        "Brotbröcklein sehen, die ich ausgestreut habe, die zeigen uns den Weg "
        "nach Haus.' Als der Mond kam, machten sie sich auf, aber sie fanden "
        "kein Bröcklein mehr, denn die viel tausend Vögel, die im Walde und im "
        "Felde umher fliegen, die hatten sie weggepickt. Hänsel sagte zu "
        "Grethel 'wir werden den Weg schon finden,' aber sie fanden ihn nicht. "
        "Sie giengen die ganze Nacht und noch einen Tag von Morgen bis Abend, "
        "aber sie kamen aus dem Wald nicht heraus, und waren so hungrig, denn "
        "sie hatten nichts als die paar Beeren, die auf der Erde standen. Und "
        "weil sie so müde waren daß die Beine sie nicht mehr tragen wollten, so "
        "legten sie sich unter einen Baum und schliefen ein.\n"
        "Nun wars schon der dritte Morgen, daß sie ihres Vaters Haus verlassen "
        "hatten. Sie fiengen wieder an zu gehen, aber sie geriethen immer "
        "tiefer in den Wald und wenn nicht bald Hilfe kam, so mußten sie "
        "verschmachten. Als es Mittag war, sahen sie ein schönes schneeweißes "
        "Vöglein auf einem Ast sitzen, das sang so schön, daß sie stehen "
        "blieben und ihm zuhörten. Und als es fertig war, schwang es seine "
        "Flügel und flog vor ihnen her, und sie giengen ihm nach, bis sie zu "
        "einem Häuschen gelangten, auf dessen Dach es sich setzte, und als sie "
        "ganz nah heran kamen, so sahen sie daß das Häuslein aus Brot gebaut "
        "war, und mit Kuchen gedeckt; aber die Fenster waren von hellem Zucker. "
        "'Da wollen wir uns dran machen,' sprach Hänsel, 'und eine gesegnete "
        "Mahlzeit halten. Ich will ein Stück vom Dach essen, Grethel, du kannst "
        "vom Fenster essen, das schmeckt süß.' Hansel reichte in die Höhe und "
        "brach sich ein wenig vom Dach ab, um zu versuchen wie es schmeckte, "
        "und Grethel stellte sich an die Scheiben und knuperte daran. Da rief "
        "eine feine Stimme aus der Stube heraus\n"
        "\n"
        "'knuper, knuper, kneischen,\n"
        "wer knupert an meinem Häuschen?'\n"
        "\n"
        "die Kinder antworteten\n"
        "\n"
        "'der Wind, der Wind,\n"
        "das himmlische Kind,'\n"
        "\n"
        "und aßen weiter, ohne sich irre machen zu lassen. Hänsel, dem das Dach "
        "sehr gut schmeckte, riß sich ein großes Stück davon herunter, und "
        "Grethel stieß eine ganze runde Fensterscheibe heraus, setzte sich "
        "nieder, und that sich wohl damit. Da gieng auf einmal die Thüre aus, "
        "und eine steinalte Frau, die sich auf eine Krücke stützte, kam heraus "
        "geschlichen. Hänsel und Grethel erschraken so gewaltig, daß sie fallen "
        "ließen was sie in den Händen hielten. Die Alte aber wackelte mit dem "
        "Kopfe und sprach 'ei, ihr lieben Kinder, wer hat euch hierher "
        "gebracht? kommt nur herein und bleibt bei mir, es geschieht euch kein "
        "Leid.' Sie faßte beide an der Hand und führte sie in ihr Häuschen. Da "
        "ward gutes Essen aufgetragen, Milch und Pfannekuchen mit Zucker, Äpfel "
        "und Nüsse. Hernach wurden zwei schöne Bettlein weiß gedeckt, und "
        "Hänsel und Grethel legten sich hinein und meinten sie wären im "
        "Himmel.\n"
        "Die Alte hatte sich nur so freundlich angestellt, sie war aber eine "
        "böse Hexe, die den Kindern auflauerte, und hatte das Brothäuslein bloß "
        "gebaut, um sie herbeizulocken. Wenn eins in ihre Gewalt kam, so machte "
        "sie es todt, kochte es und aß es, und das war ihr ein Festtag. Die "
        "Hexen haben rothe Augen und können nicht weit sehen, aber sie haben "
        "eine feine Witterung, wie die Thiere, und merkens wenn Menschen heran "
        "kommen. Als Hänsel und Grethel in ihre Nähe kamen, da lachte sie "
        "boshaft und sprach höhnisch 'die habe ich, die sollen mir nicht wieder "
        "entwischen.' Früh Morgens, ehe die Kinder erwacht waren, stand sie "
        "schon auf, und als sie beide so lieblich ruhen sah, mit den vollen "
        "rothen Backen, so murmelte sie vor sich hin 'das wird ein guter Bissen "
        "werden.' Da packte sie Hänsel mit ihrer dürren Hand und trug ihn in "
        "einen kleinen Stall und sperrte ihn mit einer Gitterthüre ein; er "
        "mochte schreien wie er wollte, es half ihm nichts. Dann gieng sie zur "
        "Grethel, rüttelte sie wach und rief 'steh auf, Faullenzerin, trag "
        "Wasser und koch deinem Bruder etwas gutes, der sitzt draußen im Stall "
        "und soll fett werden. Wenn er fett ist, so will ich ihn essen.' "
        "Grethel fieng an bitterlich zu weinen, aber es war alles vergeblich, "
        "sie mußte thun was die böse Hexe verlangte.\n"
        "Nun ward dem armen Hänsel das beste Essen gekocht, aber Grethel bekam "
        "nichts als Krebsschalen. Jeden Morgen schlich die Alte zu dem "
        "Ställchen und rief 'Hänsel, streck deine Finger heraus, damit ich "
        "fühle ob du bald fett bist.' Hänsel streckte ihr aber ein Knöchlein "
        "heraus, und die Alte, die trübe Augen hatte, konnte es nicht sehen, "
        "und meinte es wären Hänsels Finger, und verwunderte sich daß er gar "
        "nicht fett werden wollte. Als vier Wochen herum waren und Hänsel immer "
        "mager blieb, da übernahm sie die Ungeduld, und sie wollte nicht länger "
        "warten. 'Heda, Grethel,' rief sie dem Mädchen zu, 'sei flink und trag "
        "Wasser: Hänsel mag fett oder mager sein, morgen will ich ihn "
        "schlachten und kochen.' Ach, wie jammerte das arme Schwesterchen, als "
        "es das Wasser tragen mußte, und wie flossen ihm die Tränen über die "
        "Backen herunter! 'Lieber Gott, hilf uns doch,' rief sie aus, 'hätten "
        "uns nur die wilden Thiere im Wald gefressen, so wären wir doch "
        "zusammen gestorben.' 'Spar nur dein Geblärre,' sagte die Alte, 'es "
        "hilft dir alles nichts.'\n"
        "Früh Morgens mußte Grethel heraus, den Kessel mit Wasser aufhängen und "
        "Feuer anzünden. 'Erst wollen wir backen' sagte die Alte, 'ich habe den "
        "Backofen schon eingeheizt und den Teig geknätet.' Sie stieß das arme "
        "Grethel hinaus zu dem Backofen, aus dem die Feuerflammen schon heraus "
        "schlugen. 'Kriech hinein,' sagte die Hexe, 'und sieh zu ob recht "
        "eingeheizt ist, damit wir das Brot hineinschießen können.' Und wenn "
        "Grethel darin war, wollte sie den Ofen zumachen, und Grethel sollte "
        "darin braten, und dann wollte sies auch aufessen. Aber Grethel merkte "
        "was sie im Sinn hatte und sprach 'ich weiß nicht wie ichs machen soll; "
        "wie komm ich da hinein?' 'Dumme Gans,' sagte die Alte, 'die Öffnung "
        "ist groß genug, siehst du wohl, ich könnte selbst hinein,' trappelte "
        "heran und steckte den Kopf in den Backofen. Da gab ihr Grethel einen "
        "Stoß daß sie weit hinein fuhr, machte die eiserne Thür zu und schob "
        "den Riegel vor. Hu! da fieng sie an zu heulen, ganz grauselich; aber "
        "Grethel lief fort, und die gottlose Hexe mußte elendiglich "
        "verbrennen.\n"
        "Grethel aber lief schnurstracks zum Hänsel, öffnete sein Ställchen und "
        "rief 'Hänsel, wir sind erlöst, die alte Hexe ist todt.' Da sprang "
        "Hänsel heraus, wie ein Vogel aus dem Käfig, wenn ihm die Thüre "
        "aufgemacht wird. Wie haben sie sich gefreut, sind sich um den Hals "
        "gefallen, sind herumgesprungen und haben sich geküßt! Und weil sie "
        "sich nicht mehr zu fürchten brauchten, so giengen sie in das Haus der "
        "Hexe hinein, da standen in allen Ecken Kasten mit Perlen und "
        "Edelsteinen. 'Die sind noch besser als Kieselsteine' sagte Hänsel und "
        "steckte in seine Taschen was hinein wollte, und Grethel sagte 'ich "
        "will auch etwas mit nach Haus bringen' und füllte sich sein Schürzchen "
        "voll. 'Aber jetzt wollen wir fort,' sagte Hänsel, 'damit wir aus dem "
        "Hexenwald herauskommen.' Als sie aber ein paar Stunden gegangen waren, "
        "gelangten sie an ein großes Wasser. Wir können nicht hinüber,' sprach "
        "Hänsel, 'ich sehe keinen Steg und keine Brücke.' 'Hier fährt auch kein "
        "Schiffchen,' antwortete Grethel, 'aber da schwimmt eine weiße Ente, "
        "wenn ich die bitte, so hilft sie uns hinüber.' Da rief sie\n"
        "\n"
        "'Entchen, Entchen,\n"
        "da steht Gretel und Hänsel.\n"
        "Kein Steg und keine Brücke,\n"
        "nimm uns auf deinen weißen Rücken.'\n"
        "\n"
        "Das Entchen kam auch heran, und Hänsel setzte sich auf und bat sein "
        "Schwesterchen sich zu ihm zu setzen. Nein,' antwortete Grethel, 'es "
        "wird dem Entchen zu schwer, es soll uns nach einander hinüber "
        "bringen.' Das that das gute Thierchen, und als sie glücklich drüben "
        "waren und ein Weilchen fortgiengen, da kam ihnen der Wald immer "
        "bekannter und immer bekannter vor, und endlich erblickten sie von "
        "weitem ihres Vaters Haus. Da fiengen sie an zu laufen, stürzten in die "
        "Stube hinein und fielen ihrem Vater um den Hals. Der Mann hatte keine "
        "frohe Stunde gehabt, seitdem er die Kinder im Walde gelassen hatte, "
        "die Frau aber war gestorben. Grethel schüttete sein Schürzchen aus daß "
        "die Perlen und Edelsteine in der Stube herumsprangen, und Hänsel warf "
        "eine Handvoll nach der andern aus seiner Tasche dazu. Da hatten alle "
        "Sorgen ein Ende, und sie lebten in lauter Freude zusammen. Mein "
        "Märchen ist aus, dort lauft eine Maus, wer sie fängt, darf sich eine "
        "große große Pelzkappe daraus machen."
    ],
    "CAU-Leitbild": [
        "An der Christian-Albrechts-Universität zu Kiel (CAU) bilden Studium und "
        "Lehre eine universitäre Kernaufgabe. Das Lehren und Lernen sehen wir als "
        "eine unserer wesentlichen Leistungsdimensionen an. Als zukunftsorientierte "
        "Volluniversität orientieren wir uns an den aktuellen und zukünftigen "
        "Anforderungen von Wissenschaft und Gesellschaft und berücksichtigen die "
        "Freiheit der Lehre. Wir unterstützen innovative Lehr- und Lern-Angebote für "
        "alle Personen an der CAU. Dabei stehen wir für eine Förderung von "
        "Weltoffenheit, internationaler Verständigung und Frieden ein – entsprechend "
        "dem Motto der CAU „Pax Optima Rerum“. Durch einen gesamtuniversitären "
        "Ansatz ist die CAU ein ganzheitlicher Lernort für zukunftsfähiges Handeln. "
        "So leisten wir aktive Beiträge zur Gestaltung einer sozial, ökologisch und "
        "ökonomisch nachhaltigen Gesellschaft und befördern Bildung für nachhaltige "
        "Entwicklung. Wir greifen in der Lehre den aktuellen Forschungsstand in den "
        "jeweiligen Fachwissenschaften auf und bieten ein vielseitiges Lehr- und "
        "Lernumfeld, das geeignete Praxisbezüge einschließt. Aktuelle didaktische, "
        "methodische und technische Möglichkeiten nutzen wir, um vielfältige Zugänge "
        "zu Lehr-, Lern- und Bildungsressourcen für alle Lehrenden und Lernenden der "
        "CAU zu schaffen. Wir vermitteln Wissen und prägen insbesondere eine Kultur "
        "des guten Zuhörens, womit wir die Weiterentwicklung von Werten, "
        "Einstellungen und Technologien konstruktiv-kritisch verfolgen und zeitnah "
        "in die Lehr- und Lernprozesse einfließen lassen. Um dies zu ermöglichen, "
        "sehen wir Lehre und Forschung als sich gegenseitig unterstützende und "
        "ineinander greifende Systeme an, deren gleichwertige Anerkennung "
        "unabdingbar ist. Wir setzen unsere Richtlinien zur Sicherung guter "
        "wissenschaftlicher Praxis konsequent um. Hierzu halten wir Strukturen zur "
        "Unterstützung und Sicherstellung wissenschaftlicher Integrität in der Lehre "
        "und im Lernen vor, insbesondere hinsichtlich des verantwortungsvollen "
        "Umgangs mit Forschungsdaten. Zur Förderung der Eigenständigkeit und der "
        "wissenschaftlichen Qualifikation unserer Studierenden bilden die "
        "forschungsorientierte Lehre und das forschungsbasierte Lernen Schwerpunkte "
        "an der CAU. Für die Bereiche Lehren und Lernen definieren wir "
        "Digitalisierung, Diversität, Gleichstellung, Internationalisierung, "
        "Nachhaltigkeit sowie Persönlichkeits- und Kompetenzentwicklung als "
        "Querschnittsthemen und -ziele. In die Lehre und in das Lernen beziehen wir "
        "diese kontinuierlich und umfassend ein. Dies betrifft alle universitären "
        "Ebenen und damit Studierende, Lehrende, Forschende und weitere "
        "Mitarbeitende, die an der CAU oder angegliederten Einrichtungen beschäftigt "
        "sind.\n"
        "Wir fördern an der CAU die statusgruppenübergreifende Interaktion und "
        "Partizipation in der Lehre und im Lernen. Insbesondere begreifen wir "
        "studentisches Engagement in der Lehre als Chance und große Bereicherung und "
        "würdigen dieses angemessen. Durch niedrigschwellige Beteiligungs- und "
        "Mitgestaltungsmöglichkeiten entwickeln wir bestehende Strukturen "
        "kontinuierlich weiter. Dadurch gelingt es uns im Sinne einer effektiven "
        "Selbstregulation und auf der Grundlage einer akademischen Selbstverwaltung, "
        "den aktuellen und zukünftigen wissenschaftlichen und gesellschaftlichen "
        "Herausforderungen aktiv zu begegnen. So verstehen wir Lehrveranstaltungen "
        "an der CAU als dynamische Prozesse, die im Rahmen der Qualitätsentwicklung "
        "von steter Evaluation profitieren und auf die jeweiligen Beteiligten "
        "ausgerichtet sind. Wir stärken und wertschätzen Engagement und Reflexion "
        "und gehen auf konkrete Bedarfe ein. Eine annehmende und achtende "
        "Fehlerkultur mit einer selbstreflektierenden Haltung sowie eine etablierte "
        "Feedbackkultur mit adäquaten Möglichkeiten der Konflikt- und Problemlösung "
        "sehen wir als unerlässlichen Bestandteil des Lehrens und Lernens an unserer "
        "Universität an. Lehrende stellen aufgrund ihrer Vorbildfunktion sicher, "
        "dass in Lehrveranstaltungen vorhandene kontroverse Positionen und Theorien "
        "in Forschungsfeldern gegenübergestellt und konstruktiv diskutiert werden. "
        "Wir schaffen an der CAU gute Rahmenbedingungen für die universitäre "
        "Bildung, die unter anderem Kontinuität, Sicherheit und Verlässlichkeit "
        "sowie ein unterstützendes und wertschätzendes Umfeld beinhalten. Unser Ziel "
        "ist es, inklusive und diversitätssensible Bedingungen für Studium und Lehre "
        "entsprechend dem Leitbild Diversität bereitzustellen und somit den "
        "bewussten Umgang mit Diversität als selbstverständlichen Teil des Lehrens "
        "und Lernens zu etablieren. Die CAU toleriert keinerlei Diskriminierung oder "
        "Ausgrenzung.\n"
        "Wir fördern das lebenslange Lernen und die Kompetenzentwicklung aller "
        "Statusgruppen. Für uns ist es wichtig, ein pluralistisches, partizipatives "
        "Lehr-Lern-Umfeld zu schaffen. Damit ermöglichen wir es allen Mitgliedern "
        "der CAU, ihre Fähigkeiten und ihre Persönlichkeit individuell einzubringen "
        "und weiterzuentwickeln. Wir unterstützen dies durch einen inter- und "
        "transdisziplinären Ansatz sowie durch ein bedarfsgerechtes Angebot "
        "überfachlicher Lehr- und Weiterbildungsangebote. Die CAU als internationale "
        "Netzwerkuniversität ist bestrebt, in der Lehre und im Lernen ein Arbeiten "
        "im internationalen Kontext zu ermöglichen, um die weltweite "
        "wissenschaftliche Vernetzung und Mobilität zu unterstützen. Unsere Lehre "
        "bezieht internationale Forschungsergebnisse ein, die uns einen umfassenden "
        "Zugang zu wissenschaftlichen Perspektiven eröffnen und Lernende "
        "ganzheitlich auf einen internationalen Arbeitsalltag in heterogenen Teams "
        "vorbereiten.\n"
        "Die CAU als digital gestützte Präsenzuniversität fördert eine reflektierte "
        "und evaluierte Mischung aus verschiedenen Lehr- und Lernformen und schließt "
        "hybride und digitale Lehr- und Lernmöglichkeiten ausdrücklich mit ein. "
        "Durch Schaffung digitaler Lehr-/Lernräume und -umgebungen sowie die "
        "Entwicklung und Anwendung damit verbundener Methoden, Kompetenzen und "
        "Ressourcen gestalten wir die digitale Transformation aktiv mit. Wir binden "
        "die unterschiedlichen Aspekte des Lehrens und Lernens an der CAU "
        "gleichwertig in die gemeinsame Arbeit ein, um sowohl individuelle "
        "Entwicklung als auch gegenseitiges Verständnis und Akzeptanz zu "
        "unterstützen."
    ],
}

TEXTE = {name: "\n\n".join(teile) for name, teile in TEXTTEILE.items()}

GRIMM_TITEL = [
    "Der Froschkönig oder der eiserne Heinrich",
    "Der Wolf und die sieben jungen Geisslein",
    "Die Bremer Stadtmusikanten",
    "Das tapfere Schneiderlein",
    "Daumesdick",
    "Rapunzel",
    "Brüderchen und Schwesterchen",
    "Aschenputtel",
    "Hänsel und Gretel",
    "Dornröschen",
    "Sneewittchen",
    "Der Fuchs und das Pferd",
    "Frau Holle",
    "König Drosselbart",
    "Sechse kommen durch die ganze Welt",
    "Jorinde und Joringel",
    "Rotkäppchen",
    "Rumpelstilzchen",
    "Schneeweißchen und Rosenrot",
    "Hans im Glück",
]

QUELLEN = {
    "Grimms Märchen": {
        "url": "https://www.gutenberg.org/ebooks/77905",
        "umfang": "20 ausgewählte Märchen; historische Schreibweise",
    },
    "Alice im Wunderland": {
        "url": "https://www.gutenberg.org/ebooks/19778",
        "umfang": "Alle 12 Kapitel; Übersetzung: Antonie Zimmermann (1869)",
    },
    "Die Verwandlung": {
        "url": "https://www.gutenberg.org/ebooks/22367",
        "umfang": "Vollständige Erzählung; Textausgabe von 1917",
    },
    "Max und Moritz": {
        "url": "https://www.gutenberg.org/ebooks/17161",
        "umfang": "Vorwort, alle sieben Streiche und Schluss; Ausgabe von 1906",
    },
    "Hänsel und Gretel": {
        "url": "https://maerchen.com/grimm/haensel-und-gretel.php",
        "umfang": "Brüder Grimm, Kinder- und Hausmärchen, Band 1 (1850); "
        "Online-Text: maerchen.com",
    },
    "CAU-Leitbild": {
        "url": "https://www.uni-kiel.de/de/universitaet/profil/leitbilder",
        "umfang": "Christian-Albrechts-Universität zu Kiel: Leitbild Lehren und "
        "Lernen; bereitgestellte Textfassung",
    },
}

print("Bereit – weiter mit Station 1.")


<br><br>

## 1 · Funktionsweise von Markow-Textgeneratoren

Für diese Rechnung verwenden wir einen bewusst kleinen Text:

> der Hase läuft . der Hase schläft . der Fuchs läuft .

1. Berechne $P(\text{Hase}\mid\text{der})$ und $P(\text{Fuchs}\mid\text{der})$. Warum sind die beiden Nachfolger nicht gleich wahrscheinlich? <span style="color: #707070;">(leicht)</span>
2. Der Generator startet mit **der**. Wie wahrscheinlich steht **zwei Schritte später** das Wort **schläft**? Zeichne die möglichen Wege im Baumdiagramm und multipliziere die Wahrscheinlichkeiten entlang eines Weges. <span style="color: #707070;">(leicht)</span>
3. Hänge den Satz **der Fuchs schläft .** mehrfach an den Trainingstext an. Wie viele Kopien sind mindestens nötig, damit anschließend $P(\text{schläft}\mid\text{Fuchs})\geq 3/4$ gilt? Wie verändert sich dadurch auch $P(\text{Fuchs}\mid\text{der})$? <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_1 = station_zaehlen()

<br><br>

## 2 · Ein Wort oder ein Wortpaar als Zustand?

Ein Modell **erster Ordnung** berücksichtigt nur das letzte Wort. Ein Modell **zweiter Ordnung** berücksichtigt die letzten beiden Wörter. Auf den Zustand $(u,v)$ folgt beim Auslosen von $w$ der neue Zustand $(v,w)$. Die Folge der Wortpaare ist damit wieder eine Markowkette erster Ordnung.

1. **Vermute, welche Ausgabe flüssiger klingt,** die des Modells erster Ordnung oder des Modells zweiter Ordnung. Vergleiche dann beide Modelle zuerst anhand von *Max und Moritz* und anschließend anhand von *Grimms Märchen*. Beide beginnen mit denselben zwei vorgegebenen Einheiten; ab dem dritten verwendet das erste Modell nur das letzte Wort, das zweite das ganze Paar. <span style="color: #707070;">(leicht)</span>
2. Die Tabelle zählt Zustände mit beobachteter Fortsetzung und den Anteil davon mit nur einem verschiedenen Nachfolger. Warum kann mehr Kontext längere Originalpassagen reproduzieren, statt mehr Abwechslung zu erzeugen? <span style="color: #707070;">(schwerer)</span>
3. Wähle **Max und Moritz**. Schalte **Groß-/Kleinschreibung ignorieren** und **Satzzeichen als eigene Einheiten** erst einzeln, dann zusammen ein. Vergleiche mit beiden ausgeschalteten Schaltern: Was ändert sich an der Schreibweise von Namen und Satzanfängen sowie an der Stellung von Kommas und Punkten? Erkläre die Unterschiede anhand der Zustände, aus denen das Modell lernt. Welche Einstellung führt zu generiertem Text, der am natürlichsten klingt? <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_2 = station_ordnungen()

<br><br>

## 3 · Warum übernimmt das Modell manchmal ganze Passagen?

Wähle ein Wort oder Wortpaar als Zustand. Die Tabelle zeigt seine Nachfolger. Bei Ordnung 2 siehst du zusätzlich, wie viele Nachfolger allein das letzte Wort hätte.

1. Wähle **Grimms Märchen**, **1. Ordnung** und **Alle Wörter**. Untersuche den Zustand **Königstochter**. Wechsle zu **2. Ordnung** und wähle **der Königstochter**. Welche Fortsetzungen fallen weg? Verfolge die erzwungenen Schritte mithilfe der Tabelle: Aus $(u,v)$ wird nach dem nächsten Wort $w$ der Zustand $(v,w)$. Notiere die Folge und prüfe sie anschließend vom Startpaar aus mit **„Fortsetzung prüfen“**. <span style="color: #707070;">(leicht)</span>
2. Wähle **Genau ein Nachfolger** und untersuche ein weiteres Wortpaar deiner Wahl. Vergleiche die angezeigten Nachfolgerzahlen für das Paar und sein letztes Wort. Erkläre damit den in Station 2 sichtbaren Effekt: Warum reproduziert ein Modell 2. Ordnung häufig lange Textpassagen aus dem Originaltext? <span style="color: #707070;">(schwerer)</span>
3. Muss *jedes* Wortpaar weniger verschiedene Nachfolger haben als sein letztes Wort? Warum kann ein beobachtetes Wortpaar nie mehr verschiedene Nachfolger haben als sein letztes Wort? <span style="color: #707070;">(schwerer)</span>

In dieser Station bleiben die beiden Schalter aus Station 2 unberücksichtigt. Wir trennen nur an Leerraum: `der` und `Der` sowie `Wort` und `Wort,` zählen als getrennte Zustände. Zwischen verschiedenen Märchen werden keine Übergänge gezählt.


In [ ]:
station_3 = station_nachfolger()

<br><br>

## 4 · Einen eigenen Generator bauen

Nun kannst du den Markow-Textgenerator auf einem eigenen Text trainieren.

1. Gib einen eigenen Text ein und drücke **„Text übernehmen“**. Der Text sollte mindestens eine Verzweigung beinhalten, mindestens ein Wort soll also mehrere verschiedene Nachfolger haben. <span style="color: #707070;">(leicht)</span>
2. Vergleiche die Ordnungen 1, 2 und 3. Was geschieht, wenn jeder längere Kontext nur einmal vorkommt? Der Generator beginnt mit den ersten ein, zwei bzw. drei Einheiten deines Textes. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_4 = station_eigener_text()

<br><br>

## Kleine Codeaufgabe · Eine Wahrscheinlichkeit berechnen (optional) <span style="color: #707070;">(leicht)</span>

Ergänze `p_hase` als **Anzahl für „Hase“ geteilt durch die Gesamtzahl**. Mit `nachfolger["Hase"]` liest du die entsprechende Anzahl aus; `/` bedeutet Division.

Führe die Zelle aus und ändere anschließend die Anzahl für „Hase“ von 2 auf 5. Wie ändern sich die Wahrscheinlichkeiten? Warum wäre `1 / len(nachfolger)` im Allgemeinen die falsche Rechnung?

In [ ]:
nachfolger = {"Hase": 2, "Fuchs": 1}
gesamt = sum(nachfolger.values())
p_hase = None # Ergänze hier deine Rechnung.

if p_hase is None:
    print("Ersetze None durch die passende Rechnung.")
else:
    print("Dein P(Hase | der):", p_hase)
    print("Dein P(Fuchs | der):", 1 - p_hase)

<details>
<summary>Texte und Quellen</summary>

- **Grimms Märchen:** 20 ausgewählte Märchen aus [Deutsche Märchen gesammelt durch die Brüder Grimm](https://www.gutenberg.org/ebooks/77905), herausgegeben von M. Thilo-Luyken, Ausgabe 1921. Digitalisierung: Alexander Bauer, Andrew Sly und das Online Distributed Proofreading Team. Auswahl: Der Froschkönig oder der eiserne Heinrich, Der Wolf und die sieben jungen Geisslein, Die Bremer Stadtmusikanten, Das tapfere Schneiderlein, Daumesdick, Rapunzel, Brüderchen und Schwesterchen, Aschenputtel, Hänsel und Gretel, Dornröschen, Sneewittchen, Der Fuchs und das Pferd, Frau Holle, König Drosselbart, Sechse kommen durch die ganze Welt, Jorinde und Joringel, Rotkäppchen, Rumpelstilzchen, Schneeweißchen und Rosenrot, Hans im Glück.
- **Alice im Wunderland:** [Alice’s Abenteuer im Wunderland](https://www.gutenberg.org/ebooks/19778), Lewis Carroll, deutsche Übersetzung von Antonie Zimmermann (1869), alle zwölf Kapitel. Digitalisierung: Ralph Janke, David Starner, Marilynda Fraser-Cunliffe und das Online Distributed Proofreading Team.
- **Die Verwandlung:** [Franz Kafka, Ausgabe 1917](https://www.gutenberg.org/ebooks/22367), vollständige Erzählung. Digitalisierung: Jana Srna, Alexander Bauer und das Online Distributed Proofreading Team.
- **Max und Moritz:** [Wilhelm Busch, Ausgabe 1906](https://www.gutenberg.org/ebooks/17161), Vorwort, sieben Streiche und Schluss. Digitalisierung: Inka Weide, Robert Kropf, Markus Brenner und das Online Distributed Proofreading Team.
- **Hänsel und Gretel:** Jacob und Wilhelm Grimm, *Kinder- und Hausmärchen*, große Ausgabe, Band 1 (1850). [Online-Text bei maerchen.com](https://maerchen.com/grimm/haensel-und-gretel.php).
- **CAU-Leitbild:** Christian-Albrechts-Universität zu Kiel, [*Leitbild Lehren und Lernen*](https://www.uni-kiel.de/de/universitaet/profil/leitbilder).

Quellenstand: 25.09.2026. [Project-Gutenberg-Nutzungsbedingungen](https://www.gutenberg.org/policy/license.html).

</details>